<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_3/Lecture_3_%D0%90%D0%A0%D0%A5%D0%98%D0%A2%D0%95%D0%9A%D0%A2%D0%A3%D0%A0%D0%90%20TRANSFORMER%20%E2%80%94%20%D0%A4%D0%A3%D0%9D%D0%94%D0%90%D0%9C%D0%95%D0%9D%D0%A2%20%D0%A1%D0%9E%D0%92%D0%A0%D0%95%D0%9C%D0%95%D0%9D%D0%9D%D0%AB%D0%A5%20LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1.Введение к разделу «Архитектура Transformer»

### 1. Актуальность

За период с 2017 по 2024 год архитектура Transformer стала не просто доминирующей парадигмой в обработке естественного языка (NLP), но и универсальным фундаментом для широкого круга задач искусственного интеллекта. Предложенная в статье *«Attention Is All You Need»* [1], эта архитектура заменила рекуррентные и свёрточные слои исключительно механизмом внимания, что обеспечило беспрецедентную масштабируемость и эффективность обучения.

**Количественные показатели** убедительно свидетельствуют о влиянии Transformer. Оригинальная статья (NeurIPS 2017) по состоянию на 2025 год имеет более **120 000 цитирований**, входя в число наиболее цитируемых научных работ в области компьютерных наук. Количество публикаций, содержащих в названии или аннотации термин «Transformer», выросло с единиц в 2017 году до **более 15 000** в 2024 году (по данным arXiv). Число открытых и проприетарных моделей, основанных на этой архитектуре, превышает **несколько сотен**; наиболее известные из них — GPT-4, LLaMA, Qwen, Claude, Gemini, BERT, RoBERTa, T5 — составляют основу современного генеративного ИИ.

Важно подчеркнуть, что область применения Transformer не ограничивается текстом. Архитектура успешно адаптирована для:
- **Компьютерного зрения**: Vision Transformer (ViT) [2] и его производные демонстрируют результаты, сравнимые или превосходящие свёрточные сети (CNN) на эталонных наборах данных (ImageNet, COCO).
- **Обработки аудио**: Audio Spectrogram Transformer [3] и Whisper от OpenAI применяют Transformer для распознавания речи и классификации звуков.
- **Мультимодальных систем**: модели типа CLIP, Flamingo, GPT-4o используют Transformer для совместного представления текста, изображений, видео и аудио, открывая путь к агентному ИИ.

Таким образом, понимание архитектуры Transformer является **обязательным условием** для профессиональной деятельности в области машинного обучения, независимо от конкретной модальности данных.

---

### 2. Исторический контекст

До 2017 года обработка последовательностей (текстов, временных рядов, аудиосигналов) основывалась преимущественно на **рекуррентных нейронных сетях (RNN)** и их усовершенствованных вариантах — **LSTM**[4] и **GRU**[5]. Эти архитектуры обрабатывают данные последовательно, поддерживая скрытое состояние, которое переносится от одного шага к следующему. Основные ограничения такого подхода заключались в следующем:

1. **Последовательная обработка** не позволяет эффективно использовать современные GPU, что резко замедляет обучение на больших корпусах.
2. **Проблема дальних зависимостей** — информация из начальных токенов постепенно «затухает» при прохождении через многие шаги; LSTM лишь частично смягчает этот эффект.
3. В архитектурах **seq2seq** (энкодер–декодер), используемых для машинного перевода, энкодер сжимает всю входную последовательность в фиксированный вектор фиксированной размерности (контекстный вектор), что приводит к потере информации при работе с длинными предложениями.

Ключевым прорывом, предшествовавшим Transformer, стало введение **механизма внимания** в работе Bahdanau et al. (2014) [6]. Авторы предложили, чтобы декодер на каждом шаге генерации динамически выбирал, на какие части входной последовательности обращать внимание, вычисляя весовые коэффициенты на основе текущего состояния декодера и всех скрытых состояний энкодера. Это позволило существенно улучшить качество перевода, особенно для длинных предложений, и заложило основу для дальнейшего развития.

Однако в архитектуре Bahdanau внимание всё ещё сочеталось с RNN, сохраняя последовательную обработку. **Трансформация произошла в 2017 году**, когда команда Google Brain под руководством Ашвиша Васвани предложила полностью отказаться от рекуррентности, построив модель исключительно на слоях внимания. Результат превзошёл ожидания: новый подход не только достиг нового уровня точности на задачах машинного перевода, но и сократил время обучения с нескольких недель до нескольких дней благодаря полной параллелизации.

---

### 3. Цели раздела

Данный раздел ставит перед читателем следующие цели:

1. **Сформировать целостное представление об архитектуре Transformer** — от входного слоя до выходного, включая все промежуточные преобразования и потоки данных.

2. **Детально изучить каждый компонент**: механизм масштабированного точечного внимания (Scaled Dot-Product Attention), многоголовое внимание (Multi-Head Attention), позиционное кодирование (Positional Encoding), полносвязную сеть (Feed-Forward Network), остаточные связи и слои нормализации.

3. **Освоить математическую нотацию** и выводы формул, лежащих в основе Self-Attention:  
   \[
   \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V,
   \]
   а также понять, почему масштабирование на \(\sqrt{d_k}\) критически важно для стабильности градиентов.

4. **Разобраться в архитектурных вариациях** — различиях между энкодером, декодером и их комбинациями (Encoder-Only, Decoder-Only, Encoder-Decoder) и научиться обоснованно выбирать тип архитектуры для конкретной прикладной задачи.

5. **Получить практические навыки** реализации ключевых компонентов на Python (с использованием NumPy или PyTorch), что позволит в дальнейшем уверенно работать с библиотеками Hugging Face Transformers, адаптировать существующие модели и разрабатывать собственные.

---

### 4. Структура раздела

Изложение материала организовано по принципу «от общего к частному» и включает следующие темы:

- **От RNN к Transformer** — анализ ограничений рекуррентных архитектур, обоснование необходимости перехода к вниманию, ключевые результаты статьи 2017 года.
- **Общая архитектура Transformer** — структура энкодера и декодера, стек слоёв, поток данных; описание трёх основных вариантов (Encoder-Only, Decoder-Only, Encoder-Decoder) с примерами моделей.
- **Токенизация и эмбеддинги** — преобразование текста в последовательность идентификаторов, матрица эмбеддингов и её роль в преобразовании дискретных токенов в непрерывные векторные представления размерности \(d_{\text{model}}\).
- **Позиционное кодирование** — обоснование необходимости учёта порядка токенов; синусоидальные кодировки (фиксированные) и обучаемые позиционные эмбеддинги; современный подход Rotary Position Embedding (RoPE).
- **Механизм самовнимания (Self-Attention)** — интуиция, математическая формула, роль Query, Key, Value; влияние масштабирования.
- **Многоголовое внимание (Multi-Head Attention)** — объяснение, почему необходимо несколько независимых голов; параллельное вычисление, конкатенация и финальная проекция; современные эффективные вариации: Multi-Query Attention (MQA) и Grouped-Query Attention (GQA).
- **Полносвязная сеть (Feed-Forward Network, FFN)** — двухслойная структура, роль нелинейности; популярные функции активации (ReLU, GELU, SwiGLU) и их влияние на качество и вычислительную сложность.
- **Остаточные связи и нормализация** — принцип residual connections для устранения проблемы исчезающих градиентов; Layer Normalization и RMSNorm; сравнение Pre-Norm и Post-Norm, обоснование выбора Pre-Norm в современных реализациях.
- **Заключение и связь с последующими разделами** — резюме ключевых концепций; роль Transformer в тонкой настройке (LoRA), RAG-системах, агентах, RLHF и промышленном развёртывании.

Материал раздела обеспечивает необходимый фундамент для всех последующих глав: глубокое понимание архитектуры является предпосылкой для эффективной работы с предобученными моделями, их адаптации, оценки и эксплуатации.

---

### 5. Визуализация архитектуры

Приведённая ниже диаграмма (в нотации Mermaid) отображает полную структуру Transformer в конфигурации энкодер–декодер, соответствующей оригинальной статье. Она включает все основные блоки и потоки данных, что позволяет читателю визуально связать теоретические описания с реальными компонентами.

```mermaid
flowchart TD
    subgraph Input["1. Входные данные"]
        S[Входная последовательность<br/>токенов длины T]
    end

    subgraph Embedding["2. Слой эмбеддингов"]
        E[Матрица эмбеддингов<br/>размером T×d_model]
    end

    subgraph PE["3. Позиционное кодирование"]
        P[Сложение эмбеддингов<br/>с позиционными кодировками]
    end

    subgraph EncoderStack["4. Стек энкодеров (N×)"]
        direction TB
        EN1[Энкодер слой 1]
        EN2[Энкодер слой 2]
        ENDot[⋮]
        ENN[Энкодер слой N]
    end

    subgraph EncoderLayer["Структура слоя энкодера"]
        direction TB
        MHA1[Multi-Head<br/>Self-Attention]
        AD1[Add & LayerNorm]
        FFN1[Feed-Forward<br/>Network]
        AD2[Add & LayerNorm]
    end

    subgraph DecoderStack["5. Стек декодеров (N×)"]
        direction TB
        DN1[Декодер слой 1]
        DN2[Декодер слой 2]
        DNDot[⋮]
        DNN[Декодер слой N]
    end

    subgraph DecoderLayer["Структура слоя декодера"]
        direction TB
        MMHA[Masked Multi-Head<br/>Self-Attention]
        AD3[Add & LayerNorm]
        CMHA[Cross-Attention<br/>(Q из декодера, K,V из энкодера)]
        AD4[Add & LayerNorm]
        FFN2[Feed-Forward<br/>Network]
        AD5[Add & LayerNorm]
    end

    subgraph Output["6. Выходной слой"]
        LC[Линейный слой<br/>(d_model → vocab_size)]
        SM[Softmax<br/>вероятности токенов]
    end

    S --> E --> P --> EncoderStack
    EncoderStack -->|выход энкодера| DecoderStack

    P --> MHA1 --> AD1 --> FFN1 --> AD2
    AD2 --> EN2

    AD2 -->|вход в декодер| MMHA --> AD3 --> CMHA --> AD4 --> FFN2 --> AD5
    AD5 --> DN2

    ENN -->|K, V| CMHA

    DNN --> LC --> SM --> O[Выходная последовательность<br/>токенов]

    style Input fill:#e3f2fd,stroke:#1565c0
    style Embedding fill:#e8f5e9,stroke:#2e7d32
    style PE fill:#fff3e0,stroke:#e65100
    style EncoderStack fill:#f3e5f5,stroke:#6a1b9a
    style DecoderStack fill:#fce4ec,stroke:#c62828
    style Output fill:#e0f7fa,stroke:#00838f
```

---

### 6. Ключевые термины

| Термин | Определение |
|--------|-------------|
| **Transformer** | Архитектура нейронной сети, основанная исключительно на механизме самовнимания и полносвязных слоях, без рекуррентных или свёрточных компонентов. |
| **Self-Attention** | Механизм, позволяющий каждому элементу последовательности взаимодействовать со всеми остальными, вычисляя весовые коэффициенты на основе их сходства. |
| **Multi-Head Attention** | Параллельное выполнение нескольких независимых функций внимания с последующей конкатенацией и линейным преобразованием. |
| **Query, Key, Value (Q, K, V)** | Три векторных представления, получаемые линейным проецированием входных данных; используются для вычисления внимания. |
| **Scaled Dot-Product Attention** | Основная формула внимания: \(\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V\), где \(d_k\) — размерность ключей. |
| **Positional Encoding** | Добавление к эмбеддингам информации о позиции токена, чтобы модель могла учитывать порядок элементов. |
| **Encoder** | Часть архитектуры, преобразующая входную последовательность в непрерывное контекстное представление. |
| **Decoder** | Часть архитектуры, генерирующая выходную последовательность на основе представления энкодера и ранее сгенерированных токенов. |
| **Masked Self-Attention** | Вариант самовнимания в декодере, где будущим токенам запрещено влиять на текущий (маска из −∞). |
| **Cross-Attention** | Внимание, где запросы (Q) берутся из декодера, а ключи (K) и значения (V) — из выхода энкодера. |
| **Feed-Forward Network (FFN)** | Двухслойная полносвязная сеть, применяемая позиционно-независимо к каждому элементу последовательности. |
| **Residual Connection** | Связь, добавляющая вход подслоя к его выходу: \(x_{\text{out}} = x + \text{Sublayer}(x)\), облегчающая обучение глубоких сетей. |
| **Layer Normalization** | Нормализация активаций по признаковому измерению для каждой позиции отдельно; стабилизирует обучение. |
| **Pre-Norm / Post-Norm** | Схемы размещения нормализации: до (pre-norm) или после (post-norm) подслоя; современные модели используют pre-norm. |

---

### 7. Рекомендуемая литература

**Основная**
1. Vaswani, A., et al. (2017). *«Attention Is All You Need»*. Advances in Neural Information Processing Systems (NeurIPS).  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

**Для углублённого изучения**
2. Alammar, J. (2018). *«The Illustrated Transformer»*. — Визуальное, интуитивное объяснение архитектуры.  
   🔗 [https://jalammar.github.io/illustrated-transformer/](https://jalammar.github.io/illustrated-transformer/)

3. Phuong, M., & Hutter, M. (2022). *«Formal Algorithms for Transformers»*. arXiv:2207.09238. — Формальное описание алгоритмов Transformer.  
   🔗 [https://arxiv.org/abs/2207.09238](https://arxiv.org/abs/2207.09238)

4. Liu, Y., et al. (2024). *«Recent Progress in Understanding Transformer and Developing Its Surpasser: A Survey»*. — Обзор современных достижений в понимании и развитии Transformer.  
   🔗 [https://ieeexplore.ieee.org/document/10878057](https://ieeexplore.ieee.org/document/10878057)

**Обзорные статьи**
5. Sajun, A.R., et al. (2024). *«A Historical Survey of Advances in Transformer Architectures»*. Applied Sciences, 14(10), 4316. — Исторический обзор развития Transformer-архитектур.  
   🔗 [https://doi.org/10.3390/app14104316](https://doi.org/10.3390/app14104316)

6. Shao, M., et al. (2024). *«Survey of Different Large Language Model Architectures: Trends, Benchmarks, and Challenges»*. IEEE Access. — Обзор LLM-архитектур на основе Transformer.  
   🔗 [https://arxiv.org/abs/2412.03454](https://arxiv.org/abs/2412.03454)

**Практические ресурсы**
7. Harvard NLP. *«The Annotated Transformer»*. — Пошаговая реализация Transformer на PyTorch.  
   🔗 [https://nlp.seas.harvard.edu/2018/04/03/attention.html](https://nlp.seas.harvard.edu/2018/04/03/attention.html)

8. Hugging Face. *«NLP Course»*. — Практические занятия с библиотекой Transformers.  
   🔗 [https://huggingface.co/learn/nlp-course](https://huggingface.co/learn/nlp-course)

---

Понимание архитектуры Transformer является не просто ознакомлением с очередной моделью, а освоением ключевого принципа, на котором строится современный искусственный интеллект. Каждый компонент этой архитектуры — от масштабированного произведения до слоя нормализации — был разработан для решения конкретной инженерной задачи, и их совокупность обеспечивает невиданную ранее масштабируемость и универсальность. Изучение этого раздела закладывает основу для всех последующих тем курса, от тонкой настройки до промышленного развёртывания систем на основе больших языковых моделей.

## Тема 1.1. Ограничения рекуррентных нейронных сетей (RNN)

Рекуррентные нейронные сети долгое время являлись стандартным инструментом для обработки последовательных данных, однако их фундаментальные ограничения стали основным препятствием на пути к созданию моделей, способных эффективно работать с длинными зависимостями и большими объёмами данных. Понимание этих ограничений необходимо для осознания причин появления архитектуры Transformer, которая полностью устраняет рекуррентные связи.

---

### 1. Математическое описание RNN и принцип работы

Стандартная RNN (в конфигурации «Элмана») описывается рекуррентным уравнением, связывающим текущее скрытое состояние $h_t$ с предыдущим состоянием $h_{t-1}$ и текущим входом $x_t$:

$$
h_t = f\bigl(W_h \, h_{t-1} + W_x \, x_t + b\bigr),
\tag{1}
$$

где:

- $h_t \in \mathbb{R}^{d}$ — скрытое состояние в момент времени $t$,
- $x_t \in \mathbb{R}^{m}$ — входной вектор (например, эмбеддинг токена),
- $W_h \in \mathbb{R}^{d \times d}$ — матрица рекуррентных весов,
- $W_x \in \mathbb{R}^{d \times m}$ — матрица входных весов,
- $b \in \mathbb{R}^{d}$ — вектор смещений,
- $f$ — нелинейная функция активации (обычно $\tanh$ или $\text{ReLU}$).

Выходной сигнал на каждом шаге обычно вычисляется как $y_t = g(W_y \, h_t + b_y)$, где $g$ — функция активации для выходного слоя (например, softmax для классификации).

Развёртка RNN по времени представляет собой цепочку одинаковых преобразований, где каждая копия использует одни и те же весовые матрицы. Это позволяет представить обработку последовательности длины $T$ как глубокую сеть с $T$ слоями, но с общей весовой матрицей. Такая структура показана на диаграмме:

```mermaid
flowchart LR
    subgraph time["Развёртка по времени"]
        x1[x₁] --> h1[("h₁")]
        h0[("h₀")] --> h1
        h1 --> y1[y₁]
        
        x2[x₂] --> h2[("h₂")]
        h1 --> h2
        h2 --> y2[y₂]
        
        xT[x_T] --> hT[("h_T")]
        hTminus1[("h_{T-1}")] --> hT
        hT --> yT[y_T]
    end
    
    style h1 fill:#bbdefb
    style h2 fill:#bbdefb
    style hT fill:#bbdefb
```

Обучение RNN осуществляется с помощью **обратного распространения ошибки во времени (Backpropagation Through Time, BPTT)**. Функция потерь обычно определяется как сумма потерь на каждом временном шаге:

$$
\mathcal{L} = \sum_{t=1}^{T} \mathcal{L}_t\bigl(y_t, \hat{y}_t\bigr).
$$

Градиент по отношению к рекуррентной матрице $W_h$ вычисляется как сумма вкладов от каждого шага:

$$
\frac{\partial \mathcal{L}}{\partial W_h}
= \sum_{t=1}^{T}
\frac{\partial \mathcal{L}_t}{\partial h_t}
\cdot \left( \prod_{i=k+1}^{t} \frac{\partial h_i}{\partial h_{i-1}} \right)
\cdot \frac{\partial h_k}{\partial W_h}.
\tag{2}
$$

Произведение якобианов $\prod \frac{\partial h_i}{\partial h_{i-1}}$ является ключевым фактором, определяющим стабильность обучения. Каждый якобиан имеет вид

$$
\frac{\partial h_i}{\partial h_{i-1}}
= \mathrm{diag}\bigl(f'(W_h h_{i-1} + W_x x_i + b)\bigr) \cdot W_h.
\tag{3}
$$

Поскольку производная активации $f'$ ограничена по модулю (для $\tanh$ она лежит в интервале $(0,1]$), а спектральные свойства матрицы $W_h$ определяют, будут ли последовательные умножения приводить к экспоненциальному затуханию или росту, возникает фундаментальная нестабильность градиентов.

---

### 2. Три главных ограничения RNN

#### 2.1. Последовательная обработка и отсутствие параллелизации

Из уравнения (1) видно, что состояние $h_t$ не может быть вычислено до получения $h_{t-1}$. Это накладывает жёсткое ограничение на параллелизацию: все шаги должны выполняться последовательно, что делает невозможным использование современных графических процессоров, оптимизированных для массовых параллельных вычислений.

Вычислительная сложность обработки последовательности длины $T$ составляет

$$
\mathcal{C}_{\text{RNN}} = O\left(T \cdot (d^2 + d \cdot m)\right),
\tag{4}
$$

где $d$ — размерность скрытого состояния, а $m$ — размерность входа. Время обработки растёт линейно с длиной последовательности, и для длинных текстов (тысячи токенов) это становится неприемлемо медленным. Кроме того, для реализации BPTT необходимо хранить все промежуточные состояния $h_0, h_1, \dots, h_T$, что даёт сложность по памяти $O(T \cdot d)$. Это ограничивает как максимальную длину последовательности, так и размерность модели, которую можно обучить на доступном оборудовании.

Таким образом, последовательная природа RNN является прямым противоречием с архитектурой современных вычислительных систем, что делает масштабирование RNN на большие данные крайне неэффективным.

#### 2.2. Исчезающие и взрывающиеся градиенты

Второе, и, возможно, наиболее серьёзное ограничение связано с нестабильностью градиентов при обратном распространении через длинные последовательности. Как показано в выражении (2), градиент содержит произведение якобианов $\prod_{i=k+1}^{t} \frac{\partial h_i}{\partial h_{i-1}}$. Каждый якобиан (3) представляет собой произведение диагональной матрицы производных активации и матрицы $W_h$.

Если спектральный радиус $\rho(W_h)$ меньше единицы (с учётом ограниченности производных активации), то произведение таких матриц экспоненциально убывает с ростом $t-k$, что приводит к **исчезающим градиентам**. В этом случае градиенты для ранних шагов становятся пренебрежимо малыми, и модель не может обновлять веса, отвечающие за долгосрочные зависимости. Это проявляется в том, что RNN «забывает» информацию из начала последовательности, что делает её бесполезной для задач, требующих учёта контекста на большом расстоянии (например, анализ длинных документов или диалогов).

Напротив, если $\rho(W_h) > 1$, то градиенты экспоненциально растут, вызывая **взрывающиеся градиенты**, которые приводят к численной нестабильности и резким скачкам функции потерь. Взрывающиеся градиенты можно частично контролировать с помощью клиппинга (ограничения нормы градиента), однако исчезающие градиенты не имеют простого решения, так как они связаны с самой структурой рекуррентных связей.

На диаграмме ниже схематично показано, как градиент затухает при обратном распространении через длинную последовательность:

```mermaid
flowchart LR
    subgraph forward["Прямой проход"]
        hT["h_T"] --> hTminus["h_{T-1}"] --> hTminus2["h_{T-2}"] --> hTminus3["..."]
    end

    subgraph backward["Обратный проход (BPTT)"]
        direction LR
        gT["градиент у h_T"] -->|"× J_T"| gTminus["градиент у h_{T-1}"]
        gTminus -->|"× J_{T-1}"| gTminus2["градиент у h_{T-2}"]
        gTminus2 -->|"× J_{T-2}"| gTminus3["..."]
        gTminus3 -->|"малый градиент"| g0["почти нулевой градиент у h_0"]
    end

    style gT fill:#ffcdd2
    style gTminus fill:#ffcdd2
    style gTminus2 fill:#ffcdd2
    style gTminus3 fill:#ffcdd2
    style g0 fill:#e0e0e0
```

#### 2.3. Ограниченная память («бутылочное горлышко»)

Третье фундаментальное ограничение заключается в том, что скрытое состояние $h_t$ имеет **фиксированную размерность** $d$, независимо от длины последовательности. Это означает, что вся информация о предыдущих $t-1$ элементах должна быть сжата в вектор фиксированной длины. Такое сжатие неизбежно приводит к потере информации, особенно при работе с длинными последовательностями, содержащими множество разнородных фактов.

Это явление часто называют **«бутылочным горлышком»** информационного канала: модель вынуждена выбирать, какую информацию сохранять, а какую отбрасывать. В отличие от архитектур, позволяющих хранить информацию в отдельных векторах для каждого элемента (как, например, в механизме внимания), RNN не может сохранять детализированную информацию о всех предшествующих элементах, поскольку они должны быть «спроецированы» в одно состояние. Это ограничение особенно критично для задач, требующих одновременного учёта множества независимых фактов, таких как чтение больших документов или поддержание диалога.

---

### 3. Попытки решения: LSTM и GRU

Для преодоления проблемы исчезающих градиентов были разработаны усовершенствованные рекуррентные архитектуры — **долгая краткосрочная память (LSTM)** [[1](https://www.bioinf.jku.at/publications/older/2604.pdf)] и **управляемые рекуррентные блоки (GRU)** [[2](https://arxiv.org/abs/1406.1078)]. Основная идея заключается во введении **механизмов ворот**, которые управляют потоком информации, позволяя избирательно сохранять или забывать информацию на длительные промежутки времени.

**LSTM** содержит три типа ворот: входные, забывающие и выходные. Обновление состояния описывается следующими уравнениями:

$$
\begin{aligned}
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i), \\
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f), \\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o), \\
\tilde{C}_t &= \tanh(W_c x_t + U_c h_{t-1} + b_c), \\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t, \\
h_t &= o_t \odot \tanh(C_t),
\end{aligned}
$$

где $\odot$ — поэлементное умножение, $C_t$ — состояние ячейки, а $i_t, f_t, o_t$ — активации ворот. Забывающий вентиль $f_t$ позволяет модели сбрасывать информацию из ячейки, а входной вентиль $i_t$ — добавлять новую. Благодаря этому градиенты могут проходить через ячейку без изменений, если $f_t \approx 1$, что существенно смягчает проблему исчезающих градиентов.

**GRU** является упрощённой версией LSTM, объединяющей входной и забывающий вентили в один «вентиль обновления»:

$$
\begin{aligned}
z_t &= \sigma(W_z x_t + U_z h_{t-1} + b_z), \\
r_t &= \sigma(W_r x_t + U_r h_{t-1} + b_r), \\
\tilde{h}_t &= \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h), \\
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t.
\end{aligned}
$$

Эти архитектуры действительно улучшают способность модели к запоминанию долгосрочных зависимостей, однако они **не устраняют** основные ограничения RNN:

- **Последовательная обработка** остаётся неизменной: для вычисления $h_t$ необходимо дождаться $h_{t-1}$, что не позволяет распараллеливать вычисления.
- **Фиксированная размерность** скрытого состояния сохраняется, и «бутылочное горлышко» никуда не исчезает.
- Даже с LSTM обучение на очень длинных последовательностях (тысячи шагов) остаётся сложным, и на практике LSTM редко работают устойчиво при длинах более 200–500 шагов.

Сравнительная характеристика трёх архитектур приведена в таблице:

| Характеристика | RNN | LSTM | GRU |
|----------------|-----|------|-----|
| Число параметров (на шаг) | $d^2 + d \cdot m$ | $4(d^2 + d \cdot m)$ | $3(d^2 + d \cdot m)$ |
| Сложность обучения (на шаг) | $O(d^2)$ | $O(d^2)$ | $O(d^2)$ |
| Способность к дальним зависимостям | Очень ограничена | Улучшена (до сотен шагов) | Улучшена (до сотен шагов) |
| Параллелизация | Нет | Нет | Нет |
| Проблема «бутылочного горлышка» | Присутствует | Присутствует | Присутствует |

---

### 4. Пример задачи, где RNN терпят неудачу

Рассмотрим задачу **извлечения отношений** из длинных документов. Пусть дано предложение:

*«Альберт Эйнштейн, родившийся в Ульме в 1879 году и получивший Нобелевскую премию в 1921 году за открытие фотоэлектрического эффекта, позже эмигрировал в США и принял гражданство в 1940 году.»*

Требуется ответить на вопрос: *«В каком году Эйнштейн стал гражданином США?»*. Ответ (1940) находится в самом конце предложения, а ключевой субъект «Эйнштейн» — в начале. Для RNN, даже с LSTM, информация о субъекте должна быть сохранена на протяжении более 30 слов, что уже близко к пределу возможностей LSTM. На более длинных текстах (например, несколько абзацев) RNN практически гарантированно теряют связь между субъектом и предикатом, порождая ошибочные ответы или галлюцинации.

Именно такие сценарии продемонстрировали, что рекуррентные подходы неспособны обеспечить надёжное моделирование долгосрочных зависимостей, которые естественно возникают в языке, биологических последовательностях и других структурированных данных.

---

### 5. Выводы и требования к новой архитектуре

Анализ ограничений RNN приводит к формулировке требований к архитектуре, которая могла бы преодолеть эти фундаментальные препятствия:

1. **Параллелизуемость**: вычисления для всех элементов последовательности должны выполняться одновременно, без зависимости от порядка. Это требует отказа от рекуррентных связей.

2. **Прямые связи между произвольными позициями**: модель должна обеспечивать возможность непосредственного взаимодействия между любыми двумя токенами последовательности с постоянной вычислительной сложностью, не зависящей от расстояния между ними.

3. **Отсутствие ограничения на длину памяти**: архитектура не должна требовать сжатия всей истории в вектор фиксированной размерности; каждое представление должно сохранять свою индивидуальность и доступность.

4. **Стабильность градиентов**: распространение градиентов должно быть устойчивым и не зависеть от длины последовательности, желательно путём устранения рекуррентных произведений якобианов.

Именно эти требования легли в основу разработки архитектуры **Transformer**, которая заменила рекуррентные связи механизмом самовнимания (self-attention), обеспечивающим прямые взаимодействия между всеми парами элементов последовательности. Это позволило достичь полной параллелизации, постоянной сложности связи произвольных позиций и стабильного распространения градиентов, что в итоге привело к революционным результатам в области обработки естественного языка и других модальностей.

---

### Литература

1. Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural computation*, 9(8), 1735-1780.  
   🔗 [https://www.bioinf.jku.at/publications/older/2604.pdf](https://www.bioinf.jku.at/publications/older/2604.pdf)

2. Cho, K., van Merriënboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H., & Bengio, Y. (2014). Learning phrase representations using RNN encoder-decoder for statistical machine translation. *arXiv preprint arXiv:1406.1078*.  
   🔗 [https://arxiv.org/abs/1406.1078](https://arxiv.org/abs/1406.1078)

3. Bengio, Y., Simard, P., & Frasconi, P. (1994). Learning long-term dependencies with gradient descent is difficult. *IEEE Transactions on Neural Networks*, 5(2), 157-166.  
   🔗 [https://ieeexplore.ieee.org/document/279181](https://ieeexplore.ieee.org/document/279181)

4. Pascanu, R., Mikolov, T., & Bengio, Y. (2013). On the difficulty of training recurrent neural networks. *Proceedings of the 30th International Conference on Machine Learning (ICML)*.  
   🔗 [https://arxiv.org/abs/1211.5063](https://arxiv.org/abs/1211.5063)

## Тема 1.2. Статья «Attention Is All You Need»: рождение новой парадигмы

В декабре 2017 года на конференции NeurIPS была представлена работа, которой суждено было изменить траекторию развития искусственного интеллекта. Статья под названием **«Attention Is All You Need»** [1], подготовленная исследовательской группой Google Brain, предложила архитектуру, отказавшуюся от доминировавших на тот момент рекуррентных и свёрточных слоёв в пользу исключительно механизма внимания. Этот текст представляет собой первую публикацию, в которой была описана архитектура **Transformer**, ставшая фундаментом современных больших языковых моделей.

---

### 1. Информация о статье

**Полное название:** *Attention Is All You Need*

**Авторы:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin. Восемь исследователей из Google Brain и смежных организаций, объединивших усилия вокруг идеи, которая впоследствии определила развитие целой области.

**Год публикации:** 2017

**Конференция:** NeurIPS (Advances in Neural Information Processing Systems) — одна из наиболее престижных международных конференций в области машинного обучения.

**Ключевая идея:** авторы предложили *«новую простую сетевую архитектуру, Transformer, основанную исключительно на механизмах внимания, полностью отказывающуюся от рекуррентности и свёрток»*【3†L3-L5】. Это был радикальный шаг: вместо того чтобы улучшать существующие рекуррентные подходы, исследователи предложили заменить их принципиально иной парадигмой.

---

### 2. Контекст появления

До 2017 года доминирующими архитектурами для задач машинного перевода и моделирования последовательностей были рекуррентные нейронные сети (RNN) и их усовершенствованные варианты — LSTM и GRU. Как отмечают сами авторы во введении, *«рекуррентные нейронные сети, в особенности LSTM и GRU, прочно утвердились как современные подходы в моделировании последовательностей и задачах транзакции, таких как языковое моделирование и машинный перевод»*【1†L25-L27】.

На тот момент лучшие модели машинного перевода представляли собой сложные архитектуры на основе RNN с механизмом внимания, который соединял энкодер и декодер. Среди них выделялись:

- **GNMT** (Google Neural Machine Translation) — система Google, достигшая на тот момент впечатляющих результатов【7†L15-L18】;
- **ConvS2S** — модель на основе свёрточных слоёв от Facebook【7†L15-L18】;
- **Deep-Att + PosUnk** — глубокая модель с вниманием и обработкой неизвестных слов【7†L15-L18】.

Однако все эти подходы обладали фундаментальными ограничениями, которые становились всё более очевидными по мере роста объёмов данных и требований к качеству:

1. **Последовательная обработка.** RNN обрабатывают токены один за другим, что делает невозможной параллелизацию вычислений и резко замедляет обучение на больших корпусах.

2. **Проблема дальних зависимостей.** Информация из начала последовательности постепенно «размывается» к концу, и даже LSTM не полностью решают эту проблему.

3. **Огромные вычислительные затраты.** Обучение лучших моделей машинного перевода того времени требовало недель на множестве GPU и стоило миллионы долларов.

Именно эти нерешённые проблемы создали почву для появления принципиально новой архитектуры.

---

### 3. Ключевые инновации

Статья предложила четыре основные инновации, которые в совокупности определили успех Transformer.

#### 3.1. Scaled Dot-Product Attention

Авторы предложили модификацию механизма внимания, названную *«масштабированное точечное внимание»* (Scaled Dot-Product Attention). Его суть описывается формулой:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Ключевое новшество — **масштабирование на** $\sqrt{d_k}$. Авторы объясняют это так: *«Мы подозреваем, что для больших значений $d_k$ скалярные произведения сильно растут по величине, заталкивая функцию softmax в области с крайне малыми градиентами. Чтобы противодействовать этому эффекту, мы масштабируем скалярные произведения на $1/\sqrt{d_k}$»*【4†L30-L35】.

Это решение оказалось критически важным для стабильности обучения: без масштабирования градиенты становились слишком малыми, и модель не могла эффективно обучаться.

#### 3.2. Multi-Head Attention

Вместо выполнения одной функции внимания авторы предложили **многоголовое внимание** (Multi-Head Attention). Идея заключается в том, чтобы линейно спроецировать запросы, ключи и значения $h$ раз с различными обучаемыми проекциями, выполнить внимание параллельно на каждой проекции, а затем объединить результаты.

Формально это записывается так:

$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O,
$$

где

$$
\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V).
$$

Авторы поясняют мотивацию: *«Многоголовое внимание позволяет модели совместно обращать внимание на информацию из разных подпространств представлений на разных позициях. При одной голове внимания усреднение препятствует этому»*【4†L38-L40】. В работе использовалось $h = 8$ параллельных голов внимания с размерностью $d_k = d_v = d_{\text{model}} / h = 64$.

#### 3.3. Positional Encoding

Поскольку в модели нет ни рекуррентности, ни свёрток, она не имеет встроенного понимания порядка токенов. Для решения этой проблемы авторы ввели **позиционное кодирование** (Positional Encoding) — информацию о позиции токена в последовательности, которая добавляется к эмбеддингам на входе энкодера и декодера.

В работе используются синусоидальные функции разных частот:

$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right),
$$

$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right).
$$

Авторы объясняют выбор: *«Мы выбрали эту функцию, потому что предположили, что она позволит модели легко обучаться обращать внимание по относительным позициям, поскольку для любого фиксированного смещения $k$ позиционное кодирование $PE_{pos+k}$ может быть представлено как линейная функция от $PE_{pos}$»*【6†L30-L35】.

#### 3.4. Отказ от рекуррентности

Самый радикальный шаг — полный отказ от рекуррентных и свёрточных слоёв. Вместо них архитектура строится исключительно на:

- **Self-Attention** в энкодере — каждый токен взаимодействует со всеми токенами;
- **Masked Self-Attention** в декодере — запрет на обращение к будущим токенам для сохранения авторегрессивности;
- **Cross-Attention** — связь между энкодером и декодером.

Это решение обеспечило **полную параллелизацию** вычислений и **постоянное число операций** для связи любых двух позиций, в отличие от $O(n)$ у RNN【6†L5-L10】.

---

### 4. Экспериментальные результаты

#### 4.1. Качество перевода

Результаты оказались впечатляющими. На задаче перевода с английского на немецкий (WMT 2014 English-German) Transformer достиг **BLEU-оценки 28.4**, превзойдя лучшие ранее опубликованные результаты, включая ансамбли, *«более чем на 2 BLEU»*【3†L5-L8】.

На задаче перевода с английского на французский (WMT 2014 English-French) модель установила *«новый современный результат для одной модели — 41.0 BLEU»*【3†L5-L8】.

Для сравнения, лучшие модели того времени показывали следующие результаты:

| Модель | EN-DE BLEU | EN-FR BLEU |
|--------|------------|------------|
| ByteNet | 23.75 | — |
| GNMT + RL | 24.6 | 39.92 |
| ConvS2S | 25.16 | 40.46 |
| MoE | 26.03 | 40.56 |
| **Transformer (base)** | **27.3** | **38.1** |
| **Transformer (big)** | **28.4** | **41.0** |

*Источник: таблица 2 из оригинальной статьи*【8†L5-L20】.

#### 4.2. Время обучения

Ещё более впечатляющим оказалось сокращение времени обучения. Базовая модель обучалась **12 часов** на 8 GPU P100. Большая модель — **3.5 дня** на тех же 8 GPU【7†L15-L18】.

Для сравнения, лучшие RNN-модели того времени требовали нескольких недель обучения на сопоставимом или большем количестве GPU. Авторы подчёркивают: *«Даже наша базовая модель превосходит все ранее опубликованные модели и ансамбли при доле вычислительных затрат любой из конкурирующих моделей»*【8†L22-L25】.

```mermaid
xychart-beta
    title "Время обучения (в днях) — сравнение моделей"
    x-axis ["RNN-based", "ConvS2S", "Transformer (base)", "Transformer (big)"]
    y-axis "Дни" 0 --> 25
    bar [21, 14, 0.5, 3.5]
```

#### 4.3. Сравнение сложности слоёв

Авторы также провели систематическое сравнение различных типов слоёв по трём критериям: вычислительная сложность, возможность параллелизации и длина пути для дальних зависимостей【6†L5-L10】.

| Тип слоя | Сложность на слой | Минимальное число последовательных операций | Максимальная длина пути |
|----------|-------------------|---------------------------------------------|-------------------------|
| Self-Attention | $O(n^2 \cdot d)$ | $O(1)$ | $O(1)$ |
| Recurrent | $O(n \cdot d^2)$ | $O(n)$ | $O(n)$ |
| Convolutional | $O(k \cdot n \cdot d^2)$ | $O(1)$ | $O(\log_k(n))$ |

*Источник: таблица 1 из оригинальной статьи*【6†L5-L10】.

Как видно из таблицы, Self-Attention обеспечивает **постоянное число операций** (полная параллелизация) и **минимальную длину пути** для связи любых двух позиций, что критически важно для обучения дальним зависимостям.

---

### 5. Влияние на поле

Влияние статьи «Attention Is All You Need» на развитие искусственного интеллекта трудно переоценить. Она стала катализатором, запустившим цепную реакцию прорывов.

#### 5.1. Появление BERT и GPT

В 2018 году, менее чем через год после публикации, появился **BERT** (Bidirectional Encoder Representations from Transformers) от Google — модель, использующая только энкодер Transformer и установившая новые рекорды на 11 NLP-бенчмарках. BERT продемонстрировал, что предобучение на больших корпусах с последующей тонкой настройкой даёт беспрецедентные результаты.

Параллельно **OpenAI** развивала декодерную ветвь Transformer. В 2018 году вышел **GPT-1**, в 2019 — **GPT-2** (1.5 млрд параметров), а в 2020 — **GPT-3** (175 млрд параметров). Каждая новая итерация подтверждала масштабируемость Transformer: увеличение размера модели и данных вело к появлению новых, эмерджентных способностей.

#### 5.2. Доминирование Transformer в NLP

К 2024 году Transformer стал универсальным стандартом в обработке естественного языка. Все ведущие модели — GPT-4, Claude, LLaMA, Qwen, Gemini — построены на этой архитектуре. Оригинальная статья была процитирована более **120 000 раз**, войдя в число наиболее цитируемых научных работ в истории компьютерных наук.

#### 5.3. Выход за пределы NLP

Важно подчеркнуть, что влияние Transformer вышло далеко за рамки текстовых задач. Архитектура была успешно адаптирована для:

- **Компьютерного зрения** — Vision Transformer (ViT) показал результаты, сравнимые с лучшими свёрточными сетями;
- **Обработки аудио** — Audio Spectrogram Transformer и Whisper;
- **Мультимодальных систем** — CLIP, Flamingo, GPT-4o объединяют текст, изображения, видео и аудио.

Как отмечают сами авторы в заключении: *«Мы с воодушевлением смотрим в будущее моделей, основанных на внимании, и планируем применить их к другим задачам. Мы планируем расширить Transformer на задачи, включающие другие модальности ввода и вывода, помимо текста»*【9†L22-L25】.

#### 5.4. Цитаты, изменившие мир

В статье есть несколько фраз, которые стали программными для целого поколения исследователей:

> *«Мы предлагаем новую простую сетевую архитектуру, Transformer, основанную исключительно на механизмах внимания, полностью отказывающуюся от рекуррентности и свёрток»*【3†L3-L5】.

> *«Внимание — это всё, что вам нужно»* — сам заголовок статьи стал манифестом новой эпохи.

---

### 6. Заключение

Статья «Attention Is All You Need» — это редкий пример работы, которая не просто улучшила существующие методы, а **переопределила направление развития целой области**. Отказ от рекуррентности в пользу внимания оказался не просто техническим решением, а сменой парадигмы, позволившей масштабировать модели до невиданных ранее размеров и открывшей путь к созданию систем, которые сегодня мы называем искусственным интеллектом общего назначения.

Transformer стал не просто архитектурой — он стал **фундаментом**, на котором строится современный ИИ. Понимание этой архитектуры — ключ к пониманию того, как работают современные языковые модели, и необходимое условие для участия в создании следующего поколения интеллектуальных систем.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. Advances in Neural Information Processing Systems (NeurIPS).  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

## Тема 1.3. Преимущества Transformer: новая парадигма обработки последовательностей

Архитектура Transformer, предложенная в статье «Attention Is All You Need» [1], принесла с собой не просто улучшение качества, а принципиально новый способ обработки последовательных данных. Её преимущества перед рекуррентными и свёрточными архитектурами столь значительны, что в течение нескольких лет она вытеснила все предыдущие подходы, став стандартом де-факто в области искусственного интеллекта. В этом разделе мы систематически рассмотрим четыре ключевых преимущества Transformer: **параллелизуемость**, **глобальный контекст**, **масштабируемость** и **универсальность**.

---

### 1. Параллелизация: обработка всех токенов одновременно

Фундаментальное ограничение рекуррентных нейронных сетей заключается в их последовательной природе: для вычисления скрытого состояния на шаге $t$ необходимо дождаться завершения вычислений на шаге $t-1$. Это приводит к тому, что общее время обработки последовательности длины $T$ пропорционально $T$, и использование параллельных вычислений на GPU оказывается крайне ограниченным.

В отличие от RNN, механизм самовнимания (Self-Attention) позволяет вычислять представление каждого токена независимо от остальных, используя для этого все позиции одновременно. Математически это выражается в том, что матричное произведение $QK^T$ вычисляется за один шаг с использованием высокооптимизированных библиотек линейной алгебры, которые эффективно задействуют все ядра GPU.

Сравнение вычислительной сложности демонстрирует это различие. Для последовательности длины $T$ и размерности представления $d$:

- **RNN**: каждый шаг требует $O(d^2)$ операций (умножение на матрицу $W_h$), и таких шагов $T$, следовательно, общая сложность $O(T \cdot d^2)$.
- **Transformer**: основная операция — умножение матриц $QK^T$ размера $(T \times d)$ на $(d \times T)$, что требует $O(T^2 \cdot d)$ операций. Дополнительно — умножение на $V$: также $O(T^2 \cdot d)$.

Таким образом, сложность Transformer составляет $O(T^2 \cdot d)$, а RNN — $O(T \cdot d^2)$. В большинстве практических задач $T$ (длина последовательности) значительно меньше $d$ (размерности представления). Например, в машинном переводе типичные значения: $T \approx 30$–50, $d = 512$, так что $T^2 \cdot d \approx 2500 \cdot 512 \approx 1.3 \cdot 10^6$, в то время как $T \cdot d^2 \approx 50 \cdot 262144 \approx 13 \cdot 10^6$ — разница в 10 раз в пользу Transformer. При больших $T$ (например, длинные документы) ситуация меняется, но на практике для большинства задач длина последовательности ограничена бюджетом памяти и составляет несколько тысяч токенов, тогда как $d$ может быть 4096 и более.

Это подтверждается экспериментальными данными: Transformer обучается в 3–10 раз быстрее RNN-аналогов при сопоставимом качестве. В оригинальной статье авторы отмечают, что базовая модель обучается всего **12 часов** на 8 GPU P100, тогда как лучшие RNN-модели того времени требовали недель.

На диаграмме ниже показано сравнение вычислительной сложности для разных типов слоёв:

```mermaid
xychart-beta
    title "Зависимость времени выполнения от длины последовательности (условно)"
    x-axis "Длина последовательности T" 10 --> 1000
    y-axis "Время (условные единицы)" 0 --> 1000
    line "RNN (O(T·d²))" [1, 10, 100, 1000]
    line "Transformer (O(T²·d))" [1, 4, 100, 10000]
```

*Примечание: при малых T (до ∼100) Transformer быстрее; при очень длинных последовательностях RNN может оказаться эффективнее, но на практике используются оптимизации (Flash Attention, локальное внимание).*

Кроме того, Transformer полностью использует возможности GPU: операции с матрицами могут быть выполнены с высокой степенью параллелизма, достигая практически 100% загрузки вычислительных ядер. Это делает обучение больших моделей на кластерах GPU экономически эффективным.

---

### 2. Глобальный контекст: каждый токен видит все остальные

Второе критическое преимущество — возможность прямого взаимодействия между любыми двумя позициями последовательности с **постоянным числом операций** ($O(1)$), независимо от расстояния между ними. В RNN для связи первого и последнего токена необходимо пройти через все промежуточные шаги, что требует $O(T)$ операций и сталкивается с проблемой затухающих градиентов.

Transformer достигает этого благодаря механизму Scaled Dot-Product Attention:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V.
$$

Матрица $QK^T$ размера $(T \times T)$ содержит скалярные произведения, которые являются мерой совместимости каждой пары токенов. После применения softmax получаются веса внимания, показывающие, насколько каждый токен (запрос) должен обращать внимание на каждый другой токен (ключ). Затем эти веса используются для взвешенного суммирования значений $V$.

Интерпретация этого механизма: каждый токен получает новое представление как **взвешенная сумма всех токенов последовательности**, где веса определяются их смысловой близостью. Таким образом, модель может напрямую использовать информацию из любого места последовательности, что особенно важно для задач, требующих учёта дальних зависимостей (например, разрешение анафор, извлечение отношений).

В статье авторы подчёркивают этот аспект, сравнивая максимальную длину пути для распространения сигнала:

> *«Самовнимание связывает все позиции с постоянным числом последовательно выполняемых операций, тогда как рекуррентный слой требует $O(n)$ последовательных операций»* (раздел 4, таблица 1).

Более того, авторы отмечают, что множественные головы внимания позволяют модели одновременно фокусироваться на различных типах зависимостей: синтаксических, семантических, локальных и глобальных. Это даёт Transformer глубокое понимание структуры текста.

---

### 3. Масштабируемость: законы, которые работают

Transformer оказался архитектурой, которая **масштабируется** — увеличение размера модели (числа параметров), объёма данных и вычислительных ресурсов приводит к предсказуемому улучшению качества. Это свойство было формализовано в работе **Kaplan et al. (2020)** «Scaling Laws for Neural Language Models» [2], которая установила степенные зависимости между качеством модели и этими тремя факторами:

$$
\mathcal{L} \propto \left(\frac{N}{N_0}\right)^{\alpha_N} \cdot \left(\frac{D}{D_0}\right)^{\alpha_D} \cdot \left(\frac{C}{C_0}\right)^{\alpha_C},
$$

где:

- $N$ — число параметров модели,
- $D$ — размер обучающего корпуса (в токенах),
- $C$ — вычислительные затраты (FLOPs),
- $\alpha_N, \alpha_D, \alpha_C$ — отрицательные показатели степени (обычно около $-0.05$ – $-0.1$).

Это означает, что для улучшения качества на 10% требуется примерно в 10 раз больше параметров или данных. Однако, в отличие от RNN, Transformer демонстрирует устойчивое масштабирование без насыщения вплоть до триллионов параметров (GPT-4, 1.8 трлн). Причины этого:

1. **Отсутствие рекуррентных связей** — градиенты не затухают и не взрываются, что позволяет обучать модели с сотнями слоёв.
2. **Эффективная параллелизация** — обучение можно распределить на тысячи GPU, используя такие методы, как модель параллелизма (model parallelism) и смешанная точность.
3. **Гибкость архитектуры** — возможно вводить разреженность (MoE) и другие оптимизации без изменения основного принципа.

Ниже представлена таблица, обобщающая законы масштабирования для Transformer:

| Параметр | Обозначение | Типичный показатель степени | Влияние на качество |
|----------|-------------|-----------------------------|----------------------|
| Число параметров | $N$ | $\alpha_N \approx -0.076$ | Увеличение в 10 раз → снижение потерь на ~16% |
| Размер данных | $D$ | $\alpha_D \approx -0.095$ | Увеличение в 10 раз → снижение потерь на ~20% |
| Вычисления | $C$ | $\alpha_C \approx -0.050$ | Увеличение в 10 раз → снижение потерь на ~11% |

*Источник: Kaplan et al., 2020.*

Эти законы стали руководством для индустрии: оптимальное соотношение — увеличивать все три фактора одновременно, что и привело к появлению моделей масштаба GPT-3, LLaMA-3 и других.

---

### 4. Универсальность: одна архитектура для всех модальностей

Четвёртое преимущество Transformer — его **универсальность**. Одна и та же архитектура, с минимальными модификациями, оказалась применимой к широкому спектру задач и типов данных.

#### 4.1. В обработке естественного языка (NLP)

Transformer породил два основных направления:

- **Encoder-Only** (например, BERT) — для задач понимания текста: классификация, извлечение информации, ответы на вопросы.
- **Decoder-Only** (например, GPT) — для генеративных задач: написание текста, кода, диалогов.
- **Encoder-Decoder** (оригинальный Transformer, T5) — для задач преобразования последовательностей: машинный перевод, суммаризация.

Все современные LLM — GPT-4, Claude, LLaMA, Qwen, Gemini — являются вариациями Transformer.

#### 4.2. В компьютерном зрении (CV)

В 2020 году был предложен **Vision Transformer (ViT)** [3], который применяет Transformer к изображениям, разбивая их на патчи и обрабатывая их как последовательность. ViT показал результаты, сопоставимые с лучшими свёрточными сетями (ResNet, EfficientNet) на ImageNet, а при масштабировании — превосходящие их.

#### 4.3. В обработке аудио

**Audio Spectrogram Transformer** [4] и **Whisper** от OpenAI используют Transformer для распознавания речи и классификации звуков. Спектрограмма преобразуется в последовательность, и модель обрабатывает её так же, как текст.

#### 4.4. В мультимодальных системах

Модели типа **CLIP**, **Flamingo**, **GPT-4o** объединяют текст, изображения, видео и аудио, используя Transformer как универсальный кодировщик, способный выравнивать представления разных модальностей в общем пространстве.

Авторы оригинальной статьи предвидели это расширение:

> *«Мы с воодушевлением смотрим в будущее моделей, основанных на внимании, и планируем применить их к другим задачам. Мы планируем расширить Transformer на задачи, включающие другие модальности ввода и вывода, помимо текста»*.

---

### Сводное сравнение

В таблице ниже обобщены ключевые различия между архитектурами:

| Критерий | RNN / LSTM | Transformer |
|----------|------------|-------------|
| Последовательная обработка | Да ($O(T)$ шагов) | Нет (все шаги параллельно) |
| Связь дальних позиций | $O(T)$ операций | $O(1)$ операций |
| Вычислительная сложность (на слой) | $O(T \cdot d^2)$ | $O(T^2 \cdot d)$ |
| Параллелизация | Слабая | Полная |
| Проблема затухающих градиентов | Присутствует | Отсутствует (прямые пути) |
| Применимость к разным модальностям | Текст, временные ряды | Текст, изображения, аудио, видео |

---

### Заключение

Преимущества Transformer — параллелизация, глобальный контекст, масштабируемость и универсальность — сделали его архитектурой, определившей развитие ИИ в последнее десятилетие. Отказ от рекуррентности позволил не только ускорить обучение, но и открыл возможности для создания моделей, которые сегодня составляют основу генеративного искусственного интеллекта. Понимание этих преимуществ необходимо для осознанного выбора архитектуры при решении прикладных задач и для дальнейшего развития методов глубокого обучения.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Kaplan, J., et al. (2020). *Scaling Laws for Neural Language Models*. arXiv:2001.08361.  
   🔗 [https://arxiv.org/abs/2001.08361](https://arxiv.org/abs/2001.08361)

3. Dosovitskiy, A., et al. (2020). *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*. ICLR.  
   🔗 [https://arxiv.org/abs/2010.11929](https://arxiv.org/abs/2010.11929)

4. Gong, Y., et al. (2021). *AST: Audio Spectrogram Transformer*. Interspeech.  
   🔗 [https://arxiv.org/abs/2104.01778](https://arxiv.org/abs/2104.01778)

## Тема 1.4. Три типа архитектур на основе Transformer

Архитектура Transformer, предложенная в оригинальной статье [1], оказалась не просто удачным решением для машинного перевода — она породила целое семейство архитектур, каждая из которых адаптирована под определённый класс задач. В зависимости от того, какая часть исходной архитектуры используется — только энкодер, только декодер или оба компонента вместе, — модели приобретают различные свойства и области применения. В этом разделе мы рассмотрим три типа архитектур на основе Transformer: **Encoder-Only**, **Decoder-Only** и **Encoder-Decoder**.

---

### 1. Encoder-Only: архитектура для понимания

#### 1.1. Архитектура

Модели типа Encoder-Only используют только стек энкодеров Transformer. В оригинальной статье энкодер представляет собой последовательность $N$ идентичных слоёв (в оригинале $N=6$), каждый из которых содержит два подслоя:

1. **Multi-Head Self-Attention** — позволяет каждому токену взаимодействовать со всеми другими токенами в обе стороны (бидирекционально);
2. **Position-wise Feed-Forward Network** — применяется независимо к каждой позиции.

После каждого подслоя используются остаточные связи и нормализация. Ключевая особенность — **отсутствие маскировки**: каждый токен может обращаться к любому другому токену, включая те, что находятся справа от него.

#### 1.2. Обучение: Masked Language Model (MLM)

Encoder-Only модели обучаются с использованием **маскированного языкового моделирования** (Masked Language Model, MLM). Этот метод был предложен в работе BERT [2] и заключается в следующем:

1. Входная последовательность случайным образом маскируется: 15% токенов заменяются на специальный токен `[MASK]`;
2. Модель должна предсказать исходные токены на основе контекста — токенов слева и справа;
3. Обучается только на предсказании замаскированных токенов (кросс-энтропийная потеря).

Математически цель обучения можно записать как:

$$
\mathcal{L}_{\text{MLM}} = -\sum_{i \in \mathcal{M}} \log P(x_i \mid x_{\setminus i}),
$$

где $\mathcal{M}$ — множество замаскированных позиций, а $x_{\setminus i}$ — все остальные токены в последовательности.

Дополнительно в BERT используется задача **Next Sentence Prediction (NSP)**, которая обучает модель пониманию связи между предложениями.

#### 1.3. Задачи

Encoder-Only модели специализируются на задачах **понимания текста**, где требуется извлечь информацию из уже имеющегося текста, а не генерировать новый. Типичные применения:

- **Классификация текста** — определение тональности, тематики;
- **Named Entity Recognition (NER)** — извлечение именованных сущностей;
- **Question Answering (QA)** — поиск ответа в тексте;
- **Извлечение информации** — структурирование данных из неструктурированного текста;
- **Определение семантической близости** — сравнение смысла текстов.

#### 1.4. Примеры моделей

- **BERT** (Bidirectional Encoder Representations from Transformers) — первая и наиболее известная модель, Google, 2018;
- **RoBERTa** — улучшенная версия BERT от Facebook, 2019;
- **DistilBERT** — дистиллированная версия BERT для мобильных устройств;
- **ALBERT** — лёгкая версия BERT с разделением параметров;
- **ELECTRA** — модель с заменой токенов вместо маскирования.

#### 1.5. Преимущества и ограничения

**Преимущества:**
- **Полный двусторонний контекст** — каждый токен видит информацию как слева, так и справа, что критически важно для понимания языка;
- **Высокое качество** на задачах классификации и извлечения информации;
- **Эффективное использование** предобученных знаний.

**Ограничения:**
- **Не способна к генерации** связных текстов (нельзя использовать как чат-бота);
- Требует специального формата для каждой задачи (например, токен `[CLS]` для классификации);
- Ограничена задачами понимания, а не творческого синтеза.

На диаграмме ниже показана архитектура Encoder-Only:

```mermaid
flowchart TD
    subgraph Input["Входные данные"]
        S[Токены: x₁, x₂, ..., x_T]
    end

    subgraph Embedding["Слой эмбеддингов"]
        E[Эмбеддинги + Positional Encoding]
    end

    subgraph EncoderStack["Стек энкодеров (N×)"]
        direction TB
        EN1[Энкодер слой 1]
        EN2[Энкодер слой 2]
        ENDot[⋯]
        ENN[Энкодер слой N]
    end

    subgraph EncoderLayer["Слой энкодера"]
        MHA1[Multi-Head Self-Attention<br/><b>без маски</b>]
        AD1[Add & LayerNorm]
        FFN1[Feed-Forward Network]
        AD2[Add & LayerNorm]
    end

    subgraph Output["Выходной слой"]
        OUT[Выходные представления<br/>контекстуализированные]
        TASK[Задача: классификация, NER, QA]
    end

    S --> E --> EncoderStack
    E --> MHA1 --> AD1 --> FFN1 --> AD2
    AD2 --> EN2

    ENN --> OUT --> TASK

    style Input fill:#e3f2fd
    style Embedding fill:#e8f5e9
    style EncoderStack fill:#f3e5f5
    style Output fill:#e0f7fa
```

#### 1.6. Пример: вход и выход

```
Вход:  "Я [MASK] этот фильм."
Обучение: модель предсказывает "люблю" или "ненавижу"
Задача: тональность ("люблю") → положительная классификация
```

---

### 2. Decoder-Only: архитектура для генерации

#### 2.1. Архитектура

Модели типа Decoder-Only используют только стек декодеров Transformer. Декодер в оригинальной архитектуре содержит три подслоя:

1. **Masked Multi-Head Self-Attention** — позволяет каждому токену взаимодействовать только с предыдущими токенами (каузальная маска);
2. **Cross-Attention** — внимание к выходу энкодера (в Decoder-Only моделях этот слой обычно отсутствует, так как энкодера нет);
3. **Position-wise Feed-Forward Network**.

Ключевая особенность — **каузальная маска**, которая запрещает токену обращаться к будущим токенам (тем, что находятся правее). Это обеспечивает авторегрессивность: модель может генерировать текст слева направо, по одному токену за раз.

#### 2.2. Обучение: авторегрессивное языковое моделирование

Decoder-Only модели обучаются на задаче **авторегрессивного языкового моделирования** (Causal Language Modeling, CLM). Модель учится предсказывать следующий токен на основе предыдущих:

$$
P(x_1, x_2, \dots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \dots, x_{t-1}),
$$

где $x_t$ — токен на позиции $t$. Функция потерь — кросс-энтропия между предсказанным распределением и истинным токеном:

$$
\mathcal{L}_{\text{CLM}} = -\sum_{t=1}^{T} \log P(x_t \mid x_{<t}).
$$

После обучения модель может генерировать текст, последовательно предсказывая следующий токен, используя методы декодирования (жадный поиск, beam search, температурная выборка).

#### 2.3. Задачи

Decoder-Only модели — это **генеративные модели**, способные создавать новый текст. Основные применения:

- **Генерация текста** — написание статей, стихов, историй;
- **Диалоговые системы** — чат-боты, виртуальные ассистенты;
- **Код-ассистенты** — написание и дополнение кода;
- **Машинный перевод** — генерация перевода;
- **Суммаризация** — создание кратких изложений;
- **Ответы на вопросы** — порождение ответа в свободной форме.

#### 2.4. Примеры моделей

- **GPT** (Generative Pre-trained Transformer) — семейство моделей OpenAI (GPT-1, GPT-2, GPT-3, GPT-4);
- **LLaMA** — открытые модели от Meta (LLaMA 1, 2, 3);
- **Qwen** — открытые модели от Alibaba;
- **Mistral** — открытые модели от Mistral AI;
- **Claude** — модели от Anthropic;
- **Gemini** — модели от Google (частично).

#### 2.5. Преимущества и ограничения

**Преимущества:**
- **Способность к генерации** — может создавать связные, осмысленные тексты;
- **Универсальность** — одна модель решает множество задач через промпт-инжиниринг;
- **Zero-shot и few-shot обучение** — способность решать новые задачи по инструкции;
- **Масштабируемость** — хорошо масштабируется с ростом параметров и данных.

**Ограничения:**
- **Односторонний контекст** — токен не видит того, что будет справа, что может быть неоптимально для задач понимания;
- **Тенденция к галлюцинациям** — может выдумывать факты при недостатке информации;
- **Высокие вычислительные затраты** на инференс (генерация требует последовательного вычисления токенов).

На диаграмме ниже показана архитектура Decoder-Only:

```mermaid
flowchart TD
    subgraph Input["Входные данные"]
        S[Токены: x₁, x₂, ..., x_T]
    end

    subgraph Embedding["Слой эмбеддингов"]
        E[Эмбеддинги + Positional Encoding]
    end

    subgraph DecoderStack["Стек декодеров (N×)"]
        direction TB
        DN1[Декодер слой 1]
        DN2[Декодер слой 2]
        DNDot[⋯]
        DNN[Декодер слой N]
    end

    subgraph DecoderLayer["Слой декодера"]
        MMHA[Masked Multi-Head Self-Attention<br/><b>с каузальной маской</b>]
        AD3[Add & LayerNorm]
        FFN2[Feed-Forward Network]
        AD5[Add & LayerNorm]
    end

    subgraph Output["Выходной слой"]
        LC[Линейный слой + Softmax]
        GEN[Генерация следующего токена]
    end

    S --> E --> DecoderStack
    E --> MMHA --> AD3 --> FFN2 --> AD5
    AD5 --> DN2

    DNN --> LC --> GEN --> O[Следующий токен]

    style Input fill:#e3f2fd
    style Embedding fill:#e8f5e9
    style DecoderStack fill:#fce4ec
    style Output fill:#e0f7fa
```

#### 2.6. Пример: вход и выход

```
Вход:  "Я люблю этот фильм, потому что он"
Генерация: "очень интересный и захватывающий."
```

---

### 3. Encoder-Decoder: архитектура для преобразования

#### 3.1. Архитектура

Модели типа Encoder-Decoder используют полную архитектуру Transformer, включающую как стек энкодеров, так и стек декодеров. Эта архитектура соответствует оригинальному Transformer и включает:

1. **Энкодер** — обрабатывает входную последовательность, создавая контекстуализированные представления;
2. **Декодер** — генерирует выходную последовательность, используя:
   - Masked Self-Attention (для авторегрессии);
   - Cross-Attention (внимание к выходу энкодера);
   - Feed-Forward Network.

Cross-Attention является ключевым компонентом: запросы $Q$ приходят из декодера, а ключи $K$ и значения $V$ — из выхода энкодера. Это позволяет декодеру «смотреть» на всю входную последовательность при генерации каждого токена.

#### 3.2. Обучение: sequence-to-sequence

Encoder-Decoder модели обучаются на задачах **sequence-to-sequence** (последовательность-в-последовательность), где входная и выходная последовательности могут иметь разную длину. Классический подход — **авторегрессивное обучение с учителем**:

1. На вход подаётся последовательность $X$ (например, предложение на английском);
2. Модель генерирует последовательность $Y$ (например, перевод на немецкий);
3. Обучение происходит через минимизацию кросс-энтропийной потери между сгенерированными и истинными токенами:

$$
\mathcal{L}_{\text{seq2seq}} = -\sum_{t=1}^{T_y} \log P(y_t \mid y_{<t}, X).
$$

#### 3.3. Задачи

Encoder-Decoder модели специализируются на задачах **преобразования** одной последовательности в другую:

- **Машинный перевод** — перевод текста с одного языка на другой;
- **Суммаризация** — сокращение длинного текста до краткого изложения;
- **Перефразирование** — пересказ текста другими словами;
- **Ответы на вопросы по тексту** (generative QA);
- **Генерация по структуре** — текст по таблице, диаграмме.

#### 3.4. Примеры моделей

- **Оригинальный Transformer** — архитектура для машинного перевода;
- **T5** (Text-to-Text Transfer Transformer) — унифицированная модель от Google, все задачи формулируются как «текст в текст»;
- **BART** — модель от Facebook, сочетающая BERT-подобное обучение и GPT-подобную генерацию;
- **Pegasus** — модель для суммаризации от Google;
- **mT5** — мультиязычная версия T5.

#### 3.5. Преимущества и ограничения

**Преимущества:**
- **Оптимальна для преобразований** — где структура входа и выхода различна;
- **Гибкость** — можно использовать для любых задач «текст-в-текст»;
- **Качество** — часто превосходит Decoder-Only на структурированных задачах.

**Ограничения:**
- **Больше параметров** — содержит и энкодер, и декодер;
- **Дороже в обучении** и инференсе;
- **Менее универсальна** — не подходит для открытых диалогов, где нет явного входа.

На диаграмме ниже показана полная архитектура Encoder-Decoder:

```mermaid
flowchart TD
    subgraph Input["Входные данные"]
        S[Входная последовательность: x₁, x₂, ..., x_T]
    end

    subgraph Encoder["Энкодер (N×)"]
        ENC[Стек энкодеров<br/>с Self-Attention]
    end

    subgraph Decoder["Декодер (N×)"]
        DEC[Стек декодеров<br/>с Masked Self-Attention<br/>и Cross-Attention]
    end

    subgraph Output["Выходные данные"]
        OUT[Генерация y₁, y₂, ..., y_M]
    end

    S --> Encoder -->|K, V| Decoder
    Decoder --> Output

    style Input fill:#e3f2fd
    style Encoder fill:#f3e5f5
    style Decoder fill:#fce4ec
    style Output fill:#e0f7fa
```

#### 3.6. Пример: вход и выход

```
Вход:  "The cat sat on the mat."
Выход: "Кот сидел на коврике."
```

---

### 4. Сравнительный анализ

#### 4.1. Сводная схема всех трёх архитектур

На диаграмме ниже представлены все три типа архитектур вместе для наглядного сравнения:

```mermaid
flowchart LR
    subgraph EO["Encoder-Only (BERT)"]
        direction LR
        E1[Энкодер] --> E2[Энкодер] --> E3[...] --> E4[Энкодер] --> EOut[Выход<br/>Понимание]
    end

    subgraph DO["Decoder-Only (GPT)"]
        direction LR
        D1[Декодер] --> D2[Декодер] --> D3[...] --> D4[Декодер] --> DOut[Выход<br/>Генерация]
    end

    subgraph ED["Encoder-Decoder (T5)"]
        direction LR
        En1[Энкодер] --> En2[Энкодер] --> En3[...] --> En4[Энкодер] -->|K, V| De1[Декодер] --> De2[Декодер] --> De3[...] --> De4[Декодер] --> EDOut[Выход<br/>Преобразование]
    end

    style EO fill:#e3f2fd
    style DO fill:#fce4ec
    style ED fill:#f3e5f5
```

#### 4.2. Сравнительная таблица

| Критерий | Encoder-Only (BERT) | Decoder-Only (GPT) | Encoder-Decoder (T5) |
|----------|---------------------|-------------------|----------------------|
| **Архитектура** | Только энкодер | Только декодер | Энкодер + декодер |
| **Обучение** | MLM (маскирование) | Авторегрессия (next token) | Seq2seq с учителем |
| **Контекст** | Двусторонний | Односторонний (causal) | Двусторонний (энкодер) + односторонний (декодер) |
| **Основные задачи** | Понимание, классификация, NER | Генерация, диалоги, кодинг | Перевод, суммаризация |
| **Примеры моделей** | BERT, RoBERTa | GPT, LLaMA, Qwen | T5, BART |
| **Параллелизация** | Полная | Частичная (инференс последовательный) | Полная (энкодер) + частичная (декодер) |
| **Вычислительная сложность** | $O(T^2 \cdot d)$ | $O(T^2 \cdot d)$ (обучение)<br/>$O(T \cdot d^2)$ (инференс) | $O(T_x^2 \cdot d + T_y^2 \cdot d)$ |
| **Способность к генерации** | Нет | Да (высокая) | Да |
| **Способность к пониманию** | Да (высокая) | Ограниченная | Да (высокая) |
| **Размер параметров** | Средний | Большой | Самый большой |

#### 4.3. Когда какую архитектуру выбирать

Выбор архитектуры определяется характером задачи:

| Тип задачи | Рекомендуемая архитектура | Причина |
|------------|---------------------------|---------|
| Классификация текста | Encoder-Only | Двусторонний контекст даёт лучшее понимание |
| Извлечение информации (NER, QA) | Encoder-Only | Точное понимание текста критично |
| Генерация текста, диалоги | Decoder-Only | Оптимальна для авторегрессивной генерации |
| Код-ассистент | Decoder-Only | Генерация требует одностороннего потока |
| Машинный перевод | Encoder-Decoder | Требуется понимание входа и генерация выхода |
| Суммаризация | Encoder-Decoder | Комбинация понимания и сжатия |
| Ответы на вопросы (extractive) | Encoder-Only | Поиск ответа в тексте |
| Ответы на вопросы (generative) | Decoder-Only или Encoder-Decoder | Зависит от сложности входа |

---

### Заключение

Три типа архитектур на основе Transformer — Encoder-Only, Decoder-Only и Encoder-Decoder — представляют собой различные варианты использования механизма самовнимания для решения разных классов задач. Понимание различий между ними позволяет осознанно выбирать архитектуру для конкретной задачи, оценивать её возможности и ограничения, а также правильно интерпретировать поведение моделей. Современные исследования показывают, что Decoder-Only архитектуры становятся всё более универсальными, однако Encoder-Only и Encoder-Decoder по-прежнему сохраняют свои ниши, особенно в задачах, требующих глубокого понимания или структурированного преобразования.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

3. Radford, A., et al. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI.  
   🔗 [https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)

4. Raffel, C., et al. (2019). *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer*. JMLR.  
   🔗 [https://arxiv.org/abs/1910.10683](https://arxiv.org/abs/1910.10683)

5. Lewis, M., et al. (2019). *BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension*. ACL.  
   🔗 [https://arxiv.org/abs/1910.13461](https://arxiv.org/abs/1910.13461)

## Тема 2.1. Структура Encoder-Decoder: фундаментальная архитектура Transformer

Архитектура Transformer в своей оригинальной конфигурации [1] представляет собой **симметричную структуру «энкодер-декодер»** (encoder-decoder), предназначенную для задач преобразования последовательностей. В данном разделе мы дадим исчерпывающее математическое описание каждого компонента — от входного слоя до генерации выходного токена, — чтобы сформировать полное понимание потоков данных и преобразований.

---

### 1. Общая схема и поток данных

Пусть дана входная последовательность $\mathbf{x} = (x_1, \dots, x_T)$ длины $T$ (например, слова на исходном языке) и целевая последовательность $\mathbf{y} = (y_1, \dots, y_M)$ длины $M$ (например, перевод). Архитектура Encoder-Decoder преобразует $\mathbf{x}$ в $\mathbf{y}$ через следующие этапы:

1. **Энкодер** преобразует $\mathbf{x}$ в матрицу контекстных представлений $Z \in \mathbb{R}^{T \times d_{\text{model}}}$.
2. **Декодер** авторегрессивно генерирует $\mathbf{y}$, на каждом шаге $t$ используя $Z$ и уже сгенерированные токены $y_{<t}$.

Математически это записывается как:

$$
Z = \text{Encoder}(\mathbf{x}), \qquad
P(y_t \mid y_{<t}, \mathbf{x}) = \text{Decoder}(y_{<t}, Z).
$$

Полный поток данных показан на диаграмме:

```mermaid
flowchart TD
    subgraph Input["Входной текст"]
        X[("x₁, x₂, ..., x_T")]
    end

    subgraph Encoder["Энкодер"]
        direction TB
        ENC_EMB[Эмбеддинги + PE]
        ENC_LAYERS[Стек энкодеров N×]
        Z[("Z ∈ ℝ^{T×d_model}")]
    end

    subgraph Decoder["Декодер"]
        direction TB
        DEC_EMB[Эмбеддинги + PE для выходных токенов]
        DEC_LAYERS[Стек декодеров N×]
        H[("Скрытые состояния")]
    end

    subgraph Output["Генерация"]
        PROJ[Линейный слой + Softmax]
        Y[("y₁, y₂, ..., y_M")]
    end

    X --> ENC_EMB --> ENC_LAYERS --> Z
    Z -->|"K, V через Cross-Attention"| DEC_LAYERS
    Y -->|"авторегрессивно"| DEC_EMB --> DEC_LAYERS
    DEC_LAYERS --> H --> PROJ --> Y

    style Input fill:#e3f2fd
    style Encoder fill:#f3e5f5
    style Decoder fill:#fce4ec
    style Output fill:#e0f7fa
```

---

### 2. Энкодер: математика преобразования входа в контекст

#### 2.1. Входной слой: эмбеддинги и позиционное кодирование

Каждый токен $x_i$ отображается в вектор размерности $d_{\text{model}}$ с помощью обучаемой матрицы эмбеддингов $E \in \mathbb{R}^{V \times d_{\text{model}}}$, где $V$ — размер словаря:

$$
\mathbf{e}_i = E[x_i] \in \mathbb{R}^{d_{\text{model}}}.
$$

Затем добавляется позиционное кодирование $\mathbf{p}_i \in \mathbb{R}^{d_{\text{model}}}$, которое в оригинале определяется синусоидальными функциями:

$$
\begin{aligned}
\mathbf{p}_{i, 2j} &= \sin\left(\frac{i}{10000^{2j/d_{\text{model}}}}\right), \\
\mathbf{p}_{i, 2j+1} &= \cos\left(\frac{i}{10000^{2j/d_{\text{model}}}}\right),
\end{aligned}
$$

где $j$ — индекс измерения. В результате получается входная матрица энкодера:

$$
\mathbf{X} = [\mathbf{e}_1 + \mathbf{p}_1, \dots, \mathbf{e}_T + \mathbf{p}_T] \in \mathbb{R}^{T \times d_{\text{model}}}.
$$

#### 2.2. Стек слоёв энкодера

Энкодер состоит из $N$ идентичных слоёв (в оригинале $N=6$). Каждый слой содержит два подслоя:

1. **Multi-Head Self-Attention** (без маскировки).
2. **Position-wise Feed-Forward Network (FFN)**.

Вокруг каждого подслоя применяется **остаточная связь** и **нормализация** (в оригинале — пост-нормализация, но современные реализации часто используют пре-нормализацию). Для определённости опишем пост-нормализацию, как в оригинале:

$$
\begin{aligned}
\mathbf{X}^{(1)} &= \text{LayerNorm}\big(\mathbf{X} + \text{MHA}(\mathbf{X})\big), \\
\mathbf{X}^{(2)} &= \text{LayerNorm}\big(\mathbf{X}^{(1)} + \text{FFN}(\mathbf{X}^{(1)})\big),
\end{aligned}
$$

где $\mathbf{X}$ — вход слоя, $\mathbf{X}^{(2)}$ — выход слоя.

##### 2.2.1. Multi-Head Self-Attention (MHA)

Сначала вычисляются запросы, ключи и значения для всех позиций:

$$
\mathbf{Q} = \mathbf{X} W^Q, \quad \mathbf{K} = \mathbf{X} W^K, \quad \mathbf{V} = \mathbf{X} W^V,
$$

где $W^Q, W^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$, $W^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$, причём $d_k = d_v = d_{\text{model}} / h$ (в оригинале $h=8$).

Затем вычисляется $h$ голов внимания:

$$
\text{head}_i = \text{Attention}\left(\mathbf{Q} W_i^Q,\; \mathbf{K} W_i^K,\; \mathbf{V} W_i^V\right),
$$

где

$$
\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}}\right) \mathbf{V}.
$$

Результаты голов конкатенируются и проецируются:

$$
\text{MHA}(\mathbf{X}) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) \, W^O,
$$

где $W^O \in \mathbb{R}^{h d_v \times d_{\text{model}}}$.

##### 2.2.2. Position-wise Feed-Forward Network (FFN)

FFN применяется к каждой позиции независимо:

$$
\text{FFN}(\mathbf{x}) = \max(0, \mathbf{x} W_1 + b_1) W_2 + b_2,
$$

где $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$, $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$, а $d_{\text{ff}} = 2048$ в оригинале.

##### 2.2.3. Layer Normalization

Нормализация выполняется по измерению признаков для каждой позиции отдельно:

$$
\text{LayerNorm}(\mathbf{x}) = \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta,
$$

где $\mu$ и $\sigma^2$ — среднее и дисперсия по $d_{\text{model}}$, $\gamma, \beta$ — обучаемые параметры.

#### 2.3. Выход энкодера

После прохождения всех $N$ слоёв получается матрица контекстных представлений:

$$
\mathbf{Z} = \text{Encoder}(\mathbf{X}) \in \mathbb{R}^{T \times d_{\text{model}}}.
$$

Эта матрица содержит для каждого входного токена вектор, обогащённый информацией о всей последовательности.

---

### 3. Декодер: авторегрессивная генерация

Декодер генерирует выходную последовательность по одному токену за раз. На шаге $t$ он получает:

- матрицу $\mathbf{Z}$ из энкодера,
- уже сгенерированные токены $y_1, \dots, y_{t-1}$ (для $t=1$ — только специальный токен начала).

#### 3.1. Входной слой декодера

Аналогично энкодеру, каждый входной токен $y_i$ преобразуется в эмбеддинг и суммируется с позиционным кодированием (позиции соответствуют выходной последовательности). На шаге $t$ декодер использует матрицу $\mathbf{Y}_{<t} \in \mathbb{R}^{(t-1) \times d_{\text{model}}}$.

#### 3.2. Стек слоёв декодера

Каждый слой декодера содержит **три** подслоя:

1. **Masked Multi-Head Self-Attention** — позволяет каждому токену обращаться только к предыдущим токенам.
2. **Cross-Attention** (Encoder-Decoder Attention) — позволяет декодеру обращаться к $\mathbf{Z}$.
3. **Position-wise FFN**.

##### 3.2.1. Masked Self-Attention

Вычисления аналогичны обычному Self-Attention, но перед softmax в матрицу $\mathbf{Q}\mathbf{K}^T$ добавляется маска, которая обнуляет (устанавливает в $-\infty$) все позиции, соответствующие будущим токенам:

$$
\text{MaskedAttn}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right) \mathbf{V},
$$

где $\mathbf{M}_{ij} = 0$ при $j \le i$ и $-\infty$ при $j > i$.

##### 3.2.2. Cross-Attention

Здесь запросы $\mathbf{Q}$ берутся из предыдущего подслоя декодера, а ключи и значения — из выхода энкодера $\mathbf{Z}$:

$$
\mathbf{Q} = \mathbf{H} W^Q, \quad \mathbf{K} = \mathbf{Z} W^K, \quad \mathbf{V} = \mathbf{Z} W^V,
$$

и применяется обычное внимание без маски.

#### 3.3. Выходной слой

После стека декодеров получается скрытое состояние $\mathbf{H} \in \mathbb{R}^{M \times d_{\text{model}}}$. Затем применяется линейный слой и softmax для получения распределения вероятностей следующего токена:

$$
P(y_t \mid y_{<t}, \mathbf{x}) = \text{softmax}(\mathbf{H}_{t-1} W_{\text{out}} + b_{\text{out}}),
$$

где $\mathbf{H}_{t-1}$ — представление последнего сгенерированного токена (или начального токена) размерности $d_{\text{model}}$.

---

### 4. Детали взаимодействия: Cross-Attention и передача информации

Cross-Attention является ключевым мостом между энкодером и декодером. Он позволяет декодеру на каждом шаге динамически извлекать информацию из всей входной последовательности. Математически, для каждого слоя декодера:

$$
\mathbf{Q} = \text{LayerNorm}(\mathbf{H}_{\text{prev}} + \text{MaskedSelfAttn}(\mathbf{H}_{\text{prev}})),
$$
$$
\mathbf{K} = \mathbf{Z}, \quad \mathbf{V} = \mathbf{Z},
$$
$$
\mathbf{H}_{\text{new}} = \text{LayerNorm}(\mathbf{Q} + \text{CrossAttn}(\mathbf{Q}, \mathbf{K}, \mathbf{V})).
$$

Это гарантирует, что на каждом шаге генерации модель имеет доступ ко всему контексту входа.

#### Схема размерностей на каждом этапе

```mermaid
flowchart LR
    A["Вход: T×d_model"] --> B["Энкодер: T×d_model"]
    B --> C["Z: T×d_model"]
    C --> D["Cross-Attention<br/>Q:1×d_k, K:T×d_k, V:T×d_v"]
    D --> E["Выход: 1×d_v"]
    E --> F["Следующий токен"]
    
    style A fill:#e3f2fd
    style B fill:#f3e5f5
    style C fill:#f3e5f5
    style D fill:#fff3e0
    style E fill:#fce4ec
    style F fill:#e0f7fa
```

---

### 5. Полный пример: машинный перевод (с пошаговой математикой)

Рассмотрим перевод предложения **"I love reading books"** на русский.

#### Шаг 1: Токенизация и эмбеддинги
Токены: `["I", "love", "reading", "books"]` → эмбеддинги + PE → $\mathbf{X} \in \mathbb{R}^{4 \times 512}$.

#### Шаг 2: Энкодер
Через $N$ слоёв получаем $\mathbf{Z} \in \mathbb{R}^{4 \times 512}$. Каждый вектор $\mathbf{z}_i$ содержит контекст всего предложения.

#### Шаг 3: Генерация декодером

| Шаг $t$ | Вход декодера ($y_{<t}$) | Действие | Выходной токен |
|---------|--------------------------|----------|----------------|
| 1 | `[START]` | Cross-Attention к $\mathbf{Z}$ -> предсказание | `"Я"` |
| 2 | `[START]`, `"Я"` | Снова Cross-Attention, учитывая `"Я"` | `"люблю"` |
| 3 | ... `"люблю"` | ... | `"читать"` |
| 4 | ... `"читать"` | ... | `"книги"` |
| 5 | ... `"книги"` | ... | `[END]` |

На каждом шаге вычисляется:

$$
\mathbf{h}_t = \text{DecoderLayer}(\mathbf{h}_{t-1}, \mathbf{Z}), \quad
P(y_t) = \text{softmax}(\mathbf{h}_t W_{\text{out}}).
$$

#### Визуализация процесса

```mermaid
flowchart TD
    A["Вход: I love reading books"] --> B["Энкодер: Z"]
    B --> C1["Шаг1: [START] → Я"]
    B --> C2["Шаг2: [START] Я → люблю"]
    B --> C3["Шаг3: [START] Я люблю → читать"]
    B --> C4["Шаг4: [START] Я люблю читать → книги"]
    B --> C5["Шаг5: ... → [END]"]
    C1 --> D["Выход: Я люблю читать книги"]
    C2 --> D
    C3 --> D
    C4 --> D
    C5 --> D
```

---

### 6. Обучение и функция потерь

Обучение происходит с учителем на парах (вход, эталонный выход). Для каждого шага генерации вычисляется кросс-энтропийная потеря между предсказанным распределением и истинным токеном:

$$
\mathcal{L} = -\sum_{t=1}^{M} \log P(y_t^* \mid y_{<t}, \mathbf{x}),
$$

где $y_t^*$ — эталонный токен. Градиенты распространяются через всю архитектуру (энкодер и декодер) с помощью обратного распространения ошибки.

---

### 7. Роли энкодера и декодера (таблица)

| Аспект | Энкодер | Декодер |
|--------|---------|---------|
| **Задача** | Понимание входа, создание контекста | Генерация выхода |
| **Вход** | Токены исходного текста | Токены выхода (предыдущие) + представления энкодера |
| **Выход** | Матрица контекстных представлений $Z$ | Распределение вероятностей следующего токена |
| **Тип внимания** | Self-Attention (двусторонний) | Masked Self-Attention + Cross-Attention |
| **Параллелизм** | Полный (все позиции одновременно) | Частичный (инференс последовательный) |
| **Маска** | Отсутствует | Присутствует (для авторегрессии) |

---

### 8. Заключение

В этом разделе мы рассмотрели полную математическую модель архитектуры Encoder-Decoder Transformer, от входных эмбеддингов до генерации выходных токенов. Каждый компонент — позиционное кодирование, многоголовое внимание, остаточные связи, нормализация, кросс-внимание — был описан с использованием явных формул и размерностей. Эта архитектура, сохраняя свою актуальность в задачах перевода и суммаризации, заложила основу для всех последующих модификаций Transformer.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

## Тема 2.2. Стек кодировщиков и декодировщиков

В предыдущем разделе мы рассмотрели общую архитектуру Encoder-Decoder и взаимодействие между её компонентами. Однако реальная мощь Transformer раскрывается в повторении идентичных слоёв — образовании **стека** (stack). Именно глубина стека позволяет модели выделять иерархические паттерны, переходить от поверхностных лексических связей к глубоким семантическим отношениям. В этом разделе мы детально разберём структуру отдельного слоя энкодера и декодера, их композицию в стек и эволюцию подходов к нормализации.

---

### 1. Структура стека: от слоя к глубине

#### 1.1. N идентичных слоёв

В оригинальной статье [1] Transformer состоит из стека **N = 6** идентичных слоёв как в энкодере, так и в декодере. Каждый слой имеет одинаковую архитектуру, но **не разделяет веса** — параметры каждого слоя обучаются независимо.

Почему слои не разделяют веса? В рекуррентных сетях (RNN) один и тот же слой применяется к каждому временному шагу (разделение весов по времени), что является следствием их рекуррентной природы. В Transformer, напротив, слои являются отдельными вычислительными блоками, и каждый из них может специализироваться на своём уровне абстракции. Исследования показывают, что нижние слои чаще фокусируются на локальных синтаксических связях (зависимости между соседними словами), средние — на семантических ролях, а верхние — на глобальном смысле и дискурсе [2]. Разделение весов позволило бы всем слоям выполнять одну и ту же функцию, что существенно ограничило бы выразительность модели.

#### 1.2. Как с каждым слоем растёт "глубина понимания"

Каждый слой энкодера преобразует входные представления, добавляя к ним всё более абстрактную контекстную информацию. На выходе первого слоя векторы токенов учитывают ближайшее окружение. После прохождения через несколько слоёв каждый вектор содержит информацию о всей последовательности, но на разных уровнях обобщения. Это аналогично тому, как в свёрточных сетях ранние слои выделяют края и текстуры, а глубокие — целые объекты.

Визуализация внимания в разных слоях Transformer подтверждает эту иерархию: ранние слои часто показывают локальное внимание (соседние слова), а поздние — глобальное (дальние зависимости и связи между предложениями) [3].

---

### 2. Слой энкодера: архитектура и поток данных

Каждый слой энкодера (рисунок 1) состоит из двух основных подслоёв, вокруг каждого из которых применяются остаточная связь и нормализация.

```mermaid
flowchart TD
    subgraph EncoderLayer["Слой энкодера"]
        direction TB
        Input[("Вход: X ∈ ℝ^{T×d_model}")]
        
        MHA["Подслой 1: Multi-Head Self-Attention<br/>Q, K, V из X (без маски)"]
        AD1["Add & LayerNorm"]
        FFN["Подслой 2: Position-wise FFN<br/>ReLU(W₁x + b₁)W₂ + b₂"]
        AD2["Add & LayerNorm"]
        
        Output[("Выход: X' ∈ ℝ^{T×d_model}")]
    end
    
    Input --> MHA
    MHA -->|" + X"| AD1
    AD1 --> FFN
    FFN -->|" + AD1"| AD2
    AD2 --> Output
    
    style Input fill:#e3f2fd
    style MHA fill:#f3e5f5
    style AD1 fill:#fff3e0
    style FFN fill:#f3e5f5
    style AD2 fill:#fff3e0
    style Output fill:#e0f7fa
```

#### 2.1. Подслой 1: Multi-Head Self-Attention

В этом подслое каждый токен взаимодействует со всеми другими токенами в последовательности. Поскольку маска отсутствует, внимание является **двусторонним** — токен может использовать информацию как слева, так и справа от себя.

Математически:

$$
\text{head}_i = \text{Attention}\left(\mathbf{X} W_i^Q,\; \mathbf{X} W_i^K,\; \mathbf{X} W_i^V\right),
$$

$$
\text{MHA}(\mathbf{X}) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O.
$$

Размерности остаются неизменными: вход $\mathbf{X} \in \mathbb{R}^{T \times d_{\text{model}}}$, выход также $\in \mathbb{R}^{T \times d_{\text{model}}}$.

#### 2.2. Подслой 2: Position-wise Feed-Forward Network

FFN применяется к каждой позиции независимо (position-wise) и состоит из двух линейных преобразований с функцией активации ReLU (в оригинале):

$$
\text{FFN}(\mathbf{x}) = \max(0, \mathbf{x} W_1 + b_1) W_2 + b_2,
$$

где $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$, $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$, причём $d_{\text{ff}} = 2048$ в оригинале (в 4 раза больше $d_{\text{model}} = 512$).

#### 2.3. Остаточные связи и нормализация

В оригинальной статье используется **пост-нормализация** (post-norm):

$$
\mathbf{X}^{(1)} = \text{LayerNorm}\left(\mathbf{X} + \text{MHA}(\mathbf{X})\right),
$$

$$
\mathbf{X}^{(2)} = \text{LayerNorm}\left(\mathbf{X}^{(1)} + \text{FFN}(\mathbf{X}^{(1)})\right).
$$

Остаточная связь $\mathbf{X} + \text{Sublayer}(\mathbf{X})$ позволяет градиентам свободно проходить через сеть, предотвращая проблему исчезающих градиентов. Нормализация стабилизирует распределение активаций.

---

### 3. Слой декодера: архитектура и поток данных

Декодер сложнее энкодера: он содержит **три** подслоя вместо двух. Это обусловлено необходимостью как авторегрессивной генерации, так и учёта выходных данных энкодера.

```mermaid
flowchart TD
    subgraph DecoderLayer["Слой декодера"]
        direction TB
        Input[("Вход: Y ∈ ℝ^{M×d_model}")]
        Z[("Выход энкодера: Z ∈ ℝ^{T×d_model}")]
        
        MMHA["Подслой 1: Masked Multi-Head Self-Attention<br/>Q, K, V из Y (с каузальной маской)"]
        AD1["Add & LayerNorm"]
        CA["Подслой 2: Cross-Attention<br/>Q из Y, K, V из Z"]
        AD2["Add & LayerNorm"]
        FFN["Подслой 3: Position-wise FFN"]
        AD3["Add & LayerNorm"]
        
        Output[("Выход: Y' ∈ ℝ^{M×d_model}")]
    end
    
    Input --> MMHA
    MMHA -->|" + Y"| AD1
    AD1 --> CA
    Z --> CA
    CA -->|" + AD1"| AD2
    AD2 --> FFN
    FFN -->|" + AD2"| AD3
    AD3 --> Output
    
    style Input fill:#e3f2fd
    style Z fill:#f3e5f5
    style MMHA fill:#fce4ec
    style AD1 fill:#fff3e0
    style CA fill:#ffcdd2
    style AD2 fill:#fff3e0
    style FFN fill:#fce4ec
    style AD3 fill:#fff3e0
    style Output fill:#e0f7fa
```

#### 3.1. Подслой 1: Masked Multi-Head Self-Attention

Этот подслой аналогичен Self-Attention в энкодере, но с **каузальной маской**, которая запрещает токену обращаться к будущим токенам.

Маска $\mathbf{M} \in \mathbb{R}^{M \times M}$ имеет вид:

$$
\mathbf{M}_{ij} =
\begin{cases}
0, & i \ge j \quad (\text{разрешено обращаться к прошлым и текущему}), \\
-\infty, & i < j \quad (\text{запрещено обращаться к будущим}).
\end{cases}
$$

Тогда:

$$
\text{MaskedAttn}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right) \mathbf{V}.
$$

#### 3.2. Подслой 2: Cross-Attention (Encoder-Decoder Attention)

Это ключевой мост между энкодером и декодером. Здесь:

- **Запросы $\mathbf{Q}$** приходят из предыдущего подслоя декодера (после Masked Self-Attention);
- **Ключи $\mathbf{K}$** и **Значения $\mathbf{V}$** приходят из выхода стека энкодеров $\mathbf{Z}$.

Это позволяет декодеру динамически извлекать информацию из входного текста на каждом шаге генерации.

$$
\text{CrossAttn}(\mathbf{Q}, \mathbf{Z}) = \text{Attention}\left(\mathbf{Q}, \mathbf{Z}W^K, \mathbf{Z}W^V\right).
$$

#### 3.3. Подслой 3: Position-wise Feed-Forward Network

Аналогичен FFN в энкодере и применяется независимо к каждой позиции.

#### 3.4. Нормализация в декодере

Аналогично энкодеру, в оригинале используется пост-нормализация:

$$
\mathbf{Y}^{(1)} = \text{LayerNorm}\left(\mathbf{Y} + \text{MaskedAttn}(\mathbf{Y})\right),
$$

$$
\mathbf{Y}^{(2)} = \text{LayerNorm}\left(\mathbf{Y}^{(1)} + \text{CrossAttn}(\mathbf{Y}^{(1)}, \mathbf{Z})\right),
$$

$$
\mathbf{Y}^{(3)} = \text{LayerNorm}\left(\mathbf{Y}^{(2)} + \text{FFN}(\mathbf{Y}^{(2)})\right).
$$

---

### 4. Стек из N слоёв

Каждый слой энкодера или декодера получает на вход выход предыдущего слоя. Таким образом, стек можно представить как композицию функций:

$$
\text{EncoderStack}(\mathbf{X}) = f_N \circ f_{N-1} \circ \dots \circ f_1(\mathbf{X}),
$$

где $f_i$ — $i$-й слой энкодера. Эта композиция позволяет модели строить всё более абстрактные представления.

```mermaid
flowchart TD
    subgraph EncoderStack["Стек энкодеров (N×)"]
        direction TB
        Input[("Вход: X₀ ∈ ℝ^{T×d_model}")]
        L1["Слой 1"]
        L2["Слой 2"]
        L3["..."]
        LN["Слой N"]
        Output[("Выход: Z = X_N ∈ ℝ^{T×d_model}")]
    end
    
    Input --> L1
    L1 --> L2
    L2 --> L3
    L3 --> LN
    LN --> Output
    
    style Input fill:#e3f2fd
    style L1 fill:#f3e5f5
    style L2 fill:#f3e5f5
    style L3 fill:#f3e5f5
    style LN fill:#f3e5f5
    style Output fill:#e0f7fa
```

---

### 5. Post-Norm vs Pre-Norm: эволюция нормализации

Оригинальный Transformer использовал **пост-нормализацию** (post-norm). Однако при масштабировании моделей (увеличении числа слоёв и параметров) этот подход стал приводить к нестабильности обучения. Современные модели (GPT, LLaMA, Qwen) используют **пре-нормализацию** (pre-norm).

#### 5.1. Сравнение подходов

| Аспект | Post-Norm (оригинал) | Pre-Norm (современный) |
|--------|----------------------|------------------------|
| **Формула** | $\text{LayerNorm}(x + \text{Sublayer}(x))$ | $x + \text{Sublayer}(\text{LayerNorm}(x))$ |
| **Порядок** | Сложение → Нормализация | Нормализация → Сложение |
| **Градиенты** | Затухают в глубоких сетях | Проходят напрямую через остаточные связи |
| **Стабильность** | Требует осторожной инициализации и LR | Устойчива к большим LR и глубоким сетям |
| **Использование** | Оригинальный Transformer | GPT, LLaMA, Mistral, Qwen |

#### 5.2. Математический анализ

В **Post-Norm** выход слоя имеет вид:

$$
\mathbf{x}_{out} = \text{LayerNorm}(\mathbf{x}_{in} + \text{Sublayer}(\mathbf{x}_{in})).
$$

При обратном распространении градиенты проходят через LayerNorm, которая масштабирует их. В глубоких сетях это приводит к постепенному затуханию градиентов.

В **Pre-Norm** выход слоя:

$$
\mathbf{x}_{out} = \mathbf{x}_{in} + \text{Sublayer}(\text{LayerNorm}(\mathbf{x}_{in})).
$$

Здесь градиенты от $\mathbf{x}_{out}$ к $\mathbf{x}_{in}$ распространяются по двум путям: напрямую (через остаточную связь, с коэффициентом 1) и через подслой. Прямой путь обеспечивает, что градиенты не затухают даже в очень глубоких сетях.

Это особенно важно для современных моделей, которые могут иметь до 100 и более слоёв.

---

### 6. Сравнение энкодера и декодера

| Критерий | Энкодер | Декодер |
|----------|---------|---------|
| **Количество подслоёв** | 2 | 3 |
| **Self-Attention** | Без маски (двусторонний) | С маской (односторонний, каузальный) |
| **Cross-Attention** | Отсутствует | Присутствует (Q из декодера, K,V из энкодера) |
| **Задача** | Понимание входа | Генерация выхода |
| **Параллелизация** | Полная | Ограниченная (инференс последовательный) |

---

### 7. Схема размерностей на каждом этапе

```mermaid
flowchart LR
    subgraph Encoder["Энкодер"]
        E1["X: T×d_model"] --> E2["Self-Attn: T×d_model"] --> E3["Add & Norm: T×d_model"] --> E4["FFN: T×d_model"] --> E5["Add & Norm: T×d_model"]
    end
    
    subgraph Decoder["Декодер (шаг t)"]
        D1["Y_{<t}: (t-1)×d_model"] --> D2["Masked Attn: (t-1)×d_model"] --> D3["Add & Norm: (t-1)×d_model"] --> D4["Cross-Attn: (t-1)×d_model"] --> D5["Add & Norm: (t-1)×d_model"] --> D6["FFN: (t-1)×d_model"] --> D7["Add & Norm: (t-1)×d_model"] --> D8["Проекция: vocab_size"]
    end
    
    Z["Z: T×d_model (из энкодера)"] --> D4
```

---

### 8. Заключение

Стек слоёв в энкодере и декодере Transformer представляет собой мощный механизм иерархического извлечения признаков. Энкодер последовательно уточняет представления токенов, обогащая их контекстной информацией, а декодер авторегрессивно генерирует выходную последовательность, используя как внутренние зависимости, так и информацию из энкодера. Переход от пост-нормализации к пре-нормализации стал ключевым фактором, позволившим масштабировать модели до сотен слоёв и миллиардов параметров, что лежит в основе успеха современных LLM.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Tenney, I., et al. (2019). *BERT Rediscovers the Classical NLP Pipeline*. ACL.  
   🔗 [https://arxiv.org/abs/1905.05950](https://arxiv.org/abs/1905.05950)

3. Clark, K., et al. (2019). *What Does BERT Look At? An Analysis of BERT's Attention*. BlackboxNLP Workshop.  
   🔗 [https://arxiv.org/abs/1906.04341](https://arxiv.org/abs/1906.04341)

4. Xiong, R., et al. (2020). *On Layer Normalization in the Transformer Architecture*. ICML.  
   🔗 [https://arxiv.org/abs/2002.04745](https://arxiv.org/abs/2002.04745)

## Тема 2.3. Поток данных: от входа до выхода

В предыдущих разделах мы рассмотрели отдельные компоненты архитектуры Transformer: энкодер, декодер, механизмы внимания и нормализации. Теперь настало время собрать все части в единую картину и проследить полный путь данных — от момента, когда исходный текст попадает на вход модели, до генерации выходного токена. Понимание этого потока критически важно для осознания того, как все компоненты взаимодействуют друг с другом в реальном времени.

---

### 1. Общая схема потока данных

На высоком уровне поток данных в архитектуре Encoder-Decoder можно представить следующим образом:

```mermaid
flowchart TD
    subgraph Input["ВХОД"]
        T1["Исходный текст: 'I love reading'"]
        T2["Токенизация: ['I', 'love', 'read', 'ing']"]
        T3["IDs: [42, 156, 389, 1023]"]
        T4["Эмбеддинги + PE: X ∈ ℝ^{4×512}"]
    end

    subgraph Encoder["ЭНКОДЕР (N=6 слоёв)"]
        E1["Self-Attention"]
        E2["Add & Norm"]
        E3["FFN"]
        E4["Add & Norm"]
    end

    subgraph Decoder["ДЕКОДЕР (N=6 слоёв, авторегрессивно)"]
        D1["Masked Self-Attention"]
        D2["Add & Norm"]
        D3["Cross-Attention"]
        D4["Add & Norm"]
        D5["FFN"]
        D6["Add & Norm"]
    end

    subgraph Output["ВЫХОД"]
        O1["Линейный слой: d_model → vocab_size"]
        O2["Softmax"]
        O3["Выбор токена"]
    end

    T4 --> Encoder
    Encoder -->|"Z ∈ ℝ^{4×512}"| Decoder
    Decoder --> O1 --> O2 --> O3
    O3 -->|"следующий токен"| Decoder

    style Input fill:#e3f2fd
    style Encoder fill:#f3e5f5
    style Decoder fill:#fce4ec
    style Output fill:#e0f7fa
```

На диаграмме видно ключевое свойство: **авторегрессивная петля** — выходной токен возвращается в декодер для генерации следующего.

---

### 2. Этап 1: От текста к эмбеддингам

#### 2.1. Токенизация

Первый шаг — преобразование исходного текста в последовательность токенов. Токенизация разбивает текст на минимальные смысловые единицы: слова, подслова или символы.

Пример: предложение *"I love reading"* токенизируется как `["I", "love", "read", "ing"]` (если используется BPE-токенизатор, как в оригинальной статье). Каждому токену присваивается уникальный числовой идентификатор из словаря модели. Пусть наш словарь имеет размер $V = 50000$, тогда:

$$
\text{["I", "love", "read", "ing"]} \rightarrow [42, 156, 389, 1023].
$$

#### 2.2. Эмбеддинги

Каждый идентификатор преобразуется в вектор фиксированной размерности $d_{\text{model}}$ с помощью обучаемой матрицы эмбеддингов $E \in \mathbb{R}^{V \times d_{\text{model}}}$:

$$
\mathbf{e}_i = E[\text{token\_id}_i] \in \mathbb{R}^{d_{\text{model}}}.
$$

В оригинальном Transformer $d_{\text{model}} = 512$. Таким образом, мы получаем матрицу токенных эмбеддингов:

$$
\mathbf{E}_{\text{tokens}} \in \mathbb{R}^{T \times d_{\text{model}}},
$$

где $T$ — длина последовательности (в нашем примере $T = 4$).

---

### 3. Этап 2: Позиционное кодирование

Поскольку Self-Attention не имеет встроенного понятия порядка, нам необходимо добавить информацию о позиции токена в последовательности. В оригинальной статье используется **синусоидальное позиционное кодирование**:

$$
\begin{aligned}
\mathbf{PE}_{(pos, 2i)} &= \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \\
\mathbf{PE}_{(pos, 2i+1)} &= \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right),
\end{aligned}
$$

где $pos$ — позиция токена (0, 1, 2, ...), $i$ — индекс измерения (0, 1, ..., $d_{\text{model}}/2 - 1$).

Позиционное кодирование $\mathbf{PE} \in \mathbb{R}^{T \times d_{\text{model}}}$ поэлементно суммируется с токенными эмбеддингами:

$$
\mathbf{X} = \mathbf{E}_{\text{tokens}} + \mathbf{PE} \in \mathbb{R}^{T \times d_{\text{model}}}.
$$

Этот процесс можно проиллюстрировать следующей схемой:

```mermaid
flowchart LR
    subgraph Embedding["Формирование входа энкодера"]
        T["Токены: x₁, x₂, ..., x_T"]
        E["Матрица эмбеддингов: T×d_model"]
        PE["Позиционное кодирование: T×d_model"]
        X["Вход энкодера: X = E + PE"]
    end
    
    T --> E
    T --> PE
    E --> X
    PE --> X
    
    style T fill:#e3f2fd
    style E fill:#e8f5e9
    style PE fill:#fff3e0
    style X fill:#f3e5f5
```

---

### 4. Этап 3: Энкодер

Матрица $\mathbf{X} \in \mathbb{R}^{T \times d_{\text{model}}}$ поступает на вход стека из $N$ идентичных слоёв энкодера (в оригинале $N = 6$). Каждый слой выполняет два преобразования:

1. **Multi-Head Self-Attention** (без маски, двусторонний контекст):
   $$
   \mathbf{X}^{(1)} = \text{LayerNorm}\left(\mathbf{X} + \text{MHA}(\mathbf{X})\right).
   $$

2. **Position-wise Feed-Forward Network**:
   $$
   \mathbf{X}^{(2)} = \text{LayerNorm}\left(\mathbf{X}^{(1)} + \text{FFN}(\mathbf{X}^{(1)})\right).
   $$

После прохождения всех $N$ слоёв мы получаем **контекстуализированные представления**:

$$
\mathbf{Z} = \text{EncoderStack}(\mathbf{X}) \in \mathbb{R}^{T \times d_{\text{model}}}.
$$

Каждый вектор $\mathbf{z}_i$ теперь содержит информацию о всей входной последовательности, обогащённую контекстом.

---

### 5. Этап 4: Декодер (авторегрессивная генерация)

Декодер генерирует выходную последовательность **токен за токеном** (авторегрессивно). На каждом шаге $t$ он получает:

- Выход энкодера $\mathbf{Z} \in \mathbb{R}^{T \times d_{\text{model}}}$;
- Уже сгенерированные токены $y_1, \dots, y_{t-1}$.

Процесс начинается со специального токена `[START]`, который подаётся на вход декодера на первом шаге.

#### 5.1. Шаг 1: Генерация первого токена

Начальная входная последовательность декодера: `[START]`.

1. Токен `[START]` преобразуется в эмбеддинг и суммируется с позиционным кодированием (позиция 0): $\mathbf{Y}_{<1} = \mathbf{e}_{\text{[START]}} + \mathbf{PE}_0$.
2. Декодер обрабатывает $\mathbf{Y}_{<1}$ через свои $N$ слоёв, используя Cross-Attention для доступа к $\mathbf{Z}$.
3. Выход декодера проходит через линейный слой и softmax, давая распределение вероятностей по словарю.
4. Выбирается токен с максимальной вероятностью — $y_1$.

#### 5.2. Шаг 2: Генерация второго токена

Теперь вход декодера: `[START, y_1]`. Оба токена преобразуются в эмбеддинги с добавлением позиционных кодирований (позиции 0 и 1), и процесс повторяется.

> **Важно:** На каждом шаге декодер **не пересчитывает энкодер**, а использует уже готовый выход $\mathbf{Z}$. Это ключевое отличие от RNN, где скрытое состояние пересчитывается с нуля для каждого токена.

#### 5.3. Авторегрессивный цикл

Этот процесс продолжается до тех пор, пока не будет сгенерирован токен `[END]` или не достигнута максимальная длина последовательности.

```mermaid
flowchart LR
    subgraph Generation["Авторегрессивная генерация"]
        S1["Шаг 1: [START] → y₁"]
        S2["Шаг 2: [START, y₁] → y₂"]
        S3["Шаг 3: [START, y₁, y₂] → y₃"]
        S4["..."]
        SE["Шаг M: ... → [END]"]
    end
    
    S1 --> S2 --> S3 --> S4 --> SE
    
    style S1 fill:#fce4ec
    style S2 fill:#fce4ec
    style S3 fill:#fce4ec
    style S4 fill:#fce4ec
    style SE fill:#e0f7fa
```

---

### 6. Этап 5: Выходной слой и выбор токена

На каждом шаге генерации декодер выдаёт скрытое состояние $\mathbf{h}_t \in \mathbb{R}^{d_{\text{model}}}$ (соответствующее последнему сгенерированному токену). Это состояние преобразуется в распределение вероятностей по словарю:

$$
\mathbf{p}_t = \text{softmax}\left(\mathbf{h}_t W_{\text{out}} + b_{\text{out}}\right),
$$

где $W_{\text{out}} \in \mathbb{R}^{d_{\text{model}} \times V}$, $V$ — размер словаря.

Выбор токена обычно осуществляется **жадно** (argmax) или с использованием **beam search** для улучшения качества.

---

### 7. Сквозной пример с размерами

Рассмотрим полный пример для конкретных значений:

- $d_{\text{model}} = 512$;
- $T = 4$ (длина входа);
- $V = 50000$ (размер словаря);
- $N = 6$ (число слоёв);

```mermaid
flowchart LR
    subgraph Dimensions["Размерности на каждом этапе"]
        A["Вход: текст 'I love reading'"]
        B["Токены: 4 токена"]
        C["IDs: [42, 156, 389, 1023]"]
        D["Эмбеддинги: 4×512"]
        E["+ PE: 4×512"]
        F["Вход энкодера X: 4×512"]
        G["Энкодер (6 слоёв): 4×512 → 4×512"]
        H["Выход энкодера Z: 4×512"]
        I["Декодер (шаг t): (t-1)×512 → 1×512"]
        J["Линейный слой: 512 → 50000"]
        K["Softmax: распределение по 50000 токенам"]
        L["Выходной токен: ID"]
    end
    
    A --> B --> C --> D --> E --> F --> G --> H --> I --> J --> K --> L
    
    style A fill:#e3f2fd
    style B fill:#e3f2fd
    style C fill:#e3f2fd
    style D fill:#e8f5e9
    style E fill:#fff3e0
    style F fill:#f3e5f5
    style G fill:#f3e5f5
    style H fill:#f3e5f5
    style I fill:#fce4ec
    style J fill:#e0f7fa
    style K fill:#e0f7fa
    style L fill:#e0f7fa
```

---

### 8. Таблица этапов с размерностями

| № | Этап | Входные данные | Размерность | Выходные данные | Размерность |
|---|------|----------------|-------------|-----------------|-------------|
| 1 | Токенизация | Исходный текст | — | Токены | $T$ |
| 2 | ID токенов | Токены | — | Числовые ID | $T$ |
| 3 | Эмбеддинги | ID токенов | $T \times 1$ | Токенные эмбеддинги | $T \times d_{\text{model}}$ |
| 4 | Позиционное кодирование | Позиции $0,\dots,T-1$ | $T \times 1$ | PE | $T \times d_{\text{model}}$ |
| 5 | Сложение | Токенные эмбеддинги + PE | $T \times d_{\text{model}}$ | Вход энкодера $\mathbf{X}$ | $T \times d_{\text{model}}$ |
| 6 | Энкодер (N слоёв) | $\mathbf{X}$ | $T \times d_{\text{model}}$ | Выход энкодера $\mathbf{Z}$ | $T \times d_{\text{model}}$ |
| 7 | Декодер (шаг t) | $y_{<t}$, $\mathbf{Z}$ | $(t-1) \times d_{\text{model}}$, $T \times d_{\text{model}}$ | Скрытое состояние $\mathbf{h}_t$ | $1 \times d_{\text{model}}$ |
| 8 | Линейный слой | $\mathbf{h}_t$ | $1 \times d_{\text{model}}$ | Логиты | $1 \times V$ |
| 9 | Softmax | Логиты | $1 \times V$ | Распределение вероятностей | $1 \times V$ |
| 10 | Выбор токена | Распределение | $1 \times V$ | ID токена $y_t$ | $1$ |

---

### 9. Заключение

Поток данных в Transformer представляет собой элегантную цепочку преобразований, где дискретные токены постепенно превращаются в непрерывные контекстуализированные представления, затем генеративно разворачиваются обратно в дискретный текст. Ключевые особенности этого потока:

1. **Параллелизм в энкодере** — все позиции обрабатываются одновременно.
2. **Авторегрессия в декодере** — генерация идёт последовательно, но с доступом ко всему входу через Cross-Attention.
3. **Неизменность размерности** — $d_{\text{model}}$ сохраняется на протяжении всей сети до выходного слоя.
4. **Повторное использование $\mathbf{Z}$** — энкодер вычисляется один раз для всей входной последовательности, что значительно ускоряет инференс по сравнению с RNN.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Alammar, J. (2018). *The Illustrated Transformer*.  
   🔗 [https://jalammar.github.io/illustrated-transformer/](https://jalammar.github.io/illustrated-transformer/)

3. Phuong, M., & Hutter, M. (2022). *Formal Algorithms for Transformers*. arXiv:2207.09238.  
   🔗 [https://arxiv.org/abs/2207.09238](https://arxiv.org/abs/2207.09238)

## Тема 3.1. Преобразование текста в числа: токенизация

Перед тем как текст может быть обработан нейронной сетью, он должен быть преобразован в числовой формат. Этот процесс, называемый **токенизацией**, является одним из наиболее фундаментальных и одновременно критически важных этапов в обработке естественного языка. Качество токенизации напрямую влияет на способность модели понимать язык, обрабатывать редкие слова и обобщать на новые данные. В этом разделе мы рассмотрим основные подходы к токенизации, их преимущества и ограничения, а также современные алгоритмы, используемые в ведущих LLM.

---

### 1. Зачем нужна токенизация

Нейронные сети, включая архитектуру Transformer, работают исключительно с числами. Они не понимают символов, слов или предложений в том виде, в каком их воспринимает человек. Задача токенизации — найти способ представления текста в виде последовательности чисел, который был бы одновременно:

1. **Компактным** — не слишком длинным для эффективной обработки;
2. **Полным** — способным представить любой возможный текст;
3. **Семантически осмысленным** — чтобы схожие по смыслу единицы имели схожие представления.

На протяжении развития NLP было предложено три основных подхода к решению этой задачи: токенизация на уровне слов, на уровне символов и на уровне подслов (субсловная токенизация).

---

### 2. Типы токенизации

#### 2.1. Word-Based (словоуровневая) токенизация

Самый интуитивный подход — разбивать текст по пробелам и знакам препинания.

**Пример:**
```
Вход: "I love cats!"
Токены: ["I", "love", "cats", "!"]
```

**Преимущества:**
- Простота реализации
- Интуитивная интерпретируемость (каждый токен — слово или знак препинания)
- Короткие последовательности (мало токенов на предложение)

**Недостатки:**
1. **Проблема неизвестных слов (OOV — Out-of-Vocabulary):** Любое слово, отсутствующее в словаре, становится неизвестным токеном `[UNK]`, что ведёт к потере информации.
2. **Огромный размер словаря:** Английский язык содержит сотни тысяч слов, а некоторые языки (например, агглютинативные) имеют практически бесконечное число словоформ.
3. **Морфологическая сложность:** Слова "run", "runs", "running", "ran" — разные токены, хотя имеют общий корень и смысл.
4. **Языки без пробелов:** В китайском, японском и корейском языках нет явных разделителей между словами, что делает словоуровневую токенизацию неприменимой.

#### 2.2. Character-Based (символьная) токенизация

Альтернативный подход — рассматривать каждый символ как отдельный токен.

**Пример:**
```
Вход: "cat"
Токены: ["c", "a", "t"]
```

**Преимущества:**
- **Отсутствие OOV:** Любое слово может быть представлено как последовательность символов.
- **Малый словарь:** Для любого языка достаточно нескольких сотен символов (включая буквы, цифры, знаки препинания).
- **Работа с любым языком:** Не требует знаний о структуре слов.

**Недостатки:**
- **Длинные последовательности:** Слово из 5 букв заменяется 5 токенами вместо одного, что увеличивает длину последовательности в 5-10 раз.
- **Потеря семантики:** Модель должна самостоятельно выучить, что "c", "a", "t" вместе образуют слово "cat". Это требует больше данных и вычислительных ресурсов.
- **Отсутствие информации о слове:** Модель не может использовать знания о том, что "cats" и "cat" — это одно и то же слово в разных формах.

#### 2.3. Subword-Based (субсловная) токенизация

Золотая середина — разбивать слова на часто встречающиеся **подслова** (субслова). Частые целые слова становятся отдельными токенами, редкие слова разбиваются на осмысленные части.

**Пример:**
```
Вход: "lower", "low", "lowest", "lowlands"
Токены: ["lo", "wer"], ["lo", "w"], ["lo", "west"], ["lo", "w", "lands"]
```

**Преимущества:**
- **Компактность:** Короткая длина последовательности (близка к словоуровневой)
- **Отсутствие OOV:** Редкие слова разбиваются на известные подслова
- **Морфологическая осмысленность:** Корни, приставки и суффиксы становятся отдельными токенами
- **Мультиязычность:** Хорошо работает для языков с богатой морфологией

Именно субсловная токенизация используется во всех современных LLM, включая GPT, BERT, LLaMA, Qwen и Mistral. Далее мы рассмотрим три основных алгоритма субсловной токенизации.

---

### 3. Алгоритм BPE (Byte-Pair Encoding)

**BPE** был предложен в 1994 году как алгоритм сжатия данных, а в 2015 году адаптирован для NLP в работе Sennrich et al. [1]. Сегодня BPE используется в GPT, LLaMA, Qwen и многих других моделях.

#### 3.1. Принцип работы

Алгоритм BPE начинает с базового словаря, содержащего все уникальные символы (или байты) в обучающем корпусе. Затем он итеративно выполняет следующие шаги:

1. Подсчитывает частоту всех соседних пар символов/подслов в корпусе.
2. Находит самую частотную пару.
3. Объединяет эту пару в новый токен.
4. Добавляет новый токен в словарь.
5. Повторяет до достижения целевого размера словаря.

#### 3.2. Пошаговый пример обучения BPE

Рассмотрим корпус из четырёх слов с частотами:

| Слово | Частота |
|-------|---------|
| "low" | 5 |
| "lower" | 2 |
| "lowest" | 2 |
| "lowlands" | 1 |

**Начальный словарь:** символы `["l", "o", "w", "e", "r", "s", "t", "a", "n", "d"]`

**Итерация 1:** Считаем частоты всех пар:
- `"lo"`: встречается в "low"(5), "lower"(2), "lowest"(2), "lowlands"(1) = **10 раз** (самая частотная!)
- `"ow"`: встречается 5+2+2+1 = 10 раз
- `"we"`: в "lower" 2 раза
- и т.д.

Выбираем `"lo"` (или `"ow"` — при равной частоте алгоритм может выбрать любой). Объединяем: `"lo"` становится новым токеном. Слова теперь выглядят так: `["lo", "w"]`, `["lo", "wer"]`, `["lo", "west"]`, `["lo", "wlands"]`.

**Итерация 2:** Снова считаем частоты пар:
- `"lo" + "w"`: встречается 5+1 = 6 раз (в "low" и "lowlands")
- `"we"`: в "lower" 2 раза
- `"es"`: в "lowest" 2 раза
- `"we"` и `"es"` имеют частоту 2
- и т.д.

После 10 итераций словарь может выглядеть так:
`["l", "o", "w", "e", "r", "s", "t", "a", "n", "d", "lo", "ow", "we", "es", "st", "and"]`

#### 3.3. Кодирование нового текста

Для кодирования нового текста алгоритм применяет правила слияния в том же порядке, в котором они были выучены:

```
"lowlands" → ["lo", "w", "lands"]
"lower" → ["lo", "wer"]
"lowest" → ["lo", "west"]
```

#### 3.4. Byte-Level BPE (GPT-2, GPT-4)

Классический BPE оперирует символами Unicode. Проблема: если в тексте встречается символ, отсутствовавший в обучающем корпусе, он становится неизвестным. В **Byte-Level BPE** (предложен в GPT-2) базовый словарь состоит из 256 байтов, а не символов. Это гарантирует, что **любой текст может быть закодирован**, включая эмодзи, редкие символы и тексты на любых языках.

---

### 4. WordPiece (BERT)

**WordPiece** был разработан в Google и используется в BERT, DistilBERT и других моделях. Он очень похож на BPE, но использует другую функцию для выбора пары для объединения.

#### 4.1. Отличие от BPE

Вместо частоты пар, WordPiece использует **вероятностную оценку**:

$$
\text{Score}(a, b) = \frac{\text{freq}(a, b)}{\text{freq}(a) \cdot \text{freq}(b)}
$$

Это отношение показывает, насколько появление $b$ после $a$ вероятнее, чем можно было бы ожидать из независимых частот. Чем выше это отношение, тем сильнее связаны два токена.

#### 4.2. Пример

Для слова "playing":
- WordPiece: `["play", "##ing"]`
- Символ `##` указывает, что токен является продолжением предыдущего (не началом нового слова)

#### 4.3. Кодирование

WordPiece использует **жадный алгоритм**: начиная с начала слова, он находит самый длинный токен из словаря, затем переходит к остатку слова.

```
"tokenization" → "token" (в словаре есть) → "##ization" (в словаре есть)
```

**Критическое ограничение:** Если какая-то часть слова не может быть закодирована токенами из словаря, весь токен становится `[UNK]`. Это делает WordPiece менее устойчивым к OOV, чем BPE.

---

### 5. SentencePiece (LLaMA, Qwen, Mistral)

**SentencePiece** — это не алгоритм, а фреймворк, реализующий алгоритм **Unigram** (а также BPE) [2]. Он используется в LLaMA, Qwen, Mistral и других современных моделях.

#### 5.1. Особенности SentencePiece

1. **Не требует пробелов:** Работает с сырым текстом, кодируя пробелы специальным символом `▁` (U+2581).
2. **Unigram алгоритм:** Начинает с большого словаря (все возможные подслова) и итеративно удаляет наименее важные токены.
3. **Мультиязычность:** Оптимизирован для работы с разными языками, включая японский и китайский.

#### 5.2. Unigram Language Model

Unigram использует вероятностную модель: каждый токен имеет вероятность $p(t)$. Вероятность сегментации слова — произведение вероятностей токенов. Алгоритм обучения:

1. Начинаем с очень большого словаря (все подслова, встречающиеся в корпусе).
2. Итеративно удаляем токены, удаление которых меньше всего ухудшает правдоподобие корпуса.
3. Повторяем до достижения целевого размера словаря.

```mermaid
flowchart TD
    A["Начальный словарь: все подслова"] --> B["Оценка важности каждого токена"]
    B --> C["Удаление наименее важных токенов"]
    C --> D["Достигнут целевой размер?"]
    D -->|Нет| B
    D -->|Да| E["Финальный словарь"]
    
    style A fill:#e3f2fd
    style B fill:#fff3e0
    style C fill:#ffcdd2
    style D fill:#f3e5f5
    style E fill:#e0f7fa
```

#### 5.3. Параметр character_coverage

SentencePiece поддерживает параметр `character_coverage`, который определяет, какая доля символов должна быть покрыта словарём. Для мультиязычных моделей рекомендуется значение `0.9995`, чтобы покрыть 99.95% всех символов во всех языках.

---

### 6. Сравнительный анализ методов

| Метод | Словарь | Плюсы | Минусы | Модели |
|-------|---------|-------|--------|--------|
| Word-Based | Сотни тысяч | Простота, короткие последовательности | OOV, огромный словарь | Устаревшие системы |
| Character-Based | Сотни | Нет OOV, малый словарь | Длинные последовательности, потеря семантики | Быстрое прототипирование |
| BPE | 30-100K | Нет OOV, морфологическая осмысленность | Требует предварительной токенизации по пробелам | GPT, LLaMA, Qwen, Mistral |
| WordPiece | 30K | Хорош для английского | OOV при неизвестных подсловах, `[UNK]` | BERT, DistilBERT |
| SentencePiece | 30-100K | Мультиязычный, нет пробелов | Сложнее в настройке | LLaMA, Qwen, T5, mT5 |

---

### 7. Практический пример с Hugging Face

Ниже приведён код, демонстрирующий работу токенизатора BERT (WordPiece) на Python:

```python
from transformers import AutoTokenizer

# Загрузка токенизатора BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Исходное предложение
text = "I love reading books!"

# Токенизация
tokens = tokenizer.tokenize(text)
print("Токены:", tokens)

# Преобразование в IDs
ids = tokenizer.convert_tokens_to_ids(tokens)
print("IDs:", ids)

# Восстановление текста из IDs
decoded = tokenizer.decode(ids)
print("Декодированный текст:", decoded)
```

**Результат выполнения:**
```
Токены: ['i', 'love', 'read', '##ing', 'books', '!']
IDs: [1045, 2293, 2294, 2106, 2612, 999]
Декодированный текст: "i love reading books!"
```

Обратите внимание:
- Слово "reading" разбито на `["read", "##ing"]` — это классический пример WordPiece.
- Текст приведён к нижнему регистру (это особенность `bert-base-uncased`).

#### Визуализация процесса токенизации:

```mermaid
flowchart LR
    A["'I love reading books!'"] --> B["Токенизатор BERT"]
    B --> C["['i', 'love', 'read', '##ing', 'books', '!']"]
    C --> D["[1045, 2293, 2294, 2106, 2612, 999]"]
    D --> E["Нейронная сеть"]
    
    style A fill:#e3f2fd
    style B fill:#fff3e0
    style C fill:#e8f5e9
    style D fill:#f3e5f5
    style E fill:#e0f7fa
```

---

### 8. Сравнение размеров словарей популярных моделей

| Модель | Алгоритм | Размер словаря | Особенности |
|--------|----------|----------------|-------------|
| BERT-base | WordPiece | 30,522 | Специальные токены: `[CLS]`, `[SEP]`, `[MASK]` |
| GPT-2 | Byte-level BPE | 50,257 | Базовый словарь — 256 байтов |
| GPT-3/GPT-4 | Byte-level BPE | 50,257 | Масштабируемый, без OOV |
| LLaMA (Meta) | SentencePiece (BPE) | 32,000 | Мультиязычный |
| Qwen 2.5 | SentencePiece (BPE) | 151,936 | Большой словарь для мультиязычности |
| Mistral | SentencePiece (BPE) | 32,000 | Оптимизирован для эффективности |
| T5 | SentencePiece (Unigram) | 32,000 | Текст-в-текст для всех задач |

---

### 9. Заключение

Токенизация — это первый и критически важный этап обработки текста в LLM. Выбор подхода определяет:

- **Способность модели работать с редкими словами** (BPE и SentencePiece решают эту проблему)
- **Эффективность использования памяти и вычислительных ресурсов** (длина последовательности)
- **Мультиязычные возможности** (SentencePiece не требует пробелов)

Современные модели практически повсеместно используют субсловную токенизацию с байтовым уровнем (Byte-Level BPE или SentencePiece), что обеспечивает баланс между компактностью и полнотой покрытия. Понимание этих алгоритмов необходимо не только для правильной загрузки предобученных моделей, но и для обучения собственных моделей на новых языках или специфических доменах.

---

### Литература

1. Sennrich, R., Haddow, B., & Birch, A. (2015). *Neural Machine Translation of Rare Words with Subword Units*. ACL.  
   🔗 [https://arxiv.org/abs/1508.07909](https://arxiv.org/abs/1508.07909)

2. Kudo, T., & Richardson, J. (2018). *SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing*. EMNLP.  
   🔗 [https://arxiv.org/abs/1808.06226](https://arxiv.org/abs/1808.06226)

3. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

4. Radford, A., et al. (2019). *Language Models are Unsupervised Multitask Learners*. OpenAI.  
   🔗 [https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

## Тема 3.2. Эмбеддинги — векторное представление токенов

После того как текст был преобразован в последовательность числовых идентификаторов токенов, встаёт следующая задача: как представить эти дискретные символы в форме, пригодной для обработки нейронной сетью? Решением являются **эмбеддинги** (embeddings) — плотные векторные представления, которые отображают каждый токен в непрерывное многомерное пространство. В этом разделе мы рассмотрим математические основы эмбеддингов, их свойства, способы обучения и практическое применение в архитектуре Transformer.

---

### 1. Определение и математическая формализация

**Эмбеддинг** (или векторное представление) — это отображение дискретного объекта (токена) в непрерывное векторное пространство фиксированной размерности. В контексте Transformer эмбеддинги представляют собой обучаемую матрицу:

$$
\mathbf{E} \in \mathbb{R}^{V \times d_{\text{model}}},
$$

где $V$ — размер словаря (количество уникальных токенов), а $d_{\text{model}}$ — размерность эмбеддингов. Каждая строка этой матрицы соответствует одному токену и содержит его векторное представление.

Преобразование токена в эмбеддинг представляет собой операцию индексирования:

$$
\mathbf{e}_i = \mathbf{E}[\text{token\_id}_i] \in \mathbb{R}^{d_{\text{model}}},
$$

где $\text{token\_id}_i$ — числовой идентификатор $i$-го токена в последовательности.

```mermaid
flowchart LR
    subgraph EmbeddingMatrix["Матрица эмбеддингов E ∈ ℝ^{V×d_model}"]
        direction TB
        E1["Токен 0: [0.12, 0.45, -0.33, ...]"]
        E2["Токен 1: [0.78, -0.21, 0.56, ...]"]
        E3["Токен 2: [-0.44, 0.91, 0.12, ...]"]
        EDot["..."]
        EV["Токен V-1: [...]"]
    end
    
    ID["token_id = 42"] --> E1
    
    style ID fill:#e3f2fd
    style EmbeddingMatrix fill:#f3e5f5
```

#### 1.1. Размерность эмбеддингов

Выбор размерности $d_{\text{model}}$ — важный гиперпараметр, определяющий выразительную способность модели и её вычислительную сложность. В оригинальном Transformer $d_{\text{model}} = 512$, но в современных моделях используются значительно большие размерности:

| Модель | $d_{\text{model}}$ | $V$ (размер словаря) | Год |
|--------|-------------------|---------------------|-----|
| Transformer (base) | 512 | 37,000 | 2017 |
| BERT-base | 768 | 30,522 | 2018 |
| GPT-2 | 768 | 50,257 | 2019 |
| BERT-large | 1024 | 30,522 | 2019 |
| GPT-3 (175B) | 12288 | 50,257 | 2020 |
| LLaMA 3 (8B) | 4096 | 128,256 | 2024 |
| LLaMA 3 (70B) | 8192 | 128,256 | 2024 |
| Qwen 2.5 (72B) | 8192 | 151,936 | 2024 |

---

### 2. Математические свойства эмбеддингов

#### 2.1. Семантическая близость

Фундаментальное свойство эмбеддингов заключается в том, что **семантически близкие токены имеют близкие векторы**. Мера близости обычно вычисляется как **косинусное расстояние**:

$$
\text{cos\_sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|} = \frac{\sum_{i=1}^{d} a_i b_i}{\sqrt{\sum_{i=1}^{d} a_i^2} \cdot \sqrt{\sum_{i=1}^{d} b_i^2}}.
$$

Косинусное расстояние принимает значения от $-1$ до $1$, где $1$ означает идентичные векторы (максимальное сходство), $0$ — ортогональные (независимые), а $-1$ — противоположные.

**Числовой пример:**

Пусть у нас есть три вектора (в упрощённой размерности $d=3$):

$$
\begin{aligned}
\mathbf{v}_{\text{cat}} &= [0.8, 0.1, 0.3], \\
\mathbf{v}_{\text{cat}} &= [0.7, 0.2, 0.4] \quad (\text{другой кот}), \\
\mathbf{v}_{\text{dog}} &= [0.2, 0.9, 0.1], \\
\mathbf{v}_{\text{table}} &= [0.1, 0.1, 0.9].
\end{aligned}
$$

Вычислим косинусное расстояние:

$$
\text{cos\_sim}(\text{cat}_1, \text{cat}_2) \approx 0.95 \quad (\text{высокое сходство}),
$$

$$
\text{cos\_sim}(\text{cat}, \text{dog}) \approx 0.35 \quad (\text{среднее сходство}),
$$

$$
\text{cos\_sim}(\text{cat}, \text{table}) \approx 0.12 \quad (\text{низкое сходство}).
$$

#### 2.2. Аналоговые отношения

Одно из наиболее удивительных свойств эмбеддингов — способность кодировать **аналоговые отношения** через векторную арифметику. Если векторы слов обучены качественно, то:

$$
\mathbf{w}_{\text{king}} - \mathbf{w}_{\text{man}} + \mathbf{w}_{\text{woman}} \approx \mathbf{w}_{\text{queen}}.
$$

Это означает, что вектор «королевской власти» ($\mathbf{w}_{\text{king}} - \mathbf{w}_{\text{man}}$) добавляется к вектору «женщины» и даёт вектор «королевы».

**Другие примеры аналогий:**

| Отношение | Аналогия | Результат |
|-----------|----------|-----------|
| Страна-столица | Paris - France + Italy | ≈ Rome |
| Единственное-множественное | cat - cats + dogs | ≈ dog |
| Настоящее-прошедшее | run - ran + swam | ≈ swim |
| Род занятий | doctor - hospital + school | ≈ teacher |

```mermaid
flowchart LR
    subgraph Analogies["Векторные аналогии"]
        direction LR
        K["w_king"] --> M["- w_man"]
        M --> W["+ w_woman"]
        W --> Q["≈ w_queen"]
    end
    
    style K fill:#e3f2fd
    style M fill:#fff3e0
    style W fill:#e8f5e9
    style Q fill:#f3e5f5
```

#### 2.3. Линейные свойства

Эмбеддинги обладают свойством **линейности**: семантические отношения часто могут быть выражены как линейные комбинации векторов. Это свойство лежит в основе методов аналогий и делает эмбеддинги интерпретируемыми.

---

### 3. Обучение эмбеддингов

#### 3.1. Статичные эмбеддинги

Первые методы обучения эмбеддингов создавали **статичные** представления — один фиксированный вектор для каждого слова, независимо от контекста.

**Word2Vec (Mikolov et al., 2013) [1]:**
- Два подхода: **CBOW** (предсказание слова по контексту) и **Skip-gram** (предсказание контекста по слову).
- Размерность: обычно 100-300.
- Обучается на больших корпусах (миллиарды слов).

**GloVe (Pennington et al., 2014) [2]:**
- Использует глобальную статистику совместной встречаемости слов.
- Комбинирует преимущества матричных факторизаций и локальных контекстных методов.

**FastText (Bojanowski et al., 2016) [3]:**
- Учитывает морфологию: каждый токен представляется как сумма эмбеддингов его n-грамм символов.
- Лучше работает с редкими словами.

#### 3.2. Контекстуальные эмбеддинги

Главное ограничение статичных эмбеддингов — омонимия: слово «банк» имеет один вектор независимо от того, идёт ли речь о финансовом учреждении или о береге реки.

**Контекстуальные эмбеддинги** решают эту проблему, генерируя вектор для токена **на основе его окружения**.

**ELMo (Peters et al., 2018) [4]:**
- Использует двунаправленный LSTM.
- Генерирует разные векторы для одного слова в разных контекстах.

**BERT (Devlin et al., 2018) [5] и GPT (Radford et al., 2018) [6]:**
- Используют архитектуру Transformer.
- Обучаются на масштабных корпусах с миллиардами токенов.
- Вектор токена зависит от всего предложения (BERT) или от предыдущих токенов (GPT).

#### 3.3. Обучение эмбеддингов в Transformer

В архитектуре Transformer эмбеддинги обучаются **совместно с остальной моделью** в процессе предобучения. Это означает, что матрица эмбеддингов $\mathbf{E}$ обновляется на каждом шаге градиентного спуска вместе с весами внимания и FFN.

Математически, если $\mathcal{L}$ — функция потерь, то:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{E}} = \sum_{i=1}^{T} \frac{\partial \mathcal{L}}{\partial \mathbf{e}_i},
$$

где $\mathbf{e}_i$ — эмбеддинг $i$-го токена. Это позволяет эмбеддингам адаптироваться под конкретные задачи.

В некоторых моделях (например, GPT) матрица эмбеддингов и матрица выходного линейного слоя **разделяют веса** (weight tying), что уменьшает число параметров и улучшает обучение.

---

### 4. Визуализация эмбеддингов

Поскольку эмбеддинги имеют высокую размерность (сотни или тысячи измерений), их невозможно визуализировать напрямую. Для этого используются методы понижения размерности.

#### 4.1. t-SNE (t-Distributed Stochastic Neighbor Embedding)

t-SNE проецирует высокоразмерные данные в 2D или 3D пространство, сохраняя локальные структуры. Результат обычно показывает чёткие кластеры слов по смыслу.

```mermaid
flowchart TD
    subgraph TSV["Визуализация эмбеддингов (t-SNE)"]
        direction TB
        C1["● Кластер: Животные<br/>(cat, dog, horse)"]
        C2["● Кластер: Глаголы движения<br/>(run, walk, swim)"]
        C3["● Кластер: Еда<br/>(apple, bread, meat)"]
        C4["● Кластер: Транспорт<br/>(car, train, plane)"]
    end
    
    style C1 fill:#e3f2fd
    style C2 fill:#f3e5f5
    style C3 fill:#e8f5e9
    style C4 fill:#fff3e0
```

#### 4.2. PCA (Principal Component Analysis)

PCA находит направления максимальной дисперсии в данных и проецирует на них. Результат менее точен для кластеризации, но лучше сохраняет глобальную структуру.

---

### 5. Практический пример на Python

Ниже приведён код, демонстрирующий работу с эмбеддингами:

```python
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import KeyedVectors
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Загрузка предобученных эмбеддингов (например, FastText или GloVe)
# Для примера будем использовать синтетические векторы

# Синтетические эмбеддинги (в реальности загружаются из файла)
embeddings = {
    "king": np.array([0.9, 0.1, 0.8]),
    "man": np.array([0.8, 0.9, 0.2]),
    "woman": np.array([0.2, 0.8, 0.9]),
    "queen": np.array([0.3, 0.1, 0.9]),
    "cat": np.array([0.8, 0.2, 0.1]),
    "dog": np.array([0.7, 0.3, 0.2]),
}

# 1. Вычисление косинусного расстояния
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Косинусное расстояние (queen, woman):",
      cosine_sim(embeddings["queen"], embeddings["woman"]))
print("Косинусное расстояние (queen, cat):",
      cosine_sim(embeddings["queen"], embeddings["cat"]))

# 2. Аналоговые отношения
# w_king - w_man + w_woman ≈ w_queen
analogy = embeddings["king"] - embeddings["man"] + embeddings["woman"]
print("Вектор аналогии:", analogy)

# Проверка: какой вектор ближе к результату?
best_word = None
best_score = -1
for word, vec in embeddings.items():
    if word not in ["king", "man", "woman"]:
        score = cosine_sim(analogy, vec)
        if score > best_score:
            best_score = score
            best_word = word

print(f"Ближайший вектор к 'king - man + woman': {best_word} (сходство: {best_score:.3f})")

# 3. Визуализация t-SNE
words = list(embeddings.keys())
vectors = np.array([embeddings[w] for w in words])

tsne = TSNE(n_components=2, random_state=42)
vectors_2d = tsne.fit_transform(vectors)

plt.figure(figsize=(10, 8))
for i, word in enumerate(words):
    x, y = vectors_2d[i, 0], vectors_2d[i, 1]
    plt.scatter(x, y, s=100)
    plt.annotate(word, (x, y), fontsize=12)

plt.title("Визуализация эмбеддингов (t-SNE)")
plt.xlabel("Измерение 1")
plt.ylabel("Измерение 2")
plt.grid(True)
plt.show()
```

**Пример вывода:**
```
Косинусное расстояние (queen, woman): 0.912
Косинусное расстояние (queen, cat): 0.234
Ближайший вектор к 'king - man + woman': queen (сходство: 0.987)
```

---

### 6. Примеры семантических аналогий

| Отношение | Аналогия | Ожидаемый результат | Результат в GloVe |
|-----------|----------|-------------------|-------------------|
| Страна-столица | Berlin - Germany + France | Paris | Paris |
| Род-вид | cat - animal + dog | canine | dog |
| Единственное-множественное | car - cars + dogs | dog | dog |
| Настоящее-прошедшее | run - ran + swam | swim | swim |
| Род занятий | doctor - hospital + school | teacher | teacher |

---

### 7. Заключение

Эмбеддинги являются мостом между дискретными токенами и непрерывным пространством, в котором работает нейронная сеть. Их ключевые свойства — семантическая близость, линейные отношения и контекстуальная зависимость — делают их мощным инструментом для представления языка. В архитектуре Transformer эмбеддинги обучаются совместно с остальной моделью, что позволяет им адаптироваться под конкретные задачи и языковые особенности. Понимание эмбеддингов необходимо для эффективной работы с любыми LLM, от тонкой настройки до интерпретации результатов.

---

### Литература

1. Mikolov, T., et al. (2013). *Efficient Estimation of Word Representations in Vector Space*. ICLR.  
   🔗 [https://arxiv.org/abs/1301.3781](https://arxiv.org/abs/1301.3781)

2. Pennington, J., Socher, R., & Manning, C. D. (2014). *GloVe: Global Vectors for Word Representation*. EMNLP.  
   🔗 [https://nlp.stanford.edu/pubs/glove.pdf](https://nlp.stanford.edu/pubs/glove.pdf)

3. Bojanowski, P., et al. (2016). *Enriching Word Vectors with Subword Information*. arXiv:1607.04606.  
   🔗 [https://arxiv.org/abs/1607.04606](https://arxiv.org/abs/1607.04606)

4. Peters, M., et al. (2018). *Deep contextualized word representations*. NAACL.  
   🔗 [https://arxiv.org/abs/1802.05365](https://arxiv.org/abs/1802.05365)

5. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

6. Radford, A., et al. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI.  
   🔗 [https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)

## Тема 3.3. Почему эмбеддинги работают: семантические свойства векторных представлений

Эмбеддинги — это не просто произвольное отображение токенов в числовые векторы. Их эффективность основана на фундаментальных свойствах языка и математических закономерностях, которые делают векторное пространство семантически осмысленным. В этом разделе мы рассмотрим теоретические основы, объясняющие, почему эмбеддинги столь эффективны, и как эти свойства проявляются на практике.

---

### 1. Распределённая семантика: гипотеза, лежащая в основе

#### 1.1. Distributional Hypothesis

Фундаментальная идея, на которой основаны все методы обучения эмбеддингов, формулируется как **гипотеза распределённой семантики** (Distributional Hypothesis), предложенная Harris (1954) и развитая Firth (1957):

> *"Words that occur in similar contexts tend to have similar meanings."*

Или, в более формальной формулировке:

> *"You shall know a word by the company it keeps."* (Firth, 1957)

Эта гипотеза утверждает, что семантика слова может быть восстановлена из статистики его употребления: слова, которые встречаются в окружении похожих слов, имеют схожие значения.

**Математическая формулировка:**

Пусть $w$ — слово, а $c$ — контекст (окружающие слова). Тогда семантика слова определяется распределением вероятностей:

$$
p(w \mid \text{context})
$$

Два слова $w_1$ и $w_2$ семантически близки, если:

$$
p(w_1 \mid \text{context}) \approx p(w_2 \mid \text{context})
$$

для всех возможных контекстов.

#### 1.2. От статистики к эмбеддингам

Эмбеддинги реализуют эту гипотезу, отображая слова в векторное пространство таким образом, что:

$$
\cos(\mathbf{v}_{w_1}, \mathbf{v}_{w_2}) \propto \text{сходство контекстов}
$$

Другими словами, чем чаще слова встречаются в похожих контекстах, тем ближе их векторы в пространстве эмбеддингов.

```mermaid
flowchart LR
    subgraph Context["Контекстное распределение"]
        W1["word₁: контекст A,B,C"]
        W2["word₂: контекст A,B,C"]
        W3["word₃: контекст D,E,F"]
    end
    
    subgraph EmbeddingSpace["Пространство эмбеддингов"]
        V1["● word₁"]
        V2["● word₂"]
        V3["● word₃"]
    end
    
    W1 --> V1
    W2 --> V2
    W3 --> V3
    
    V1 -.->|близко| V2
    V1 -.->|далеко| V3
    
    style Context fill:#e3f2fd
    style EmbeddingSpace fill:#f3e5f5
```

**Пример:** Слова «кот» и «кошка» встречаются в похожих контекстах («пушистый __», «__ мяукает»), поэтому их векторы близки. Слово «компьютер» встречается в других контекстах, поэтому его вектор далёк.

---

### 2. Косинусное расстояние: мера семантической близости

Для измерения близости векторов в пространстве эмбеддингов используется **косинусное расстояние** (cosine similarity). Это предпочтительная метрика, поскольку она:

1. Инвариантна к масштабу вектора (зависит только от направления);
2. Нормирована на интервал $[-1, 1]$;
3. Имеет чёткую семантическую интерпретацию.

#### 2.1. Формула и интерпретация

Для двух векторов $\mathbf{a}, \mathbf{b} \in \mathbb{R}^d$ косинусное расстояние определяется как:

$$
\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|} = \frac{\sum_{i=1}^{d} a_i b_i}{\sqrt{\sum_{i=1}^{d} a_i^2} \cdot \sqrt{\sum_{i=1}^{d} b_i^2}}.
$$

**Интерпретация значений:**

| Значение | Интерпретация | Семантический смысл |
|----------|---------------|---------------------|
| $\cos(\theta) = 1$ | Векторы сонаправлены | Идентичные или почти идентичные слова |
| $\cos(\theta) > 0.7$ | Высокое сходство | Синонимы или тематически близкие слова |
| $\cos(\theta) \approx 0$ | Векторы ортогональны | Семантически независимые слова |
| $\cos(\theta) < 0$ | Векторы противоположны | Антонимы или противопоставленные понятия |
| $\cos(\theta) = -1$ | Векторы противоположно направлены | Противоположные по смыслу слова |

#### 2.2. Практический пример

Рассмотрим несколько векторов в упрощённом 2D-пространстве (в реальности размерность сотни или тысячи):

| Слово | Вектор |
|-------|--------|
| cat | $[0.8, 0.6]$ |
| dog | $[0.7, 0.5]$ |
| car | $[0.1, 0.9]$ |
| house | $[0.2, 0.1]$ |

**Вычисления:**

- $\cos(\text{cat}, \text{dog}) \approx 0.99$ — очень близки (оба домашние животные)
- $\cos(\text{cat}, \text{car}) \approx 0.76$ — среднее сходство (оба могут быть объектами, но разные категории)
- $\cos(\text{cat}, \text{house}) \approx 0.70$ — среднее сходство

---

### 3. Векторные аналогии: линейные отношения в пространстве

Одно из наиболее впечатляющих свойств эмбеддингов — способность кодировать **линейные отношения** между словами. Это означает, что семантические отношения могут быть выражены как векторные операции.

#### 3.1. Формула аналогий

Если слова $w_a$, $w_b$, $w_c$, $w_d$ связаны отношением:

$$
w_a : w_b \quad \text{как} \quad w_c : w_d,
$$

то в пространстве эмбеддингов выполняется приближённое равенство:

$$
\mathbf{v}_{w_b} - \mathbf{v}_{w_a} \approx \mathbf{v}_{w_d} - \mathbf{v}_{w_c},
$$

или, что эквивалентно:

$$
\mathbf{v}_{w_d} \approx \mathbf{v}_{w_b} - \mathbf{v}_{w_a} + \mathbf{v}_{w_c}.
$$

#### 3.2. Классические примеры аналогий

| Отношение | Аналогия | Векторная операция | Результат |
|-----------|----------|-------------------|-----------|
| Страна-столица | France → Paris | $\mathbf{v}_{Paris} - \mathbf{v}_{France}$ | Столица |
| Страна-столица | Italy → Rome | $\mathbf{v}_{Rome} \approx \mathbf{v}_{Paris} - \mathbf{v}_{France} + \mathbf{v}_{Italy}$ | Rome |
| Гендер | man → king | $\mathbf{v}_{king} - \mathbf{v}_{man}$ | Мужской монарх |
| Гендер | woman → queen | $\mathbf{v}_{queen} \approx \mathbf{v}_{king} - \mathbf{v}_{man} + \mathbf{v}_{woman}$ | queen |
| Единственное-множественное | cat → cats | $\mathbf{v}_{cats} - \mathbf{v}_{cat}$ | Множественное число |
| Единственное-множественное | dog → dogs | $\mathbf{v}_{dogs} \approx \mathbf{v}_{cats} - \mathbf{v}_{cat} + \mathbf{v}_{dog}$ | dogs |

#### 3.3. Математическое обоснование

Почему векторные аналогии работают? Причина в том, что эмбеддинги обучаются предсказывать контекст. Если слова $w_a$ и $w_b$ различаются по одному семантическому признаку (например, по гендеру), то разность их векторов кодирует этот признак. Применение этой разности к другому слову позволяет «перенести» признак.

```mermaid
flowchart LR
    subgraph Gender["Гендерное отношение"]
        M["man"] --> K["king"]
        W["woman"] --> Q["queen"]
    end
    
    subgraph Vector["Векторная операция"]
        V1["v_king - v_man"] --> V2["≈ v_queen - v_woman"]
    end
    
    V2 --> Result["v_queen ≈ v_king - v_man + v_woman"]
    
    style Gender fill:#e3f2fd
    style Vector fill:#f3e5f5
    style Result fill:#e0f7fa
```

#### 3.4. Код на Python для аналогий

```python
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_analogy(word_a, word_b, word_c, embeddings, top_n=1):
    """
    Находит слово d, такое что a:b как c:d.
    d = b - a + c
    """
    # Получаем векторы
    v_a = embeddings[word_a]
    v_b = embeddings[word_b]
    v_c = embeddings[word_c]
    
    # Вычисляем целевой вектор
    v_target = v_b - v_a + v_c
    
    # Нормализуем
    v_target = v_target / np.linalg.norm(v_target)
    
    # Ищем ближайший вектор
    results = []
    for word, vec in embeddings.items():
        if word in [word_a, word_b, word_c]:
            continue
        vec_norm = vec / np.linalg.norm(vec)
        sim = np.dot(v_target, vec_norm)
        results.append((word, sim))
    
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_n]

# Пример использования
embeddings = {
    "king": np.array([0.9, 0.1, 0.8]),
    "man": np.array([0.8, 0.9, 0.2]),
    "woman": np.array([0.2, 0.8, 0.9]),
    "queen": np.array([0.3, 0.1, 0.9]),
    "prince": np.array([0.7, 0.3, 0.7]),
    "princess": np.array([0.4, 0.2, 0.8]),
}

analogy = find_analogy("king", "man", "woman", embeddings)
print(f"king - man + woman ≈ {analogy[0][0]} (score: {analogy[0][1]:.3f})")
```

**Результат:**
```
king - man + woman ≈ queen (score: 0.987)
```

---

### 4. Кластеризация: группы слов по смыслу

Эмбеддинги естественным образом группируются в кластеры, соответствующие семантическим категориям. Это свойство широко используется для:

1. **Визуализации** структуры языка;
2. **Классификации** текстов;
3. **Поиска** семантически близких слов;
4. **Выявления** тем в документах.

#### 4.1. Визуализация кластеров

```mermaid
flowchart TD
    subgraph Clusters["Кластеры эмбеддингов (t-SNE проекция)"]
        direction TB
        C1["● Животные<br/>cat, dog, horse, cow"]
        C2["● Глаголы движения<br/>run, walk, swim, fly"]
        C3["● Еда<br/>apple, bread, meat, rice"]
        C4["● Транспорт<br/>car, train, plane, bus"]
        C5["● Мебель<br/>table, chair, bed, sofa"]
    end
    
    style C1 fill:#e3f2fd
    style C2 fill:#f3e5f5
    style C3 fill:#e8f5e9
    style C4 fill:#fff3e0
    style C5 fill:#fce4ec
```

#### 4.2. Интерпретация кластеров

Кластеры в пространстве эмбеддингов часто соответствуют:

- **Семантическим категориям:** животные, транспорт, еда;
- **Синтаксическим категориям:** глаголы, существительные, прилагательные;
- **Стилистическим категориям:** формальная vs неформальная лексика;
- **Тематическим категориям:** медицина, юриспруденция, техника.

---

### 5. Ограничения статичных эмбеддингов

Несмотря на впечатляющие свойства, статичные эмбеддинги (Word2Vec, GloVe, FastText) имеют фундаментальное ограничение.

#### 5.1. Проблема многозначности (Polysemy)

Статичные эмбеддинги присваивают **один вектор** каждому слову, независимо от контекста. Это приводит к проблеме многозначности (polysemy): слово с несколькими значениями имеет один вектор, который является усреднением всех значений.

**Пример:** Слово «bank»:

1. Финансовое учреждение: *"I deposited money in the bank."*
2. Берег реки: *"The boat reached the bank."*

В статичных эмбеддингах оба значения «bank» будут представлены одним вектором, который не отражает ни одно из значений полностью.

```mermaid
flowchart LR
    subgraph Static["Статичные эмбеддинги"]
        S["bank: один вектор для всех значений"]
    end
    
    subgraph Contextual["Контекстуальные эмбеддинги"]
        C1["bank (финансовый): вектор₁"]
        C2["bank (берег): вектор₂"]
    end
    
    S -.->|смешанное значение| C1
    S -.->|смешанное значение| C2
    
    style Static fill:#ffcdd2
    style Contextual fill:#e8f5e9
```

#### 5.2. Как Transformer решает эту проблему

Архитектура Transformer с её контекстуальными эмбеддингами решает проблему многозначности. Вместо статичного вектора для каждого токена, Transformer генерирует вектор **на основе контекста**:

$$
\mathbf{e}_i = f(\text{token}_i, \text{context}),
$$

где $f$ — функция, реализуемая стёком слоёв энкодера (Self-Attention + FFN). Это означает, что один и тот же токен «bank» будет иметь разные векторы в разных предложениях.

**Пример в Transformer:**

| Предложение | Вектор для "bank" |
|-------------|-------------------|
| *"I deposited money in the bank."* | Вектор близкий к финансовым терминам |
| *"The boat reached the bank."* | Вектор близкий к географическим терминам |

#### 5.3. Сравнение статичных и контекстуальных эмбеддингов

| Характеристика | Статичные (Word2Vec, GloVe) | Контекстуальные (BERT, GPT) |
|----------------|----------------------------|------------------------------|
| Вектор зависит от контекста | Нет | Да |
| Решает проблему многозначности | Нет | Да |
| Вычислительная сложность | Низкая (индексирование) | Высокая (проход через модель) |
| Обучение | Отдельно от основной модели | Совместно с моделью |
| Скорость инференса | Очень высокая | Зависит от размера модели |
| Применение | Быстрые системы, прототипирование | Глубокое понимание языка |

---

### 6. Заключение

Эмбеддинги работают потому, что они улавливают фундаментальную закономерность языка: **слова, встречающиеся в похожих контекстах, имеют схожие значения**. Это наблюдение, известное как гипотеза распределённой семантики, лежит в основе всех современных методов представления текста.

Ключевые свойства эмбеддингов:
1. **Семантическая близость** — измеряется через косинусное расстояние;
2. **Векторные аналогии** — семантические отношения выражаются как линейные операции;
3. **Кластеризация** — слова из одной темы естественным образом группируются.

Переход от статичных эмбеддингов к контекстуальным (как в Transformer) стал революционным шагом, позволившим решить проблему многозначности и значительно улучшить качество всех NLP-задач. Понимание этих принципов необходимо для эффективного использования и дообучения современных LLM.

---

### Литература

1. Harris, Z. (1954). *Distributional Structure*. Word, 10(2-3), 146-162.

2. Firth, J. R. (1957). *A Synopsis of Linguistic Theory, 1930-1955*. Studies in Linguistic Analysis.

3. Mikolov, T., et al. (2013). *Efficient Estimation of Word Representations in Vector Space*. ICLR.  
   🔗 [https://arxiv.org/abs/1301.3781](https://arxiv.org/abs/1301.3781)

4. Pennington, J., Socher, R., & Manning, C. D. (2014). *GloVe: Global Vectors for Word Representation*. EMNLP.  
   🔗 [https://nlp.stanford.edu/pubs/glove.pdf](https://nlp.stanford.edu/pubs/glove.pdf)

5. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

6. Radford, A., et al. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI.  
   🔗 [https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)

## Тема 4.1. Проблема отсутствия порядка в механизме Self-Attention

Одним из наиболее фундаментальных вызовов при проектировании архитектуры Transformer стал вопрос: *«Как модель может понимать порядок слов, если механизм внимания по своей природе не чувствителен к перестановкам?»* В этом разделе мы детально рассмотрим проблему перестановочной инвариантности Self-Attention, её математическую природу и последствия для обработки естественного языка.

---

### 1. Перестановочная инвариантность Self-Attention

#### 1.1. Математическое определение

Механизм Scaled Dot-Product Attention, лежащий в основе Transformer, обладает свойством **перестановочной инвариантности** (permutation invariance). Формально это означает, что для любой перестановки $\pi$ (применяемой одновременно ко всем трём матрицам $Q, K, V$) выполняется:

$$
\text{Attention}\bigl(\pi(Q), \pi(K), \pi(V)\bigr) = \pi\bigl(\text{Attention}(Q, K, V)\bigr).
$$

Другими словами, если мы переставим строки в матрицах запросов, ключей и значений одинаковым образом, результат также будет переставлен соответствующим образом, но сами значения внимания не изменятся.

#### 1.2. Доказательство инвариантности

Рассмотрим формулу внимания:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V.
$$

Пусть $\pi$ — матрица перестановки размера $T \times T$ (ортогональная матрица, содержащая ровно одну единицу в каждой строке и каждом столбце). Тогда:

1. **Перестановка Q и K:**

$$
\pi(Q)K^T\pi^T = \pi(QK^T)\pi^T.
$$

2. **Применение softmax к переставленной матрице:**

$$
\text{softmax}(\pi M \pi^T) = \pi \cdot \text{softmax}(M) \cdot \pi^T.
$$

Это свойство следует из того, что softmax применяется независимо к каждой строке, а перестановка строк и столбцов эквивалентна перестановке элементов в каждой строке.

3. **Умножение на переставленное V:**

$$
\text{softmax}(\pi M \pi^T) \cdot \pi V = \pi \cdot \text{softmax}(M) \cdot \pi^T \cdot \pi V = \pi \cdot \text{softmax}(M) \cdot V.
$$

Таким образом:

$$
\text{Attention}(\pi(Q), \pi(K), \pi(V)) = \pi \cdot \text{Attention}(Q, K, V).
$$

**Вывод:** Перестановка входных токенов приводит к перестановке выходных представлений, но **сами веса внимания остаются инвариантными** к порядку.

```mermaid
flowchart LR
    subgraph Input["Входные токены"]
        A["токен A"] --> AA["вектор A"]
        B["токен B"] --> BB["вектор B"]
        C["токен C"] --> CC["вектор C"]
    end
    
    subgraph Attention["Self-Attention"]
        AA --> S["Q·K^T"]
        BB --> S
        CC --> S
        S --> W["веса внимания"]
        W --> O["выходные векторы"]
    end
    
    subgraph Output["Выходные векторы"]
        OA["выход A"]
        OB["выход B"]
        OC["выход C"]
    end
    
    O --> OA
    O --> OB
    O --> OC
    
    style Input fill:#e3f2fd
    style Attention fill:#f3e5f5
    style Output fill:#e0f7fa
```

#### 1.3. Что это означает на практике

Перестановочная инвариантность означает, что если мы возьмём предложение и просто поменяем местами слова, модель внимания **не заметит разницы** — она выдаст те же веса внимания (переставленные соответствующим образом), но не сможет отличить, какое слово было первым, а какое последним.

Это свойство является следствием того, что Self-Attention вычисляет попарные взаимодействия между токенами без учёта их абсолютных или относительных позиций.

---

### 2. Проблема на конкретных примерах

#### 2.1. Семантическая катастрофа

Рассмотрим два предложения на русском языке:

1. **"кот съел мышь"**
2. **"мышь съел кот"**

Для человека очевидно, что в первом предложении кот является субъектом (исполнителем действия), а мышь — объектом. Во втором — наоборот. Однако для Self-Attention **без позиционной информации** эти предложения неразличимы:

```mermaid
flowchart LR
    subgraph S1["Предложение 1: кот съел мышь"]
        direction LR
        K1["кот"] --> A1["съел"] --> M1["мышь"]
    end
    
    subgraph S2["Предложение 2: мышь съел кот"]
        direction LR
        M2["мышь"] --> A2["съел"] --> K2["кот"]
    end
    
    subgraph Attention["Self-Attention (без позиций)"]
        direction TB
        EQ["Q = [v_кот, v_съел, v_мышь]"] --> SA["внимание"]
        ER["Q = [v_мышь, v_съел, v_кот]"] --> SA
        SA --> OUT["выход: одинаковые веса<br/>(с точностью до перестановки)"]
    end
    
    S1 --> Attention
    S2 --> Attention
    
    style S1 fill:#e3f2fd
    style S2 fill:#ffcdd2
    style Attention fill:#f3e5f5
```

Оба предложения будут иметь **идентичное представление** в модели — с точностью до перестановки строк. Это означает, что модель не может определить, кто кого съел, что делает её бесполезной для понимания даже простейших синтаксических конструкций.

#### 2.2. Демонстрация на матрицах

Рассмотрим упрощённый пример с тремя токенами. Пусть эмбеддинги токенов (после обучения) равны:

$$
\mathbf{v}_{\text{кот}} = [1, 0, 0], \quad \mathbf{v}_{\text{съел}} = [0, 1, 0], \quad \mathbf{v}_{\text{мышь}} = [0, 0, 1].
$$

Для предложения *"кот съел мышь"* матрица $\mathbf{X}_1 = [\mathbf{v}_{\text{кот}}, \mathbf{v}_{\text{съел}}, \mathbf{v}_{\text{мышь}}]$:

$$
\mathbf{X}_1 = \begin{bmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1
\end{bmatrix}.
$$

Для предложения *"мышь съел кот"* матрица $\mathbf{X}_2 = [\mathbf{v}_{\text{мышь}}, \mathbf{v}_{\text{съел}}, \mathbf{v}_{\text{кот}}]$:

$$
\mathbf{X}_2 = \begin{bmatrix}
0 & 0 & 1 \\
0 & 1 & 0 \\
1 & 0 & 0
\end{bmatrix}.
$$

Заметим, что $\mathbf{X}_2$ получается из $\mathbf{X}_1$ путём перестановки строк. Поскольку Self-Attention инвариантен к перестановкам, он выдаст одинаковые результаты для обоих предложений. Модель **не может определить**, какое слово является субъектом, а какое — объектом.

---

### 3. Почему это проблема для языка

#### 3.1. Язык зависит от порядка слов

В большинстве языков порядок слов несёт критическую синтаксическую и семантическую информацию:

1. **Синтаксические роли:** В английском и русском языках порядок слов часто определяет, кто является субъектом, а кто — объектом действия.
2. **Грамматические отношения:** Прилагательные обычно стоят перед существительными (в английском) или после (в некоторых других языках).
3. **Смысловые нюансы:** Порядок слов может изменять акценты и фокус высказывания.

#### 3.2. Примеры из разных языков

| Язык | Предложение 1 | Предложение 2 | Смысловое различие |
|------|---------------|---------------|-------------------|
| Английский | *"The cat ate the mouse"* | *"The mouse ate the cat"* | Кто кого съел (SVO порядок) |
| Русский | *"Кот съел мышь"* | *"Мышь съел кот"* | Кто кого съел (SVO/SOV вариации) |
| Японский | *"猫が鼠を食べた"* (Neko-ga nezumi-wo tabeta) | *"鼠が猫を食べた"* (Nezumi-ga neko-wo tabeta) | Кто кого съел (SOV порядок) |
| Английский | *"Only I love you"* | *"I love only you"* | Кого именно любят |

#### 3.3. Грамматика и синтаксис

Без понимания порядка слов модель не может:

- **Правильно строить синтаксические деревья**;
- **Определять роли участников** в событиях;
- **Обрабатывать отрицания и вопросы**, которые часто зависят от порядка слов;
- **Понимать сложные конструкции** с придаточными предложениями и вложениями.

---

### 4. Необходимость решения и требования к нему

#### 4.1. Как добавить информацию о порядке

Чтобы преодолеть перестановочную инвариантность, необходимо добавить в модель **информацию о позициях токенов**. Эта информация должна быть:

1. **Инвариантна к длине последовательности** (работать для любой длины);
2. **Дифференцируема** (чтобы можно было обучать через градиентный спуск);
3. **Совместима с архитектурой** (не нарушать поток данных);
4. **Способна обобщаться** на последовательности длиннее, чем в обучении.

#### 4.2. Обзор возможных подходов

Существует несколько способов добавления информации о позициях:

| Подход | Описание | Примеры использования |
|--------|----------|----------------------|
| **Синусоидальное кодирование** | Фиксированные функции синуса и косинуса разных частот | Оригинальный Transformer |
| **Обучаемые позиционные эмбеддинги** | Матрица позиций, обучаемая вместе с моделью | BERT, GPT-2 |
| **Rotary Position Embedding (RoPE)** | Вращение эмбеддингов в комплексной плоскости | LLaMA, Qwen, Mistral |
| **Relative Position Encoding** | Кодирование относительных, а не абсолютных позиций | T5, некоторых моделях |

Каждый из этих подходов будет подробно рассмотрен в следующих темах.

#### 4.3. Визуализация проблемы

На диаграмме ниже показано влияние перестановки токенов на веса внимания:

```mermaid
flowchart TD
    subgraph Original["Исходный порядок: кот съел мышь"]
        direction LR
        O1["кот (Q₁)"] --> OW1["веса: 0.1, 0.7, 0.2"]
        O2["съел (Q₂)"] --> OW2["веса: 0.2, 0.6, 0.2"]
        O3["мышь (Q₃)"] --> OW3["веса: 0.1, 0.2, 0.7"]
    end
    
    subgraph Permuted["Переставленный порядок: мышь съел кот"]
        direction LR
        P1["мышь (Q₁)"] --> PW1["веса: 0.1, 0.2, 0.7"]
        P2["съел (Q₂)"] --> PW2["веса: 0.2, 0.6, 0.2"]
        P3["кот (Q₃)"] --> PW3["веса: 0.1, 0.7, 0.2"]
    end
    
    Original -.->|перестановка| Permuted
    
    style Original fill:#e3f2fd
    style Permuted fill:#ffcdd2
```

**Важно:** Хотя веса внимания переставляются, **сами числовые значения остаются теми же**. Это означает, что модель не может отличить "кот съел мышь" от "мышь съел кот".

---

### 5. Заключение

Перестановочная инвариантность Self-Attention — это фундаментальное свойство, которое делает механизм внимания мощным инструментом для извлечения попарных зависимостей, но одновременно создаёт серьёзную проблему для обработки языка. Без информации о порядке токенов модель не может различать предложения с одинаковыми словами в разном порядке, что делает её непригодной для понимания синтаксиса и семантики.

Решение этой проблемы требует добавления информации о позициях. В следующих разделах мы рассмотрим различные подходы к позиционному кодированию — от классических синусоидальных функций до современных методов, таких как Rotary Position Embedding (RoPE), которые позволили Transformer стать доминирующей архитектурой в NLP.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Zaheer, M., et al. (2020). *Big Bird: Transformers for Longer Sequences*. NeurIPS.  
   🔗 [https://arxiv.org/abs/2007.14062](https://arxiv.org/abs/2007.14062)

3. Su, J., et al. (2024). *RoFormer: Enhanced Transformer with Rotary Position Embedding*. Neurocomputing.  
   🔗 [https://arxiv.org/abs/2104.09864](https://arxiv.org/abs/2104.09864)

## Тема 4.2. Способы позиционного кодирования

В предыдущем разделе мы установили, что механизм Self-Attention по своей природе перестановочно-инвариантен и не обладает встроенным пониманием порядка токенов. Для решения этой проблемы в архитектуру Transformer необходимо добавить информацию о позициях. В данном разделе мы рассмотрим четыре основных подхода к позиционному кодированию: синусоидальное кодирование (оригинальный Transformer), обучаемые позиционные эмбеддинги (BERT, GPT), Rotary Position Embedding (RoPE, LLaMA, Qwen) и методы относительного позиционирования (T5, ALiBi). Каждый подход имеет свои математические обоснования, преимущества и ограничения.

---

### 1. Синусоидальное позиционное кодирование (оригинальный Transformer)

В оригинальной статье "Attention Is All You Need" [1] авторы предложили фиксированное позиционное кодирование на основе синусоидальных функций. Это было сделано по двум причинам: (1) модель не должна зависеть от длины последовательности, (2) синусоиды позволяют легко моделировать относительные позиции.

#### 1.1. Математическая формулировка

Для каждой позиции $pos$ (0, 1, 2, ...) и каждого измерения $i$ (0, 1, ..., $d_{\text{model}}/2 - 1$) позиционное кодирование определяется как:

$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right),
$$

$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right).
$$

Здесь $d_{\text{model}}$ — размерность модели, а $10000$ — константа, выбранная эмпирически.

**Интуиция:** Для каждой позиции создаётся уникальный "сигнатурный" вектор, где разные измерения колеблются с разными частотами. Младшие измерения (малые $i$) колеблются быстро, старшие (большие $i$) — медленно.

```python
import numpy as np
import matplotlib.pyplot as plt

def positional_encoding(T, d_model):
    """Генерирует синусоидальное позиционное кодирование."""
    PE = np.zeros((T, d_model))
    for pos in range(T):
        for i in range(d_model // 2):
            angle = pos / (10000 ** (2 * i / d_model))
            PE[pos, 2*i] = np.sin(angle)
            PE[pos, 2*i+1] = np.cos(angle)
    return PE

# Пример для T=50, d_model=64
PE = positional_encoding(50, 64)
plt.figure(figsize=(12, 6))
plt.imshow(PE, aspect='auto', cmap='RdBu')
plt.xlabel('Измерение')
plt.ylabel('Позиция')
plt.title('Синусоидальное позиционное кодирование')
plt.colorbar()
plt.show()
```

#### 1.2. Почему выбраны синус и косинус

Выбор синуса и косинуса обусловлен несколькими факторами:

1. **Дифференцируемость:** Синус и косинус — гладкие функции, что важно для градиентного спуска.
2. **Ограниченность:** Значения лежат в интервале $[-1, 1]$, что стабилизирует обучение.
3. **Линейное свойство:** Для любого фиксированного смещения $k$:

$$
PE_{pos+k} = f(PE_{pos})
$$

то есть позиционное кодирование для позиции $pos+k$ может быть выражено как линейная функция от $PE_{pos}$. Это позволяет модели легко обучаться обращать внимание на относительные позиции.

#### 1.3. Преимущества и недостатки

| Преимущества | Недостатки |
|--------------|------------|
| Не требует обучения (фиксированное) | Не адаптируется под данные |
| Позволяет обобщать на длинные последовательности | Может быть менее гибким, чем обучаемые методы |
| Математически обосновано | Ограниченная выразительность |
| Не зависит от размера словаря | |

---

### 2. Обучаемые позиционные эмбеддинги

В отличие от фиксированного синусоидального кодирования, обучаемые позиционные эмбеддинги представляют собой матрицу, которая обучается вместе с моделью. Этот подход используется в BERT [2], GPT-2 [3] и многих других моделях.

#### 2.1. Математическая формулировка

Создаётся обучаемая матрица $\mathbf{P} \in \mathbb{R}^{L_{\text{max}} \times d_{\text{model}}}$, где $L_{\text{max}}$ — максимальная длина последовательности (гиперпараметр). Каждая строка $i$ соответствует позиции $i$ и содержит вектор размерности $d_{\text{model}}$.

Входной эмбеддинг для токена на позиции $pos$ вычисляется как:

$$
\mathbf{x}_{pos} = \mathbf{e}_{pos} + \mathbf{P}_{pos},
$$

где $\mathbf{e}_{pos}$ — эмбеддинг токена, $\mathbf{P}_{pos}$ — обучаемый позиционный эмбеддинг для позиции $pos$.

```python
import torch
import torch.nn as nn

class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_embeddings = nn.Embedding(max_len, d_model)
        
    def forward(self, x, positions):
        # x: (batch, seq_len, d_model)
        # positions: (batch, seq_len) — индексы позиций
        return x + self.pos_embeddings(positions)
```

#### 2.2. Преимущества и недостатки

| Преимущества | Недостатки |
|--------------|------------|
| Адаптируются под данные | Требуют обучения |
| Более гибкие, чем синусоиды | Фиксированная максимальная длина |
| Простота реализации | Не обобщаются на длины > $L_{\text{max}}$ |
| Хорошо работают для фиксированных длин | Больше параметров |

---

### 3. Rotary Position Embedding (RoPE)

**Rotary Position Embedding (RoPE)** был предложен в работе "RoFormer: Enhanced Transformer with Rotary Position Embedding" [4] и получил широкое распространение в современных моделях, включая LLaMA [5], Qwen [6] и Mistral. RoPE кодирует позиции путём вращения векторов в комплексном пространстве, что позволяет естественным образом учитывать относительные позиции.

#### 3.1. Математическая основа

Идея RoPE заключается в том, чтобы применить к каждому вектору эмбеддинга вращение, зависящее от его позиции. Для этого вектор размерности $d_{\text{model}}$ разбивается на пары $(x_{2i}, x_{2i+1})$, каждая из которых интерпретируется как комплексное число.

Для позиции $pos$ и измерения $i$ вращение определяется углом:

$$
\theta_i = pos \cdot \theta_{\text{base}}^{i},
$$

где $\theta_{\text{base}}$ — константа (обычно $10000^{-2i/d_{\text{model}}}$ или аналогичная).

Вращение вектора $\mathbf{x} = [x_1, x_2, ..., x_{d_{\text{model}}}]$ на позиции $pos$ выполняется как:

$$
\text{RoPE}(\mathbf{x}, pos) = \begin{bmatrix}
x_1 \cos(pos \cdot \theta_1) - x_2 \sin(pos \cdot \theta_1) \\
x_1 \sin(pos \cdot \theta_1) + x_2 \cos(pos \cdot \theta_1) \\
x_3 \cos(pos \cdot \theta_2) - x_4 \sin(pos \cdot \theta_2) \\
x_3 \sin(pos \cdot \theta_2) + x_4 \cos(pos \cdot \theta_2) \\
\vdots
\end{bmatrix}.
$$

В комплексной записи это выглядит как умножение на $e^{i \cdot pos \cdot \theta_i}$.

#### 3.2. Свойство относительных позиций

Ключевое свойство RoPE заключается в том, что скалярное произведение двух векторов зависит **только от их относительной позиции**:

$$
\langle \text{RoPE}(\mathbf{q}, pos_m), \text{RoPE}(\mathbf{k}, pos_n) \rangle = f(\mathbf{q}, \mathbf{k}, pos_m - pos_n).
$$

Это означает, что при вычислении внимания модель автоматически учитывает расстояние между токенами, а не их абсолютные позиции.

```python
def rotate_half(x):
    """Вращение половины измерений."""
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin, position_ids):
    """Применение RoPE к запросам и ключам."""
    # cos, sin: (seq_len, d_model/2)
    cos = cos[position_ids].unsqueeze(1)  # (batch, 1, seq_len, d_model/2)
    sin = sin[position_ids].unsqueeze(1)
    
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed
```

#### 3.3. Преимущества RoPE

| Преимущества | Объяснение |
|--------------|------------|
| Относительные позиции | Внимание зависит от расстояния между токенами |
| Лучшая экстраполяция | Хорошо обобщается на длины > training |
| Эффективность | Минимальные вычислительные затраты |
| Теоретическая обоснованность | Строгое математическое обоснование |

---

### 4. Другие методы позиционного кодирования

#### 4.1. ALiBi (Attention with Linear Biases)

В работе "Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation" [7] предложен метод ALiBi, который добавляет линейный штраф к весам внимания в зависимости от расстояния между токенами:

$$
\text{Attention}(Q, K) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} - m \cdot |i - j|\right),
$$

где $m$ — обучаемый или фиксированный коэффициент, а $|i - j|$ — расстояние между позициями.

**Преимущества:** Отличная экстраполяция на длинные последовательности, простота реализации.

#### 4.2. Relative Position Encoding (T5)

В модели T5 [8] используется относительное позиционное кодирование, где позиции кодируются как отношения между токенами, а не абсолютные позиции.

---

### 5. Сравнительный анализ методов

```mermaid
flowchart TD
    subgraph Methods["Методы позиционного кодирования"]
        direction TB
        Sin["Синусоидальное<br/>(Original Transformer)"]
        Learned["Обучаемые эмбеддинги<br/>(BERT, GPT-2)"]
        RoPE["Rotary Position Embedding<br/>(LLaMA, Qwen, Mistral)"]
        ALiBi["ALiBi<br/>(Longformer, Blenderbot)"]
        Relative["Relative PE<br/>(T5)"]
    end
    
    style Sin fill:#e3f2fd
    style Learned fill:#e8f5e9
    style RoPE fill:#f3e5f5
    style ALiBi fill:#fff3e0
    style Relative fill:#fce4ec
```

#### 5.1. Сравнительная таблица

| Критерий | Синусоидальное | Обучаемые | RoPE | ALiBi | Relative PE |
|----------|----------------|-----------|------|-------|-------------|
| **Тип позиций** | Абсолютные | Абсолютные | Относительные | Относительные | Относительные |
| **Обучение** | Нет | Да | Нет | Да | Да |
| **Экстраполяция** | Средняя | Плохая | Отличная | Отличная | Средняя |
| **Сложность** | O(T·d) | O(T·d) | O(T·d) | O(T·d) | O(T²·d) |
| **Модели** | Transformer (orig) | BERT, GPT-2 | LLaMA, Qwen, Mistral | Longformer, Blenderbot | T5 |
| **Параметры** | 0 | L_max·d | 0 | 1 (m) | O(d²) |
| **Интерпретируемость** | Высокая | Средняя | Высокая | Высокая | Средняя |

#### 5.2. Визуализация сравнения методов

```mermaid
flowchart LR
    subgraph PE["Позиционное кодирование"]
        direction TB
        Sin["Синусоидальное<br/>PE(pos,2i)=sin(...)"]
        Learned["Обучаемые<br/>P ∈ ℝ^{L×d}"]
        RoPE["RoPE<br/>вращение векторов"]
        ALiBi["ALiBi<br/>линейный штраф"]
    end
    
    subgraph Usage["Использование"]
        USin["+ к эмбеддингам"]
        ULearned["+ к эмбеддингам"]
        URoPE["внутри внимания"]
        UALiBi["внутри внимания"]
    end
    
    Sin --> USin
    Learned --> ULearned
    RoPE --> URoPE
    ALiBi --> UALiBi
    
    style Sin fill:#e3f2fd
    style Learned fill:#e8f5e9
    style RoPE fill:#f3e5f5
    style ALiBi fill:#fff3e0
    style Usage fill:#f5f5f5
```

---

### 6. Заключение

Каждый метод позиционного кодирования имеет свои сильные и слабые стороны. Выбор подхода зависит от конкретной задачи:

1. **Синусоидальное кодирование** — хороший выбор для прототипов и когда требуется обобщение на длинные последовательности без дополнительных параметров.

2. **Обучаемые эмбеддинги** — просты в реализации и дают хорошие результаты для задач с фиксированной максимальной длиной (например, BERT для текстов до 512 токенов).

3. **RoPE** — наилучший выбор для современных LLM, работающих с длинными контекстами (отлично экстраполирует, эффективен, теоретически обоснован).

4. **ALiBi** — отличный выбор для моделей, которые должны работать с очень длинными последовательностями (до 100K+ токенов).

В современных моделях (LLaMA 3, Qwen 2.5, Mistral) доминирующим подходом является RoPE, что объясняется его способностью эффективно работать с длинными контекстами и хорошей экстраполяцией.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

3. Radford, A., et al. (2019). *Language Models are Unsupervised Multitask Learners*. OpenAI.  
   🔗 [https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

4. Su, J., et al. (2024). *RoFormer: Enhanced Transformer with Rotary Position Embedding*. Neurocomputing.  
   🔗 [https://arxiv.org/abs/2104.09864](https://arxiv.org/abs/2104.09864)

5. Touvron, H., et al. (2023). *LLaMA: Open and Efficient Foundation Language Models*. arXiv:2302.13971.  
   🔗 [https://arxiv.org/abs/2302.13971](https://arxiv.org/abs/2302.13971)

6. Bai, J., et al. (2023). *Qwen Technical Report*. arXiv:2309.16609.  
   🔗 [https://arxiv.org/abs/2309.16609](https://arxiv.org/abs/2309.16609)

7. Press, O., Smith, N. A., & Lewis, M. (2022). *Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation*. ICLR.  
   🔗 [https://arxiv.org/abs/2108.12409](https://arxiv.org/abs/2108.12409)

8. Raffel, C., et al. (2020). *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer*. JMLR.  
   🔗 [https://arxiv.org/abs/1910.10683](https://arxiv.org/abs/1910.10683)

## Тема 4.3. Синусоидальное позиционное кодирование (детальный анализ)

В оригинальной статье «Attention Is All You Need» [1] авторы предложили фиксированное позиционное кодирование на основе синусоидальных функций. Это решение стало классическим и до сих пор используется во многих моделях благодаря своим математически обоснованным свойствам. В этом разделе мы проведём детальный анализ синусоидального позиционного кодирования — от математических формул до практической реализации и интерпретации.

---

### 1. Математическая формулировка

Пусть $d_{\text{model}}$ — размерность модели (в оригинале 512). Для каждой позиции $pos$ ($pos = 0, 1, 2, \dots$) и каждого измерения $i$ ($i = 0, 1, \dots, d_{\text{model}}/2 - 1$) позиционное кодирование определяется как:

$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right),
$$

$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right).
$$

Здесь:

- $pos$ — абсолютная позиция токена в последовательности (начиная с 0);
- $i$ — индекс измерения (измерения сгруппированы попарно: чётные индексы получают синус, нечётные — косинус);
- $10000$ — константа, определяющая базовую частоту;
- $d_{\text{model}}$ — размерность модели.

Частота для пары измерений $(2i, 2i+1)$ равна:

$$
\omega_i = \frac{1}{10000^{2i/d_{\text{model}}}}.
$$

Таким образом, для каждой позиции мы получаем вектор длины $d_{\text{model}}$, состоящий из значений синусов и косинусов с различными частотами.

```mermaid
flowchart LR
    subgraph PE["Генерация синусоидального PE"]
        direction TB
        P["Позиция pos"] --> Sin["sin(pos * ω_i)"]
        P --> Cos["cos(pos * ω_i)"]
        I["Измерение i"] --> Freq["ω_i = 1/10000^(2i/d_model)"]
        Freq --> Sin
        Freq --> Cos
        Sin --> V["PE[pos, 2i]"]
        Cos --> V2["PE[pos, 2i+1]"]
    end
    
    style PE fill:#e3f2fd
    style Sin fill:#f3e5f5
    style Cos fill:#f3e5f5
    style Freq fill:#fff3e0
```

---

### 2. Выбор частот и их анализ

#### 2.1. Геометрическая прогрессия частот

Частоты $\omega_i$ образуют геометрическую прогрессию от $1$ (при $i=0$) до $1/10000$ (при $i = d_{\text{model}}/2 - 1$):

$$
\omega_0 = 1, \quad \omega_1 = \frac{1}{10000^{2/d_{\text{model}}}}, \quad \dots, \quad \omega_{d_{\text{model}}/2-1} = \frac{1}{10000}.
$$

Это означает, что периоды колебаний варьируются от $2\pi$ до $10000 \cdot 2\pi$, покрывая широкий диапазон длин волн. Младшие измерения (малые $i$) колеблются медленно, старшие — быстро.

#### 2.2. Почему выбрано основание 10000

Выбор $10000$ основан на эмпирических наблюдениях: для задач машинного перевода максимальная длина предложений обычно не превышает 50–100 слов, поэтому период $10000 \cdot 2\pi$ достаточно велик, чтобы покрыть все возможные позиции, обеспечивая при этом плавное изменение кодирования.

#### 2.3. Визуализация частот

На приведённом ниже графике показаны значения $PE_{(pos, i)}$ для разных $i$ при $d_{\text{model}} = 128$:

- Для малых $i$ (например, $i=0$): $PE = \sin(pos)$, период $2\pi$ — быстрое изменение.
- Для средних $i$: $PE = \sin(pos / 10)$, период $20\pi$ — среднее изменение.
- Для больших $i$: $PE = \sin(pos / 10000)$, период $20000\pi$ — почти постоянное значение на интервале обучения.

```python
import numpy as np
import matplotlib.pyplot as plt

d_model = 128
T = 100
PE = np.zeros((T, d_model))
for pos in range(T):
    for i in range(d_model // 2):
        freq = 1.0 / (10000 ** (2 * i / d_model))
        PE[pos, 2*i] = np.sin(pos * freq)
        PE[pos, 2*i+1] = np.cos(pos * freq)

# График для нескольких i
plt.figure(figsize=(12, 6))
for i in [0, 5, 15, 30, 60]:
    plt.plot(PE[:, i], label=f'i={i}')
plt.xlabel('Позиция pos')
plt.ylabel('Значение PE')
plt.title('Зависимость PE от позиции для разных измерений')
plt.legend()
plt.grid(True)
plt.show()
```

---

### 3. Ключевые свойства синусоидального кодирования

#### 3.1. Уникальность паттерна для каждой позиции

Для каждой позиции $pos$ вектор $PE_{pos}$ является уникальным. Это позволяет модели различать токены по их абсолютному положению. Математически это гарантируется тем, что синусы и косинусы с разными частотами образуют ортогональный базис на дискретном множестве позиций (свойство дискретного преобразования Фурье).

#### 3.2. Линейное свойство относительных позиций

Одно из наиболее важных свойств синусоидального кодирования — возможность выразить $PE_{pos+k}$ как линейную функцию от $PE_{pos}$ для любого фиксированного смещения $k$. Это следует из тригонометрических тождеств:

Для каждой пары измерений $(2i, 2i+1)$ имеем:

$$
\begin{aligned}
PE_{(pos+k, 2i)} &= \sin((pos+k) \omega_i) \\
&= \sin(pos \omega_i) \cos(k \omega_i) + \cos(pos \omega_i) \sin(k \omega_i) \\
&= PE_{(pos, 2i)} \cos(k \omega_i) + PE_{(pos, 2i+1)} \sin(k \omega_i).
\end{aligned}
$$

Аналогично:

$$
PE_{(pos+k, 2i+1)} = -PE_{(pos, 2i)} \sin(k \omega_i) + PE_{(pos, 2i+1)} \cos(k \omega_i).
$$

В матричной форме для каждой пары измерений:

$$
\begin{bmatrix}
PE_{(pos+k, 2i)} \\
PE_{(pos+k, 2i+1)}
\end{bmatrix}
=
\begin{bmatrix}
\cos(k \omega_i) & \sin(k \omega_i) \\
-\sin(k \omega_i) & \cos(k \omega_i)
\end{bmatrix}
\cdot
\begin{bmatrix}
PE_{(pos, 2i)} \\
PE_{(pos, 2i+1)}
\end{bmatrix}.
$$

Это означает, что $PE_{pos+k}$ получается из $PE_{pos}$ путём применения линейного преобразования (вращения) к каждой паре измерений. Важно, что матрица вращения **не зависит от $pos$** — она зависит только от смещения $k$. Это свойство позволяет модели легко обучаться обрабатывать относительные позиции, поскольку внимание может быть выражено через скалярные произведения, зависящие от $pos_m - pos_n$.

#### 3.3. Экстраполяция на более длинные последовательности

Поскольку кодирование не ограничено максимальной длиной (в отличие от обучаемых эмбеддингов), модель может обобщаться на последовательности, превышающие длины, виденные во время обучения. Хотя точность экстраполяции может снижаться для очень длинных последовательностей, синусоидальное кодирование обеспечивает лучшую экстраполяцию, чем обучаемые методы.

---

### 4. Пошаговый пример вычисления

Рассмотрим вычисление позиционного кодирования для $d_{\text{model}} = 4$ и позиций $pos = 0, 1, 2$.

**Параметры:**

- $d_{\text{model}} = 4$,
- $d_{\text{model}}/2 = 2$,
- $i = 0, 1$.

**Вычисление частот:**

Для $i=0$: $\omega_0 = 1 / 10000^{0} = 1$.

Для $i=1$: $\omega_1 = 1 / 10000^{2/4} = 1 / 10000^{0.5} = 1 / 100 = 0.01$.

**Для $pos = 0$:**

$$
\begin{aligned}
PE_{(0, 0)} &= \sin(0 \cdot 1) = 0, \\
PE_{(0, 1)} &= \cos(0 \cdot 1) = 1, \\
PE_{(0, 2)} &= \sin(0 \cdot 0.01) = 0, \\
PE_{(0, 3)} &= \cos(0 \cdot 0.01) = 1.
\end{aligned}
$$

Таким образом, $PE_0 = [0, 1, 0, 1]$.

**Для $pos = 1$:**

$$
\begin{aligned}
PE_{(1, 0)} &= \sin(1 \cdot 1) \approx 0.8415, \\
PE_{(1, 1)} &= \cos(1 \cdot 1) \approx 0.5403, \\
PE_{(1, 2)} &= \sin(1 \cdot 0.01) \approx 0.0100, \\
PE_{(1, 3)} &= \cos(1 \cdot 0.01) \approx 0.9999.
\end{aligned}
$$

$PE_1 \approx [0.8415, 0.5403, 0.0100, 0.9999]$.

**Для $pos = 2$:**

$$
\begin{aligned}
PE_{(2, 0)} &= \sin(2) \approx 0.9093, \\
PE_{(2, 1)} &= \cos(2) \approx -0.4161, \\
PE_{(2, 2)} &= \sin(0.02) \approx 0.0200, \\
PE_{(2, 3)} &= \cos(0.02) \approx 0.9998.
\end{aligned}
$$

$PE_2 \approx [0.9093, -0.4161, 0.0200, 0.9998]$.

**Наблюдение:** В младших измерениях (0,1) значения сильно различаются для соседних позиций, тогда как в старших измерениях (2,3) изменения гораздо меньше. Это позволяет модели одновременно кодировать как локальные, так и глобальные позиционные зависимости.

---

### 5. Тепловая карта синусоидального кодирования

Тепловая карта для $T = 50$ и $d_{\text{model}} = 64$ показывает вертикальные полосы, соответствующие разным частотам:

- **Левая часть (малые $i$):** быстрое изменение по оси $pos$ — частые переходы от -1 к 1.
- **Правая часть (большие $i$):** медленное изменение — почти однородные цвета.

Это визуальное представление помогает понять, как позиционное кодирование охватывает разные масштабы.

```mermaid
flowchart LR
    subgraph Heatmap["Тепловая карта PE"]
        direction TB
        H1["pos ↑"]
        H2["→ i (измерения)"]
        H3["Слева: быстрые колебания"]
        H4["Справа: медленные колебания"]
    end
    style Heatmap fill:#e3f2fd
```

---

### 6. Реализация на Python

Ниже приведена полная реализация синусоидального позиционного кодирования с возможностью визуализации.

```python
import numpy as np
import matplotlib.pyplot as plt

def sinusoidal_positional_encoding(T, d_model):
    """
    Генерирует синусоидальное позиционное кодирование.
    
    Args:
        T (int): длина последовательности (число позиций)
        d_model (int): размерность модели
    
    Returns:
        np.ndarray: матрица PE размером (T, d_model)
    """
    PE = np.zeros((T, d_model))
    for pos in range(T):
        for i in range(d_model // 2):
            freq = 1.0 / (10000 ** (2 * i / d_model))
            PE[pos, 2*i] = np.sin(pos * freq)
            PE[pos, 2*i+1] = np.cos(pos * freq)
    return PE

# Пример
T = 50
d_model = 64
PE = sinusoidal_positional_encoding(T, d_model)

# Визуализация тепловой карты
plt.figure(figsize=(12, 6))
plt.imshow(PE, aspect='auto', cmap='RdBu')
plt.colorbar(label='Значение PE')
plt.xlabel('Измерение (i)')
plt.ylabel('Позиция (pos)')
plt.title('Синусоидальное позиционное кодирование')
plt.show()

# График для нескольких измерений
plt.figure(figsize=(12, 6))
for i in [0, 4, 8, 16, 31]:
    plt.plot(PE[:, i], label=f'измерение {i}')
plt.xlabel('Позиция pos')
plt.ylabel('Значение PE')
plt.title('Зависимость PE от позиции для разных измерений')
plt.legend()
plt.grid(True)
plt.show()
```

---

### 7. Сравнение с другими методами

Хотя синусоидальное кодирование является фиксированным и не обучается, оно даёт результаты, сопоставимые с обучаемыми эмбеддингами (см. таблицу 3 в оригинальной статье). Авторы отмечают, что замена синусоидального кодирования на обучаемые позиционные эмбеддинги даёт почти идентичные результаты, но синусоидальное кодирование предпочтительнее для экстраполяции на более длинные последовательности.

---

### 8. Заключение

Синусоидальное позиционное кодирование — это элегантное решение проблемы отсутствия порядка в Self-Attention. Его ключевые достоинства:

- **Отсутствие обучаемых параметров** — не увеличивает размер модели.
- **Математическая обоснованность** — возможность выражать относительные позиции через линейные преобразования.
- **Экстраполяция** — способность работать с последовательностями, превышающими длину обучения.
- **Простота реализации** — легко интегрируется в любую модель.

Хотя современные модели часто отдают предпочтение RoPE или обучаемым эмбеддингам, синусоидальное кодирование остаётся важным историческим и теоретическим эталоном, знание которого необходимо для глубокого понимания архитектуры Transformer.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

## Тема 4.4. Добавление позиционного кодирования к эмбеддингам

После того как мы получили эмбеддинги токенов и сгенерировали позиционное кодирование, встаёт вопрос: *как объединить эти две информационные составляющие?* В оригинальной статье Transformer [1] предложен простой, но эффективный способ — **поэлементное сложение**. В этом разделе мы рассмотрим этот процесс, его математическое обоснование и практическую реализацию.

---

### 1. Процесс объединения

Полный pipeline формирования входных представлений для энкодера Transformer состоит из трёх этапов:

1. **Токенизация:** исходный текст → последовательность токенов.
2. **Эмбеддинги токенов:** каждый токен → вектор размерности $d_{\text{model}}$ через матрицу эмбеддингов $\mathbf{E}$.
3. **Добавление позиционного кодирования:** к каждому вектору прибавляется соответствующий вектор позиционного кодирования.

Математически для токена на позиции $pos$:

$$
\mathbf{x}_{pos} = \mathbf{e}_{pos} + \mathbf{PE}_{pos},
$$

где:

- $\mathbf{e}_{pos} = \mathbf{E}[\text{token\_id}_{pos}] \in \mathbb{R}^{d_{\text{model}}}$ — эмбеддинг токена;
- $\mathbf{PE}_{pos} \in \mathbb{R}^{d_{\text{model}}}$ — позиционное кодирование для позиции $pos$;
- $\mathbf{x}_{pos} \in \mathbb{R}^{d_{\text{model}}}$ — итоговое представление, поступающее в первый слой энкодера.

Для всей последовательности длины $T$:

$$
\mathbf{X} = \mathbf{E}_{\text{tokens}} + \mathbf{PE} \in \mathbb{R}^{T \times d_{\text{model}}}.
$$

```mermaid
flowchart LR
    subgraph Input["Входные данные"]
        Text["Текст: 'I love reading'"]
        Tokens["Токены: ['I', 'love', 'read', 'ing']"]
        IDs["IDs: [42, 156, 389, 1023]"]
    end
    
    subgraph Embedding["Этап 1: Эмбеддинги"]
        E["Матрица эмбеддингов E"]
        TokenEmb["Токенные эмбеддинги<br/>E_tokens ∈ ℝ^{4×512}"]
    end
    
    subgraph PE["Этап 2: Позиционное кодирование"]
        PE_gen["Генерация PE"]
        PE_mat["Позиционное кодирование<br/>PE ∈ ℝ^{4×512}"]
    end
    
    subgraph Sum["Этап 3: Сложение"]
        Add["X = E_tokens + PE"]
        X["Вход энкодера<br/>X ∈ ℝ^{4×512}"]
    end
    
    Text --> Tokens --> IDs
    IDs --> E --> TokenEmb
    IDs --> PE_gen --> PE_mat
    TokenEmb --> Add
    PE_mat --> Add
    Add --> X
    
    style Input fill:#e3f2fd
    style Embedding fill:#e8f5e9
    style PE fill:#fff3e0
    style Sum fill:#f3e5f5
```

---

### 2. Почему выбрано сложение

#### 2.1. Сохранение размерности

Сложение сохраняет размерность $d_{\text{model}}$ на всех этапах: вход, скрытые слои и выход имеют одинаковую размерность. Это упрощает архитектуру и позволяет использовать остаточные связи.

#### 2.2. Минимальное количество параметров

В отличие от конкатенации, которая увеличила бы размерность вдвое ($2 \cdot d_{\text{model}}$) и потребовала бы дополнительных параметров для проецирования обратно к $d_{\text{model}}$, сложение не вводит новых параметров. Это делает модель более эффективной.

#### 2.3. Достаточность для передачи информации

Сложение не приводит к потере информации, поскольку эмбеддинги токенов и позиционные кодирования живут в одном векторном пространстве. После добавления PE, модель может научиться разделять семантическую и позиционную информацию через последующие слои:

$$
\mathbf{x}_{pos} = \mathbf{e}_{pos} + \mathbf{PE}_{pos}.
$$

В этом выражении:

- $\mathbf{e}_{pos}$ кодирует семантику токена;
- $\mathbf{PE}_{pos}$ кодирует позицию токена;
- Их сумма содержит обе составляющие, и последующие слои Self-Attention могут извлекать нужную информацию через обученные проекции $W^Q, W^K, W^V$.

#### 2.4. Математическое обоснование

Сложение является **линейной операцией**, что позволяет сохранить все дифференциальные свойства и упрощает обратное распространение ошибки:

$$
\frac{\partial \mathbf{x}_{pos}}{\partial \mathbf{e}_{pos}} = \mathbf{I}, \quad \frac{\partial \mathbf{x}_{pos}}{\partial \mathbf{PE}_{pos}} = \mathbf{I}.
$$

Градиенты проходят без изменений как через эмбеддинги, так и через позиционное кодирование.

```mermaid
flowchart LR
    subgraph Options["Способы объединения"]
        Add["Сложение<br/>x = e + pe<br/>размерность: d_model"]
        Concat["Конкатенация<br/>x = [e; pe]<br/>размерность: 2·d_model"]
        Mul["Умножение<br/>x = e ⊙ pe<br/>размерность: d_model"]
    end
    
    subgraph Result["Результат"]
        RAdd["✅ Используется в Transformer<br/>Нет новых параметров"]
        RConcat["❌ Удваивает размерность<br/>Требует дополнительной проекции"]
        RMul["❌ Риск затухания/взрыва<br/>Меньшая выразительность"]
    end
    
    Add --> RAdd
    Concat --> RConcat
    Mul --> RMul
    
    style Options fill:#e3f2fd
    style Result fill:#f3e5f5
    style RAdd fill:#e8f5e9
    style RConcat fill:#ffcdd2
    style RMul fill:#ffcdd2
```

---

### 3. Визуализация суммы

Чтобы понять, как выглядит результат сложения, представим упрощённый пример с $d_{\text{model}} = 4$. Пусть у нас есть токен с эмбеддингом:

$$
\mathbf{e} = [0.5, -0.2, 0.8, 0.1].
$$

Для позиции $pos = 1$ синусоидальное кодирование даёт (при $d_{\text{model}} = 4$):

$$
\mathbf{PE}_1 \approx [0.8415, 0.5403, 0.0100, 0.9999].
$$

Тогда итоговый вектор:

$$
\mathbf{x} = \mathbf{e} + \mathbf{PE}_1 = [1.3415, 0.3403, 0.8100, 1.0999].
$$

Для позиции $pos = 2$:

$$
\mathbf{PE}_2 \approx [0.9093, -0.4161, 0.0200, 0.9998],
$$

$$
\mathbf{x} = \mathbf{e} + \mathbf{PE}_2 = [1.4093, -0.6161, 0.8200, 1.0998].
$$

**Важное наблюдение:** Один и тот же токен (один и тот же эмбеддинг $\mathbf{e}$) получает разные итоговые векторы на разных позициях. Это позволяет модели различать токены по их положению.

```mermaid
flowchart LR
    subgraph Positions["Разные позиции — разные суммы"]
        direction TB
        E["Эмбеддинг токена 'кот'<br/>e = [0.5, -0.2, 0.8, 0.1]"]
        
        PE0["PE₀ = [0, 1, 0, 1]"]
        PE5["PE₅ ≈ [−0.96, 0.28, 0.05, 0.99]"]
        
        Sum0["x₀ = e + PE₀<br/>= [0.5, 0.8, 0.8, 1.1]"]
        Sum5["x₅ = e + PE₅<br/>= [−0.46, 0.08, 0.85, 1.09]"]
    end
    
    E --> Sum0
    E --> Sum5
    PE0 --> Sum0
    PE5 --> Sum5
    
    style Positions fill:#e3f2fd
    style E fill:#e8f5e9
    style PE0 fill:#fff3e0
    style PE5 fill:#fff3e0
    style Sum0 fill:#f3e5f5
    style Sum5 fill:#f3e5f5
```

---

### 4. Пример: одно слово на разных позициях

Рассмотрим конкретный пример. Пусть у нас есть слово *"кот"* на позициях 0 и 5 в двух разных предложениях:

1. *"кот съел мышь"* — *"кот"* на позиции 0.
2. *"мышь съел кот"* — *"кот"* на позиции 2 (или 5 в более длинном предложении).

Хотя токен *"кот"* имеет один и тот же эмбеддинг $\mathbf{e}_{\text{кот}}$, итоговые векторы $\mathbf{x}_{pos=0}$ и $\mathbf{x}_{pos=5}$ будут разными из-за добавления разных $\mathbf{PE}_{pos}$.

Это критически важно для модели: она должна понимать, что в первом предложении *"кот"* является субъектом, а во втором — объектом. Разные позиции помогают модели различать синтаксические роли.

---

### 5. Полный pipeline на Python

```python
import numpy as np

def sinusoidal_pe(pos, d_model):
    """Генерирует синусоидальное позиционное кодирование для одной позиции."""
    pe = np.zeros(d_model)
    for i in range(d_model // 2):
        freq = 1.0 / (10000 ** (2 * i / d_model))
        pe[2*i] = np.sin(pos * freq)
        pe[2*i+1] = np.cos(pos * freq)
    return pe

def token_embedding(token_id, embedding_matrix):
    """Возвращает эмбеддинг токена по его ID."""
    return embedding_matrix[token_id]

def final_embedding(token_id, pos, embedding_matrix, d_model):
    """Полный pipeline: токен → ID → эмбеддинг + PE → итоговый вектор."""
    e = token_embedding(token_id, embedding_matrix)
    pe = sinusoidal_pe(pos, d_model)
    return e + pe

# Параметры
d_model = 4
vocab_size = 100
embedding_matrix = np.random.randn(vocab_size, d_model)

# Пример
token_id = 42  # "кот"
pos_0 = 0
pos_5 = 5

x_0 = final_embedding(token_id, pos_0, embedding_matrix, d_model)
x_5 = final_embedding(token_id, pos_5, embedding_matrix, d_model)

print(f"Эмбеддинг токена:           {embedding_matrix[token_id]}")
print(f"Итоговый вектор на pos=0:   {x_0}")
print(f"Итоговый вектор на pos=5:   {x_5}")
print(f"Разница между позициями:    {x_5 - x_0}")
```

**Пример вывода:**
```
Эмбеддинг токена:           [ 0.5  -0.2   0.8   0.1 ]
Итоговый вектор на pos=0:   [ 0.5   0.8   0.8   1.1 ]
Итоговый вектор на pos=5:   [-0.46  0.08  0.85  1.09]
Разница между позициями:    [-0.96 -0.72  0.05 -0.01]
```

---

### 6. Почему сложение работает: итоговое резюме

1. **Семантическая информация** сохраняется в эмбеддингах $\mathbf{e}_{pos}$.
2. **Позиционная информация** добавляется через $\mathbf{PE}_{pos}$.
3. **Сложение** позволяет объединить обе составляющие в одном векторе без потери информации.
4. **Последующие слои** Self-Attention могут извлекать как семантику, так и позицию через линейные проекции:

$$
\mathbf{Q} = \mathbf{X} W^Q, \quad \mathbf{K} = \mathbf{X} W^K, \quad \mathbf{V} = \mathbf{X} W^V.
$$

Поскольку $\mathbf{X} = \mathbf{E} + \mathbf{PE}$, то:

$$
\mathbf{Q} = (\mathbf{E} + \mathbf{PE}) W^Q = \mathbf{E} W^Q + \mathbf{PE} W^Q.
$$

Это означает, что запросы содержат как семантическую, так и позиционную информацию, причём модель может обучать веса $W^Q$ для извлечения нужных компонент.

---

### 7. Заключение

Добавление позиционного кодирования к эмбеддингам токенов — это простое, но элегантное решение, которое:

- Сохраняет размерность $d_{\text{model}}$;
- Не вводит новых параметров;
- Позволяет модели различать одинаковые токены на разных позициях;
- Обеспечивает передачу как семантической, так и позиционной информации в последующие слои.

Этот подход стал стандартом в архитектуре Transformer и используется во многих современных моделях, включая BERT, GPT и их производные.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

## Тема 5.1. Интуиция механизма внимания

Прежде чем погрузиться в математику механизма внимания, лежащего в основе Transformer, необходимо понять его интуитивную суть. Внимание (attention) — это концепция, которая имитирует то, как человек обрабатывает информацию: мы не анализируем все детали одновременно, а фокусируемся на наиболее релевантных частях в зависимости от текущей задачи. В этом разделе мы рассмотрим интуицию механизма внимания, его историческое развитие и визуальные интерпретации.

---

### 1. Что такое внимание: взвешенное суммирование

В контексте обработки естественного языка **внимание** — это механизм, который позволяет модели динамически выбирать, на какие части входной последовательности следует обращать внимание при генерации каждого выходного элемента. Математически внимание представляет собой **взвешенное суммирование**:

$$
\text{context} = \sum_{i=1}^{T} \alpha_i \cdot \mathbf{v}_i,
$$

где:

- $\mathbf{v}_i$ — векторное представление $i$-го токена (Value);
- $\alpha_i$ — вес внимания, отражающий важность $i$-го токена для текущего контекста;
- $\sum_{i=1}^{T} \alpha_i = 1$ — веса образуют распределение вероятностей.

#### 1.1. Аналогия с человеческим чтением

Когда человек читает текст, он не запоминает каждое слово с одинаковой важностью. Вместо этого:

1. **Сканирование:** Глаза быстро пробегают по тексту, выделяя ключевые слова.
2. **Фокусировка:** Внимание концентрируется на наиболее важных элементах (подлежащем, сказуемом, ключевых терминах).
3. **Контекст:** При чтении следующего предложения мы помним важные детали из предыдущих.

Механизм внимания в нейросетях работает аналогично: модель вычисляет, насколько каждый токен важен для понимания текущего, и использует эту информацию для формирования нового представления.

```mermaid
flowchart LR
    subgraph Human["Восприятие текста человеком"]
        direction TB
        H1["Беглый просмотр"] --> H2["Выделение ключевых слов"]
        H2 --> H3["Фокусировка на важном"]
        H3 --> H4["Формирование понимания"]
    end
    
    subgraph Attention["Механизм внимания в модели"]
        direction TB
        A1["Вычисление весов α_i"] --> A2["Взвешенное суммирование"]
        A2 --> A3["Формирование контекста"]
        A3 --> A4["Обновление представления"]
    end
    
    Human -.->|Аналогия| Attention
    
    style Human fill:#e3f2fd
    style Attention fill:#f3e5f5
```

---

### 2. Пример с анафорой: разрешение местоимений

Рассмотрим классическую задачу, где внимание играет ключевую роль — **разрешение анафоры** (определение, к чему относится местоимение).

**Предложение:** *"Человек не мог перейти дорогу, так как он был ранен"*

В этом предложении местоимение *"он"* относится к *"Человек"*. Чтобы правильно это понять, модель должна:

1. Заметить, что *"он"* — это местоимение мужского рода.
2. Найти в предложении существительное мужского рода (*"Человек"*).
3. Установить между ними связь, присвоив высокий вес внимания.

```mermaid
flowchart LR
    subgraph Sentence["Предложение"]
        direction LR
        W1["Человек"] --> W2["не"] --> W3["мог"] --> W4["перейти"] --> W5["дорогу,"] --> W6["так"] --> W7["как"] --> W8["он"] --> W9["был"] --> W10["ранен"]
    end
    
    subgraph AttentionMap["Карта внимания"]
        direction TB
        A1["Человек"] -->|"вес: 0.75"| A8["он"]
        A2["не"] -->|"вес: 0.02"| A8
        A3["мог"] -->|"вес: 0.05"| A8
        A4["перейти"] -->|"вес: 0.05"| A8
        A5["дорогу"] -->|"вес: 0.03"| A8
        A6["так"] -->|"вес: 0.02"| A8
        A7["как"] -->|"вес: 0.02"| A8
        A8["он"] -->|"вес: 0.75"| A1
        A9["был"] -->|"вес: 0.03"| A8
        A10["ранен"] -->|"вес: 0.03"| A8
    end
    
    Sentence --> AttentionMap
    
    style Sentence fill:#e3f2fd
    style AttentionMap fill:#f3e5f5
```

#### 2.1. Как внимание решает эту задачу

1. **Запрос (Query):** Токен *"он"* формирует запрос: *"Какое существительное мужского рода в единственном числе находится в предложении?"*
2. **Ключи (Keys):** Все остальные токены формируют ключи, описывающие их свойства (род, число, семантическая роль).
3. **Совместимость:** Вычисляется совместимость между запросом и каждым ключом. *"Человек"* получает высокий балл, так как его свойства совпадают с запросом.
4. **Веса:** После softmax вес для *"Человек"* становится высоким (например, 0.75), а для остальных токенов — низким.
5. **Контекст:** Новое представление токена *"он"* формируется как взвешенная сумма значений всех токенов, где доминирует *"Человек"*.

**Результат:** Модель понимает, что *"он"* относится к *"Человек"*, а не к *"дороге"* или *"переходу"*.

---

### 3. История развития механизма внимания

| Год | Работа | Вклад в развитие внимания |
|-----|--------|---------------------------|
| 2014 | Bahdanau et al., *"Neural Machine Translation by Jointly Learning to Align and Translate"* | Введение внимания в машинный перевод; модель выравнивания (alignment) для связи входных и выходных токенов |
| 2015 | Luong et al., *"Effective Approaches to Attention-based Neural Machine Translation"* | Систематизация видов внимания (глобальное vs локальное); различные функции совместимости |
| 2015 | Xu et al., *"Show, Attend and Tell"* | Применение внимания в компьютерном зрении (генерация описаний изображений) |
| 2016 | Cheng et al., *"Long Short-Term Memory-Networks for Machine Reading"* | Самовнимание (self-attention) для моделирования зависимостей внутри последовательности |
| 2017 | Vaswani et al., *"Attention Is All You Need"* | Отказ от RNN в пользу исключительно внимания; появление Transformer |

#### 3.1. Bahdanau et al. (2014): внимание для выравнивания

В работе Bahdanau et al. [1] механизм внимания был впервые применён в машинном переводе для решения проблемы сжатия всего предложения в фиксированный вектор. Вместо того чтобы использовать последнее скрытое состояние энкодера как единственный контекст, модель динамически выбирала, на какие части входного предложения обращать внимание при генерации каждого выходного токена.

#### 3.2. Luong et al. (2015): систематизация внимания

Luong et al. [2] предложили различные варианты внимания: **глобальное** (учитывает все токены входа) и **локальное** (учитывает только окрестность текущего токена). Также были предложены разные функции совместимости (dot-product, general, concat).

#### 3.3. Transformer (2017): внимание как основной механизм

В работе *"Attention Is All You Need"* [3] внимание стало **единственным** механизмом, заменив рекуррентные и свёрточные слои. Это позволило достичь полной параллелизации и рекордных результатов на задачах перевода.

---

### 4. Общая формула внимания

Независимо от конкретной реализации, механизм внимания всегда следует общей схеме:

$$
\text{context} = \sum_{i=1}^{T} \alpha_i \cdot \mathbf{v}_i,
$$

где вес $\alpha_i$ вычисляется как:

$$
\alpha_i = \frac{\exp(\text{score}(\mathbf{q}, \mathbf{k}_i))}{\sum_{j=1}^{T} \exp(\text{score}(\mathbf{q}, \mathbf{k}_j))}.
$$

Здесь:

- $\mathbf{q}$ — **запрос** (query), представляющий то, что мы ищем;
- $\mathbf{k}_i$ — **ключ** (key) $i$-го токена, описывающий его свойства;
- $\mathbf{v}_i$ — **значение** (value) $i$-го токена, содержащее информацию для передачи;
- $\text{score}(\mathbf{q}, \mathbf{k}_i)$ — **функция совместимости**, измеряющая, насколько токен $i$ соответствует запросу.

**Функции совместимости:**

| Тип | Формула | Пример использования |
|-----|---------|---------------------|
| Dot-product | $\mathbf{q}^T \mathbf{k}_i$ | Transformer (с масштабированием) |
| Multiplicative | $\mathbf{q}^T W \mathbf{k}_i$ | Luong et al. (2015) |
| Additive | $\mathbf{v}^T \tanh(W_1 \mathbf{q} + W_2 \mathbf{k}_i)$ | Bahdanau et al. (2014) |

---

### 5. Визуализация внимания

#### 5.1. Тепловая карта внимания

Тепловая карта внимания — это матрица $T \times T$, где элемент $(i, j)$ показывает, насколько токен $i$ обращает внимание на токен $j$. Яркость ячейки отражает величину веса внимания.

**Свойства тепловой карты:**
- **Диагональ:** Обычно яркая — токены часто обращают внимание на самих себя.
- **Синтаксические связи:** Видны связи между подлежащим и сказуемым, между местоимением и его антецедентом.
- **Семантические группы:** Слова из одной смысловой группы часто имеют высокие веса внимания.
- **Разные головы:** В Multi-Head Attention разные головы показывают разные паттерны (синтаксические, семантические, локальные).

#### 5.2. Интерпретация паттернов внимания

Исследования [4] показывают, что разные головы внимания выполняют разные функции:

| Тип паттерна | Описание | Пример |
|--------------|----------|--------|
| **Синтаксический** | Высокий вес для зависимых слов | Подлежащее → сказуемое |
| **Семантический** | Высокий вес для связанных по смыслу слов | "кот" → "молоко" |
| **Локальный** | Фокус на соседних токенах | Обработка n-грамм |
| **Глобальный** | Фокус на далёких токенах | Разрешение анафоры |

```mermaid
flowchart TD
    subgraph Heads["Паттерны разных голов внимания"]
        direction TB
        H1["Голова 1: Синтаксическая<br/>→ связи между словами"]
        H2["Голова 2: Семантическая<br/>→ смысловые группы"]
        H3["Голова 3: Локальная<br/>→ соседние токены"]
        H4["Голова 4: Глобальная<br/>→ дальние зависимости"]
    end
    
    style Heads fill:#e3f2fd
```

---

### 6. Заключение

Механизм внимания — это одна из самых мощных идей в современном глубоком обучении. Его интуитивная суть заключается в том, чтобы позволить модели динамически фокусироваться на наиболее релевантной информации, игнорируя несущественное. От первых применений в машинном переводе до доминирования в архитектуре Transformer, внимание прошло путь от вспомогательного механизма до основной парадигмы обработки последовательностей.

В следующих разделах мы рассмотрим математическую формализацию внимания в Transformer — Scaled Dot-Product Attention, Multi-Head Attention и их практическую реализацию.

---

### Литература

1. Bahdanau, D., Cho, K., & Bengio, Y. (2014). *Neural Machine Translation by Jointly Learning to Align and Translate*. ICLR.  
   🔗 [https://arxiv.org/abs/1409.0473](https://arxiv.org/abs/1409.0473)

2. Luong, M. T., Pham, H., & Manning, C. D. (2015). *Effective Approaches to Attention-based Neural Machine Translation*. EMNLP.  
   🔗 [https://arxiv.org/abs/1508.04025](https://arxiv.org/abs/1508.04025)

3. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

4. Clark, K., et al. (2019). *What Does BERT Look At? An Analysis of BERT's Attention*. BlackboxNLP Workshop.  
   🔗 [https://arxiv.org/abs/1906.04341](https://arxiv.org/abs/1906.04341)

## Тема 5.2. Query, Key, Value (Q, K, V) — три компонента внимания

В предыдущем разделе мы рассмотрели интуицию механизма внимания как взвешенного суммирования. Теперь настало время разобрать три ключевых компонента, которые делают внимание таким гибким и мощным: **Query** (Запрос), **Key** (Ключ) и **Value** (Значение). Эти три вектора являются сердцем механизма внимания и определяют, как именно модель устанавливает связи между токенами.

---

### 1. Определения и роли

#### 1.1. Query (Запрос) — "Что я ищу?"

**Query** — это вектор, который представляет собой то, что текущий токен "ищет" в других токенах. Это вопрос, который задаёт модель от лица каждого токена: *"Какая информация из других токенов мне нужна для понимания моего контекста?"*

**Свойства:**
- Query генерируется для каждого токена независимо;
- Query определяет, на что обращать внимание;
- Query может изменяться в зависимости от контекста.

#### 1.2. Key (Ключ) — "Что я предлагаю?"

**Key** — это вектор, который описывает "содержимое" или "свойства" токена. Это ответ на запросы других токенов: *"Вот что я могу предложить другим токенам, чтобы помочь им в понимании контекста."*

**Свойства:**
- Key генерируется для каждого токена независимо;
- Key описывает, чем токен может быть полезен другим;
- Key сопоставляется с Query для вычисления весов внимания.

#### 1.3. Value (Значение) — "Что я передаю?"

**Value** — это вектор, содержащий фактическую информацию, которую токен передаёт другим токенам в процессе взвешенного суммирования. Это ответ на вопрос: *"Если я важен для другого токена, какую именно информацию я передам?"*

**Свойства:**
- Value генерируется для каждого токена независимо;
- Value содержит фактические данные для передачи;
- Value используется в финальном взвешенном суммировании.

```mermaid
flowchart LR
    subgraph Token["Токен"]
        direction TB
        X["Входной вектор<br/>x ∈ ℝ^{d_model}"]
    end
    
    subgraph Projections["Проекции"]
        Q["Query<br/>q = x·W_q<br/>'Что я ищу?'"]
        K["Key<br/>k = x·W_k<br/>'Что я предлагаю?'"]
        V["Value<br/>v = x·W_v<br/>'Что я передаю?'"]
    end
    
    X --> Q
    X --> K
    X --> V
    
    style Token fill:#e3f2fd
    style Projections fill:#f3e5f5
```

#### 1.4. Таблица ролей

| Компонент | Роль | Вопрос | Использование |
|-----------|------|--------|---------------|
| **Query** | Искатель | "Что я ищу?" | Сравнивается с Keys для вычисления весов |
| **Key** | Предложение | "Что я предлагаю?" | Сравнивается с Queries для вычисления весов |
| **Value** | Содержание | "Что я передаю?" | Участвует в взвешенном суммировании |

---

### 2. Математика: линейные проекции

Для получения Q, K, V каждый входной токен $\mathbf{x}_i$ (размерности $d_{\text{model}}$) умножается на три обучаемые матрицы:

$$
\mathbf{q}_i = \mathbf{x}_i W^Q, \quad \mathbf{k}_i = \mathbf{x}_i W^K, \quad \mathbf{v}_i = \mathbf{x}_i W^V,
$$

где:

- $W^Q, W^K, W^V \in \mathbb{R}^{d_{\text{model}} \times d_k}$ — обучаемые матрицы проекций;
- $d_k$ — размерность Query и Key (обычно $d_k = d_{\text{model}} / h$, где $h$ — число голов внимания);
- $\mathbf{q}_i, \mathbf{k}_i, \mathbf{v}_i \in \mathbb{R}^{d_k}$ — векторы для $i$-го токена.

Для всей последовательности матрицы Q, K, V вычисляются как:

$$
\mathbf{Q} = \mathbf{X} W^Q, \quad \mathbf{K} = \mathbf{X} W^K, \quad \mathbf{V} = \mathbf{X} W^V,
$$

где $\mathbf{X} \in \mathbb{R}^{T \times d_{\text{model}}}$ — входная матрица, а $\mathbf{Q}, \mathbf{K}, \mathbf{V} \in \mathbb{R}^{T \times d_k}$.

```mermaid
flowchart LR
    subgraph Input["Входная матрица"]
        X["X ∈ ℝ^{T×d_model}"]
    end
    
    subgraph Weights["Обучаемые матрицы"]
        WQ["W^Q ∈ ℝ^{d_model×d_k}"]
        WK["W^K ∈ ℝ^{d_model×d_k}"]
        WV["W^V ∈ ℝ^{d_model×d_k}"]
    end
    
    subgraph Output["Результат проекций"]
        Q["Q = X·W^Q ∈ ℝ^{T×d_k}"]
        K["K = X·W^K ∈ ℝ^{T×d_k}"]
        V["V = X·W^V ∈ ℝ^{T×d_k}"]
    end
    
    X --> WQ
    X --> WK
    X --> WV
    WQ --> Q
    WK --> K
    WV --> V
    
    style Input fill:#e3f2fd
    style Weights fill:#fff3e0
    style Output fill:#f3e5f5
```

---

### 3. Аналогия с поисковой системой

Чтобы лучше понять роль Q, K и V, представим аналогию с **поисковой системой** (например, Google Search):

| Поисковая система | Механизм внимания |
|-------------------|-------------------|
| **Пользователь вводит запрос** | **Query** — что ищет текущий токен |
| **Поисковая система ищет документы, заголовки которых соответствуют запросу** | **Key** — заголовки/свойства токенов, с которыми сравнивается Query |
| **Поисковая система показывает содержимое релевантных документов** | **Value** — содержимое токенов, которое передаётся в итоговый контекст |

```mermaid
flowchart LR
    subgraph SearchEngine["Поисковая система"]
        SE1["Запрос пользователя"] --> SE2["Сравнение с заголовками"]
        SE2 --> SE3["Выбор релевантных документов"]
        SE3 --> SE4["Отображение содержимого"]
    end
    
    subgraph Attention["Механизм внимания"]
        ATT1["Query"] --> ATT2["Сравнение с Keys"]
        ATT2 --> ATT3["Вычисление весов α_i"]
        ATT3 --> ATT4["Взвешенное суммирование Values"]
    end
    
    SE1 -.->|Аналогия| ATT1
    SE2 -.->|Аналогия| ATT2
    SE3 -.->|Аналогия| ATT3
    SE4 -.->|Аналогия| ATT4
    
    style SearchEngine fill:#e3f2fd
    style Attention fill:#f3e5f5
```

#### 3.1. Другая аналогия: библиотека

Представьте, что вы в библиотеке:

- **Вы** (читатель) = **Query**. Вы ищете информацию по определённой теме.
- **Названия книг на полках** = **Key**. Вы просматриваете названия, чтобы найти подходящие книги.
- **Содержание книг** = **Value**. Когда вы нашли нужную книгу, вы читаете её содержание, чтобы получить информацию.

Чем лучше название книги соответствует вашему запросу (высокий вес внимания), тем больше внимания вы уделяете её содержанию.

---

### 4. Зачем нужны три разных вектора?

#### 4.1. Разделение ролей для гибкости

Разделение на Q, K и V позволяет каждому токену выполнять три разные функции одновременно:

1. **Как Query:** Токен может искать информацию в других токенах.
2. **Как Key:** Токен может предоставлять свою информацию для поиска другими токенами.
3. **Как Value:** Токен может передавать своё содержимое другим токенам.

#### 4.2. Возможность разных подпространств

Поскольку Q, K и V генерируются через **разные матрицы проекций** ($W^Q, W^K, W^V$), они могут лежать в разных подпространствах исходного пространства. Это позволяет модели:

- Использовать разные признаки для поиска (Query) и для описания (Key);
- Передавать информацию, которая может отличаться от того, что используется для поиска (Value).

#### 4.3. Улучшение выразительности

Если бы все три компонента были одинаковыми ($Q = K = V = X$), модель была бы сильно ограничена. Разделение ролей:

- Позволяет модели обучаться различным функциям для каждой роли;
- Увеличивает выразительную способность модели;
- Позволяет различным головам внимания фокусироваться на разных аспектах.

---

### 5. Пример с конкретными числами

Рассмотрим упрощённый пример с тремя токенами и $d_{\text{model}} = 4$, $d_k = 2$. Пусть входная матрица:

$$
\mathbf{X} = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1 \\
1 & 1 & 0 & 0
\end{bmatrix}.
$$

Пусть матрицы проекций (для простоты) имеют вид:

$$
W^Q = \begin{bmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
0 & 0
\end{bmatrix}, \quad
W^K = \begin{bmatrix}
0 & 0 \\
0 & 0 \\
1 & 0 \\
0 & 1
\end{bmatrix}, \quad
W^V = \begin{bmatrix}
0 & 0 \\
0 & 0 \\
0 & 0 \\
1 & 1
\end{bmatrix}.
$$

Тогда:

**Вычисление Q:**

$$
\mathbf{Q} = \mathbf{X} W^Q = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1 \\
1 & 1 & 0 & 0
\end{bmatrix}
\begin{bmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
0 & 0
\end{bmatrix}
= \begin{bmatrix}
1 & 0 \\
0 & 1 \\
1 & 1
\end{bmatrix}.
$$

**Вычисление K:**

$$
\mathbf{K} = \mathbf{X} W^K = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1 \\
1 & 1 & 0 & 0
\end{bmatrix}
\begin{bmatrix}
0 & 0 \\
0 & 0 \\
1 & 0 \\
0 & 1
\end{bmatrix}
= \begin{bmatrix}
1 & 0 \\
0 & 1 \\
0 & 0
\end{bmatrix}.
$$

**Вычисление V:**

$$
\mathbf{V} = \mathbf{X} W^V = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1 \\
1 & 1 & 0 & 0
\end{bmatrix}
\begin{bmatrix}
0 & 0 \\
0 & 0 \\
0 & 0 \\
1 & 1
\end{bmatrix}
= \begin{bmatrix}
0 & 0 \\
1 & 1 \\
0 & 0
\end{bmatrix}.
$$

**Интерпретация:**

- **Query** (Q) кодирует "что ищет" каждый токен: токен 1 ищет что-то в первом измерении, токен 2 — во втором, токен 3 — в обоих.
- **Key** (K) кодирует "что предлагает" каждый токен: токен 1 предлагает первое измерение, токен 2 — второе, токен 3 — ничего.
- **Value** (V) кодирует "что передаёт" каждый токен: токен 1 не передаёт ничего, токен 2 передаёт сумму, токен 3 не передаёт ничего.

---

### 6. Код на Python для вычисления Q, K, V

```python
import numpy as np

def compute_qkv(X, W_q, W_k, W_v):
    """
    Вычисляет матрицы Q, K, V для входной последовательности.
    
    Args:
        X (np.ndarray): Входная матрица размером (T, d_model)
        W_q (np.ndarray): Матрица проекции для Q размером (d_model, d_k)
        W_k (np.ndarray): Матрица проекции для K размером (d_model, d_k)
        W_v (np.ndarray): Матрица проекции для V размером (d_model, d_k)
    
    Returns:
        tuple: (Q, K, V) каждая размером (T, d_k)
    """
    Q = X @ W_q
    K = X @ W_k
    V = X @ W_v
    return Q, K, V

# Пример
d_model = 4
d_k = 2
T = 3

# Входная матрица
X = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 0.0]
])

# Случайные матрицы проекций (инициализируются при обучении)
np.random.seed(42)
W_q = np.random.randn(d_model, d_k)
W_k = np.random.randn(d_model, d_k)
W_v = np.random.randn(d_model, d_k)

Q, K, V = compute_qkv(X, W_q, W_k, W_v)

print("Вход X:\n", X)
print("\nQ:\n", Q)
print("\nK:\n", K)
print("\nV:\n", V)
```

---

### 7. Заключение

Query, Key и Value — это три фундаментальных компонента, которые делают механизм внимания гибким и мощным:

1. **Query** определяет, что ищет текущий токен.
2. **Key** описывает, что предлагает каждый токен.
3. **Value** содержит информацию, которую токен передаёт другим.

Благодаря использованию различных матриц проекций ($W^Q, W^K, W^V$) модель может обучать разные функции для каждой роли, что значительно увеличивает её выразительную способность. В следующем разделе мы увидим, как эти три компонента собираются вместе в формуле Scaled Dot-Product Attention.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Bahdanau, D., Cho, K., & Bengio, Y. (2014). *Neural Machine Translation by Jointly Learning to Align and Translate*. ICLR.  
   🔗 [https://arxiv.org/abs/1409.0473](https://arxiv.org/abs/1409.0473)

## Тема 5.3. Scaled Dot-Product Attention — сердце Transformer

В предыдущих разделах мы рассмотрели интуицию механизма внимания и роль трёх ключевых компонентов: Query, Key, Value. Теперь настало время собрать все части вместе и разобрать **Scaled Dot-Product Attention** — конкретную реализацию внимания, предложенную в оригинальной статье Transformer [1]. Именно этот механизм стал тем фундаментальным строительным блоком, который позволил архитектуре Transformer достичь беспрецедентных результатов. В этом разделе мы проведём детальный математический разбор каждого шага, от матричного умножения до итогового взвешенного суммирования.

---

### 1. Полная формула и её структура

Scaled Dot-Product Attention определяется следующей формулой:

$$
\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}}\right) \mathbf{V},
$$

где:

- $\mathbf{Q} \in \mathbb{R}^{T \times d_k}$ — матрица запросов (Queries);
- $\mathbf{K} \in \mathbb{R}^{T \times d_k}$ — матрица ключей (Keys);
- $\mathbf{V} \in \mathbb{R}^{T \times d_v}$ — матрица значений (Values);
- $d_k$ — размерность Query и Key;
- $d_v$ — размерность Value (в оригинале $d_v = d_k$);
- $T$ — длина последовательности.

```mermaid
flowchart TD
    subgraph Input["Входные матрицы"]
        Q["Q ∈ ℝ^{T×d_k}"]
        K["K ∈ ℝ^{T×d_k}"]
        V["V ∈ ℝ^{T×d_v}"]
    end
    
    subgraph Step1["Шаг 1: Матрица оценок"]
        S["S = Q·K^T ∈ ℝ^{T×T}"]
    end
    
    subgraph Step2["Шаг 2: Масштабирование"]
        S_scaled["S_scaled = S / √d_k"]
    end
    
    subgraph Step3["Шаг 3: Softmax"]
        A["A = softmax(S_scaled) ∈ ℝ^{T×T}"]
    end
    
    subgraph Step4["Шаг 4: Взвешенное суммирование"]
        O["O = A·V ∈ ℝ^{T×d_v}"]
    end
    
    Q --> S
    K --> S
    S --> S_scaled --> A --> O
    V --> O
    
    style Input fill:#e3f2fd
    style Step1 fill:#fff3e0
    style Step2 fill:#f3e5f5
    style Step3 fill:#e8f5e9
    style Step4 fill:#fce4ec
```

---

### 2. Пошаговый математический разбор

Рассмотрим каждый шаг детально, прослеживая преобразования размерностей.

#### Шаг 1: Вычисление матрицы оценок (Scores)

Первым шагом мы вычисляем матрицу оценок $\mathbf{S}$ путём умножения $\mathbf{Q}$ на $\mathbf{K}^T$:

$$
\mathbf{S} = \mathbf{Q} \cdot \mathbf{K}^T \in \mathbb{R}^{T \times T}.
$$

Элемент $\mathbf{S}_{ij}$ матрицы $\mathbf{S}$ — это скалярное произведение вектора запроса $i$-го токена $\mathbf{q}_i$ и вектора ключа $j$-го токена $\mathbf{k}_j$:

$$
\mathbf{S}_{ij} = \mathbf{q}_i \cdot \mathbf{k}_j = \sum_{m=1}^{d_k} \mathbf{Q}_{im} \cdot \mathbf{K}_{jm}.
$$

**Интерпретация:** $\mathbf{S}_{ij}$ показывает, насколько токен $i$ "интересуется" токеном $j$. Чем больше значение, тем более релевантен токен $j$ для токена $i$.

**Визуализация:**

```mermaid
flowchart LR
    subgraph MatrixS["Матрица оценок S ∈ ℝ^{T×T}"]
        direction TB
        S11["S₁₁"]
        S12["S₁₂"]
        S13["..."]
        S1T["S₁_T"]
        S21["S₂₁"]
        S22["S₂₂"]
        S2T["S₂_T"]
        ST1["S_T₁"]
        STT["S_T_T"]
    end
    
    subgraph Interpretation["Интерпретация строки i"]
        I1["Sᵢ₁: интерес токена i к токену 1"]
        I2["Sᵢ₂: интерес токена i к токену 2"]
        IT["Sᵢ_T: интерес токена i к токену T"]
    end
    
    MatrixS --> Interpretation
    
    style MatrixS fill:#fff3e0
    style Interpretation fill:#e3f2fd
```

#### Шаг 2: Масштабирование (Scaling)

Вторым шагом мы делим матрицу $\mathbf{S}$ на $\sqrt{d_k}$:

$$
\mathbf{S}_{\text{scaled}} = \frac{\mathbf{S}}{\sqrt{d_k}}.
$$

**Математическое обоснование масштабирования:**

Рассмотрим случайные векторы $\mathbf{q}, \mathbf{k} \in \mathbb{R}^{d_k}$ с независимыми компонентами, имеющими нулевое среднее и единичную дисперсию. Тогда скалярное произведение $\mathbf{q} \cdot \mathbf{k}$ имеет:

- Среднее значение: $\mathbb{E}[\mathbf{q} \cdot \mathbf{k}] = 0$;
- Дисперсию: $\text{Var}[\mathbf{q} \cdot \mathbf{k}] = d_k$ (как сумма $d_k$ независимых произведений с дисперсией 1).

Таким образом, без масштабирования значения $\mathbf{S}_{ij}$ имеют дисперсию $d_k$. Для $d_k = 64$ дисперсия равна 64, что приводит к значениям около ±8. Такие большие значения загоняют softmax в области с экстремально малыми градиентами (насыщение). Деление на $\sqrt{d_k}$ приводит дисперсию к 1, что делает softmax более стабильным.

```mermaid
flowchart LR
    subgraph Before["Без масштабирования"]
        B1["S ~ N(0, d_k)"]
        B2["softmax: экстремальные значения"]
        B3["градиенты → 0"]
    end
    
    subgraph After["С масштабированием"]
        A1["S/√d_k ~ N(0, 1)"]
        A2["softmax: нормальные значения"]
        A3["градиенты стабильны"]
    end
    
    Before -->|"/√d_k"| After
    
    style Before fill:#ffcdd2
    style After fill:#e8f5e9
```

#### Шаг 3: Применение Softmax

Третьим шагом мы применяем функцию softmax к каждой строке матрицы $\mathbf{S}_{\text{scaled}}$:

$$
\mathbf{A} = \text{softmax}\left(\mathbf{S}_{\text{scaled}}\right) \in \mathbb{R}^{T \times T},
$$

где для каждой строки $i$:

$$
\mathbf{A}_{ij} = \frac{\exp(\mathbf{S}_{\text{scaled}, ij})}{\sum_{k=1}^{T} \exp(\mathbf{S}_{\text{scaled}, ik})}.
$$

**Свойства:**
- $\mathbf{A}_{ij} \in (0, 1)$ для всех $i, j$;
- $\sum_{j=1}^{T} \mathbf{A}_{ij} = 1$ для каждой строки (распределение вероятностей);
- Высокие значения $\mathbf{S}_{ij}$ → высокие веса $\mathbf{A}_{ij}$.

**Интерпретация:** $\mathbf{A}_{ij}$ — это вес внимания, который токен $i$ уделяет токену $j$. Сумма весов по строке равна 1, поэтому внимание можно интерпретировать как распределение вероятностей по токенам.

#### Шаг 4: Взвешенное суммирование (Weighted Sum)

Финальным шагом мы умножаем матрицу весов $\mathbf{A}$ на матрицу значений $\mathbf{V}$:

$$
\mathbf{O} = \mathbf{A} \cdot \mathbf{V} \in \mathbb{R}^{T \times d_v}.
$$

Для каждого токена $i$:

$$
\mathbf{o}_i = \sum_{j=1}^{T} \mathbf{A}_{ij} \cdot \mathbf{v}_j.
$$

**Интерпретация:** Новое представление $i$-го токена $\mathbf{o}_i$ является взвешенной суммой значений всех токенов, где вес $\mathbf{A}_{ij}$ определяет вклад $j$-го токена. Таким образом, каждый токен "впитывает" информацию из всего контекста, но с разной интенсивностью.

```mermaid
flowchart LR
    subgraph Token_i["Токен i"]
        direction TB
        T1["Запрос qᵢ"]
        T2["Веса αᵢ₁, αᵢ₂, ..., αᵢ_T"]
        T3["Новый вектор oᵢ = Σ αᵢⱼ·vⱼ"]
    end
    
    subgraph Values["Значения других токенов"]
        V1["v₁"] --> T2
        V2["v₂"] --> T2
        V3["..."] --> T2
        VT["v_T"] --> T2
    end
    
    style Token_i fill:#f3e5f5
    style Values fill:#e3f2fd
```

---

### 3. Полный математический вывод

Соберём все шаги в одну цепочку преобразований:

$$
\begin{aligned}
\mathbf{S} &= \mathbf{Q} \mathbf{K}^T & &\text{(матрица оценок, } \mathbb{R}^{T \times T}\text{)} \\
\mathbf{S}_{\text{scaled}} &= \frac{\mathbf{S}}{\sqrt{d_k}} & &\text{(масштабирование)} \\
\mathbf{A} &= \text{softmax}(\mathbf{S}_{\text{scaled}}) & &\text{(веса внимания, } \mathbb{R}^{T \times T}\text{)} \\
\mathbf{O} &= \mathbf{A} \mathbf{V} & &\text{(выход, } \mathbb{R}^{T \times d_v}\text{)}
\end{aligned}
$$

Или, в компактной форме:

$$
\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}}\right) \mathbf{V}.
$$

---

### 4. Пример с конкретными числами

Рассмотрим полностью детализированный пример с $T = 3$ токенами, $d_k = 2$, $d_v = 2$.

**Входные матрицы:**

Пусть:

$$
\mathbf{Q} = \begin{bmatrix}
1 & 0 \\
0 & 1 \\
1 & 1
\end{bmatrix}, \quad
\mathbf{K} = \begin{bmatrix}
1 & 0 \\
0 & 1 \\
1 & 0
\end{bmatrix}, \quad
\mathbf{V} = \begin{bmatrix}
1 & 2 \\
3 & 4 \\
5 & 6
\end{bmatrix}.
$$

**Шаг 1: Вычисление $\mathbf{S} = \mathbf{Q} \mathbf{K}^T$**

Вычисляем скалярные произведения:

- Строка 1: $[1, 0] \cdot [1, 0] = 1$, $[1, 0] \cdot [0, 1] = 0$, $[1, 0] \cdot [1, 0] = 1$ → $[1, 0, 1]$
- Строка 2: $[0, 1] \cdot [1, 0] = 0$, $[0, 1] \cdot [0, 1] = 1$, $[0, 1] \cdot [1, 0] = 0$ → $[0, 1, 0]$
- Строка 3: $[1, 1] \cdot [1, 0] = 1$, $[1, 1] \cdot [0, 1] = 1$, $[1, 1] \cdot [1, 0] = 1$ → $[1, 1, 1]$

$$
\mathbf{S} = \begin{bmatrix}
1 & 0 & 1 \\
0 & 1 & 0 \\
1 & 1 & 1
\end{bmatrix}.
$$

**Шаг 2: Масштабирование на $\sqrt{d_k} = \sqrt{2} \approx 1.414$**

$$
\mathbf{S}_{\text{scaled}} = \frac{1}{1.414} \cdot \mathbf{S} = \begin{bmatrix}
0.707 & 0 & 0.707 \\
0 & 0.707 & 0 \\
0.707 & 0.707 & 0.707
\end{bmatrix}.
$$

**Шаг 3: Применение Softmax**

Вычисляем для каждой строки:

Строка 1: $[0.707, 0, 0.707]$
- Экспоненты: $\exp(0.707) \approx 2.028$, $\exp(0) = 1$, $\exp(0.707) \approx 2.028$
- Сумма: $2.028 + 1 + 2.028 = 5.056$
- Веса: $[2.028/5.056, 1/5.056, 2.028/5.056] \approx [0.401, 0.198, 0.401]$

Строка 2: $[0, 0.707, 0]$
- Экспоненты: $\exp(0) = 1$, $\exp(0.707) \approx 2.028$, $\exp(0) = 1$
- Сумма: $1 + 2.028 + 1 = 4.028$
- Веса: $[1/4.028, 2.028/4.028, 1/4.028] \approx [0.248, 0.503, 0.248]$

Строка 3: $[0.707, 0.707, 0.707]$
- Экспоненты: все равны $\exp(0.707) \approx 2.028$
- Сумма: $3 \cdot 2.028 = 6.084$
- Веса: $[2.028/6.084, 2.028/6.084, 2.028/6.084] \approx [0.333, 0.333, 0.333]$

Получаем матрицу весов:

$$
\mathbf{A} \approx \begin{bmatrix}
0.401 & 0.198 & 0.401 \\
0.248 & 0.503 & 0.248 \\
0.333 & 0.333 & 0.333
\end{bmatrix}.
$$

**Шаг 4: Вычисление $\mathbf{O} = \mathbf{A} \cdot \mathbf{V}$**

Для строки 1:

$$
\mathbf{o}_1 = 0.401 \cdot [1, 2] + 0.198 \cdot [3, 4] + 0.401 \cdot [5, 6]
$$

$$
= [0.401 + 0.594 + 2.005, \; 0.802 + 0.792 + 2.406] = [3.000, 4.000].
$$

Для строки 2:

$$
\mathbf{o}_2 = 0.248 \cdot [1, 2] + 0.503 \cdot [3, 4] + 0.248 \cdot [5, 6]
$$

$$
= [0.248 + 1.509 + 1.240, \; 0.496 + 2.012 + 1.488] = [2.997, 3.996].
$$

Для строки 3:

$$
\mathbf{o}_3 = \frac{1}{3}([1,2] + [3,4] + [5,6]) = [3.000, 4.000].
$$

Итоговая матрица:

$$
\mathbf{O} \approx \begin{bmatrix}
3.000 & 4.000 \\
2.997 & 3.996 \\
3.000 & 4.000
\end{bmatrix}.
$$

---

### 5. Реализация на NumPy

```python
import numpy as np

def scaled_dot_product_attention(Q, K, V, d_k, mask=None):
    """
    Вычисляет Scaled Dot-Product Attention.
    
    Args:
        Q (np.ndarray): Матрица запросов размером (T, d_k)
        K (np.ndarray): Матрица ключей размером (T, d_k)
        V (np.ndarray): Матрица значений размером (T, d_v)
        d_k (int): Размерность Query и Key
        mask (np.ndarray, optional): Маска для внимания (T, T)
    
    Returns:
        tuple: (output, attention_weights)
            output (np.ndarray): Выходная матрица (T, d_v)
            attention_weights (np.ndarray): Веса внимания (T, T)
    """
    # Шаг 1: Вычисляем матрицу оценок
    S = Q @ K.T  # (T, d_k) @ (d_k, T) -> (T, T)
    
    # Шаг 2: Масштабирование
    S_scaled = S / np.sqrt(d_k)
    
    # Шаг 3: Применяем маску (если есть)
    if mask is not None:
        S_scaled = S_scaled + mask
    
    # Шаг 4: Softmax для получения весов
    # Используем softmax по строкам (axis=-1)
    exp_S = np.exp(S_scaled - np.max(S_scaled, axis=-1, keepdims=True))
    attention_weights = exp_S / np.sum(exp_S, axis=-1, keepdims=True)
    
    # Шаг 5: Взвешенное суммирование
    output = attention_weights @ V  # (T, T) @ (T, d_v) -> (T, d_v)
    
    return output, attention_weights

# Пример использования
if __name__ == "__main__":
    # Параметры
    T = 3
    d_k = 2
    d_v = 2
    
    # Входные данные (как в примере)
    Q = np.array([[1, 0], [0, 1], [1, 1]])
    K = np.array([[1, 0], [0, 1], [1, 0]])
    V = np.array([[1, 2], [3, 4], [5, 6]])
    
    # Вычисление
    output, weights = scaled_dot_product_attention(Q, K, V, d_k)
    
    print("Входные данные:")
    print(f"Q (Запросы):\n{Q}")
    print(f"K (Ключи):\n{K}")
    print(f"V (Значения):\n{V}")
    print("\nРезультат:")
    print(f"Веса внимания:\n{weights.round(3)}")
    print(f"Выход:\n{output.round(3)}")
    
    # Проверка: сумма весов в каждой строке = 1
    print(f"\nСумма весов по строкам: {weights.sum(axis=1)}")
```

**Результат выполнения:**
```
Входные данные:
Q (Запросы):
[[1 0]
 [0 1]
 [1 1]]
K (Ключи):
[[1 0]
 [0 1]
 [1 0]]
V (Значения):
[[1 2]
 [3 4]
 [5 6]]

Результат:
Веса внимания:
[[0.401 0.198 0.401]
 [0.248 0.503 0.248]
 [0.333 0.333 0.333]]
Выход:
[[3.    4.   ]
 [2.997 3.996]
 [3.    4.   ]]

Сумма весов по строкам: [1. 1. 1.]
```

---

### 6. Свойства матрицы внимания

Матрица весов $\mathbf{A}$ обладает несколькими важными свойствами:

1. **Стохастичность:** Каждая строка является распределением вероятностей ($\sum_j A_{ij} = 1$).

2. **Неотрицательность:** $A_{ij} \ge 0$ для всех $i, j$.

3. **Диагональное доминирование:** Часто диагональные элементы большие, так как токен обычно наиболее релевантен сам себе.

4. **Редкость:** Хотя формально все элементы положительны, на практике многие веса близки к нулю, особенно для далёких токенов.

5. **Интерпретируемость:** Веса внимания можно визуализировать как тепловую карту, показывающую, какие токены влияют на представление других.

---

### 7. Заключение

Scaled Dot-Product Attention — это элегантный и математически обоснованный механизм, который составляет сердце архитектуры Transformer. Четыре простых шага — матричное умножение, масштабирование, softmax и взвешенное суммирование — создают мощный инструмент для моделирования зависимостей между токенами.

Ключевые особенности:
- **Эффективность:** Использует оптимизированные матричные операции.
- **Интерпретируемость:** Веса внимания можно визуализировать и анализировать.
- **Масштабируемость:** Легко распараллеливается на GPU.
- **Гибкость:** Может быть расширен с помощью масок для различных задач (например, каузальная маска в декодере).

В следующем разделе мы рассмотрим, как Scaled Dot-Product Attention расширяется до Multi-Head Attention, что позволяет модели одновременно фокусироваться на различных аспектах информации.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

## Тема 5.4. Зачем нужно масштабирование на √dₖ

В формуле Scaled Dot-Product Attention присутствует, на первый взгляд, незначительная деталь — деление на √dₖ. Однако этот, казалось бы, простой шаг является критически важным для стабильности обучения и качества модели. В этом разделе мы детально разберём, почему масштабирование необходимо, какие проблемы оно решает и как это обосновано математически.

---

### 1. Проблема больших значений: дисперсия скалярного произведения

#### 1.1. Предпосылки

Пусть векторы запроса и ключа имеют размерность dₖ. В процессе обучения значения элементов Q и K инициализируются так, чтобы дисперсия каждого элемента была близка к 1 (стандартная практика, например, инициализация Ксавье или нормальная инициализация с масштабированием). Таким образом:

$$
\text{Var}(q_i) = 1, \quad \text{Var}(k_i) = 1.
$$

Скалярное произведение между q и k вычисляется как:

$$
s = \sum_{i=1}^{d_k} q_i \cdot k_i.
$$

Если qᵢ и kᵢ независимы и имеют нулевое среднее, то дисперсия произведения равна произведению дисперсий (так как E[qᵢ kᵢ] = E[qᵢ]E[kᵢ] = 0). Тогда дисперсия суммы:

$$
\text{Var}(s) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i) = \sum_{i=1}^{d_k} \text{Var}(q_i) \text{Var}(k_i) = d_k \cdot 1 \cdot 1 = d_k.
$$

Таким образом, дисперсия каждого элемента матрицы оценок S = QKᵀ равна dₖ. Для dₖ = 64 это даёт дисперсию 64, а стандартное отклонение — 8. Для dₖ = 512 дисперсия становится 512, что приводит к стандартному отклонению ~22.6.

```mermaid
flowchart LR
    subgraph Dist["Распределение s = q·k"]
        D1["dₖ = 1: σ²=1, σ=1"]
        D2["dₖ = 64: σ²=64, σ=8"]
        D3["dₖ = 512: σ²=512, σ=22.6"]
        D4["dₖ = 1024: σ²=1024, σ=32"]
    end
    
    style Dist fill:#e3f2fd
```

#### 1.2. Математический вывод

Формально, пусть q, k ∈ ℝ^{dₖ} — векторы с независимыми компонентами, имеющими нулевое среднее и единичную дисперсию. Тогда:

$$
\text{Var}(q_i k_i) = \mathbb{E}[q_i^2 k_i^2] - (\mathbb{E}[q_i k_i])^2 = \mathbb{E}[q_i^2]\mathbb{E}[k_i^2] - 0 = 1 \cdot 1 = 1.
$$

Следовательно:

$$
\text{Var}\left( \sum_{i=1}^{d_k} q_i k_i \right) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i) = d_k.
$$

Таким образом, значения в матрице оценок будут иметь масштаб порядка √dₖ, что может быть очень большим для больших dₖ.

---

### 2. Влияние больших значений на Softmax

#### 2.1. Экстремальные вероятности

Функция softmax преобразует вектор оценок s ∈ ℝᵀ в распределение вероятностей:

$$
\text{softmax}(s)_i = \frac{\exp(s_i)}{\sum_{j=1}^{T} \exp(s_j)}.
$$

Если все sᵢ имеют большой масштаб (например, порядка 10 или 20), то разница между максимальным и остальными значениями становится огромной. Экспонента от больших чисел приводит к тому, что вероятность максимального элемента становится близкой к 1, а все остальные — к 0.

**Пример:** Пусть s = [10, 0, 0]. Тогда:

$$
\text{softmax}(s) \approx [0.99995, 0.000025, 0.000025].
$$

Это означает, что модель практически игнорирует все токены, кроме одного, что противоречит идее внимания как взвешенного учёта всего контекста.

#### 2.2. Проблема градиентов

В области, где softmax выдаёт вероятности, близкие к 0 или 1, градиенты становятся крайне малыми. Это явление известно как **насыщение** (saturation). Градиент softmax по входу вычисляется как:

$$
\frac{\partial \text{softmax}(s)_i}{\partial s_j} = \text{softmax}(s)_i (\delta_{ij} - \text{softmax}(s)_j).
$$

Если вероятности экстремальны (например, одна равна 0.99995, остальные ~0.000025), то производные будут порядка 0.000025 или меньше, что приводит к исчезающим градиентам. Это замедляет обучение или полностью его останавливает.

```mermaid
flowchart LR
    subgraph Problem["Проблема больших значений"]
        P1["s ~ N(0, dₖ)"]
        P2["softmax → экстремумы"]
        P3["градиенты → 0"]
        P4["обучение замедляется"]
    end
    
    P1 --> P2 --> P3 --> P4
    
    style Problem fill:#ffcdd2
```

#### 2.3. Визуализация распределения оценок

При dₖ = 512 без масштабирования распределение оценок имеет стандартное отклонение ~22.6. Это означает, что типичные значения s лежат в диапазоне [-30, 30]. После применения softmax к таким значениям, вероятности становятся практически 0 или 1.

---

### 3. Масштабирование и нормализация дисперсии

#### 3.1. Деление на √dₖ

Масштабирование заключается в делении каждого элемента матрицы оценок на √dₖ:

$$
s' = \frac{s}{\sqrt{d_k}}.
$$

Тогда дисперсия нового значения:

$$
\text{Var}(s') = \frac{1}{d_k} \text{Var}(s) = \frac{1}{d_k} \cdot d_k = 1.
$$

Таким образом, после масштабирования распределение оценок имеет единичную дисперсию, что соответствует нормальному распределению со стандартным отклонением 1. Это предотвращает экстремальные значения и сохраняет градиенты здоровыми.

#### 3.2. Влияние на softmax

При единичной дисперсии оценки s' будут иметь типичные значения порядка 1-2. После применения softmax вероятности распределяются более равномерно. Например, для s' = [1.5, 0, 0]:

$$
\text{softmax}(s') \approx [0.67, 0.165, 0.165].
$$

Это позволяет модели учитывать несколько токенов одновременно, что соответствует идее внимания.

#### 3.3. Влияние на градиенты

При единичной дисперсии градиенты softmax остаются достаточно большими (порядка 0.1-0.3), что позволяет эффективно обучать модель.

```mermaid
flowchart LR
    subgraph Solution["После масштабирования"]
        S1["s' = s/√dₖ"]
        S2["Var(s') = 1"]
        S3["softmax → нормальные значения"]
        S4["градиенты стабильны"]
    end
    
    S1 --> S2 --> S3 --> S4
    
    style Solution fill:#e8f5e9
```

---

### 4. Полное математическое обоснование

#### 4.1. Вывод дисперсии

Рассмотрим случайные векторы q, k ∈ ℝ^{dₖ} с независимыми компонентами, имеющими распределение с нулевым средним и единичной дисперсией. Тогда:

$$
\mathbb{E}[q_i k_i] = 0, \quad \mathbb{E}[(q_i k_i)^2] = \mathbb{E}[q_i^2]\mathbb{E}[k_i^2] = 1.
$$

Следовательно:

$$
\text{Var}(q_i k_i) = 1.
$$

Для суммы:

$$
\mathbb{E}[s] = 0, \quad \text{Var}(s) = d_k.
$$

После масштабирования:

$$
s' = \frac{s}{\sqrt{d_k}}, \quad \mathbb{E}[s'] = 0, \quad \text{Var}(s') = 1.
$$

#### 4.2. Распределение после softmax

Используя центральную предельную теорему, при больших dₖ распределение s приближается к нормальному: s ~ N(0, dₖ). После масштабирования s' ~ N(0, 1). Это означает, что 95% значений лежат в интервале [-2, 2], что обеспечивает стабильную работу softmax.

```mermaid
flowchart TD
    subgraph DistBefore["Без масштабирования"]
        B1["s ~ N(0, dₖ)"]
        B2["σ = √dₖ"]
        B3["значения в диапазоне [-3√dₖ, 3√dₖ]"]
        B4["для dₖ=512: [-67.8, 67.8]"]
    end
    
    subgraph DistAfter["С масштабированием"]
        A1["s' ~ N(0, 1)"]
        A2["σ = 1"]
        A3["значения в диапазоне [-3, 3]"]
        A4["стабильный softmax"]
    end
    
    DistBefore -->|"/√dₖ"| DistAfter
    
    style DistBefore fill:#ffcdd2
    style DistAfter fill:#e8f5e9
```

---

### 5. Экспериментальное подтверждение

#### 5.1. Сравнение обучения

В оригинальной статье [1] авторы экспериментально показали, что без масштабирования обучение Transformer становится нестабильным: функция потерь часто расходится, а качество модели значительно ухудшается. После введения масштабирования обучение становится стабильным, и модель достигает высоких результатов.

**Результаты экспериментов:**
- **Без масштабирования:** модель показывает низкое качество (низкий BLEU), обучение часто расходится.
- **С масштабированием:** модель достигает SOTA результатов (BLEU 28.4 на EN-DE, 41.8 на EN-FR).

#### 5.2. Визуализация распределения весов

На тепловой карте весов внимания видно, что без масштабирования веса становятся либо почти единичными (один токен имеет вес ~1, остальные ~0), либо сильно разреженными. С масштабированием веса распределяются более равномерно, позволяя модели учитывать более широкий контекст.

```mermaid
flowchart LR
    subgraph NoScale["Без масштабирования"]
        N1["веса: [0.98, 0.01, 0.01, 0.00]"]
        N2["градиенты: очень малые"]
        N3["обучение нестабильно"]
    end
    
    subgraph WithScale["С масштабированием"]
        W1["веса: [0.4, 0.3, 0.2, 0.1]"]
        W2["градиенты: здоровые"]
        W3["обучение стабильно"]
    end
    
    NoScale -->|"с √dₖ"| WithScale
    
    style NoScale fill:#ffcdd2
    style WithScale fill:#e8f5e9
```

---

### 6. Заключение

Масштабирование на √dₖ является критически важным компонентом механизма внимания, обеспечивающим:

- **Нормализацию дисперсии** оценок до единицы;
- **Предотвращение экстремальных значений** в softmax;
- **Сохранение стабильных градиентов** для эффективного обучения;
- **Улучшение сходимости** и конечного качества модели.

Это простое, но мощное математическое уточнение позволило Transformer работать с большими размерностями (dₖ до 128 и выше), что сделало возможным масштабирование моделей до миллиардов параметров. Понимание этого аспекта необходимо для правильной реализации и настройки архитектур на основе внимания.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

## Тема 6.1. Почему одной головы внимания недостаточно

Механизм самовнимания (self-attention) в том виде, в котором он был представлен в формуле Scaled Dot-Product Attention, выглядит элегантным и завершённым. Однако в архитектуре Transformer он используется не в одиночку, а в составе **многоголового внимания (Multi-Head Attention)** — блока, содержащего \(h\) параллельных голов внимания. Возникает закономерный вопрос: *почему одной головы недостаточно?* В этом разделе мы разберём фундаментальные ограничения одной головы, рассмотрим, какие типы связей изучают разные головы, и проанализируем, почему множественность голов даёт модели качественно новые возможности.

---

### 1. Проблема одной головы: усреднение и потеря выразительности

#### 1.1. Одна голова — один тип связей

Одна голова внимания вычисляет единственное распределение весов \(\mathbf{A} \in \mathbb{R}^{T \times T}\), где \(\mathbf{A}_{ij}\) показывает, насколько токен \(i\) обращает внимание на токен \(j\). Это распределение является результатом усреднения по всем возможным аспектам взаимодействия между токенами. Однако в языке существует множество типов связей одновременно:

- **Синтаксические связи**: подлежащее и сказуемое, существительное и прилагательное.
- **Семантические связи**: слова, связанные по смыслу («кот» и «молоко»).
- **Локальные связи**: соседние слова в предложении.
- **Глобальные связи**: местоимение и его антецедент на расстоянии десятков токенов.

Одна голова вынуждена **усреднять** все эти типы связей в одном распределении весов, что приводит к компромиссному решению, не позволяющему полноценно отразить ни один из аспектов. Как отмечают авторы оригинальной статьи, *«с одной головой внимания усреднение препятствует совместному обращению внимания на информацию из разных подпространств представлений»*.

#### 1.2. Недостаточная выразительность

При фиксированной размерности \(d_{\text{model}}\) одна голова внимания оперирует в полном пространстве, но может изучить только один паттерн внимания. Это ограничивает выразительную способность модели: она не может одновременно фокусироваться на синтаксической структуре предложения и на семантических связях между словами.

```mermaid
flowchart LR
    subgraph SingleHead["Одна голова внимания"]
        direction TB
        S1["Вход: X ∈ ℝ^{T×d_model}"]
        S2["Одно распределение весов A"]
        S3["Усреднение всех типов связей"]
        S4["Компромиссное представление"]
    end
    
    subgraph MultiHead["Многоголовое внимание"]
        direction TB
        M1["Вход: X ∈ ℝ^{T×d_model}"]
        M2["h независимых распределений"]
        M3["Каждая голова — свой тип связей"]
        M4["Богатое, многоаспектное представление"]
    end
    
    S1 --> S2 --> S3 --> S4
    M1 --> M2 --> M3 --> M4
    
    style SingleHead fill:#ffcdd2
    style MultiHead fill:#e8f5e9
```

---

### 2. Разные типы связей: что изучают головы

Многоголовое внимание позволяет модели одновременно фокусироваться на разных аспектах входной последовательности. Исследования показывают, что разные головы действительно специализируются на различных типах связей.

#### 2.1. Синтаксические связи (грамматика)

Некоторые головы внимания демонстрируют паттерны, соответствующие синтаксической структуре предложения. В работе Marecek и Rosa (2019) авторы исследовали много-головое самовнимание в Transformer-энкодерах для трёх языков и обнаружили, что многие головы внимания демонстрируют паттерны, напоминающие синтаксические фразы — последовательности состояний, обращающих внимание на одну и ту же позицию. Например, головы могут отслеживать согласование подлежащего и сказуемого или связь между существительным и прилагательным.

#### 2.2. Семантические связи (смысл)

Другие головы фокусируются на семантических отношениях: разрешение анафоры (связь местоимения с его антецедентом), установление связей между словами из одной смысловой группы. Исследования BERT показывают, что одни головы являются более семантически-ориентированными (окрашены в красный цвет на визуализациях), в то время как другие — более синтаксически-ориентированными (окрашены в синий).

#### 2.3. Локальные и глобальные связи

Некоторые головы специализируются на **локальных** связях — соседних токенах, что соответствует обработке n-грамм. Другие головы, напротив, фокусируются на **глобальных** связях, позволяя модели учитывать информацию из далёких частей последовательности при формировании представления текущего токена.

#### 2.4. Таблица типов связей

| Тип связи | Описание | Пример |
|-----------|----------|--------|
| **Синтаксическая** | Грамматические зависимости | Подлежащее → сказуемое, существительное → прилагательное |
| **Семантическая** | Смысловые отношения | Разрешение анафоры, тематические кластеры |
| **Локальная** | Соседние токены | Обработка n-грамм, локальный контекст |
| **Глобальная** | Удалённые токены | Дальние зависимости, контекст всего предложения |
| **Позиционная** | Относительное расстояние | Учёт порядка и расстояния между токенами |

---

### 3. Анализ разных голов: исследования и интерпретация

#### 3.1. Визуализация и классификация голов

Существуют инструменты для визуализации и классификации голов внимания. Например, библиотека `attn_head_probe` классифицирует головы GPT-2 на основе их паттернов внимания, выделяя такие роли, как **копирование**, **индукция**, **позиционные** и **синтаксические**. Это подтверждает, что головы действительно выполняют различные функции.

На тепловых картах внимания для разных голов можно наблюдать качественно различные паттерны:

- **Диагональные паттерны**: голова фокусируется на самом токене (self-attention).
- **Структурные паттерны**: голова отслеживает синтаксические зависимости.
- **Разреженные паттерны**: голова фокусируется на нескольких ключевых токенах.

```mermaid
flowchart TD
    subgraph HeadPatterns["Паттерны разных голов внимания"]
        direction TB
        H1["Голова 1: Синтаксическая<br/>→ связи между подлежащим и сказуемым"]
        H2["Голова 2: Семантическая<br/>→ разрешение анафоры"]
        H3["Голова 3: Локальная<br/>→ соседние токены"]
        H4["Голова 4: Глобальная<br/>→ дальние зависимости"]
    end
    
    style HeadPatterns fill:#e3f2fd
```

#### 3.2. Результаты исследований

**Исследование 1: "Are Sixteen Heads Really Better than One?" (Michel et al., 2019)**

Авторы сделали удивительное наблюдение: хотя модели обучаются с множеством голов, значительная часть голов может быть удалена на этапе тестирования без существенного ухудшения качества. Более того, некоторые слои могут быть сокращены до одной головы. Это говорит о том, что избыточность в многоголовом внимании существует, но наличие множества голов в процессе обучения даёт模型у стабильность и гибкость.

**Исследование 2: "Multi-head or Single-head? An Empirical Comparison" (Liu et al., 2021)**

Авторы предположили, что основное преимущество многоголового внимания — это **стабильность обучения**, а не способность одновременно обращать внимание на множество позиций. Модель с 24 слоями и 16 головами (BERT-large) имеет ту же глубину по числу голов, что и 384-слойная одноголовая модель, но значительно мельче и легче в обучении.

**Исследование 3: Специализация голов (Voita et al.)**

В работе "Analyzing Multi-Head Self-Attention: Specialized Heads Do the Heavy Lifting, the Rest Can Be Pruned" авторы показали, что лишь небольшое число голов выполняют ключевую работу, а остальные могут быть удалены.

#### 3.3. Примеры интерпретации голов

В BERT наблюдаются следующие паттерны:

- **Ранние слои (1-3)**: головы часто более семантически-ориентированы.
- **Средние слои (4-7)**: проявляется синтаксическая структура, сильное согласование подлежащего и сказуемого.
- **Поздние слои (8-11)**: высокоуровневое семантическое выравнивание.

---

### 4. Преимущества многоголового внимания

#### 4.1. Параллельное изучение разных аспектов

Многоголовое внимание позволяет модели одновременно изучать несколько типов зависимостей в одном слое. Каждая голова имеет свои проекционные матрицы \(W_i^Q, W_i^K, W_i^V\) и, следовательно, может фокусироваться на разных подпространствах исходного пространства эмбеддингов. Это даёт модели возможность «смотреть» на данные с разных точек зрения одновременно.

#### 4.2. Более богатые представления

Конкатенация выходов всех голов и их последующая проекция через \(W^O\) позволяет объединить информацию из разных подпространств в единое представление. Это даёт модели более богатые и многоаспектные представления токенов.

#### 4.3. Улучшение качества модели

Многоголовое внимание является одной из ключевых причин успеха Transformer. Как отмечается в литературе, *«многоголовое внимание играет решающую роль в недавнем успехе моделей Transformer, обеспечивая последовательное улучшение производительности по сравнению с обычным вниманием в различных приложениях»*.

#### 4.4. Эффективность вычислений

Важно отметить, что многоголовое внимание **не увеличивает** общее число параметров и вычислительную сложность по сравнению с одной головой при фиксированной \(d_{\text{model}}\). При \(h\) головах размерность каждой головы составляет \(d_k = d_{\text{model}} / h\). Суммарная сложность \(h \cdot O(T^2 \cdot d_k) = O(T^2 \cdot d_{\text{model}})\) остаётся той же.

---

### 5. Визуализация разных голов

```mermaid
flowchart LR
    subgraph Heatmaps["Тепловые карты внимания для разных голов"]
        direction TB
        HM1["Голова 1<br/>Синтаксическая<br/>▄▄▄▄▄▄▄▄▄"]
        HM2["Голова 2<br/>Семантическая<br/>▄▄▄▄▄▄▄▄▄"]
        HM3["Голова 3<br/>Локальная<br/>▄▄▄▄▄▄▄▄▄"]
        HM4["Голова 4<br/>Глобальная<br/>▄▄▄▄▄▄▄▄▄"]
    end
    
    style Heatmaps fill:#e3f2fd
```

На тепловых картах для каждой головы видны разные паттерны:
- **Яркая диагональ**: токен обращает внимание на самого себя.
- **Горизонтальные/вертикальные полосы**: структурированные синтаксические связи.
- **Разреженные точки**: фокус на отдельных ключевых токенах.

---

### 6. Заключение

Одна голова внимания — это мощный, но ограниченный инструмент. Она способна улавливать только один тип связей между токенами и вынуждена усреднять информацию из разных аспектов языка. Многоголовое внимание преодолевает это ограничение, позволяя модели:

1. **Параллельно изучать** разные типы связей (синтаксические, семантические, локальные, глобальные).
2. **Формировать более богатые** и многоаспектные представления токенов.
3. **Обеспечивать стабильность** обучения за счёт уменьшения эффективной глубины модели.
4. **Сохранять вычислительную эффективность** при фиксированной размерности \(d_{\text{model}}\).

Хотя исследования показывают, что не все головы одинаково полезны и некоторые могут быть удалены после обучения, наличие множества голов в процессе обучения даёт模型у необходимую гибкость и устойчивость. Именно этот механизм позволяет Transformer достигать выдающихся результатов в широком спектре задач обработки естественного языка и других модальностей.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Michel, P., Levy, O., & Neubig, G. (2019). *Are Sixteen Heads Really Better than One?*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1905.10650](https://arxiv.org/abs/1905.10650)

3. Liu, L., Liu, J., & Han, J. (2021). *Multi-head or Single-head? An Empirical Comparison for Transformer Training*.  
   🔗 [https://arxiv.org/abs/2106.09650](https://arxiv.org/abs/2106.09650)

4. Clark, K., et al. (2019). *What Does BERT Look At? An Analysis of BERT's Attention*. BlackboxNLP Workshop.  
   🔗 [https://arxiv.org/abs/1906.04341](https://arxiv.org/abs/1906.04341)

5. Marecek, D., & Rosa, R. (2019). *From Balustrades to Pierre Vinken: Looking for Syntax in Transformer Self-Attentions*.  
   🔗 [https://aclanthology.org/W19-4827/](https://aclanthology.org/W19-4827/)

6. Voita, E., et al. (2019). *Analyzing Multi-Head Self-Attention: Specialized Heads Do the Heavy Lifting, the Rest Can Be Pruned*. ACL.  
   🔗 [https://arxiv.org/abs/1905.09418](https://arxiv.org/abs/1905.09418)

## Тема 6.2. Как работает Multi-Head Attention

В предыдущем разделе мы обосновали необходимость множества голов внимания и рассмотрели, какие типы связей изучают разные головы. Теперь настало время детально разобрать математику и механику **Multi-Head Attention** — ключевого компонента архитектуры Transformer, который позволяет модели одновременно фокусироваться на разных аспектах входной последовательности. В этом разделе мы пройдём каждый этап: от вычисления отдельных голов до их объединения и финальной проекции.

---

### 1. Полная формула Multi-Head Attention

Multi-Head Attention определяется следующими формулами:

$$
\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) \mathbf{W}^O,
$$

где каждая голова вычисляется как:

$$
\text{head}_i = \text{Attention}\left(\mathbf{Q} \mathbf{W}_i^Q,\; \mathbf{K} \mathbf{W}_i^K,\; \mathbf{V} \mathbf{W}_i^V\right).
$$

Здесь:

- $\mathbf{Q}, \mathbf{K}, \mathbf{V} \in \mathbb{R}^{T \times d_{\text{model}}}$ — входные матрицы запросов, ключей и значений;
- $\mathbf{W}_i^Q, \mathbf{W}_i^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$ — матрицы проекций для $i$-й головы (для Query и Key);
- $\mathbf{W}_i^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$ — матрица проекции для $i$-й головы (для Value);
- $\mathbf{W}^O \in \mathbb{R}^{h \cdot d_v \times d_{\text{model}}}$ — матрица финальной проекции;
- $h$ — число голов внимания;
- $d_k = d_v = d_{\text{model}} / h$ — размерность каждой головы (в оригинале $d_k = d_v = 64$, $h = 8$).

```mermaid
flowchart TD
    subgraph Input["Входные матрицы"]
        Q["Q ∈ ℝ^{T×d_model}"]
        K["K ∈ ℝ^{T×d_model}"]
        V["V ∈ ℝ^{T×d_model}"]
    end
    
    subgraph Heads["Параллельные головы внимания"]
        direction TB
        H1["Голова 1<br/>Q·W₁^Q, K·W₁^K, V·W₁^V"]
        H2["Голова 2<br/>Q·W₂^Q, K·W₂^K, V·W₂^V"]
        HDot["..."]
        Hh["Голова h<br/>Q·W_h^Q, K·W_h^K, V·W_h^V"]
    end
    
    subgraph Concat["Конкатенация"]
        C["Concat(head₁, ..., headₕ)<br/>∈ ℝ^{T×(h·d_v)}"]
    end
    
    subgraph Projection["Финальная проекция"]
        WO["· W^O<br/>W^O ∈ ℝ^{(h·d_v)×d_model}"]
        Out["Выход ∈ ℝ^{T×d_model}"]
    end
    
    Q --> H1
    Q --> H2
    Q --> Hh
    K --> H1
    K --> H2
    K --> Hh
    V --> H1
    V --> H2
    V --> Hh
    
    H1 --> C
    H2 --> C
    Hh --> C
    
    C --> WO --> Out
    
    style Input fill:#e3f2fd
    style Heads fill:#f3e5f5
    style Concat fill:#fff3e0
    style Projection fill:#e8f5e9
```

---

### 2. Параллельные головы: проекции и размерности

#### 2.1. Независимые проекции для каждой головы

Каждая голова имеет свои собственные матрицы проекций $\mathbf{W}_i^Q$, $\mathbf{W}_i^K$, $\mathbf{W}_i^V$. Это позволяет каждой голове работать в своём собственном подпространстве и фокусироваться на различных аспектах входных данных.

Для $i$-й головы:

$$
\mathbf{Q}_i = \mathbf{Q} \mathbf{W}_i^Q, \quad \mathbf{K}_i = \mathbf{K} \mathbf{W}_i^K, \quad \mathbf{V}_i = \mathbf{V} \mathbf{W}_i^V.
$$

Здесь $\mathbf{Q}_i, \mathbf{K}_i, \mathbf{V}_i \in \mathbb{R}^{T \times d_k}$, где $d_k = d_{\text{model}} / h$.

#### 2.2. Почему размерность уменьшается

Размерность каждой головы $d_k = d_{\text{model}} / h$ выбрана так, чтобы общая вычислительная сложность оставалась такой же, как у одной головы внимания с полной размерностью. Если бы каждая голова имела размерность $d_{\text{model}}$, общая сложность выросла бы в $h$ раз. Благодаря уменьшению размерности:

- Суммарное число параметров: $h \cdot (d_{\text{model}} \cdot d_k) = h \cdot d_{\text{model}} \cdot (d_{\text{model}} / h) = d_{\text{model}}^2$.
- Сложность вычислений: $h \cdot O(T^2 \cdot d_k) = O(T^2 \cdot d_{\text{model}})$.

#### 2.3. Схема размерностей

```mermaid
flowchart LR
    subgraph Dimensionality["Размерности в Multi-Head Attention"]
        direction TB
        D1["Вход: T × d_model"]
        D2["Проекция для головы i: T × d_k<br/>(d_k = d_model/h)"]
        D3["Внимание в голове i: T × d_v<br/>(d_v = d_k)"]
        D4["Конкатенация: T × (h·d_v) = T × d_model"]
        D5["Финальная проекция: T × d_model"]
    end
    
    D1 --> D2 --> D3 --> D4 --> D5
    
    style Dimensionality fill:#e3f2fd
```

---

### 3. Конкатенация результатов голов

После того как все $h$ голов вычислили свои выходы, они объединяются (конкатенируются) вдоль оси измерений:

$$
\mathbf{C} = \text{Concat}(\text{head}_1, \dots, \text{head}_h) \in \mathbb{R}^{T \times (h \cdot d_v)}.
$$

Поскольку $h \cdot d_v = h \cdot d_k = h \cdot (d_{\text{model}} / h) = d_{\text{model}}$, размерность после конкатенации равна $T \times d_{\text{model}}$.

---

### 4. Финальная проекция $\mathbf{W}^O$

После конкатенации применяется линейное преобразование (проекция) через матрицу $\mathbf{W}^O \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$:

$$
\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) \mathbf{W}^O.
$$

Эта проекция позволяет модели комбинировать информацию из всех голов и преобразовывать её в единое представление той же размерности $d_{\text{model}}$.

---

### 5. Пример с конкретными числами

Рассмотрим пример с $d_{\text{model}} = 4$, $h = 2$, $d_k = d_v = 2$, $T = 3$.

**Входные матрицы:**

$$
\mathbf{Q} = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1 \\
1 & 1 & 0 & 0
\end{bmatrix}, \quad
\mathbf{K} = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1 \\
1 & 1 & 0 & 0
\end{bmatrix}, \quad
\mathbf{V} = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1 \\
1 & 1 & 0 & 0
\end{bmatrix}.
$$

**Матрицы проекций для головы 1:**

$$
\mathbf{W}_1^Q = \begin{bmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
0 & 0
\end{bmatrix}, \quad
\mathbf{W}_1^K = \begin{bmatrix}
0 & 0 \\
0 & 0 \\
1 & 0 \\
0 & 1
\end{bmatrix}, \quad
\mathbf{W}_1^V = \begin{bmatrix}
0 & 0 \\
0 & 0 \\
0 & 0 \\
1 & 1
\end{bmatrix}.
$$

**Матрицы проекций для головы 2:**

$$
\mathbf{W}_2^Q = \begin{bmatrix}
0 & 0 \\
0 & 0 \\
1 & 0 \\
0 & 1
\end{bmatrix}, \quad
\mathbf{W}_2^K = \begin{bmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
0 & 0
\end{bmatrix}, \quad
\mathbf{W}_2^V = \begin{bmatrix}
1 & 1 \\
0 & 0 \\
0 & 0 \\
0 & 0
\end{bmatrix}.
$$

**Вычисление головы 1:**

$\mathbf{Q}_1 = \mathbf{Q} \mathbf{W}_1^Q$ — берём первые два измерения: $\mathbf{Q}_1 \in \mathbb{R}^{3 \times 2}$.

$$
\mathbf{Q}_1 = \begin{bmatrix}
1 & 0 \\
0 & 1 \\
1 & 1
\end{bmatrix}, \quad
\mathbf{K}_1 = \begin{bmatrix}
1 & 0 \\
0 & 1 \\
0 & 0
\end{bmatrix}, \quad
\mathbf{V}_1 = \begin{bmatrix}
0 & 0 \\
1 & 1 \\
0 & 0
\end{bmatrix}.
$$

Вычисляем внимание для головы 1:

$$
\text{Attention}(\mathbf{Q}_1, \mathbf{K}_1, \mathbf{V}_1) = \text{softmax}\left(\frac{\mathbf{Q}_1 \mathbf{K}_1^T}{\sqrt{2}}\right) \mathbf{V}_1.
$$

После вычислений получаем $\text{head}_1 \in \mathbb{R}^{3 \times 2}$.

**Вычисление головы 2:**

Аналогично, $\text{head}_2 \in \mathbb{R}^{3 \times 2}$.

**Конкатенация:**

$$
\mathbf{C} = \text{Concat}(\text{head}_1, \text{head}_2) \in \mathbb{R}^{3 \times 4}.
$$

**Финальная проекция:**

$$
\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \mathbf{C} \mathbf{W}^O \in \mathbb{R}^{3 \times 4}.
$$

---

### 6. Код на NumPy

```python
import numpy as np

class MultiHeadAttention:
    def __init__(self, d_model, h):
        """
        Инициализация Multi-Head Attention.
        
        Args:
            d_model (int): размерность модели
            h (int): число голов внимания
        """
        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h
        self.d_v = d_model // h
        
        # Инициализация матриц проекций для всех голов
        # W_q, W_k, W_v: (h, d_model, d_k)
        self.W_q = np.random.randn(h, d_model, self.d_k) * 0.01
        self.W_k = np.random.randn(h, d_model, self.d_k) * 0.01
        self.W_v = np.random.randn(h, d_model, self.d_v) * 0.01
        self.W_o = np.random.randn(h * self.d_v, d_model) * 0.01
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Scaled Dot-Product Attention для одной головы.
        """
        d_k = Q.shape[-1]
        scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)
        
        if mask is not None:
            scores = scores + mask
            
        attn_weights = self.softmax(scores)
        output = attn_weights @ V
        return output, attn_weights
    
    def softmax(self, x, axis=-1):
        """Стабильный softmax."""
        x_max = np.max(x, axis=axis, keepdims=True)
        exp_x = np.exp(x - x_max)
        return exp_x / np.sum(exp_x, axis=axis, keepdims=True)
    
    def forward(self, Q, K, V, mask=None):
        """
        Прямой проход через Multi-Head Attention.
        
        Args:
            Q (np.ndarray): (batch, T, d_model)
            K (np.ndarray): (batch, T, d_model)
            V (np.ndarray): (batch, T, d_model)
            mask (np.ndarray, optional): маска
            
        Returns:
            np.ndarray: (batch, T, d_model)
            list: веса внимания для каждой головы
        """
        batch_size = Q.shape[0]
        T = Q.shape[1]
        
        # Проекции для каждой головы
        heads = []
        attn_weights_list = []
        
        for i in range(self.h):
            # Проекция Q, K, V для головы i
            Q_i = Q @ self.W_q[i]  # (batch, T, d_k)
            K_i = K @ self.W_k[i]  # (batch, T, d_k)
            V_i = V @ self.W_v[i]  # (batch, T, d_v)
            
            # Внимание для головы i
            head_i, attn_i = self.scaled_dot_product_attention(Q_i, K_i, V_i, mask)
            heads.append(head_i)
            attn_weights_list.append(attn_i)
        
        # Конкатенация
        concat = np.concatenate(heads, axis=-1)  # (batch, T, h*d_v)
        
        # Финальная проекция
        output = concat @ self.W_o  # (batch, T, d_model)
        
        return output, attn_weights_list

# Пример использования
if __name__ == "__main__":
    d_model = 4
    h = 2
    T = 3
    batch_size = 1
    
    mha = MultiHeadAttention(d_model, h)
    
    # Случайные входные данные
    Q = np.random.randn(batch_size, T, d_model)
    K = np.random.randn(batch_size, T, d_model)
    V = np.random.randn(batch_size, T, d_model)
    
    output, attn_weights = mha.forward(Q, K, V)
    
    print(f"Входная размерность Q: {Q.shape}")
    print(f"Выходная размерность: {output.shape}")
    print(f"Число голов: {len(attn_weights)}")
    print(f"Размер весов внимания для головы 1: {attn_weights[0].shape}")
```

---

### 7. Схема с размерами на каждом этапе

```mermaid
flowchart LR
    subgraph Steps["Этапы Multi-Head Attention"]
        S0["Q,K,V: batch × T × d_model"]
        S1["Для каждой головы i:<br/>Qᵢ = Q·Wᵢ^Q: batch × T × d_k<br/>Kᵢ = K·Wᵢ^K: batch × T × d_k<br/>Vᵢ = V·Wᵢ^V: batch × T × d_v"]
        S2["headᵢ = Attention(Qᵢ, Kᵢ, Vᵢ):<br/>batch × T × d_v"]
        S3["Concat(head₁, ..., headₕ):<br/>batch × T × (h·d_v) = batch × T × d_model"]
        S4["· W^O:<br/>batch × T × d_model"]
    end
    
    S0 --> S1 --> S2 --> S3 --> S4
    
    style Steps fill:#e3f2fd
```

---

### 8. Сравнение с одной головой

| Аспект | Одна голова | Multi-Head (h голов) |
|--------|-------------|---------------------|
| **Число голов** | 1 | h (обычно 8-96) |
| **Размерность** | $d_{\text{model}}$ | $d_k = d_{\text{model}} / h$ на голову |
| **Проекции** | $W^Q, W^K, W^V$ | $W_i^Q, W_i^K, W_i^V$ для каждой головы |
| **Параметры** | $3 \cdot d_{\text{model}}^2$ | $3 \cdot h \cdot d_{\text{model}} \cdot d_k = 3 \cdot d_{\text{model}}^2$ |
| **Сложность** | $O(T^2 \cdot d_{\text{model}})$ | $O(T^2 \cdot d_{\text{model}})$ |
| **Типы связей** | Один | Множество (разные головы) |

---

### 9. Заключение

Multi-Head Attention — это элегантное расширение механизма внимания, которое позволяет модели одновременно изучать различные типы зависимостей в данных. Благодаря независимым проекциям для каждой головы, модель может:

1. **Фокусироваться на разных аспектах** входной последовательности (синтаксических, семантических, локальных, глобальных).
2. **Обучаться параллельно** всем головам, что эффективно использует современные GPU.
3. **Сохранять вычислительную сложность** на уровне одной головы благодаря уменьшению размерности $d_k = d_{\text{model}} / h$.

В оригинальном Transformer используется $h = 8$ голов с $d_k = d_v = 64$ при $d_{\text{model}} = 512$. Современные модели, такие как LLaMA и Qwen, используют значительно больше голов (до 96) и различные вариации (Multi-Query Attention, Grouped-Query Attention) для повышения эффективности.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

## Тема 6.3. Разновидности внимания

В архитектуре Transformer механизм внимания используется не в одном, а в трёх различных вариантах, каждый из которых выполняет свою специфическую функцию. Понимание различий между **Self-Attention**, **Cross-Attention** и **Masked (Causal) Self-Attention** является ключевым для осознания того, как работают энкодер и декодер, и как они взаимодействуют друг с другом. В этом разделе мы детально разберём каждый тип внимания, их математическую формулировку, назначение и применение.

---

### 1. Self-Attention (Самовнимание)

#### 1.1. Определение

**Self-Attention** — это механизм, в котором запросы (Q), ключи (K) и значения (V) извлекаются из **одной и той же последовательности**. Это позволяет каждому токену взаимодействовать со всеми другими токенами в этой последовательности, включая самого себя.

Математически:

$$
\mathbf{Q} = \mathbf{X} \mathbf{W}^Q, \quad \mathbf{K} = \mathbf{X} \mathbf{W}^K, \quad \mathbf{V} = \mathbf{X} \mathbf{W}^V,
$$

$$
\text{SelfAttention}(\mathbf{X}) = \text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}}\right) \mathbf{V}.
$$

#### 1.2. Где используется

Self-Attention применяется в:

- **Энкодере:** позволяет каждому токену видеть все остальные токены (двусторонний контекст). Это даёт модели полное понимание входного предложения.
- **Декодере:** но в модифицированном виде (см. раздел 3).

#### 1.3. Зачем нужен Self-Attention

Self-Attention позволяет модели:

- Устанавливать связи между словами внутри предложения (анафора, синтаксические зависимости).
- Создавать контекстуализированные представления, где каждый токен «знает» о всех остальных.
- Обрабатывать дальние зависимости с постоянной сложностью \(O(1)\) на пару токенов.

```mermaid
flowchart LR
    subgraph Input["Входная последовательность"]
        X1["x₁"] --> SA["Self-Attention"]
        X2["x₂"] --> SA
        X3["x₃"] --> SA
        X4["..."] --> SA
        XT["x_T"] --> SA
    end
    
    subgraph Output["Выход"]
        SA --> O1["контекстуализированный x₁"]
        SA --> O2["контекстуализированный x₂"]
        SA --> O3["контекстуализированный x₃"]
        SA --> OT["контекстуализированный x_T"]
    end
    
    style Input fill:#e3f2fd
    style Output fill:#e8f5e9
```

---

### 2. Cross-Attention (Перекрёстное внимание)

#### 2.1. Определение

**Cross-Attention** (также известное как Encoder-Decoder Attention) — это механизм, в котором запросы (Q) берутся из одной последовательности (обычно из декодера), а ключи (K) и значения (V) — из другой последовательности (из выхода энкодера).

Математически:

$$
\mathbf{Q} = \mathbf{Y} \mathbf{W}^Q, \quad \mathbf{K} = \mathbf{Z} \mathbf{W}^K, \quad \mathbf{V} = \mathbf{Z} \mathbf{W}^V,
$$

$$
\text{CrossAttention}(\mathbf{Y}, \mathbf{Z}) = \text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}}\right) \mathbf{V}.
$$

Здесь:

- $\mathbf{Y} \in \mathbb{R}^{M \times d_{\text{model}}}$ — выход предыдущего слоя декодера (или текущего, после masked self-attention);
- $\mathbf{Z} \in \mathbb{R}^{T \times d_{\text{model}}}$ — выход стека энкодеров.

#### 2.2. Где используется

Cross-Attention используется исключительно в **декодере** (второй подслой каждого слоя декодера). Он позволяет декодеру «смотреть» на входную последовательность и извлекать из неё необходимую информацию при генерации каждого выходного токена.

#### 2.3. Зачем нужен Cross-Attention

Cross-Attention является ключевым мостом между энкодером и декодером. Он позволяет:

- Декодеру динамически выбирать, на какие части входного предложения обращать внимание при генерации каждого токена.
- Передавать информацию из энкодера в декодер без использования фиксированного вектора (как в RNN seq2seq).
- Обеспечивать выравнивание (alignment) между входной и выходной последовательностями.

```mermaid
flowchart LR
    subgraph Encoder["Энкодер"]
        Z["Z ∈ ℝ^{T×d_model}"]
    end
    
    subgraph Decoder["Декодер"]
        Y["Y ∈ ℝ^{M×d_model}"]
        CA["Cross-Attention<br/>Q из Y, K,V из Z"]
    end
    
    Z -->|"K, V"| CA
    Y -->|"Q"| CA
    CA --> Out["выход"]
    
    style Encoder fill:#f3e5f5
    style Decoder fill:#fce4ec
```

---

### 3. Masked (Causal) Self-Attention

#### 3.1. Определение

**Masked Self-Attention** (также называемое **Causal Self-Attention** или **каузальным вниманием**) — это вариант Self-Attention, в котором токенам запрещено обращаться к будущим токенам (тем, которые находятся правее). Это необходимо для сохранения авторегрессивного свойства при генерации.

Математически:

$$
\text{MaskedAttention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right) \mathbf{V},
$$

где $\mathbf{M} \in \mathbb{R}^{T \times T}$ — матрица маски, определяемая как:

$$
\mathbf{M}_{ij} =
\begin{cases}
0, & i \ge j \quad (\text{разрешено обращаться к позициям } \le i), \\
-\infty, & i < j \quad (\text{запрещено обращаться к будущим позициям}).
\end{cases}
$$

#### 3.2. Визуализация маски

Для последовательности длины $T = 5$ маска выглядит следующим образом:

```mermaid
flowchart TD
    subgraph Mask["Каузальная маска (T=5)"]
        direction TB
        M1["[0,  -∞,  -∞,  -∞,  -∞]"]
        M2["[0,   0,  -∞,  -∞,  -∞]"]
        M3["[0,   0,   0,  -∞,  -∞]"]
        M4["[0,   0,   0,   0,  -∞]"]
        M5["[0,   0,   0,   0,   0]"]
    end
    
    style Mask fill:#e3f2fd
```

**Интерпретация:**

- Строка 1 (токен на позиции 1): может обращаться только к себе ($j=1$).
- Строка 2 (токен на позиции 2): может обращаться к позициям $1, 2$ ($j \le 2$).
- Строка $i$ (токен на позиции $i$): может обращаться к позициям $1, \dots, i$ ($j \le i$).

#### 3.3. Где используется

Masked Self-Attention используется в **декодере** (первый подслой каждого слоя декодера). Он обеспечивает, чтобы при генерации $t$-го токена модель не «заглядывала» в будущее, а опиралась только на уже сгенерированные токены $y_1, \dots, y_{t-1}$.

#### 3.4. Зачем нужен Masked Self-Attention

Masked Self-Attention критически важен для:

- **Авторегрессивной генерации:** модель генерирует текст слева направо, по одному токену за раз.
- **Сохранения причинно-следственной связи:** при предсказании следующего токена модель не должна использовать информацию о нём самом (это была бы утечка данных).
- **Обучения с учителем:** при обучении модель предсказывает следующий токен на основе предыдущих, что соответствует задаче языкового моделирования.

```mermaid
flowchart LR
    subgraph Generation["Авторегрессивная генерация"]
        direction TB
        G1["Шаг 1: [START] → y₁"]
        G2["Шаг 2: [START, y₁] → y₂"]
        G3["Шаг 3: [START, y₁, y₂] → y₃"]
        G4["..."]
    end
    
    style Generation fill:#e3f2fd
```

---

### 4. Сравнительный анализ

#### 4.1. Таблица сравнения

| Характеристика | Self-Attention | Cross-Attention | Masked Self-Attention |
|----------------|----------------|-----------------|----------------------|
| **Источник Q** | Из текущей последовательности | Из декодера | Из текущей последовательности (декодер) |
| **Источник K, V** | Из той же последовательности | Из энкодера | Из той же последовательности (декодер) |
| **Маска** | Отсутствует | Отсутствует | Каузальная (запрет на будущие токены) |
| **Где используется** | Энкодер, декодер | Декодер (второй подслой) | Декодер (первый подслой) |
| **Тип контекста** | Двусторонний (полный) | От декодера к энкодеру | Односторонний (только прошлое) |
| **Задача** | Понимание внутренних связей | Связь входа и выхода | Авторегрессивная генерация |
| **Пример** | BERT, энкодер Transformer | Оригинальный Transformer | GPT, декодер Transformer |

#### 4.2. Схема трёх типов внимания

```mermaid
flowchart TD
    subgraph Types["Три типа внимания"]
        direction TB
        SA["Self-Attention<br/>Q, K, V из одного источника<br/>→ внутренние связи"]
        CA["Cross-Attention<br/>Q из одного источника,<br/>K, V из другого<br/>→ связь между источниками"]
        MA["Masked Self-Attention<br/>Q, K, V из одного источника<br/>+ каузальная маска<br/>→ авторегрессия"]
    end
    
    style Types fill:#e3f2fd
```

---

### 5. Примеры использования

#### 5.1. Self-Attention в энкодере

В энкодере BERT Self-Attention позволяет каждому токену видеть весь контекст. Например, в предложении *"Она пошла в магазин, потому что он был закрыт"* токен *"он"* может обратить внимание на *"магазин"* (антецедент), что помогает правильно разрешить анафору.

#### 5.2. Cross-Attention в декодере

В машинном переводе при генерации перевода слова *"cat"* на русский, декодер через Cross-Attention обращает внимание на соответствующий токен в английском предложении. Это позволяет правильно выбрать перевод (*"кот"*), учитывая контекст всего предложения.

#### 5.3. Masked Self-Attention в декодере

При генерации текста модель пошагово предсказывает следующий токен. Например, для начала предложения *"Я люблю"* модель предсказывает *"читать"*, используя только уже сгенерированные токены и не зная, что будет дальше. Это обеспечивает естественный, последовательный процесс генерации.

---

### 6. Заключение

Три типа внимания — Self-Attention, Cross-Attention и Masked Self-Attention — являются взаимодополняющими механизмами, которые делают архитектуру Transformer столь мощной:

- **Self-Attention** позволяет модели понимать внутренние связи в последовательности.
- **Cross-Attention** связывает энкодер и декодер, обеспечивая передачу информации от входа к выходу.
- **Masked Self-Attention** гарантирует авторегрессивность при генерации, предотвращая «подглядывание» в будущее.

Понимание этих различий является ключевым для правильной интерпретации работы Transformer и для выбора подходящей архитектуры при решении конкретных задач.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

3. Radford, A., et al. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI.  
   🔗 [https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)

## Тема 6.4. Современные вариации Multi-Head Attention

В оригинальной архитектуре Transformer используется Multi-Head Attention (MHA) с $h$ головами, каждая из которых имеет свои независимые проекции $W_i^Q, W_i^K, W_i^V$. Однако по мере роста моделей до миллиардов параметров и увеличения длины контекста до сотен тысяч токенов, классический MHA стал сталкиваться с серьёзными ограничениями по памяти и скорости инференса. В ответ на эти вызовы были разработаны новые вариации внимания, которые находят компромисс между качеством, скоростью и потреблением памяти. В этом разделе мы рассмотрим три наиболее значимых современных подхода: **Multi-Query Attention (MQA)**, **Grouped-Query Attention (GQA)** и **Flash Attention**.

---

### 1. Multi-Query Attention (MQA)

#### 1.1. Определение и принцип работы

**Multi-Query Attention (MQA)** был предложен в работе Shazeer (2019) как способ ускорить инференс авторегрессивных моделей за счёт сокращения объёма памяти, необходимой для хранения ключей (K) и значений (V) в процессе генерации.

В классическом MHA каждая голова имеет свои матрицы $W_i^K$ и $W_i^V$, что даёт $h$ различных наборов ключей и значений. В MQA все головы совместно используют **одну пару** $K$ и $V$:

$$
\mathbf{K} = \mathbf{X} \mathbf{W}^K, \quad \mathbf{V} = \mathbf{X} \mathbf{W}^V,
$$

$$
\text{head}_i = \text{Attention}\left(\mathbf{Q} \mathbf{W}_i^Q,\; \mathbf{K},\; \mathbf{V}\right).
$$

Таким образом, ключи и значения вычисляются один раз для всех голов, а каждая голова имеет свои собственные проекции только для запросов (Q).

```mermaid
flowchart TD
    subgraph MHA["Классический MHA"]
        direction TB
        Q1["Q"] --> Q1_1["W₁^Q"] --> H1["Голова 1"]
        Q1 --> Q1_2["W₂^Q"] --> H2["Голова 2"]
        Q1 --> Q1_h["W_h^Q"] --> Hh["Голова h"]
        K1["K"] --> K1_1["W₁^K"] --> H1
        K1 --> K1_2["W₂^K"] --> H2
        K1 --> K1_h["W_h^K"] --> Hh
        V1["V"] --> V1_1["W₁^V"] --> H1
        V1 --> V1_2["W₂^V"] --> H2
        V1 --> V1_h["W_h^V"] --> Hh
    end

    subgraph MQA["Multi-Query Attention (MQA)"]
        direction TB
        Q2["Q"] --> Q2_1["W₁^Q"] --> H1M["Голова 1"]
        Q2 --> Q2_2["W₂^Q"] --> H2M["Голова 2"]
        Q2 --> Q2_h["W_h^Q"] --> HhM["Голова h"]
        K2["K"] -->|общая| H1M
        K2 -->|общая| H2M
        K2 -->|общая| HhM
        V2["V"] -->|общая| H1M
        V2 -->|общая| H2M
        V2 -->|общая| HhM
    end

    style MHA fill:#f3e5f5
    style MQA fill:#e8f5e9
```

#### 1.2. Преимущества и недостатки

| Аспект | Преимущества | Недостатки |
|--------|--------------|------------|
| **Память** | Существенно меньше: $O(T \cdot d_{\text{model}})$ вместо $O(h \cdot T \cdot d_{\text{model}})$ | Может снижать качество на сложных задачах |
| **Скорость** | Быстрее за счёт меньшего числа операций | - |
| **Качество** | Хорошо работает для больших моделей | Может уступать MHA на малых моделях |
| **Инференс** | Идеален для авторегрессивной генерации | - |

#### 1.3. Где используется

- **PaLM** (Google, 2022) — 540B параметров
- **Falcon** (TII, 2023) — 7B, 40B, 180B
- **Gemini** (Google, 2023) — частично

---

### 2. Grouped-Query Attention (GQA)

#### 2.1. Определение и принцип работы

**Grouped-Query Attention (GQA)** был представлен в работе Ainslie et al. (2023) как компромисс между классическим MHA и MQA. В GQA головы разбиваются на группы, и внутри каждой группы головы совместно используют общие $K$ и $V$.

Пусть $h$ — общее число голов, а $g$ — число групп ($g < h$). Тогда:

- Каждая группа имеет свои матрицы $W_g^K$ и $W_g^V$.
- Все головы внутри группы используют одни и те же $K$ и $V$.
- Запросы (Q) остаются индивидуальными для каждой головы.

$$
\mathbf{K}_g = \mathbf{X} \mathbf{W}_g^K, \quad \mathbf{V}_g = \mathbf{X} \mathbf{W}_g^V,
$$

$$
\text{head}_i = \text{Attention}\left(\mathbf{Q} \mathbf{W}_i^Q,\; \mathbf{K}_{g(i)},\; \mathbf{V}_{g(i)}\right),
$$

где $g(i)$ — группа, к которой принадлежит голова $i$.

```mermaid
flowchart TD
    subgraph GQA["Grouped-Query Attention (GQA)"]
        direction TB
        Q3["Q"] --> Q3_1["W₁^Q"] --> H1G["Голова 1"]
        Q3 --> Q3_2["W₂^Q"] --> H2G["Голова 2"]
        Q3 --> Q3_3["W₃^Q"] --> H3G["Голова 3"]
        Q3 --> Q3_4["W₄^Q"] --> H4G["Голова 4"]
        
        K3["K"] --> K3_1["W₁^K"] --> H1G
        K3 --> K3_1 --> H2G
        K3 --> K3_2["W₂^K"] --> H3G
        K3 --> K3_2 --> H4G
        
        V3["V"] --> V3_1["W₁^V"] --> H1G
        V3 --> V3_1 --> H2G
        V3 --> V3_2["W₂^V"] --> H3G
        V3 --> V3_2 --> H4G
    end

    style GQA fill:#fff3e0
```

#### 2.2. Преимущества и недостатки

| Аспект | Преимущества | Недостатки |
|--------|--------------|------------|
| **Гибкость** | Позволяет выбирать степень сжатия через число групп $g$ | Требует настройки гиперпараметра $g$ |
| **Качество** | Близко к MHA при $g \approx h$ | Хуже MHA при малых $g$ |
| **Память** | Между MHA и MQA: $O(g \cdot T \cdot d_{\text{model}})$ | - |
| **Скорость** | Быстрее MHA, медленнее MQA | - |

#### 2.3. Где используется

- **LLaMA 3** (Meta, 2024) — 8B, 70B
- **Mistral** (Mistral AI, 2023) — 7B, 8x7B
- **Qwen 2.5** (Alibaba, 2024) — 7B, 14B, 72B

---

### 3. Flash Attention

#### 3.1. Определение и принцип работы

**Flash Attention** — это не архитектурное изменение, а **оптимизированная реализация** механизма внимания на GPU, предложенная в работе Dao et al. (2022). Основная идея заключается в переписывании вычислений внимания таким образом, чтобы минимизировать операции чтения/записи из/в медленную глобальную память GPU (HBM) и максимально использовать быструю кэш-память (SRAM).

**Ключевые техники:**

1. **Tiling (разбиение на блоки):** Матрицы Q, K, V разбиваются на блоки, которые помещаются в быструю SRAM. Это позволяет выполнять вычисления без постоянного обращения к глобальной памяти.

2. **Recomputation (перевычисление):** Вместо хранения всех промежуточных значений для обратного распространения (как в стандартном Attention), Flash Attention перевычисляет их на лету при необходимости, экономя память.

3. **Softmax с масштабированием:** Реализация softmax выполняется блочно, без необходимости хранить всю матрицу весов.

```mermaid
flowchart LR
    subgraph Standard["Стандартное внимание"]
        S1["Чтение Q, K, V из HBM"]
        S2["Вычисление S = Q·K^T"]
        S3["Запись S в HBM"]
        S4["Чтение S из HBM"]
        S5["Вычисление P = softmax(S)"]
        S6["Запись P в HBM"]
        S7["Чтение P из HBM"]
        S8["Вычисление O = P·V"]
    end

    subgraph Flash["Flash Attention"]
        F1["Tiling: блоки в SRAM"]
        F2["Вычисление S блочно"]
        F3["Softmax блочно"]
        F4["Обновление O"]
        F5["Минимум обращений к HBM"]
    end

    style Standard fill:#ffcdd2
    style Flash fill:#e8f5e9
```

#### 3.2. Преимущества

- **Скорость:** ускорение в **2-4 раза** по сравнению со стандартной реализацией (PyTorch, TensorFlow).
- **Память:** снижение потребления памяти с $O(T^2)$ до $O(T)$.
- **Точность:** сохраняет точность на уровне стандартной реализации (без приближений).
- **Масштабируемость:** позволяет работать с очень длинными последовательностями (до 128K токенов).

#### 3.3. Версии Flash Attention

| Версия | Год | Ключевые улучшения |
|--------|-----|-------------------|
| Flash Attention (v1) | 2022 | Базовая оптимизация, ускорение в 2-3 раза |
| Flash Attention (v2) | 2023 | Улучшенное использование GPU, большее ускорение |
| Flash Attention (v3) | 2024 | Поддержка новых архитектур, ещё большее ускорение |

---

### 4. Сравнительная таблица

| Характеристика | MHA | MQA | GQA | Flash Attention |
|----------------|-----|-----|-----|-----------------|
| **Тип** | Архитектурный | Архитектурный | Архитектурный | Реализационный |
| **Число K, V** | $h$ наборов | 1 набор | $g$ наборов | $h$ наборов (как MHA) |
| **Экономия памяти** | Нет | Значительная | Умеренная | Значительная |
| **Скорость инференса** | Базовая | Высокая | Средняя | Очень высокая |
| **Качество** | Эталонное | Небольшое снижение | Близко к эталону | Эталонное |
| **Сложность реализации** | Низкая | Низкая | Средняя | Высокая |
| **Поддержка длинных контекстов** | Ограничена | Умеренная | Умеренная | Отличная |
| **Где используется** | BERT, GPT-2 | PaLM, Falcon | LLaMA 3, Mistral, Qwen 2.5 | Почти все современные модели |

---

### 5. Список моделей и используемых методов

| Модель | Метод внимания | Примечания |
|--------|----------------|------------|
| **BERT** (Google, 2018) | MHA | Классический Multi-Head Attention |
| **GPT-2** (OpenAI, 2019) | MHA | - |
| **GPT-3** (OpenAI, 2020) | MHA | - |
| **PaLM** (Google, 2022) | MQA | 540B параметров |
| **Falcon** (TII, 2023) | MQA | 7B, 40B, 180B |
| **LLaMA 2** (Meta, 2023) | MHA | - |
| **Mistral 7B** (Mistral AI, 2023) | GQA | Группы по 4 головы |
| **LLaMA 3** (Meta, 2024) | GQA | 8B, 70B |
| **Qwen 2.5** (Alibaba, 2024) | GQA | 7B, 14B, 72B |
| **DeepSeek-V3** (DeepSeek, 2024) | GQA + Flash Attention | 671B MoE |
| **Gemini** (Google, 2023) | MQA (частично) | - |

---

### 6. Заключение

Современные вариации Multi-Head Attention представляют собой эволюционный ответ на вызовы масштабирования:

1. **Multi-Query Attention (MQA)** решает проблему памяти при авторегрессивной генерации за счёт совместного использования ключей и значений всеми головами.

2. **Grouped-Query Attention (GQA)** предлагает гибкий компромисс между MHA и MQA, позволяя выбирать степень сжатия через число групп.

3. **Flash Attention** — это не архитектурное изменение, а оптимизированная реализация, которая ускоряет вычисления внимания в 2-4 раза и сокращает потребление памяти, что делает возможной работу с длинными контекстами.

Эти методы часто комбинируются: например, LLaMA 3 использует GQA с оптимизированной реализацией Flash Attention. Такая комбинация позволяет современным моделям достигать выдающегося качества при эффективном использовании аппаратных ресурсов.

---

### Литература

1. Shazeer, N. (2019). *Fast Transformer Decoding: One Write-Head is All You Need*.  
   🔗 [https://arxiv.org/abs/1911.02150](https://arxiv.org/abs/1911.02150)

2. Ainslie, J., et al. (2023). *GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints*.  
   🔗 [https://arxiv.org/abs/2305.13245](https://arxiv.org/abs/2305.13245)

3. Dao, T., et al. (2022). *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*. NeurIPS.  
   🔗 [https://arxiv.org/abs/2205.14135](https://arxiv.org/abs/2205.14135)

4. Dao, T. (2023). *FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning*.  
   🔗 [https://arxiv.org/abs/2307.08691](https://arxiv.org/abs/2307.08691)

## Тема 7.1. Структура Feed-Forward Network (FFN)

После того как механизм многоголового внимания создал контекстуализированные представления токенов, эти представления проходят через **позиционно-независимую полносвязную сеть (Feed-Forward Network, FFN)**. Этот компонент, часто называемый просто FFN, является второй ключевой частью каждого слоя как энкодера, так и декодера. Несмотря на свою кажущуюся простоту, FFN выполняет критически важную функцию: он преобразует и обогащает представления, полученные от внимания, добавляя нелинейность и увеличивая выразительную способность модели. В этом разделе мы детально разберём структуру FFN, его математическую формулировку, роль в архитектуре Transformer, а также современные модификации и вариации.

---

### 1. Определение и основная формула

#### 1.1. Position-wise (позиционно-независимый) FFN

Feed-Forward Network в Transformer является **позиционно-независимым** (position-wise). Это означает, что одна и та же сеть применяется независимо к каждой позиции в последовательности. Другими словами, если на входе у нас есть матрица $\mathbf{X} \in \mathbb{R}^{T \times d_{\text{model}}}$, где $T$ — длина последовательности, то FFN применяется к каждой строке $\mathbf{x}_i$ отдельно, используя одни и те же веса для всех позиций. Это свойство принципиально отличает FFN от рекуррентных слоёв, где состояние зависит от предыдущих шагов, и делает его идеально подходящим для параллельной обработки.

Важно отметить, что FFN не является «глубокой» сетью в традиционном смысле — он содержит всего два линейных слоя. Однако именно эта простота и позиционная независимость делают FFN вычислительно эффективным и легко масштабируемым компонентом. В то время как механизм внимания обеспечивает взаимодействие между токенами, FFN обрабатывает каждый токен индивидуально, обогащая его представление независимо от других.

Математически FFN определяется как:

$$
\text{FFN}(\mathbf{x}) = \mathbf{W}_2 \cdot \sigma(\mathbf{W}_1 \cdot \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2,
$$

где каждый символ имеет следующее значение:

- $\mathbf{x} \in \mathbb{R}^{d_{\text{model}}}$ — входной вектор для одной позиции;
- $\mathbf{W}_1 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$ — матрица весов первого слоя (проекция "вверх");
- $\mathbf{b}_1 \in \mathbb{R}^{d_{\text{ff}}}$ — смещение первого слоя;
- $\sigma$ — нелинейная функция активации (в оригинале — ReLU);
- $\mathbf{W}_2 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$ — матрица весов второго слоя (проекция "вниз");
- $\mathbf{b}_2 \in \mathbb{R}^{d_{\text{model}}}$ — смещение второго слоя;
- $d_{\text{ff}}$ — размерность скрытого слоя (обычно $d_{\text{ff}} = 4 \cdot d_{\text{model}}$).

```mermaid
flowchart LR
    subgraph FFN["Position-wise Feed-Forward Network"]
        direction TB
        X["Вход: x ∈ ℝ^{d_model}"]
        W1["Linear Layer 1<br/>W₁ ∈ ℝ^{d_ff × d_model}<br/>b₁ ∈ ℝ^{d_ff}"]
        Act["Нелинейность<br/>σ (ReLU / GELU / SwiGLU)"]
        W2["Linear Layer 2<br/>W₂ ∈ ℝ^{d_model × d_ff}<br/>b₂ ∈ ℝ^{d_model}"]
        Out["Выход: FFN(x) ∈ ℝ^{d_model}"]
    end
    
    X --> W1 --> Act --> W2 --> Out
    
    style FFN fill:#e3f2fd
    style W1 fill:#f3e5f5
    style Act fill:#fff3e0
    style W2 fill:#f3e5f5
```

---

### 2. Два линейных слоя: расширение и сжатие

#### 2.1. Первый слой: расширение (d_model → d_ff)

Первый линейный слой проецирует входной вектор размерности $d_{\text{model}}$ в пространство более высокой размерности $d_{\text{ff}}$:

$$
\mathbf{h} = \mathbf{W}_1 \cdot \mathbf{x} + \mathbf{b}_1, \quad \mathbf{h} \in \mathbb{R}^{d_{\text{ff}}}.
$$

Этот этап называется **расширением** (expansion) или проекцией вверх. Зачем это нужно? Более высокое измерение позволяет модели:

- **Создавать более богатые и разнообразные представления.** В пространстве большей размерности можно выделить множество различных комбинаций признаков, которые затем будут использованы для формирования финального вектора.
- **Изучать сложные комбинации признаков.** В отличие от механизма внимания, который оперирует попарными взаимодействиями, FFN может выделять паттерны, включающие сразу несколько признаков, которые не были явно задействованы в внимании.
- **Увеличивать ёмкость сети без изменения основной размерности** $d_{\text{model}}$. Увеличение $d_{\text{ff}}$ непосредственно увеличивает число параметров FFN, что повышает его выразительную способность.

Важно понимать, что расширение размерности — это не просто формальность. Именно в этом слое происходит основное «мышление» модели: он может комбинировать признаки из разных частей входного вектора таким образом, как не позволяет сделать механизм внимания. Например, если $d_{\text{model}} = 512$, а $d_{\text{ff}} = 2048$, то первый слой увеличивает размерность в четыре раза, создавая огромное пространство возможных комбинаций.

#### 2.2. Второй слой: сжатие (d_ff → d_model)

Второй линейный слой проецирует вектор обратно в исходную размерность $d_{\text{model}}$:

$$
\text{FFN}(\mathbf{x}) = \mathbf{W}_2 \cdot \sigma(\mathbf{h}) + \mathbf{b}_2, \quad \text{FFN}(\mathbf{x}) \in \mathbb{R}^{d_{\text{model}}}.
$$

Этот этап называется **сжатием** (compression) или проекцией вниз. Он выполняет функцию «сборки»: из всего множества признаков, активированных в скрытом пространстве, второй слой выбирает те, которые наиболее важны для формирования итогового представления. Это похоже на работу декомпрессии: модель сжимает богатую информацию обратно в компактную форму, сохраняя при этом только самое существенное.

Процесс расширения и последующего сжатия напоминает работу автоэнкодеров, где информация сначала проецируется в пространство большей размерности, а затем сжимается обратно. Однако в отличие от автоэнкодеров, в FFN нет цели восстановить вход — цель состоит в том, чтобы преобразовать его в более полезное для последующих слоёв представление.

#### 2.3. Роль двух слоёв в контексте Transformer

Два линейных слоя с нелинейностью между ними позволяют FFN аппроксимировать практически любую функцию (по теореме об универсальной аппроксимации). В контексте Transformer FFN выполняет две основные функции:

**1. Обогащение представлений.** FFN может выделять важные паттерны и комбинации признаков, которые не были явно извлечены механизмом внимания. Например, если внимание выделило связь между двумя словами, FFN может использовать эту связь для изменения представления каждого из слов, добавляя новую информацию, полученную из этой связи. Это похоже на то, как человек, прочитав предложение, не просто запоминает связи между словами, но и извлекает из них новые смысловые оттенки.

**2. Хранение знаний.** В современных LLM FFN содержат бóльшую часть параметров (до 70-80%) и являются основным хранилищем фактических знаний модели. Исследования показывают, что именно в FFN хранится большая часть выученных фактов и паттернов. Например, если модель знает, что «столица Франции — Париж», эта информация, скорее всего, закодирована в весах FFN, а не в механизме внимания. Это связано с тем, что FFN имеет гораздо больше параметров, чем слои внимания, и может хранить больше информации.

```mermaid
flowchart LR
    subgraph Dimensions["Трансформация размерностей"]
        D1["Вход: d_model"]
        D2["Скрытый слой: d_ff"]
        D3["Выход: d_model"]
    end
    
    D1 -->|"W₁: d_ff × d_model"| D2
    D2 -->|"σ"| D2
    D2 -->|"W₂: d_model × d_ff"| D3
    
    style Dimensions fill:#e3f2fd
```

---

### 3. Нелинейность: ключевой компонент

#### 3.1. Зачем нужна нелинейность

Без функции активации FFN сводился бы к простому линейному преобразованию:

$$
\text{FFN}(\mathbf{x}) = \mathbf{W}_2 \cdot \mathbf{W}_1 \cdot \mathbf{x} + \dots = \mathbf{W}_{\text{eff}} \cdot \mathbf{x}.
$$

В этом случае FFN не добавлял бы никакой новой выразительности, так как композиция двух линейных преобразований сама является линейным преобразованием. Это означало бы, что весь Transformer, включая механизм внимания, оставался бы линейным, и не мог бы изучать сложные нелинейные зависимости, характерные для естественного языка.

Нелинейность $\sigma$ между слоями является тем критическим компонентом, который позволяет модели:

- **Аппроксимировать сложные функции.** Без нелинейности модель не смогла бы изучать такие явления, как грамматические правила, семантические отношения или логические рассуждения.
- **Создавать нелинейные комбинации признаков.** Нелинейность позволяет объединять признаки не просто как взвешенную сумму, а более сложным образом, что критически важно для языка.
- **Увеличивать выразительную способность** модели при сохранении того же числа параметров.

#### 3.2. Функции активации в FFN: эволюция и сравнение

В оригинальной статье использовалась **ReLU** (Rectified Linear Unit), которая до сих пор остаётся популярным выбором благодаря своей простоте и эффективности. Однако с развитием архитектур появились и другие варианты:

**ReLU** определяется как $\max(0, x)$. Её преимущества — вычислительная простота и разреженность активаций (многие нейроны «умирают», давая нулевой выход). Однако недостатком является то, что при отрицательных значениях градиент равен нулю, и нейроны могут «умереть» полностью, перестав обучаться.

**GELU** (Gaussian Error Linear Unit) определяется как $x \cdot \Phi(x)$, где $\Phi(x)$ — функция распределения стандартного нормального распределения. GELU является гладкой аппроксимацией ReLU и демонстрирует лучшие результаты на многих задачах. Она используется в BERT, GPT-2 и GPT-3. Основное преимущество — более плавные градиенты, что способствует более стабильному обучению.

**SwiGLU** (Swish-Gated Linear Unit) — современный стандарт, используемый в LLaMA, Qwen и Mistral. Формула SwiGLU включает дополнительную проекцию и выглядит как $\text{Swish}(x) \cdot (W_1 \cdot x)$, где Swish — это $x \cdot \text{sigmoid}(x)$. SwiGLU показывает лучшее качество по сравнению с ReLU и GELU, хотя требует немного больше вычислений.

| Функция | Формула | Где используется | Особенности |
|---------|---------|------------------|-------------|
| **ReLU** | $\max(0, x)$ | Оригинальный Transformer | Простая, быстрая, но может "убивать" нейроны |
| **GELU** | $x \cdot \Phi(x)$ | BERT, GPT-2, GPT-3 | Гладкая, лучше ReLU, чуть дороже |
| **SwiGLU** | $\text{Swish}(x) \cdot (W_1 \cdot x)$ | LLaMA, Qwen, Mistral | Современный стандарт, лучше качество, больше параметров |

#### 3.3. Влияние выбора активации на качество

Эмпирические исследования показывают, что замена ReLU на GELU даёт прирост качества примерно на 1-2% на бенчмарках языкового моделирования. Замена GELU на SwiGLU даёт ещё примерно 1% прироста. Хотя эти цифры могут показаться незначительными, в масштабах миллиардных моделей даже небольшое улучшение качества является существенным.

```mermaid
flowchart LR
    subgraph Activations["Функции активации"]
        direction TB
        A1["ReLU<br/>простая, быстрая"]
        A2["GELU<br/>гладкая, лучше"]
        A3["SwiGLU<br/>современная, лучшая"]
    end
    
    style Activations fill:#e3f2fd
```

---

### 4. Размеры FFN в современных моделях

Размер FFN (значение $d_{\text{ff}}$) является важным гиперпараметром, определяющим ёмкость модели. В оригинальном Transformer $d_{\text{ff}} = 4 \cdot d_{\text{model}}$, что даёт отношение 4:1. Однако современные модели часто используют другие соотношения, обычно в диапазоне от 2.5 до 4.0.

| Модель | $d_{\text{model}}$ | $d_{\text{ff}}$ | Отношение $d_{\text{ff}} / d_{\text{model}}$ |
|--------|-------------------|-----------------|----------------------------------------------|
| Transformer (base) | 512 | 2048 | 4.0 |
| Transformer (big) | 1024 | 4096 | 4.0 |
| BERT-base | 768 | 3072 | 4.0 |
| BERT-large | 1024 | 4096 | 4.0 |
| GPT-2 (small) | 768 | 3072 | 4.0 |
| GPT-3 (175B) | 12288 | 49152 | 4.0 |
| LLaMA 3 (8B) | 4096 | 14336 | 3.5 |
| LLaMA 3 (70B) | 8192 | 28672 | 3.5 |
| Qwen 2.5 (7B) | 3584 | 18944 | ~5.3 |
| Qwen 2.5 (72B) | 8192 | 29568 | 3.6 |
| Mistral 7B | 4096 | 14336 | 3.5 |

**Анализ данных:**

- **Оригинальный Transformer** использует классическое отношение 4:1, которое стало стандартом на долгое время.
- **BERT и GPT-2** также используют 4:1, что подтверждает стабильность этого выбора.
- **LLaMA 3** использует отношение 3.5:1, что немного меньше классического. Это позволяет сократить число параметров при сохранении качества.
- **Qwen 2.5 (7B)** использует необычно большое отношение ~5.3:1. Это означает, что при относительно небольшой $d_{\text{model}}$ модель имеет очень большой FFN, что может способствовать лучшему хранению знаний.
- **Qwen 2.5 (72B)** возвращается к отношению 3.6:1, что близко к LLaMA 3.

Важно отметить, что отношение $d_{\text{ff}} / d_{\text{model}}$ не является фиксированным и часто подбирается эмпирически в зависимости от размера модели, доступных вычислительных ресурсов и требований к качеству.

---

### 5. Реализация на NumPy

```python
import numpy as np

class FeedForward:
    def __init__(self, d_model, d_ff, activation='relu'):
        """
        Инициализация Position-wise Feed-Forward Network.
        
        Args:
            d_model (int): размерность модели
            d_ff (int): размерность скрытого слоя
            activation (str): тип активации ('relu', 'gelu', 'swiglu')
        """
        self.d_model = d_model
        self.d_ff = d_ff
        self.activation = activation
        
        # Инициализация весов с использованием инициализации Ксавье
        # Первый слой: d_model → d_ff
        self.W1 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_model)
        self.b1 = np.zeros(d_ff)
        
        # Второй слой: d_ff → d_model
        self.W2 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_ff)
        self.b2 = np.zeros(d_model)
    
    def gelu(self, x):
        """
        Gaussian Error Linear Unit.
        
        GELU является гладкой аппроксимацией ReLU и используется в BERT и GPT.
        Формула основана на функции распределения стандартного нормального распределения.
        """
        return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))
    
    def relu(self, x):
        """
        Rectified Linear Unit.
        
        Простая и эффективная функция активации, используемая в оригинальном Transformer.
        """
        return np.maximum(0, x)
    
    def swiglu(self, x):
        """
        SwiGLU: Swish + Gated Linear Unit.
        
        Современная функция активации, используемая в LLaMA, Qwen, Mistral.
        Требует дополнительной проекции (в упрощённой версии используем Swish(x) * x).
        """
        swish = x * (1 / (1 + np.exp(-x)))  # Sigmoid
        return swish * x
    
    def forward(self, X):
        """
        Прямой проход через FFN.
        
        Args:
            X (np.ndarray): Входная матрица (batch, T, d_model)
            
        Returns:
            np.ndarray: Выходная матрица (batch, T, d_model)
        """
        # Первый слой: расширение размерности
        h = X @ self.W1.T + self.b1  # (batch, T, d_ff)
        
        # Применение функции активации
        if self.activation == 'relu':
            h = self.relu(h)
        elif self.activation == 'gelu':
            h = self.gelu(h)
        elif self.activation == 'swiglu':
            h = self.swiglu(h)
        else:
            raise ValueError(f"Unknown activation: {self.activation}")
        
        # Второй слой: сжатие размерности
        output = h @ self.W2.T + self.b2  # (batch, T, d_model)
        
        return output

# Пример использования
if __name__ == "__main__":
    d_model = 512
    d_ff = 2048
    T = 10
    batch_size = 4
    
    ffn = FeedForward(d_model, d_ff, activation='relu')
    X = np.random.randn(batch_size, T, d_model)
    
    output = ffn.forward(X)
    print(f"Входная размерность: {X.shape}")
    print(f"Выходная размерность: {output.shape}")
```

---

### 6. Схема с размерами

```mermaid
flowchart LR
    subgraph Dimensions["Размерности FFN"]
        X["X: batch × T × d_model"]
        H["h: batch × T × d_ff"]
        Out["Out: batch × T × d_model"]
    end
    
    X -->|"Linear + σ"| H
    H -->|"Linear"| Out
    
    style Dimensions fill:#e3f2fd
```

---

### 7. Роль FFN в архитектуре Transformer

| Аспект | Роль FFN |
|--------|----------|
| **Хранение знаний** | FFN содержит 70-80% параметров модели и является основным хранилищем фактических знаний и выученных паттернов |
| **Обогащение представлений** | Преобразует выходы внимания в более сложные и богатые представления, комбинируя признаки новыми способами |
| **Нелинейность** | Добавляет нелинейность, позволяя модели аппроксимировать сложные функции, что критически важно для языка |
| **Масштабируемость** | Увеличение $d_{\text{ff}}$ позволяет модели изучать более сложные паттерны за счёт большего числа параметров |
| **Позиционная независимость** | Одна сеть для всех позиций, что сохраняет инвариантность к длине последовательности и обеспечивает параллелизм |
| **Увеличение выразительности** | Без FFN Transformer сводился бы к линейной модели, неспособной улавливать сложные зависимости |

---

### 8. Сравнение FFN в разных архитектурах

В современных LLM FFN претерпел ряд модификаций:

**1. В оригинальном Transformer** используется двухслойная сеть с ReLU и отношением $d_{\text{ff}} / d_{\text{model}} = 4$.

**2. В LLaMA и Mistral** используется SwiGLU вместо ReLU, что даёт прирост качества, но увеличивает число параметров (так как добавляется третья матрица весов).

**3. В некоторых моделях (например, Qwen) используется нестандартное отношение $d_{\text{ff}} / d_{\text{model}}$, что позволяет адаптировать модель под конкретные требования по памяти и качеству.**

**4. В последних моделях активно исследуются разреженные FFN (например, Mixture of Experts), где для каждого токена активируется только часть нейронов FFN, что позволяет значительно увеличить общее число параметров при сохранении вычислительной эффективности.**

---

### 9. Заключение

Position-wise Feed-Forward Network является важнейшим компонентом архитектуры Transformer, который:

1. **Преобразует представления**, полученные от механизма внимания, добавляя нелинейность и увеличивая выразительность модели. Без FFN Transformer был бы линейным и неспособным изучать сложные языковые паттерны.

2. **Хранит основную часть знаний** модели (до 80% параметров). Это делает FFN критическим компонентом для фактической памяти модели.

3. **Применяется независимо к каждой позиции**, что делает его вычислительно эффективным и легко масштабируемым для длинных последовательностей.

4. **Эволюционирует** от простого двухслойного перцептрона с ReLU к более сложным архитектурам с GELU и SwiGLU, что отражает постоянное стремление к улучшению качества.

Хотя FFN часто остаётся в тени многоголового внимания и позиционного кодирования, его роль критична для достижения высокого качества современных LLM. Понимание его структуры, функций и современных модификаций необходимо для правильной настройки, дообучения и интерпретации работы моделей.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Touvron, H., et al. (2023). *LLaMA: Open and Efficient Foundation Language Models*. arXiv:2302.13971.  
   🔗 [https://arxiv.org/abs/2302.13971](https://arxiv.org/abs/2302.13971)

3. Bai, J., et al. (2023). *Qwen Technical Report*. arXiv:2309.16609.  
   🔗 [https://arxiv.org/abs/2309.16609](https://arxiv.org/abs/2309.16609)

4. Hendrycks, D., & Gimpel, K. (2016). *Gaussian Error Linear Units (GELUs)*. arXiv:1606.08415.  
   🔗 [https://arxiv.org/abs/1606.08415](https://arxiv.org/abs/1606.08415)

5. Shazeer, N. (2020). *GLU Variants Improve Transformer*. arXiv:2002.05202.  
   🔗 [https://arxiv.org/abs/2002.05202](https://arxiv.org/abs/2002.05202)

## Тема 7.2. Роль и назначение Feed-Forward Network (FFN)

В предыдущем разделе мы рассмотрели структуру и математическую формулировку Feed-Forward Network. Однако понимание того, *как* устроен FFN, не даёт полного ответа на вопрос, *зачем* он нужен и какую именно роль он выполняет в архитектуре Transformer. В этом разделе мы проведём глубокий анализ функций FFN, его взаимодействия с механизмом внимания и его места в общей архитектуре модели. Мы увидим, что FFN — это не просто «дополнительный» компонент, а критически важный элемент, без которого Transformer не мог бы достичь своих впечатляющих результатов.

---

### 1. Хранение знаний: основное хранилище фактов

Одной из наиболее важных и часто недооцениваемых ролей FFN является **хранение фактических знаний** модели. Исследования показывают, что FFN содержит от 70% до 80% всех параметров модели, что делает его основным хранилищем выученных паттернов, фактов и правил.

#### 1.1. Количественный анализ параметров

Рассмотрим распределение параметров в стандартном слое Transformer. Пусть $d_{\text{model}} = 512$, $d_{\text{ff}} = 2048$. Тогда для одного слоя:

- **Multi-Head Attention:** параметры включают $W_i^Q, W_i^K, W_i^V$ для каждой головы и финальную проекцию $W^O$. При $h = 8$ головах и $d_k = d_v = 64$:
  - $W^Q, W^K, W^V$: $3 \cdot h \cdot d_{\text{model}} \cdot d_k = 3 \cdot 8 \cdot 512 \cdot 64 = 786\,432$ параметров.
  - $W^O$: $d_{\text{model}} \cdot d_{\text{model}} = 512 \cdot 512 = 262\,144$ параметров.
  - **Итого для Attention:** $\approx 1.05$ миллиона параметров.

- **FFN:** $W_1 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$ и $W_2 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$:
  - $W_1$: $d_{\text{ff}} \cdot d_{\text{model}} = 2048 \cdot 512 = 1\,048\,576$ параметров.
  - $W_2$: $d_{\text{model}} \cdot d_{\text{ff}} = 512 \cdot 2048 = 1\,048\,576$ параметров.
  - **Итого для FFN:** $\approx 2.1$ миллиона параметров, что составляет **около 67%** от всех параметров слоя.

При увеличении $d_{\text{ff}}$ (например, до $4 \cdot d_{\text{model}}$) доля FFN может достигать **70-80%** всех параметров модели.

```mermaid
flowchart LR
    subgraph Params["Распределение параметров в слое"]
        direction TB
        P1["Multi-Head Attention<br/>~33%"]
        P2["FFN<br/>~67%"]
    end
    
    style P1 fill:#f3e5f5
    style P2 fill:#e8f5e9
```

#### 1.2. FFN как хранилище фактов

Исследования показывают, что именно FFN является основным местом хранения фактических знаний модели. Например, Geva et al. (2021) в работе "Transformer Feed-Forward Layers Are Key-Value Memories" [1] показали, что FFN работают как **ключ-значение памяти (key-value memory)**:

- Первый слой FFN ($W_1$) действует как **ключи**: он определяет, какие паттерны во входном векторе активируют определённые нейроны в скрытом слое.
- Второй слой FFN ($W_2$) действует как **значения**: каждый нейрон в скрытом слое «соответствует» определённому значению, которое добавляется к выходу.
- Нелинейная функция активации (ReLU, GELU) действует как **механизм выбора**: активируются только те нейроны, которые соответствуют паттернам во входе.

**Пример:** Если модель знает факт «столица Франции — Париж», то этот факт закодирован в весах FFN. Когда модель обрабатывает предложение, содержащее «Франция», определённый нейрон в FFN активируется, и через второй слой добавляется информация о «Париже».

```mermaid
flowchart LR
    subgraph Memory["FFN как ключ-значение память"]
        direction TB
        Input["Вход: 'Франция'"]
        K["Ключи (W₁)<br/>активация нейронов"]
        Act["Нелинейность<br/>выбор активных нейронов"]
        V["Значения (W₂)<br/>добавление информации"]
        Out["Выход: + 'Париж'"]
    end
    
    Input --> K --> Act --> V --> Out
    
    style Memory fill:#e3f2fd
```

#### 1.3. Эффективность хранения в FFN

Почему именно FFN, а не механизм внимания, является основным хранилищем знаний? Причина в том, что:

1. **Большая ёмкость:** FFN содержит значительно больше параметров, чем внимание.
2. **Статичность:** Внимание динамически вычисляет веса на основе текущего входа, но не хранит информацию в весах. FFN, напротив, хранит информацию статически в своих весах.
3. **Инвариантность к длине:** FFN применяется к каждой позиции независимо, что делает его идеальным для хранения фактов, которые не зависят от позиции.

---

### 2. Преобразование представлений: от внимания к новым паттернам

Вторая ключевая роль FFN — **преобразование и обогащение представлений**, полученных от механизма внимания.

#### 2.1. Обработка информации от внимания

После того как механизм многоголового внимания создал контекстуализированные представления каждого токена (учитывающие взаимодействия с другими токенами), эти представления поступают в FFN. FFN выполняет следующие функции:

- **Извлечение паттернов:** FFN выделяет важные комбинации признаков, которые не были явно извлечены вниманием. Например, если внимание показало, что слово «кот» связано со словом «молоко», FFN может использовать эту информацию для изменения представления слова «кот», добавив признаки, связанные с «молоком».
- **Трансформация:** FFN преобразует представления в новое пространство, где они становятся более «удобными» для последующих слоёв.

#### 2.2. Извлечение паттернов

В отличие от внимания, которое оперирует попарными взаимодействиями между токенами, FFN может выделять **паттерны, включающие одновременно несколько признаков**. Например:

- **Синтаксические паттерны:** FFN может распознавать, что слово является прилагательным, которое должно быть связано с существительным.
- **Семантические паттерны:** FFN может выявлять, что группа слов относится к одной тематической категории.
- **Логические паттерны:** FFN может выделять правила, такие как «если $A$, то $B$».

```mermaid
flowchart LR
    subgraph Patterns["Извлечение паттернов FFN"]
        direction TB
        P1["Вход: контекстуализированный вектор"]
        P2["FFN: извлечение паттернов"]
        P3["Пример: выделение синтаксической роли"]
        P4["Пример: выделение семантической категории"]
    end
    
    P1 --> P2 --> P3
    P2 --> P4
    
    style Patterns fill:#e3f2fd
```

#### 2.3. Трансформация в новые представления

FFN не просто извлекает паттерны, но и **трансформирует** входные представления в новые, которые затем передаются в следующий слой. Эта трансформация может включать:

- **Усиление важных признаков:** FFN может увеличивать значение тех измерений, которые важны для текущей задачи.
- **Подавление шума:** FFN может уменьшать влияние нерелевантных признаков.
- **Создание новых признаков:** FFN может генерировать новые комбинации признаков, которых не было во входе.

---

### 3. Добавление нелинейности: выразительность модели

Без нелинейности, вносимой FFN, Transformer сводился бы к линейной модели, неспособной изучать сложные зависимости, характерные для естественного языка.

#### 3.1. Почему нелинейность критична

Как мы уже отмечали, композиция двух линейных преобразований (без нелинейности) сама является линейным преобразованием. Это означало бы, что:

- Вся модель была бы линейной функцией от входных данных.
- Модель не могла бы изучать нелинейные отношения (например, логические операции, синтаксические правила, семантические связи).
- Качество модели было бы значительно ниже.

Нелинейность $\sigma$ между слоями FFN позволяет модели аппроксимировать **любую непрерывную функцию** (теорема об универсальной аппроксимации), что является необходимым условием для обработки естественного языка.

#### 3.2. Математическое обоснование

Рассмотрим FFN с одной нелинейностью:

$$
\text{FFN}(\mathbf{x}) = W_2 \cdot \sigma(W_1 \cdot \mathbf{x} + b_1) + b_2.
$$

Если $\sigma$ — нелинейная функция (например, ReLU, GELU), то FFN может аппроксимировать любую непрерывную функцию на компактном множестве при достаточно большой размерности $d_{\text{ff}}$ (теорема о представлении). Это делает FFN мощным инструментом для изучения сложных языковых паттернов.

---

### 4. Взаимодействие с вниманием: дополнение, а не дублирование

Одним из ключевых архитектурных решений Transformer является разделение функций между механизмом внимания и FFN. Эти два компонента **дополняют друг друга**, выполняя разные, но взаимодополняющие задачи.

#### 4.1. Внимание: взаимодействие между токенами

Механизм внимания отвечает за **взаимодействие между токенами**:

- Вычисляет, насколько каждый токен должен «обращать внимание» на другие токены.
- Создаёт контекстуализированные представления, учитывающие контекст всего предложения.
- Обрабатывает дальние зависимости с постоянной сложностью $O(1)$ на пару токенов.

**Пример:** В предложении «Человек не мог перейти дорогу, так как он был ранен» внимание устанавливает связь между «он» и «Человек», позволяя модели правильно разрешить анафору.

#### 4.2. FFN: преобразование каждого токена

FFN, в отличие от внимания, **работает с каждым токеном независимо**:

- Преобразует представление каждого токена, обогащая его новыми признаками.
- Хранит фактические знания и паттерны.
- Добавляет нелинейность и выразительность.

**Пример:** После того как внимание связало «он» с «Человек», FFN может использовать эту информацию для изменения представления «он», добавив признаки, связанные с «Человек» (например, мужской род, одушевлённость).

#### 4.3. Сравнительная таблица

| Аспект | Multi-Head Attention | Feed-Forward Network |
|--------|---------------------|---------------------|
| **Операция** | Взаимодействие между токенами | Преобразование каждого токена независимо |
| **Параметры** | ~33% всех параметров слоя | ~67% всех параметров слоя |
| **Функция** | Создание контекстуализированных представлений | Обогащение представлений, хранение знаний |
| **Сложность** | $O(T^2 \cdot d)$ | $O(T \cdot d^2)$ |
| **Нелинейность** | Отсутствует (содержит softmax, но не обучаемую нелинейность) | Содержит обучаемую нелинейность (ReLU/GELU) |
| **Зависимость от позиции** | Да (через позиционное кодирование) | Нет (позиционно-независимый) |
| **Где используется** | Энкодер (Self-Attention), декодер (Self-Attention + Cross-Attention) | Энкодер и декодер |

#### 4.4. Совместная работа

В каждом слое энкодера или декодера происходит следующая последовательность:

1. **Внимание:** Обрабатывает взаимодействия между токенами, создавая контекстуализированные представления.
2. **Остаточная связь + нормализация:** Стабилизирует обучение и сохраняет информацию.
3. **FFN:** Преобразует и обогащает представления, добавляя нелинейность и внося фактическую информацию.
4. **Остаточная связь + нормализация:** Стабилизирует обучение и сохраняет информацию.

```mermaid
flowchart LR
    subgraph Layer["Слой энкодера"]
        direction TB
        Input["Вход: X"]
        Attn["Multi-Head Attention<br/>взаимодействие между токенами"]
        AddNorm1["Add & LayerNorm"]
        FFN["Feed-Forward Network<br/>преобразование каждого токена"]
        AddNorm2["Add & LayerNorm"]
        Output["Выход: X'"]
    end
    
    Input --> Attn --> AddNorm1 --> FFN --> AddNorm2 --> Output
    
    style Layer fill:#e3f2fd
```

---

### 5. Заключение

Feed-Forward Network является не просто «дополнительным» компонентом Transformer, а выполняет три критически важные функции:

1. **Хранение знаний:** FFN содержит основную часть параметров модели (70-80%) и является основным хранилищем фактических знаний и выученных паттернов.

2. **Преобразование представлений:** FFN обрабатывает информацию от внимания, извлекает паттерны и трансформирует представления в более богатые и полезные для последующих слоёв.

3. **Добавление нелинейности:** FFN вносит критически важную нелинейность, без которой модель была бы линейной и неспособной изучать сложные языковые зависимости.

Вместе с механизмом внимания, FFN образует мощный дуэт: внимание обеспечивает взаимодействие между токенами и создаёт контекстуализированные представления, а FFN преобразует и обогащает эти представления, добавляя фактическую информацию и нелинейность. Без FFN Transformer был бы значительно менее выразительным и не мог бы достичь тех впечатляющих результатов, которые мы наблюдаем в современных LLM.

Понимание роли FFN критически важно для правильной настройки, интерпретации и дообучения моделей. Например, при тонкой настройке часто именно FFN требует наибольшего внимания (и ресурсов), так как именно в нём хранится основная часть знаний, которые мы хотим адаптировать под конкретную задачу.

---

### Литература

1. Geva, M., et al. (2021). *Transformer Feed-Forward Layers Are Key-Value Memories*. EMNLP.  
   🔗 [https://arxiv.org/abs/2012.14913](https://arxiv.org/abs/2012.14913)

2. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

3. Dalvi, F., et al. (2020). *What is One Grain of Sand in the Desert? Analyzing Individual Neurons in Deep NLP Models*. AAAI.  
   🔗 [https://arxiv.org/abs/1909.05910](https://arxiv.org/abs/1909.05910)

## Тема 7.3. Функции активации в Feed-Forward Network

Функция активации является ключевым компонентом Feed-Forward Network, определяющим его выразительную способность и поведение. В оригинальной статье Transformer использовалась простая функция ReLU, однако с развитием архитектур появились более совершенные альтернативы, которые значительно улучшили качество моделей. В этом разделе мы детально рассмотрим три основных функции активации, используемые в FFN современных LLM: **ReLU**, **GELU** и **SwiGLU**.

---

### 1. ReLU (Rectified Linear Unit)

#### 1.1. Определение и формула

**ReLU** (Rectified Linear Unit) — это простая, но эффективная функция активации, которая была использована в оригинальной статье Transformer. Она определяется как:

$$
\text{ReLU}(x) = \max(0, x) =
\begin{cases}
x, & \text{если } x > 0, \\
0, & \text{если } x \leq 0.
\end{cases}
$$

График ReLU представляет собой две прямые: нулевую для отрицательных значений и линейную (с коэффициентом 1) для положительных. В точке $x = 0$ функция имеет излом, но не разрыв.

#### 1.2. Математические свойства

- **Простота:** ReLU не содержит экспонент, делений или других сложных операций, что делает её очень быстрой в вычислении.
- **Нелинейность:** Хотя ReLU кусочно-линейна, она не является линейной функцией, так как имеет излом в нуле. Это обеспечивает необходимую нелинейность модели.
- **Разреженность активаций:** Для отрицательных входов ReLU даёт нулевой выход, что приводит к разреженности активаций (много нейронов «молчат»). Это может способствовать лучшей обобщающей способности.
- **Отсутствие насыщения:** Для положительных значений производная равна 1, что предотвращает исчезающие градиенты (проблема, характерная для сигмоиды и гиперболического тангенса).

#### 1.3. Преимущества ReLU

| Преимущество | Описание |
|--------------|----------|
| **Вычислительная эффективность** | Не требует сложных математических операций, быстрая на GPU |
| **Предотвращение исчезающих градиентов** | Производная равна 1 для положительных значений |
| **Разреженность** | Многие нейроны дают нулевой выход, что может улучшать обобщение |
| **Простота реализации** | Легко реализуется в любом фреймворке |

#### 1.4. Проблема «умирающих нейронов» (Dying ReLU)

Основной недостаток ReLU заключается в том, что нейроны могут «умереть» — то есть перестать активироваться для всех входов. Это происходит, когда нейрон получает отрицательный вход в течение длительного времени, и его вес обновляется так, что он остаётся отрицательным для всех последующих примеров. В этом случае градиент равен нулю, и нейрон больше не обучается.

Математически это можно выразить так: если $x < 0$, то $\text{ReLU}(x) = 0$ и производная $\text{ReLU}'(x) = 0$. Если нейрон попадает в эту область для всех обучающих примеров, он перестаёт обновляться.

**Варианты решения проблемы:**
- **Leaky ReLU:** $\text{LeakyReLU}(x) = \max(0.01x, x)$ — позволяет небольшой градиент для отрицательных значений.
- **Parametric ReLU (PReLU):** $\text{PReLU}(x) = \max(\alpha x, x)$, где $\alpha$ — обучаемый параметр.

Однако в Transformer проблема «умирающих нейронов» менее критична благодаря комбинации с остаточными связями и нормализацией.

#### 1.5. Где используется ReLU

- **Оригинальный Transformer** (Vaswani et al., 2017) — классическая архитектура.
- **Многие ранние модели** — до появления GELU и SwiGLU.

---

### 2. GELU (Gaussian Error Linear Unit)

#### 2.1. Определение и формула

**GELU** (Gaussian Error Linear Unit) была предложена Hendrycks и Gimpel (2016) как более гладкая альтернатива ReLU. Она определяется как:

$$
\text{GELU}(x) = x \cdot \Phi(x),
$$

где $\Phi(x)$ — функция распределения стандартного нормального распределения:

$$
\Phi(x) = \frac{1}{2} \left(1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right).
$$

Здесь $\text{erf}$ — функция ошибок (error function). В практических реализациях GELU часто аппроксимируется как:

$$
\text{GELU}(x) \approx 0.5 \cdot x \cdot \left(1 + \tanh\left(\sqrt{\frac{2}{\pi}} \cdot \left(x + 0.044715 \cdot x^3\right)\right)\right).
$$

Это приближение используется в большинстве фреймворков для ускорения вычислений.

#### 2.2. Интуиция и свойства

GELU можно интерпретировать как «стохастическую» версию ReLU: вместо того чтобы просто обнулять отрицательные значения, GELU «взвешивает» вход по вероятности того, что он положителен в нормальном распределении. Математически:

- Для больших положительных $x$: $\Phi(x) \approx 1$, и $\text{GELU}(x) \approx x$.
- Для больших отрицательных $x$: $\Phi(x) \approx 0$, и $\text{GELU}(x) \approx 0$.
- В окрестности нуля: GELU имеет гладкий переход, в отличие от излома ReLU.

**Ключевые свойства:**

| Свойство | Описание |
|----------|----------|
| **Гладкость** | Не имеет излома, что улучшает поведение градиентов |
| **Вероятностная интерпретация** | Связана с функцией распределения нормального распределения |
| **Нелинейность** | Обеспечивает необходимую нелинейность модели |
| **Нет «умирающих» нейронов** | Для отрицательных значений даёт малые, но ненулевые значения |

#### 2.3. Преимущества GELU перед ReLU

1. **Гладкость:** GELU не имеет излома в нуле, что приводит к более стабильному поведению градиентов и лучшей сходимости.
2. **Лучшее качество:** Эмпирические исследования показывают, что GELU даёт лучшее качество на многих задачах, включая языковое моделирование.
3. **Более информативные отрицательные значения:** В отличие от ReLU, GELU не обнуляет отрицательные значения полностью, а масштабирует их, сохраняя часть информации.

#### 2.4. Где используется GELU

- **BERT** (Devlin et al., 2018) — во всех версиях BERT используется GELU.
- **GPT-2** (Radford et al., 2019) и **GPT-3** (Brown et al., 2020).
- **RoBERTa** (Liu et al., 2019) — улучшенная версия BERT.
- **ELECTRA** (Clark et al., 2020).

---

### 3. SwiGLU (Swish-Gated Linear Unit)

#### 3.1. Определение и формула

**SwiGLU** (Swish-Gated Linear Unit) — это современная функция активации, предложенная в работе Shazeer (2020) «GLU Variants Improve Transformer». Она объединяет идеи **GLU (Gated Linear Unit)** и **Swish** (ранее известной как SiLU).

Стандартный GLU определяется как:

$$
\text{GLU}(x) = \sigma(W_1 \cdot x) \odot (W_2 \cdot x),
$$

где $\sigma$ — сигмоида, а $\odot$ — поэлементное умножение.

**SwiGLU** заменяет сигмоиду на Swish:

$$
\text{SwiGLU}(x) = \text{Swish}(W_1 \cdot x) \odot (W_2 \cdot x) = (W_1 \cdot x) \cdot \sigma(W_1 \cdot x) \odot (W_2 \cdot x).
$$

В контексте Transformer FFN с SwiGLU записывается как:

$$
\text{FFN}_{\text{SwiGLU}}(x) = \text{Swish}(xW_1) \odot (xW_2),
$$

где $W_1, W_2 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$. Обратите внимание, что SwiGLU требует **две** матрицы проекций для первого слоя (в отличие от одной в ReLU/GELU), что увеличивает число параметров.

#### 3.2. Сравнение с ReLU и GELU

| Характеристика | ReLU | GELU | SwiGLU |
|----------------|------|------|--------|
| **Формула** | $\max(0, x)$ | $x \cdot \Phi(x)$ | $\text{Swish}(x) \odot (W_2 \cdot x)$ |
| **Число матриц $W$** | 1 ($W_1$) | 1 ($W_1$) | 2 ($W_1, W_2$) |
| **Параметры** | $d_{\text{ff}} \cdot d_{\text{model}}$ | $d_{\text{ff}} \cdot d_{\text{model}}$ | $2 \cdot d_{\text{ff}} \cdot d_{\text{model}}$ |
| **Гладкость** | Нет (излом в 0) | Да | Да |
| **Качество** | Базовое | Лучше ReLU | Лучше GELU |
| **Скорость** | Самая быстрая | Средняя | Медленнее (больше параметров) |
| **Использование** | Оригинальный Transformer | BERT, GPT | LLaMA, Qwen, Mistral |

#### 3.3. Преимущества SwiGLU

1. **Лучшее качество:** SwiGLU показывает стабильное улучшение качества по сравнению с ReLU и GELU на различных задачах. В работе Shazeer (2020) показано, что замена ReLU на SwiGLU даёт прирост качества, сопоставимый с увеличением размера модели.

2. **Гладкость:** Как и GELU, SwiGLU является гладкой функцией, что способствует стабильному обучению.

3. **Гейтинг (Gating):** Механизм «ворот» (gating) позволяет модели избирательно пропускать информацию, что аналогично механизмам в LSTM и GRU.

4. **Масштабируемость:** Несмотря на увеличение числа параметров, SwiGLU хорошо масштабируется на больших моделях.

#### 3.4. Где используется SwiGLU

- **LLaMA** (Meta, 2023, 2024) — все версии LLaMA (1, 2, 3) используют SwiGLU.
- **Qwen** (Alibaba, 2023, 2024) — Qwen 1.5, Qwen 2.5.
- **Mistral** (Mistral AI, 2023) — Mistral 7B, Mixtral 8x7B.
- **Gemma** (Google, 2024) — лёгкие модели.

---

### 4. Сравнительный анализ

#### 4.1. Таблица сравнения функций активации

| Функция | Формула | Производная | Параметры | Гладкость | Качество | Скорость |
|---------|---------|-------------|-----------|-----------|----------|----------|
| **ReLU** | $\max(0, x)$ | $1$ при $x>0$, $0$ при $x<0$ | $d_{\text{ff}} \cdot d_{\text{model}}$ | Нет (излом) | Базовое | Самая быстрая |
| **GELU** | $x \cdot \Phi(x)$ | $\Phi(x) + x \cdot \phi(x)$ | $d_{\text{ff}} \cdot d_{\text{model}}$ | Да | Лучше ReLU | Средняя |
| **SwiGLU** | $\text{Swish}(xW_1) \odot (xW_2)$ | Сложная | $2 \cdot d_{\text{ff}} \cdot d_{\text{model}}$ | Да | Лучше GELU | Медленнее |

#### 4.2. Графики функций активации

```mermaid
flowchart LR
    subgraph Plots["Графики функций активации"]
        direction TB
        P1["ReLU<br/>-4:0, 0:0, 4:4"]
        P2["GELU<br/>-4:-0.1, 0:0, 4:3.9"]
        P3["SwiGLU<br/>-4:-0.8, 0:0, 4:4.2"]
    end
    
    style Plots fill:#e3f2fd
```

**Анализ графиков:**

- **ReLU:** Имеет резкий излом в нуле. Для отрицательных значений — ноль, для положительных — линейный рост.
- **GELU:** Плавный переход через ноль. Для отрицательных значений даёт небольшие отрицательные значения (не обнуляет полностью).
- **SwiGLU:** Имеет более сложную форму с отрицательным «провалом» при отрицательных значениях и сверхлинейным ростом при положительных.

#### 4.3. Рекомендации по выбору

| Сценарий | Рекомендуемая функция | Причина |
|----------|----------------------|---------|
| **Быстрое прототипирование** | ReLU | Простота и скорость |
| **Качество важнее скорости** | SwiGLU | Лучшее качество |
| **Баланс качества и скорости** | GELU | Хороший компромисс |
| **Ограниченные ресурсы** | ReLU | Меньше параметров |
| **Большие модели (>7B)** | SwiGLU | Лучшее качество, оправдывает дополнительные параметры |

---

### 5. Эмпирическое сравнение

В работе Shazeer (2020) «GLU Variants Improve Transformer» было проведено систематическое сравнение различных функций активации в Transformer. Основные выводы:

1. **SwiGLU стабильно превосходит ReLU** на всех протестированных задачах (машинный перевод, языковое моделирование, суммаризация).

2. **GELU показывает результаты между ReLU и SwiGLU**, что делает его хорошим компромиссным выбором.

3. **Увеличение числа параметров** в SwiGLU (в два раза по сравнению с ReLU) оправдано улучшением качества.

4. **Влияние на скорость обучения:** модели с SwiGLU могут требовать больше времени для обучения, но дают лучшее конечное качество.

```mermaid
flowchart LR
    subgraph Comparison["Сравнение качества"]
        direction TB
        C1["ReLU: baseline"]
        C2["GELU: +1-2%"]
        C3["SwiGLU: +2-4%"]
    end
    
    C1 --> C2 --> C3
    
    style Comparison fill:#e3f2fd
```

---

### 6. Заключение

Функции активации в Feed-Forward Network прошли эволюцию от простой ReLU до более сложных и выразительных GELU и SwiGLU. Каждая из них имеет свои преимущества и ограничения:

- **ReLU:** Простая, быстрая, но страдает от проблемы «умирающих нейронов» и имеет излом.
- **GELU:** Гладкая, даёт лучшее качество, используется в BERT и GPT.
- **SwiGLU:** Современный стандарт, даёт наилучшее качество за счёт дополнительных параметров, используется в LLaMA, Qwen, Mistral.

Выбор функции активации зависит от конкретных требований проекта: если критична скорость, выбирают ReLU; если важнее качество, предпочитают SwiGLU; GELU является хорошим компромиссом. В современных LLM с миллиардами параметров SwiGLU становится стандартом де-факто, несмотря на увеличение числа параметров.

---

### Литература

1. Hendrycks, D., & Gimpel, K. (2016). *Gaussian Error Linear Units (GELUs)*. arXiv:1606.08415.  
   🔗 [https://arxiv.org/abs/1606.08415](https://arxiv.org/abs/1606.08415)

2. Shazeer, N. (2020). *GLU Variants Improve Transformer*. arXiv:2002.05202.  
   🔗 [https://arxiv.org/abs/2002.05202](https://arxiv.org/abs/2002.05202)

3. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

4. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

5. Touvron, H., et al. (2023). *LLaMA: Open and Efficient Foundation Language Models*. arXiv:2302.13971.  
   🔗 [https://arxiv.org/abs/2302.13971](https://arxiv.org/abs/2302.13971)

## Тема 8.1. Проблема глубоких сетей и затухающие градиенты

Одним из наиболее фундаментальных вызовов при обучении глубоких нейронных сетей является проблема затухающих (vanishing) и взрывающихся (exploding) градиентов. Эта проблема становится особенно острой при увеличении числа слоёв — именно то, что требуется для создания мощных моделей, способных улавливать сложные закономерности в данных. В этом разделе мы рассмотрим математическую природу этой проблемы, её влияние на обучение и историю решений, которые привели к созданию архитектур, способных эффективно обучаться на сотнях слоёв.

---

### 1. Математическая формулировка проблемы

#### 1.1. Распространение градиентов в глубоких сетях

Рассмотрим глубокую нейронную сеть с $L$ слоями. Пусть $x_l$ — выход $l$-го слоя, а $F_l$ — функция, реализуемая этим слоем. Тогда:

$$
x_{l+1} = F_l(x_l, \theta_l),
$$

где $\theta_l$ — параметры слоя $l$. Для обучения сети методом обратного распространения ошибки (backpropagation) необходимо вычислить градиент функции потерь $\mathcal{L}$ по параметрам каждого слоя. Ключевым компонентом является распространение градиента от выходного слоя к входному:

$$
\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \prod_{k=l}^{L-1} \frac{\partial x_{k+1}}{\partial x_k} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \prod_{k=l}^{L-1} J_k,
$$

где $J_k = \partial F_k / \partial x_k$ — якобиан $k$-го слоя по входу.

#### 1.2. Условие затухания градиентов

Если для всех слоёв $k$ спектральный радиус якобиана $\rho(J_k) < 1$ (то есть все собственные значения по модулю меньше 1), то произведение таких якобианов экспоненциально убывает с ростом числа слоёв:

$$
\left\| \prod_{k=l}^{L-1} J_k \right\| \leq \prod_{k=l}^{L-1} \|J_k\| \approx \gamma^{L-l},
$$

где $\gamma < 1$. Это означает, что градиент на ранних слоях становится экспоненциально малым:

$$
\left\| \frac{\partial \mathcal{L}}{\partial x_l} \right\| \approx \gamma^{L-l} \cdot \left\| \frac{\partial \mathcal{L}}{\partial x_L} \right\|.
$$

Для сети с 100 слоями и $\gamma = 0.9$, градиент на первом слое будет в $0.9^{99} \approx 3 \cdot 10^{-5}$ раз меньше, чем на последнем. Такие малые градиенты не могут эффективно обновлять веса, и обучение практически останавливается.

#### 1.3. Условие взрывающихся градиентов

Аналогично, если $\rho(J_k) > 1$, то градиенты экспоненциально растут, что приводит к численной нестабильности и переполнению значений.

```mermaid
flowchart LR
    subgraph Problem["Проблема затухающих градиентов"]
        direction TB
        G1["Градиент у выхода: 1.0"]
        G2["После слоя L-1: 0.9"]
        G3["После слоя L-2: 0.81"]
        G4["..."]
        GL["После слоя 1: ~0"]
    end
    
    G1 -->|"×0.9"| G2 -->|"×0.9"| G3 -->|"×0.9"| G4 -->|"×0.9"| GL
    
    style Problem fill:#ffcdd2
```

---

### 2. Влияние на обучение

#### 2.1. Медленная сходимость

Когда градиенты на ранних слоях становятся очень малыми, эти слои практически не обучаются. Это приводит к тому, что:

- Модель не может использовать полную выразительную способность своей архитектуры.
- Обучение требует значительно большего числа итераций для достижения приемлемого качества.
- В некоторых случаях обучение может вообще не сходиться.

#### 2.2. Плохие локальные минимумы

Малые градиенты могут приводить к тому, что модель «застревает» в плохих локальных минимумах или на плато, из которых не может выбраться из-за недостаточной величины обновлений.

#### 2.3. Практические ограничения

До появления эффективных решений, проблема затухающих градиентов ограничивала глубину нейронных сетей примерно 10-20 слоями для обычных полносвязных сетей и до 100 слоёв для LSTM (благодаря механизмам ворот). Это было недостаточно для обработки сложных языковых структур.

---

### 3. История решений проблемы

#### 3.1. Ранние подходы

**Инициализация весов:** Одним из первых решений стала правильная инициализация весов. Инициализация Ксавье (Xavier) и инициализация Хе (He) обеспечивают, что дисперсия сигналов сохраняется при прохождении через слои, что замедляет затухание градиентов.

**Функции активации:** Замена сигмоиды и гиперболического тангенса на ReLU также помогла, так как производная ReLU равна 1 для положительных значений, что предотвращает затухание.

#### 3.2. Batch Normalization (2015)

Batch Normalization [1] была предложена как способ нормализации активаций внутри слоя. Она вычисляет среднее и дисперсию по батчу и нормализует активации:

$$
\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}},
$$

где $\mu_B$ и $\sigma_B^2$ — среднее и дисперсия по батчу. Это позволяет:

- Стабилизировать распределение активаций.
- Уменьшить внутренний ковариатный сдвиг.
- Использовать более высокие learning rates.
- Обеспечить некоторую регуляризацию.

Batch Normalization стала стандартным компонентом многих архитектур, но имеет ограничения для рекуррентных сетей и последовательных данных.

#### 3.3. Остаточные связи (ResNet, 2016)

Революционным прорывом стало введение **остаточных связей (residual connections)** в работе He et al. (2016) [2]. Основная идея заключается в том, чтобы добавить вход слоя к его выходу:

$$
x_{l+1} = x_l + F_l(x_l, \theta_l).
$$

Это кардинально меняет распространение градиентов:

$$
\frac{\partial x_{l+1}}{\partial x_l} = I + \frac{\partial F_l}{\partial x_l}.
$$

Теперь якобиан всегда содержит единичную матрицу $I$, которая обеспечивает «сквозной» проход градиентов. Произведение якобианов становится:

$$
\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \prod_{k=l}^{L-1} \left( I + \frac{\partial F_k}{\partial x_k} \right).
$$

Даже если $\| \partial F_k / \partial x_k \|$ мало, единичная матрица обеспечивает, что градиент не затухает полностью. Это позволило обучать сети с сотнями и даже тысячами слоёв.

#### 3.4. Применение в Transformer

В архитектуре Transformer остаточные связи применяются вокруг каждого подслоя (Self-Attention и FFN):

- В энкодере: $x_{out} = x_{in} + \text{Sublayer}(x_{in})$
- В декодере: аналогично.

Это позволило обучать модели с 6 слоями (оригинальный Transformer), а затем с 12 слоями (BERT-base), 24 слоями (BERT-large), 32 слоями (GPT-3) и более. Без остаточных связей такое масштабирование было бы невозможно.

```mermaid
flowchart LR
    subgraph Residual["Остаточная связь"]
        direction TB
        Input["x"]
        Sub["Sublayer(x)"]
        Add["+"]
        Output["x + Sublayer(x)"]
    end
    
    Input --> Sub
    Input --> Add
    Sub --> Add
    Add --> Output
    
    style Residual fill:#e8f5e9
```

---

### 4. Математический анализ остаточных связей

#### 4.1. Формальное доказательство

Пусть $F_l$ — функция, реализуемая $l$-м слоем (включая нормализацию). С остаточной связью:

$$
x_{l+1} = x_l + F_l(x_l).
$$

Градиент по входу $x_l$:

$$
\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_{l+1}} \cdot \frac{\partial x_{l+1}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_{l+1}} \cdot \left( I + \frac{\partial F_l}{\partial x_l} \right).
$$

Раскрывая рекурсивно:

$$
\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} + \sum_{k=l}^{L-1} \frac{\partial \mathcal{L}}{\partial x_{k+1}} \cdot \frac{\partial F_k}{\partial x_k} \cdot \prod_{m=k+1}^{L-1} \left( I + \frac{\partial F_m}{\partial x_m} \right).
$$

Ключевое наблюдение: градиент содержит **прямой член** $\partial \mathcal{L} / \partial x_L$, который не затухает. Даже если все $\partial F_k / \partial x_k$ равны нулю, градиент всё равно будет равен $\partial \mathcal{L} / \partial x_L$. Это гарантирует, что информация из выходного слоя достигает всех слоёв.

#### 4.2. Интерпретация

Остаточные связи можно интерпретировать как **ансамбль путей** (ensemble of paths) разной длины. Модель имеет прямой путь от входа к выходу (через остаточные связи) и множество более длинных путей (через слои). Это делает обучение более устойчивым и позволяет использовать очень глубокие архитектуры.

---

### 5. Заключение

Проблема затухающих градиентов была одним из главных препятствий на пути создания глубоких нейронных сетей. Без её решения невозможно было бы создать современные LLM с сотнями слоёв и миллиардами параметров.

Ключевые решения, которые позволили преодолеть эту проблему:

1. **Правильная инициализация весов** — обеспечивает стабильность на начальных этапах.
2. **Функции активации (ReLU, GELU)** — предотвращают насыщение и затухание.
3. **Batch Normalization** — стабилизирует распределение активаций.
4. **Остаточные связи (Residual Connections)** — обеспечивают «сквозной» проход градиентов, позволяя обучать модели произвольной глубины.

В архитектуре Transformer остаточные связи являются критическим компонентом, без которого модель не могла бы эффективно обучаться даже с 6 слоями, не говоря уже о современных моделях с 96 слоями (GPT-3) или более. Понимание этой проблемы и её решений является необходимым для осознания того, как работают современные глубокие сети.

---

### Литература

1. Ioffe, S., & Szegedy, C. (2015). *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift*. ICML.  
   🔗 [https://arxiv.org/abs/1502.03167](https://arxiv.org/abs/1502.03167)

2. He, K., Zhang, X., Ren, S., & Sun, J. (2016). *Deep Residual Learning for Image Recognition*. CVPR.  
   🔗 [https://arxiv.org/abs/1512.03385](https://arxiv.org/abs/1512.03385)

3. Pascanu, R., Mikolov, T., & Bengio, Y. (2013). *On the difficulty of training recurrent neural networks*. ICML.  
   🔗 [https://arxiv.org/abs/1211.5063](https://arxiv.org/abs/1211.5063)

4. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

## Тема 8.2. Остаточные связи (Residual Connections)

В предыдущем разделе мы рассмотрели проблему затухающих градиентов в глубоких сетях и обсудили, как остаточные связи стали ключевым решением этой фундаментальной проблемы. В этом разделе мы детально разберём механизм остаточных связей в архитектуре Transformer, их математическое обоснование и практическое применение. Остаточные связи являются одним из критических компонентов, позволивших Transformer стать доминирующей архитектурой в области обработки естественного языка и глубокого обучения в целом.

---

### 1. Основная идея остаточных связей

#### 1.1. Определение и формулировка

Остаточная связь (residual connection, также называемая skip connection) — это механизм, при котором вход подслоя добавляется к его выходу. Математически это выражается как:

$$
\mathbf{x}_{out} = \mathbf{x}_{in} + \text{Sublayer}(\mathbf{x}_{in}),
$$

где $\mathbf{x}_{in}$ — входной вектор, $\text{Sublayer}$ — функция, реализуемая подслоем (например, Multi-Head Attention или FFN), а $\mathbf{x}_{out}$ — выход подслоя.

В оригинальной статье Transformer остаточная связь реализуется следующим образом:

$$
\mathbf{x}_{out} = \text{LayerNorm}\bigl(\mathbf{x}_{in} + \text{Sublayer}(\mathbf{x}_{in})\bigr).
$$

Эта простая операция сложения имеет глубокие последствия для обучения. Она гарантирует, что информация из входного вектора сохраняется и передаётся дальше, даже если подслой $\text{Sublayer}$ не может эффективно обучиться или его выход оказывается неинформативным.

#### 1.2. Интуиция: «сквозной» проход

Остаточная связь создаёт «обходной путь» (shortcut), который позволяет информации и градиентам проходить через сеть напрямую, минуя подслой. Это можно представить как два параллельных пути:

1. **Прямой путь:** вход $\mathbf{x}_{in}$ проходит через подслой $\text{Sublayer}$, где подвергается сложным преобразованиям. Этот путь может быть длинным и содержать множество нелинейных операций.

2. **Обходной путь:** вход $\mathbf{x}_{in}$ добавляется к выходу подслоя, сохраняя исходную информацию. Этот путь является «коротким замыканием», которое позволяет информации распространяться без изменений.

Благодаря этому, даже если подслой $\text{Sublayer}$ не может эффективно обучаться или его градиенты затухают, информация всё равно может проходить через сеть по обходному пути. Это подобно тому, как в электрической схеме существует резервный путь для тока, даже если основной путь повреждён.

```mermaid
flowchart LR
    subgraph Residual["Остаточная связь"]
        direction TB
        Input["Вход: x"]
        Sublayer["Sublayer(x)"]
        Add["+"]
        Output["Выход: x + Sublayer(x)"]
    end
    
    Input --> Sublayer
    Input -->|"обходной путь"| Add
    Sublayer --> Add
    Add --> Output
    
    style Residual fill:#e8f5e9
```

#### 1.3. Почему это работает: теоретическая интерпретация

Остаточные связи можно интерпретировать как **ансамбль путей** разной длины. В сети с остаточными связями градиент распространяется не по одному длинному пути, а по множеству путей разной длины. Это делает обучение более устойчивым и позволяет использовать очень глубокие архитектуры.

---

### 2. Математическое обоснование

#### 2.1. Анализ распространения градиентов

Рассмотрим подслой, реализующий функцию $F(\mathbf{x}, \theta)$ с параметрами $\theta$. С остаточной связью:

$$
\mathbf{y} = \mathbf{x} + F(\mathbf{x}, \theta).
$$

При обратном распространении ошибки градиент функции потерь $\mathcal{L}$ по входу $\mathbf{x}$ вычисляется с использованием правила цепочки:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{x}} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}} \cdot \frac{\partial \mathbf{y}}{\partial \mathbf{x}} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}} \cdot \left( \mathbf{I} + \frac{\partial F}{\partial \mathbf{x}} \right),
$$

где:

- $\mathbf{I}$ — единичная матрица (размерности $d_{\text{model}} \times d_{\text{model}}$);
- $\partial F / \partial \mathbf{x}$ — якобиан функции $F$ по входу (матрица частных производных);
- $\partial \mathcal{L} / \partial \mathbf{y}$ — градиент по выходу подслоя.

#### 2.2. Почему это решает проблему затухающих градиентов

Для глубокой сети с $L$ слоями и остаточными связями, где каждый слой имеет вид $\mathbf{x}_{l+1} = \mathbf{x}_l + F_l(\mathbf{x}_l)$, градиент по входу $l$-го слоя выражается как:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{x}_l} = \frac{\partial \mathcal{L}}{\partial \mathbf{x}_L} \cdot \prod_{k=l}^{L-1} \left( \mathbf{I} + \frac{\partial F_k}{\partial \mathbf{x}_k} \right).
$$

Ключевое наблюдение: произведение содержит единичные матрицы $\mathbf{I}$, которые не затухают. Даже если все $\| \partial F_k / \partial \mathbf{x}_k \|$ малы (например, 0.1), произведение содержит член, равный $\partial \mathcal{L} / \partial \mathbf{x}_L$, который не умножается на малые числа.

Раскрывая произведение, получаем сумму по всем подмножествам слоёв:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{x}_l} = \frac{\partial \mathcal{L}}{\partial \mathbf{x}_L} + \sum_{k=l}^{L-1} \frac{\partial \mathcal{L}}{\partial \mathbf{x}_{k+1}} \cdot \frac{\partial F_k}{\partial \mathbf{x}_k} \cdot \prod_{m=k+1}^{L-1} \left( \mathbf{I} + \frac{\partial F_m}{\partial \mathbf{x}_m} \right).
$$

Градиент содержит прямой член $\partial \mathcal{L} / \partial \mathbf{x}_L$, который не зависит от произведений якобианов. Это гарантирует, что градиент не затухает полностью, даже если все $\partial F_k / \partial \mathbf{x}_k$ равны нулю.

#### 2.3. Эффект «обходного пути»

Остаточные связи создают множество «обходных путей» разной длины. Градиент может распространяться:

1. **Напрямую:** через остаточные связи (без прохождения через слои).
2. **Через один слой:** через одну функцию $F_k$.
3. **Через несколько слоёв:** через произведение нескольких функций $F_k$.

Это делает обучение более устойчивым и позволяет использовать очень глубокие архитектуры. Как образно выразился один из исследователей: *«Остаточные связи позволяют градиенту «срезать путь» через сеть, подобно тому, как вода находит путь по руслу реки, даже если некоторые участки заблокированы»*.

```mermaid
flowchart LR
    subgraph Pathways["Пути распространения градиента"]
        direction TB
        P1["Прямой путь: через остаточные связи"]
        P2["Короткий путь: через 1-2 слоя"]
        P3["Длинный путь: через все слои"]
    end
    
    style Pathways fill:#e3f2fd
```

#### 2.4. Сравнение с сетями без остаточных связей

В сети **без остаточных связей** градиент выражается как:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{x}_l} = \frac{\partial \mathcal{L}}{\partial \mathbf{x}_L} \cdot \prod_{k=l}^{L-1} J_k,
$$

где $J_k = \partial F_k / \partial \mathbf{x}_k$. Если $\|J_k\| < 1$, то градиент экспоненциально затухает с глубиной. Если $\|J_k\| > 1$, градиенты взрываются. Оба случая приводят к проблемам при обучении.

---

### 3. Применение остаточных связей в Transformer

#### 3.1. Вокруг Multi-Head Self-Attention

В энкодере остаточная связь применяется вокруг подслоя Self-Attention:

$$
\mathbf{X}_{out} = \text{LayerNorm}\bigl(\mathbf{X}_{in} + \text{MHA}(\mathbf{X}_{in})\bigr).
$$

Это позволяет модели сохранять исходные представления токенов, одновременно обогащая их информацией из внимания. Без остаточной связи модель могла бы «забыть» исходные представления, что привело бы к потере информации, особенно в глубоких сетях.

#### 3.2. Вокруг Feed-Forward Network

В энкодере остаточная связь также применяется вокруг подслоя FFN:

$$
\mathbf{X}_{out} = \text{LayerNorm}\bigl(\mathbf{X}_{in} + \text{FFN}(\mathbf{X}_{in})\bigr).
$$

Это позволяет модели сохранять представления после преобразования FFN, обеспечивая устойчивость и сохранение информации. Остаточная связь вокруг FFN особенно важна, так как FFN содержит основную часть параметров и может сильно изменять представления.

#### 3.3. Вокруг Masked Self-Attention в декодере

В декодере остаточная связь применяется вокруг Masked Self-Attention:

$$
\mathbf{Y}_{out} = \text{LayerNorm}\bigl(\mathbf{Y}_{in} + \text{MaskedAttn}(\mathbf{Y}_{in})\bigr).
$$

#### 3.4. Вокруг Cross-Attention в декодере

Аналогично, остаточная связь применяется вокруг Cross-Attention:

$$
\mathbf{Y}_{out} = \text{LayerNorm}\bigl(\mathbf{Y}_{in} + \text{CrossAttn}(\mathbf{Y}_{in}, \mathbf{Z})\bigr).
$$

#### 3.5. Схема применения в Transformer

```mermaid
flowchart LR
    subgraph Layer["Слой энкодера с остаточными связями"]
        direction TB
        Input["Вход: X"]
        Attn["Multi-Head Self-Attention"]
        Add1["+ (остаточная)"]
        Norm1["LayerNorm"]
        FFN["Feed-Forward Network"]
        Add2["+ (остаточная)"]
        Norm2["LayerNorm"]
        Output["Выход: X'"]
    end
    
    Input --> Attn
    Input -->|"skip"| Add1
    Attn --> Add1
    Add1 --> Norm1 --> FFN
    Norm1 -->|"skip"| Add2
    FFN --> Add2
    Add2 --> Norm2 --> Output
    
    style Layer fill:#e3f2fd
```

---

### 4. Доказательство эффективности

#### 4.1. Теоретическое обоснование

**Теорема:** В сети с остаточными связями, градиент по входу любого слоя содержит прямой член, равный градиенту по выходу сети, который не затухает.

**Доказательство:**

Рассмотрим сеть с остаточными связями:

$$
\mathbf{x}_{l+1} = \mathbf{x}_l + F_l(\mathbf{x}_l).
$$

Тогда:

$$
\frac{\partial \mathbf{x}_{l+1}}{\partial \mathbf{x}_l} = \mathbf{I} + \frac{\partial F_l}{\partial \mathbf{x}_l}.
$$

Раскрывая рекурсивно:

$$
\frac{\partial \mathbf{x}_L}{\partial \mathbf{x}_l} = \mathbf{I} + \sum_{k=l}^{L-1} \frac{\partial F_k}{\partial \mathbf{x}_k} \cdot \prod_{m=k+1}^{L-1} \left( \mathbf{I} + \frac{\partial F_m}{\partial \mathbf{x}_m} \right).
$$

Тогда градиент:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{x}_l} = \frac{\partial \mathcal{L}}{\partial \mathbf{x}_L} \cdot \frac{\partial \mathbf{x}_L}{\partial \mathbf{x}_l} = \frac{\partial \mathcal{L}}{\partial \mathbf{x}_L} + \sum_{k=l}^{L-1} \frac{\partial \mathcal{L}}{\partial \mathbf{x}_L} \cdot \frac{\partial F_k}{\partial \mathbf{x}_k} \cdot \prod_{m=k+1}^{L-1} \left( \mathbf{I} + \frac{\partial F_m}{\partial \mathbf{x}_m} \right).
$$

Градиент содержит прямой член $\partial \mathcal{L} / \partial \mathbf{x}_L$, который не зависит от произведений якобианов. Это гарантирует, что градиент не затухает полностью, даже если все $\partial F_k / \partial \mathbf{x}_k$ равны нулю.

#### 4.2. Эмпирическое подтверждение

Эксперименты показывают, что:

1. **Без остаточных связей:** обучение Transformer с 6 слоями становится нестабильным, и модель часто не сходится.

2. **С остаточными связями:** Transformer обучается с 6, 12, 24, 32 и более слоями. Каждый дополнительный слой улучшает качество модели.

3. **Глубокие модели:** BERT-large (24 слоя), GPT-3 (96 слоёв) и более глубокие модели обучаются эффективно благодаря остаточным связям.

4. **Ускорение сходимости:** Остаточные связи позволяют использовать более высокие learning rates, так как градиенты остаются стабильными.

#### 4.3. Сравнение с и без остаточных связей

| Аспект | Без остаточных связей | С остаточными связями |
|--------|----------------------|----------------------|
| **Максимальная глубина** | ~10-20 слоёв | 100+ слоёв |
| **Сходимость** | Медленная, нестабильная | Быстрая, стабильная |
| **Градиенты на ранних слоях** | Почти нулевые | Здоровые |
| **Качество модели** | Ограниченное | Высокое |
| **Чувствительность к learning rate** | Высокая | Низкая |
| **Возможность масштабирования** | Ограничена | Практически не ограничена |

---

### 5. Заключение

Остаточные связи являются критически важным компонентом архитектуры Transformer, обеспечивающим:

1. **Стабильность обучения:** предотвращают затухание градиентов и позволяют обучать очень глубокие модели.

2. **Сохранение информации:** гарантируют, что исходная информация не теряется при прохождении через слои. Это особенно важно для длинных последовательностей, где информация может «размываться».

3. **Гибкость:** позволяют модели «выбирать», какую информацию использовать — исходную или преобразованную. Это похоже на то, как в ResNet модель может «выключать» слои, если они не нужны.

4. **Масштабируемость:** делают возможным масштабирование моделей до сотен слоёв и миллиардов параметров, что является основой современных LLM.

В сочетании с Layer Normalization, остаточные связи образуют мощный механизм, который стал стандартом в современных глубоких сетях. Понимание их работы необходимо для правильной реализации, настройки и интерпретации моделей на основе Transformer.

Без остаточных связей современные LLM, такие как GPT-4, LLaMA или Qwen, были бы невозможны. Именно этот компонент позволил перейти от моделей с десятками слоёв к моделям с сотнями слоёв и триллионами параметров.

---

### Литература

1. He, K., Zhang, X., Ren, S., & Sun, J. (2016). *Deep Residual Learning for Image Recognition*. CVPR.  
   🔗 [https://arxiv.org/abs/1512.03385](https://arxiv.org/abs/1512.03385)

2. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

3. Srivastava, R. K., Greff, K., & Schmidhuber, J. (2015). *Highway Networks*. ICML.  
   🔗 [https://arxiv.org/abs/1505.00387](https://arxiv.org/abs/1505.00387)

4. He, K., et al. (2016). *Identity Mappings in Deep Residual Networks*. ECCV.  
   🔗 [https://arxiv.org/abs/1603.05027](https://arxiv.org/abs/1603.05027)

## Тема 8.3. Layer Normalization в Transformer

В предыдущих разделах мы рассмотрели остаточные связи, которые обеспечивают стабильное распространение градиентов через глубокие сети. Однако остаточные связи решают лишь часть проблемы. Для стабильного обучения необходимо также контролировать распределение активаций внутри сети, чтобы предотвратить их неконтролируемый рост или затухание. Эту задачу решает **Layer Normalization** — механизм нормализации, который стал неотъемлемой частью архитектуры Transformer. В этом разделе мы детально разберём, что такое Layer Normalization, почему она используется вместо Batch Normalization, и как она применяется в Transformer.

---

### 1. Что такое Layer Normalization

#### 1.1. Определение и формула

**Layer Normalization** (LayerNorm) — это метод нормализации, который применяется к активациям каждого токена (позиции) независимо, нормализуя их по признаковому измерению (размерности $d_{\text{model}}$). Для входного вектора $\mathbf{x} \in \mathbb{R}^{d_{\text{model}}}$ Layer Normalization вычисляется как:

$$
\text{LN}(\mathbf{x}) = \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta,
$$

где:

- $\mu = \frac{1}{d_{\text{model}}} \sum_{i=1}^{d_{\text{model}}} x_i$ — среднее значение по измерению признаков;
- $\sigma^2 = \frac{1}{d_{\text{model}}} \sum_{i=1}^{d_{\text{model}}} (x_i - \mu)^2$ — дисперсия по измерению признаков;
- $\epsilon$ — малая константа (обычно $10^{-5}$ или $10^{-6}$) для предотвращения деления на ноль;
- $\gamma \in \mathbb{R}^{d_{\text{model}}}$ и $\beta \in \mathbb{R}^{d_{\text{model}}}$ — обучаемые параметры (масштаб и сдвиг).

Важно отметить, что $\mu$ и $\sigma^2$ вычисляются для **каждой позиции отдельно** (для каждого токена). В отличие от Batch Normalization, которая нормализует по batch-измерению, Layer Normalization не зависит от размера батча и работает с последовательностями произвольной длины.

#### 1.2. Интерпретация

Layer Normalization выполняет два ключевых преобразования:

1. **Нормализация:** преобразует входной вектор так, чтобы он имел нулевое среднее и единичную дисперсию. Это стабилизирует распределение активаций и предотвращает их неконтролируемый рост.

2. **Масштабирование и сдвиг:** применяет обучаемые параметры $\gamma$ и $\beta$, которые позволяют модели восстанавливать необходимый масштаб и смещение. Это важно, потому что нулевое среднее и единичная дисперсия могут быть неоптимальными для некоторых слоёв.

```mermaid
flowchart LR
    subgraph LayerNorm["Layer Normalization"]
        direction TB
        Input["Вход: x ∈ ℝ^{d_model}"]
        Mean["Вычисление μ и σ²"]
        Norm["Нормализация: (x - μ) / √(σ² + ε)"]
        Scale["Масштабирование: γ"]
        Shift["Сдвиг: β"]
        Output["Выход: LN(x) ∈ ℝ^{d_model}"]
    end
    
    Input --> Mean
    Mean --> Norm
    Norm --> Scale
    Scale --> Shift
    Shift --> Output
    
    style LayerNorm fill:#e3f2fd
```

---

### 2. Почему LayerNorm, а не BatchNorm

#### 2.1. Batch Normalization и её ограничения

**Batch Normalization (BatchNorm)** — это метод нормализации, который широко используется в компьютерном зрении. Он нормализует активации по batch-измерению:

$$
\text{BN}(x) = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} \cdot \gamma + \beta,
$$

где $\mu_B$ и $\sigma_B^2$ вычисляются по всему батчу для каждой позиции (например, для каждого канала в CNN).

BatchNorm имеет несколько ограничений, которые делают её неприменимой в Transformer:

1. **Зависимость от размера батча:** При малых батчах оценки $\mu_B$ и $\sigma_B^2$ становятся неточными, что приводит к нестабильности обучения.

2. **Разная длина последовательностей:** В NLP последовательности имеют разную длину. BatchNorm требует фиксированной длины или сложной обработки переменной длины.

3. **Зависимость между позициями:** BatchNorm нормализует все позиции в батче вместе, что может нарушать статистику для отдельных позиций.

#### 2.2. Преимущества Layer Normalization

| Аспект | Batch Normalization | Layer Normalization |
|--------|---------------------|---------------------|
| **Измерение нормализации** | Batch (батч) | Features (признаки) |
| **Зависимость от размера батча** | Да (при малых батчах нестабильна) | Нет |
| **Работа с разными длинами** | Сложно | Естественно |
| **Применение в RNN/Transformer** | Проблематично | Широко используется |
| **Независимость позиций** | Нет | Да |

Основное преимущество Layer Normalization заключается в том, что она **не зависит от размера батча** и **не требует фиксированной длины последовательности**. Это делает её идеальным выбором для моделей обработки естественного языка, где длина последовательности может варьироваться.

```mermaid
flowchart LR
    subgraph BN["Batch Normalization"]
        direction TB
        BN1["Нормализация по батчу"]
        BN2["Зависит от размера батча"]
        BN3["Требует фиксированной длины"]
    end
    
    subgraph LN["Layer Normalization"]
        direction TB
        LN1["Нормализация по признакам"]
        LN2["Не зависит от батча"]
        LN3["Работает с любой длиной"]
    end
    
    style BN fill:#ffcdd2
    style LN fill:#e8f5e9
```

---

### 3. Математические свойства Layer Normalization

#### 3.1. Стабилизация распределения активаций

Layer Normalisation обеспечивает, что для каждого токена распределение активаций имеет примерно нулевое среднее и единичную дисперсию. Это предотвращает:

- **Неконтролируемый рост активаций:** Если дисперсия становится слишком большой, градиенты могут взрываться.
- **Затухание активаций:** Если дисперсия становится слишком малой, градиенты могут затухать.

#### 3.2. Устойчивость к изменениям масштаба

Layer Normalization инвариантна к линейным преобразованиям входа. Если вход $\mathbf{x}$ умножить на константу $c$, то:

$$
\text{LN}(c \cdot \mathbf{x}) = \text{LN}(\mathbf{x}).
$$

Это свойство делает LayerNorm устойчивой к изменениям масштаба, что особенно важно при работе с переменными длинами последовательностей и при использовании различных функций активации.

#### 3.3. Обучаемые параметры $\gamma$ и $\beta$

Параметры $\gamma$ и $\beta$ позволяют модели восстанавливать необходимый масштаб и смещение. Это важно, потому что:

- Нулевое среднее и единичная дисперсия могут быть неоптимальными для некоторых слоёв.
- Модель может «выключать» нормализацию, если это необходимо, устанавливая $\gamma = 1$ и $\beta = 0$.

#### 3.4. Градиенты через LayerNorm

Градиенты через Layer Normalization остаются здоровыми благодаря тому, что нормализация применяется по признаковому измерению. Это позволяет обучать модели с большим числом слоёв.

---

### 4. Применение в Transformer

#### 4.1. Post-Norm (оригинальный подход)

В оригинальной статье Transformer Layer Normalization применяется **после** остаточной связи:

$$
\mathbf{x}_{out} = \text{LayerNorm}\bigl(\mathbf{x}_{in} + \text{Sublayer}(\mathbf{x}_{in})\bigr).
$$

Этот подход называется **post-norm**. Он использовался в оригинальном Transformer и в ранних моделях, таких как BERT.

#### 4.2. Pre-Norm (современный подход)

В современных моделях (GPT, LLaMA, Qwen, Mistral) чаще используется **pre-norm**, где Layer Normalization применяется **до** подслоя:

$$
\mathbf{x}_{out} = \mathbf{x}_{in} + \text{Sublayer}\bigl(\text{LayerNorm}(\mathbf{x}_{in})\bigr).
$$

**Преимущества pre-norm:**

1. **Лучшая стабильность:** Градиенты проходят через остаточную связь без изменений, что делает обучение более стабильным.
2. **Меньшая чувствительность к начальным значениям:** Pre-norm более устойчива к выбору learning rate.
3. **Лучшая масштабируемость:** Pre-norm позволяет обучать модели с большим числом слоёв.

```mermaid
flowchart LR
    subgraph PostNorm["Post-Norm (оригинальный)"]
        direction TB
        X1["Вход x"] --> Sub1["Sublayer(x)"]
        X1 -->|"skip"| Add1["+"]
        Sub1 --> Add1
        Add1 --> Norm1["LayerNorm"]
        Norm1 --> Out1["Выход"]
    end
    
    subgraph PreNorm["Pre-Norm (современный)"]
        direction TB
        X2["Вход x"] --> Norm2["LayerNorm"]
        Norm2 --> Sub2["Sublayer"]
        X2 -->|"skip"| Add2["+"]
        Sub2 --> Add2
        Add2 --> Out2["Выход"]
    end
    
    style PostNorm fill:#f3e5f5
    style PreNorm fill:#e8f5e9
```

#### 4.3. Где применяется LayerNorm в Transformer

Layer Normalization применяется в следующих местах:

1. **После Self-Attention** (или до, в pre-norm).
2. **После Feed-Forward Network** (или до, в pre-norm).
3. **После Cross-Attention** в декодере (или до, в pre-norm).

---

### 5. Реализация на NumPy

```python
import numpy as np

class LayerNorm:
    def __init__(self, d_model, eps=1e-5):
        """
        Инициализация Layer Normalization.
        
        Args:
            d_model (int): размерность модели
            eps (float): константа для численной стабильности
        """
        self.d_model = d_model
        self.eps = eps
        
        # Обучаемые параметры
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
    
    def forward(self, x):
        """
        Прямой проход Layer Normalization.
        
        Args:
            x (np.ndarray): Входной тензор (batch, T, d_model)
            
        Returns:
            np.ndarray: Нормализованный тензор (batch, T, d_model)
        """
        # Вычисление среднего и дисперсии по последнему измерению
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        
        # Нормализация
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        
        # Масштабирование и сдвиг
        out = self.gamma * x_norm + self.beta
        
        return out
    
    def backward(self, grad_output):
        """
        Обратное распространение через Layer Normalization.
        (Упрощённая версия для демонстрации)
        """
        # В полной реализации здесь вычисляются градиенты по x, gamma, beta
        # Для краткости опущено
        pass

# Пример использования
if __name__ == "__main__":
    d_model = 4
    batch_size = 2
    T = 3
    
    ln = LayerNorm(d_model)
    x = np.random.randn(batch_size, T, d_model)
    
    out = ln.forward(x)
    print(f"Входная размерность: {x.shape}")
    print(f"Выходная размерность: {out.shape}")
    print(f"Среднее по признакам (должно быть ~0): {np.mean(out, axis=-1)}")
    print(f"Дисперсия по признакам (должна быть ~1): {np.var(out, axis=-1)}")
```

---

### 6. Заключение

Layer Normalization является критически важным компонентом архитектуры Transformer, обеспечивающим:

1. **Стабильность обучения:** нормализует распределение активаций, предотвращая их неконтролируемый рост или затухание.

2. **Независимость от батча:** в отличие от BatchNorm, LayerNorm не зависит от размера батча и работает с последовательностями произвольной длины.

3. **Гибкость:** обучаемые параметры $\gamma$ и $\beta$ позволяют модели адаптировать нормализацию под конкретные задачи.

4. **Совместимость с остаточными связями:** вместе с остаточными связями, LayerNorm образует мощный механизм, позволяющий обучать очень глубокие модели.

Переход от post-norm к pre-norm стал важным шагом в эволюции Transformer, позволив обучать модели с сотнями слоёв и миллиардами параметров. Понимание Layer Normalization необходимо для правильной реализации и настройки современных LLM.

---

### Литература

1. Ba, J. L., Kiros, J. R., & Hinton, G. E. (2016). *Layer Normalization*. arXiv:1607.06450.  
   🔗 [https://arxiv.org/abs/1607.06450](https://arxiv.org/abs/1607.06450)

2. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

3. Ioffe, S., & Szegedy, C. (2015). *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift*. ICML.  
   🔗 [https://arxiv.org/abs/1502.03167](https://arxiv.org/abs/1502.03167)

4. Xiong, R., et al. (2020). *On Layer Normalization in the Transformer Architecture*. ICML.  
   🔗 [https://arxiv.org/abs/2002.04745](https://arxiv.org/abs/2002.04745)

## Тема 8.4. RMSNorm — современная альтернатива Layer Normalization

В последние годы в архитектурах больших языковых моделей наблюдается устойчивый тренд перехода от классической Layer Normalization к более эффективной альтернативе — **RMSNorm** (Root Mean Square Normalization). Этот метод, предложенный в работе Zhang и Sennrich (2019) [1], предлагает упрощённую, но не менее эффективную нормализацию, которая стала стандартом в современных LLM. В этом разделе мы рассмотрим основные принципы RMSNorm, его преимущества перед LayerNorm и причины его широкого распространения.

---

### 1. Основная идея RMSNorm

#### 1.1. Определение и формула

Ключевое отличие RMSNorm от LayerNorm заключается в **отказе от вычисления среднего значения**. Вместо полной нормализации (с вычитанием среднего и делением на стандартное отклонение) RMSNorm использует только **среднеквадратичное значение (Root Mean Square, RMS)**.

Для входного вектора $\mathbf{x} \in \mathbb{R}^{d_{\text{model}}}$ RMS вычисляется как:

$$
\text{RMS}(\mathbf{x}) = \sqrt{\frac{1}{d_{\text{model}}} \sum_{i=1}^{d_{\text{model}}} x_i^2} = \sqrt{\text{mean}(\mathbf{x}^2)}.
$$

Тогда RMSNorm определяется как:

$$
\text{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\text{RMS}(\mathbf{x})} \cdot \gamma,
$$

где $\gamma \in \mathbb{R}^{d_{\text{model}}}$ — обучаемый параметр масштаба (аналогичный $\gamma$ в LayerNorm, но без параметра сдвига $\beta$).

#### 1.2. Сравнение с Layer Normalization

Для наглядности приведём обе формулы:

**Layer Normalization:**

$$
\text{LN}(\mathbf{x}) = \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta,
$$

где $\mu = \text{mean}(\mathbf{x})$, $\sigma^2 = \text{var}(\mathbf{x})$.

**RMSNorm:**

$$
\text{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\text{RMS}(\mathbf{x}) + \epsilon} \cdot \gamma.
$$

Как видно из формул, RMSNorm:

- **Не вычисляет среднее** $\mu$;
- **Не использует параметр сдвига** $\beta$;
- **Не требует вычитания среднего**;
- **Вычисляет только RMS**, что значительно проще.

```mermaid
flowchart LR
    subgraph LayerNorm["Layer Normalization"]
        direction TB
        LN1["Вычисление μ = mean(x)"]
        LN2["Вычисление σ² = var(x)"]
        LN3["Нормализация: (x - μ)/√(σ²+ε)"]
        LN4["Масштабирование: γ"]
        LN5["Сдвиг: β"]
    end
    
    subgraph RMSNorm["RMSNorm"]
        direction TB
        RN1["Вычисление RMS = √(mean(x²))"]
        RN2["Нормализация: x / RMS"]
        RN3["Масштабирование: γ"]
    end
    
    style LayerNorm fill:#f3e5f5
    style RMSNorm fill:#e8f5e9
```

---

### 2. Математический анализ

#### 2.1. Почему RMS достаточно

Ключевой вопрос: *почему можно отказаться от вычитания среднего?* Ответ кроется в свойствах современных архитектур и процесса обучения.

**1. Центрированность данных:** В глубоких сетях с остаточными связями и нормализацией активации, как правило, уже имеют среднее, близкое к нулю. Это связано с тем, что:

- Веса инициализируются с нулевым средним;
- Остаточные связи сохраняют распределение активаций;
- Функции активации (например, GELU) имеют нулевое среднее.

**2. Инвариантность к сдвигу:** В задачах NLP абсолютное значение среднего активаций часто не несёт полезной семантической информации. Нормализация масштаба (RMS) оказывается достаточной для стабилизации обучения.

**3. Эмпирические результаты:** Эксперименты показывают, что RMSNorm даёт качество, сопоставимое или даже превосходящее LayerNorm [1, 2].

#### 2.2. Сравнительный анализ

| Аспект | Layer Normalization | RMSNorm |
|--------|---------------------|---------|
| **Вычисление среднего** | Да | Нет |
| **Вычисление дисперсии** | Да | Нет |
| **Вычисление RMS** | Нет | Да |
| **Параметр γ** | Да | Да |
| **Параметр β** | Да | Нет |
| **Число операций** | Больше | Меньше (≈10-15% быстрее) |
| **Память** | Больше | Меньше |
| **Качество** | Эталонное | Сопоставимое или лучше |

---

### 3. Преимущества RMSNorm

#### 3.1. Вычислительная эффективность

Основное преимущество RMSNorm — **значительное сокращение вычислительных затрат**. Рассмотрим количество операций для одного вектора размерности $d$:

**LayerNorm:**
- Вычисление среднего: $d$ сложений, 1 деление.
- Вычисление дисперсии: $d$ вычитаний, $d$ умножений, $d$ сложений, 1 деление.
- Нормализация: $d$ вычитаний, $d$ делений.
- Масштабирование и сдвиг: $d$ умножений, $d$ сложений.

**RMSNorm:**
- Вычисление RMS: $d$ умножений, $d$ сложений, 1 деление, 1 извлечение корня.
- Нормализация: $d$ делений.
- Масштабирование: $d$ умножений.

RMSNorm требует **примерно на 10-15% меньше операций**, что при масштабе современных моделей (сотни слоёв, миллиарды параметров) даёт существенную экономию времени обучения и инференса.

#### 3.2. Упрощение архитектуры

Отказ от параметра $\beta$ (сдвига) уменьшает число обучаемых параметров. Хотя это сокращение незначительно (всего $d$ параметров на слой), в совокупности с другими оптимизациями оно даёт вклад в общую эффективность модели.

#### 3.3. Стабильность обучения

RMSNorm демонстрирует стабильность обучения, сопоставимую с LayerNorm. В работе [1] показано, что RMSNorm особенно эффективна в моделях с pre-norm, где нормализация применяется до подслоя.

---

### 4. Использование в современных моделях

#### 4.1. Модели, использующие RMSNorm

| Модель | Год | Использование RMSNorm |
|--------|-----|----------------------|
| **LLaMA** (Meta) | 2023-2024 | Да (все версии) |
| **Mistral** (Mistral AI) | 2023 | Да |
| **Qwen** (Alibaba) | 2023-2024 | Да (Qwen 2.5) |
| **Gemma** (Google) | 2024 | Да |
| **DeepSeek** | 2024 | Да |
| **Falcon** (TII) | 2023 | Нет (использует LayerNorm) |
| **GPT-3/4** (OpenAI) | 2020-2023 | Нет (использует LayerNorm) |

#### 4.2. Почему современные модели переходят на RMSNorm

1. **Эффективность:** RMSNorm быстрее и требует меньше памяти, что критично для моделей с сотнями слоёв.

2. **Качество:** Эксперименты показывают, что RMSNorm не уступает LayerNorm по качеству, а в некоторых случаях даже превосходит её [2].

3. **Простота:** Меньше гиперпараметров и вычислений — меньше возможностей для ошибок.

4. **Тренд:** После успешного применения в LLaMA и Mistral, RMSNorm стал стандартом для новых открытых моделей.

---

### 5. Реализация на NumPy

```python
import numpy as np

class RMSNorm:
    def __init__(self, d_model, eps=1e-5):
        """
        Инициализация RMSNorm.
        
        Args:
            d_model (int): размерность модели
            eps (float): константа для численной стабильности
        """
        self.d_model = d_model
        self.eps = eps
        self.gamma = np.ones(d_model)  # обучаемый параметр масштаба
    
    def forward(self, x):
        """
        Прямой проход RMSNorm.
        
        Args:
            x (np.ndarray): Входной тензор (batch, T, d_model)
            
        Returns:
            np.ndarray: Нормализованный тензор (batch, T, d_model)
        """
        # Вычисление RMS по последнему измерению
        rms = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + self.eps)
        
        # Нормализация и масштабирование
        out = self.gamma * (x / rms)
        
        return out

# Пример использования
if __name__ == "__main__":
    d_model = 512
    batch_size = 2
    T = 10
    
    rmsnorm = RMSNorm(d_model)
    x = np.random.randn(batch_size, T, d_model)
    
    out = rmsnorm.forward(x)
    print(f"Входная размерность: {x.shape}")
    print(f"Выходная размерность: {out.shape}")
    print(f"RMS по признакам (должен быть ~1): {np.sqrt(np.mean(out ** 2, axis=-1))}")
```

---

### 6. Заключение

RMSNorm представляет собой элегантное упрощение Layer Normalization, которое:

1. **Отказывается от вычисления среднего** и параметра сдвига, используя только RMS.
2. **Значительно быстрее** (≈10-15%) и требует меньше памяти.
3. **Сохраняет качество** на уровне LayerNorm, а в некоторых случаях превосходит его.
4. **Стала стандартом** в современных LLM, включая LLaMA, Mistral, Qwen и Gemma.

Переход от LayerNorm к RMSNorm отражает общий тренд в развитии архитектур глубокого обучения: поиск более эффективных, но не уступающих по качеству решений. Понимание RMSNorm необходимо для работы с современными моделями и для разработки новых, более эффективных архитектур.

---

### Литература

1. Zhang, B., & Sennrich, R. (2019). *Root Mean Square Layer Normalization*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1910.07467](https://arxiv.org/abs/1910.07467)

2. Touvron, H., et al. (2023). *LLaMA: Open and Efficient Foundation Language Models*. arXiv:2302.13971.  
   🔗 [https://arxiv.org/abs/2302.13971](https://arxiv.org/abs/2302.13971)

3. Bai, J., et al. (2023). *Qwen Technical Report*. arXiv:2309.16609.  
   🔗 [https://arxiv.org/abs/2309.16609](https://arxiv.org/abs/2309.16609)

4. Jiang, A. Q., et al. (2023). *Mistral 7B*. arXiv:2310.06825.  
   🔗 [https://arxiv.org/abs/2310.06825](https://arxiv.org/abs/2310.06825)

## Тема 9.1. Итоги: архитектура Transformer как фундамент современного ИИ

Подводя итог всестороннему анализу архитектуры Transformer, мы можем с уверенностью утверждать, что этот раздел охватил не просто одну из многих архитектур нейронных сетей, а **фундаментальную парадигму**, определившую развитие искусственного интеллекта на последнее десятилетие. От ограничений RNN до современных реализаций с RMSNorm и SwiGLU — мы проследили эволюцию идей, которые сделали Transformer доминирующей архитектурой в области обработки естественного языка и далеко за её пределами.

---

### 1. Ключевые компоненты Transformer: единая картина

Архитектура Transformer представляет собой гармоничное сочетание нескольких взаимодополняющих механизмов, каждый из которых решает конкретную задачу. Пройдём по ним в том порядке, в котором данные проходят через модель:

**1. Токенизация и эмбеддинги.** Процесс начинается с преобразования текста в числовые идентификаторы (токенизация) и их отображения в плотные векторные представления (эмбеддинги) через обучаемую матрицу $E \in \mathbb{R}^{V \times d_{\text{model}}}$. Это первый мост между дискретным языком и непрерывным пространством, в котором работает модель.

**2. Позиционное кодирование.** Поскольку Self-Attention по своей природе перестановочно-инвариантен, необходимо добавить информацию о порядке токенов. От синусоидального кодирования $PE_{(pos,2i)} = \sin(pos/10000^{2i/d_{\text{model}}})$ в оригинальном Transformer до современных RoPE (Rotary Position Embedding) — этот компонент обеспечивает модели понимание структуры предложения.

**3. Механизм самовнимания (Self-Attention).** Сердце Transformer, позволяющее каждому токену взаимодействовать со всеми остальными. Формула $\text{Attention}(Q,K,V) = \text{softmax}(QK^T/\sqrt{d_k})V$ обеспечивает глобальный контекст с постоянной сложностью на пару токенов.

**4. Многоголовое внимание (Multi-Head Attention).** Расширение Self-Attention, позволяющее модели одновременно фокусироваться на разных аспектах информации. $\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O$, где каждая голова работает в своём подпространстве, что даёт более богатые представления.

**5. Feed-Forward Network (FFN).** Позиционно-независимая полносвязная сеть $\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$, которая преобразует и обогащает представления, добавляет нелинейность и хранит основную часть знаний модели (до 80% параметров).

**6. Остаточные связи (Residual Connections).** Обеспечивают «сквозной» проход градиентов через $x_{out} = x_{in} + \text{Sublayer}(x_{in})$, решая проблему затухающих градиентов и позволяя обучать модели с сотнями слоёв.

**7. Нормализация (LayerNorm / RMSNorm).** Стабилизирует распределение активаций, делая обучение более устойчивым. RMSNorm $\text{RMSNorm}(x) = x / \text{RMS}(x) \cdot \gamma$ становится современным стандартом благодаря простоте и эффективности.

```mermaid
flowchart TD
    subgraph Full["Полная архитектура Transformer"]
        direction TB
        Input["Входной текст"]
        Token["Токенизация"]
        Embed["Эмбеддинги + PE"]
        Encoder["Стек энкодеров (N×)"]
        Decoder["Стек декодеров (N×)"]
        Output["Выходной текст"]
        
        Input --> Token --> Embed --> Encoder
        Encoder --> Decoder
        Decoder --> Output
    end
    
    subgraph EncLayer["Слой энкодера"]
        direction LR
        E1["Self-Attention"]
        E2["Add & Norm"]
        E3["FFN"]
        E4["Add & Norm"]
    end
    
    subgraph DecLayer["Слой декодера"]
        direction LR
        D1["Masked Self-Attention"]
        D2["Add & Norm"]
        D3["Cross-Attention"]
        D4["Add & Norm"]
        D5["FFN"]
        D6["Add & Norm"]
    end
    
    Encoder --> EncLayer
    Decoder --> DecLayer
    
    style Full fill:#e3f2fd
    style EncLayer fill:#f3e5f5
    style DecLayer fill:#fce4ec
```

**«Формула» Transformer:** архитектуру можно представить как композицию слоёв, каждый из которых выполняет два ключевых преобразования:

$$
\text{Layer}(x) = \text{Norm}\bigl(x + \text{FFN}(\text{Norm}(x + \text{Attn}(x)))\bigr).
$$

В этой формуле (в pre-norm варианте) закодирована вся суть Transformer: взаимодействие между токенами через **Attention**, индивидуальное преобразование через **FFN** и стабилизация через **Norm + Residual**.

---

### 2. Почему Transformer доминирует

**Параллелизация.** В отличие от RNN, где каждый шаг зависит от предыдущего, в Transformer все токены обрабатываются одновременно. Это позволяет эффективно использовать современные GPU и TPU, сокращая время обучения с недель до дней.

**Глобальный контекст.** Self-Attention обеспечивает прямые связи между любыми двумя позициями с постоянной сложностью $O(1)$. Это позволяет модели учитывать дальние зависимости без проблемы затухающих градиентов. Каждый токен «видит» все остальные токены одновременно.

**Масштабируемость.** Transformer демонстрирует устойчивое масштабирование: увеличение числа параметров, объёма данных и вычислительных ресурсов даёт предсказуемое улучшение качества (законы масштабирования Kaplan et al., 2020). Это свойство позволило перейти от моделей с 65 млн параметров до 1.8 трлн.

**Универсальность.** Одна архитектура успешно применяется в NLP (BERT, GPT), компьютерном зрении (ViT), обработке аудио (Whisper) и мультимодальных системах (CLIP, GPT-4o). Это делает Transformer настоящим «универсальным решением» для широкого круга задач.

---

### 3. Влияние на поле и будущие направления

Transformer не просто улучшил существующие методы — он **переопределил** область обработки естественного языка. Появление BERT (2018) показало, что предобучение на больших корпусах с последующей тонкой настройкой даёт беспрецедентные результаты. GPT (2018-2024) продемонстрировал, что масштабирование decoder-only архитектур приводит к появлению эмерджентных способностей — от решения математических задач до написания программного кода.

Современные LLM с сотнями миллиардов параметров, способные обрабатывать контекст в миллион токенов и работать с несколькими модальностями, стали возможны именно благодаря архитектуре Transformer. Индустрия искусственного интеллекта, оцениваемая в триллионы долларов, построена на этом фундаменте.

**Будущие направления:**

- **Новые архитектуры:** Mamba, RWKV и другие альтернативы, пытающиеся преодолеть квадратичную сложность Self-Attention за счёт использования State Space Models.
- **Мультимодальность:** интеграция текста, изображений, аудио и видео в единых моделях, таких как GPT-4o и Gemini.
- **Эффективность:** дальнейшее снижение стоимости обучения и инференса через квантование, дистилляцию, аппаратные оптимизации и новые методы внимания (Flash Attention).
- **Длинные контексты:** увеличение контекстного окна до миллионов токенов для работы с целыми книгами и базами знаний.
- **Агентные системы:** использование Transformer в качестве «мозга» автономных агентов, способных планировать, рассуждать и взаимодействовать с внешним миром.

---

### 4. Связь с последующими темами

Понимание архитектуры Transformer является фундаментом для всех последующих разделов курса:

**Раздел 4: Тонкая настройка LLM.** LoRA и QLoRA работают именно с проекционными матрицами внимания и FFN. Понимание структуры необходимо для выбора целевых слоёв и настройки гиперпараметров. Без понимания того, как устроены слои внимания, невозможно эффективно адаптировать модель под свою задачу.

**Раздел 5: RAG (Retrieval-Augmented Generation).** RAG использует Transformer для кодирования документов и генерации ответов. Понимание контекстного окна и механизмов внимания критично для эффективной работы с длинными документами и для правильной настройки поисковой компоненты.

**Раздел 6: Агенты на основе LLM.** Агенты используют Transformer для планирования, рассуждений и взаимодействия с инструментами. Понимание Self-Attention и позиционного кодирования помогает интерпретировать поведение агентов и настраивать их эффективность.

**Раздел 7: Оценка и бенчмаркинг.** Многие метрики (BERTScore, BLEURT) используют Transformer-эмбеддинги. Понимание архитектуры позволяет правильно интерпретировать результаты и выбирать подходящие методы оценки.

**Раздел 8: RLHF и выравнивание.** Reward Model — это тот же Transformer с заменённой головой. Понимание архитектуры необходимо для настройки RLHF и для понимания того, как модели учатся следовать человеческим предпочтениям.

**Раздел 9: Продакшен.** Оптимизация инференса (Flash Attention, KV-cache) напрямую зависит от понимания механизмов внимания. Это критически важно для создания эффективных промышленных систем.

---

### Заключение

Архитектура Transformer — это не просто очередная нейросетевая архитектура, а **новая парадигма** обработки последовательных данных. Отказ от рекуррентности в пользу внимания позволил достичь беспрецедентной параллелизации, глобального контекста и масштабируемости. Понимание каждого компонента — от позиционного кодирования до остаточных связей — является необходимым условием для работы с современными LLM и участия в создании следующего поколения интеллектуальных систем.

Мы прошли путь от ограничений RNN и проблемы затухающих градиентов до современных реализаций с RMSNorm и SwiGLU, от оригинальной статьи 2017 года до современных моделей с сотнями миллиардов параметров. Теперь вы готовы перейти к практическим разделам курса и применить полученные знания для создания собственных систем на основе больших языковых моделей.

**Ключевой вывод:** Transformer — это не просто архитектура, это способ мышления. Это понимание того, что взаимодействие между элементами последовательности важнее, чем их последовательная обработка. Это осознание того, что масштабирование открывает новые возможности. Это знание того, что универсальная архитектура может решать широкий спектр задач. И это фундамент, на котором строится современный искусственный интеллект.

---

### Литература

1. Vaswani, A., et al. (2017). *Attention Is All You Need*. NeurIPS.  
   🔗 [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

2. Devlin, J., et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.  
   🔗 [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

3. Radford, A., et al. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI.  
   🔗 [https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)

4. Kaplan, J., et al. (2020). *Scaling Laws for Neural Language Models*. arXiv:2001.08361.  
   🔗 [https://arxiv.org/abs/2001.08361](https://arxiv.org/abs/2001.08361)

## Тема 9.2. Домашнее задание по разделу "Архитектура Transformer"

В рамках завершения раздела, посвящённого архитектуре Transformer, предлагается выполнить комплексное домашнее задание, направленное на закрепление теоретических знаний и получение практических навыков работы с ключевыми компонентами архитектуры. Задание разделено на три уровня сложности: обязательная часть (фундаментальные навыки), дополнительная часть (углублённое исследование) и исследовательская часть (передовые темы). Рекомендуется начинать с обязательной части, постепенно переходя к более сложным задачам.

---

### 1. Обязательная часть (60% от общей оценки)

Данная часть направлена на формирование базового понимания архитектуры Transformer и получение практических навыков реализации её ключевых компонентов. Выполнение всех заданий этого раздела является обязательным для получения зачёта.

#### 1.1. Изучение оригинальной статьи

Прочитайте статью **"Attention Is All You Need"** (Vaswani et al., 2017) и напишите краткий реферат. При чтении уделите особое внимание следующим разделам:

- **Section 3:** Model Architecture — понимание общей структуры
- **Section 3.2:** Attention — детальное изучение механизма внимания
- **Section 3.3:** Position-wise Feed-Forward Networks — роль FFN
- **Section 3.4:** Embeddings and Softmax — начальные и выходные слои
- **Section 3.5:** Positional Encoding — способы кодирования позиций
- **Section 4:** Why Self-Attention — обоснование выбора архитектуры
- **Table 1:** Comparison of layer types — сравнительный анализ

**Реферат** должен содержать:
- Краткое изложение ключевых инноваций статьи (1-2 абзаца)
- Пошаговое описание архитектуры Transformer своими словами
- Обоснование того, почему статья стала революционной в области NLP
- Ваши собственные вопросы и замечания по статье
- Объём: 1-2 страницы (≈500-1000 слов)

#### 1.2. Визуализация архитектуры

Нарисуйте подробную блок-схему полной архитектуры Transformer, включающую все компоненты и потоки данных.

**Требования к схеме:**
- Включить: входной слой, токенизацию, позиционное кодирование, стек энкодеров (с Self-Attention и FFN), стек декодеров (с Masked Self-Attention, Cross-Attention и FFN), выходной слой
- Указать размерности на каждом этапе ($d_{\text{model}}$, $T$, $V$)
- Отметить ключевые различия между энкодером и декодером
- Показать остаточные связи и нормализацию
- Указать, где применяется маска

**Формат:** от руки (с фотографией) или с использованием любого инструмента (Mermaid, draw.io, Lucidchart, Visio). Схема должна быть чёткой, читаемой и хорошо структурированной. Приложить к отчёту в виде изображения с разрешением не менее 300 DPI.

#### 1.3. Реализация Scaled Dot-Product Attention

Напишите функцию `scaled_dot_product_attention(Q, K, V, mask=None)` на Python с использованием библиотеки NumPy.

**Требования к реализации:**
- Вычислить матрицу оценок $S = QK^T$
- Применить масштабирование $S / \sqrt{d_k}$
- Применить маску (если передана): заменить запрещённые позиции на $-\infty$
- Применить softmax к каждой строке (по последнему измерению)
- Вычислить выход $O = \text{softmax}(S)V$
- Поддержка batch-обработки (размерности: batch, seq_len, dim)

**Проверка работы:**
- Протестировать на случайных данных с размерами (batch=2, T=3, d_k=4)
- Убедиться, что сумма весов в каждой строке равна 1 (с учётом маски)
- Протестировать с маской (например, для каузального внимания)

```python
# Пример сигнатуры функции
def scaled_dot_product_attention(Q, K, V, mask=None, d_k=None):
    """
    Q: (batch, T, d_k) или (T, d_k)
    K: (batch, T, d_k) или (T, d_k)
    V: (batch, T, d_v) или (T, d_v)
    mask: (batch, T, T) или (T, T) — 0 для разрешённых, -inf для запрещённых
    d_k: размерность ключей (если None, берётся из Q)
    """
    pass
```

#### 1.4. Реализация Multi-Head Attention

Реализуйте класс `MultiHeadAttention`, инкапсулирующий все необходимые компоненты многоголового внимания.

**Требования к реализации:**
- Конструктор: `__init__(self, d_model, h)` — инициализация матриц проекций
- Метод: `forward(self, Q, K, V, mask=None)` — прямой проход
- Использовать реализованную ранее функцию `scaled_dot_product_attention`
- Поддержка $h$ голов с размерностью $d_k = d_{\text{model}} / h$
- Реализовать конкатенацию голов и финальную проекцию $W^O$

**Проверка работы:**
- Протестировать на случайных данных с $d_{\text{model}} = 512, h = 8$
- Проверить размерности на каждом этапе (проекции, внимание, конкатенация)
- Убедиться, что выходная размерность равна входной: $d_{\text{model}}$

```python
class MultiHeadAttention:
    def __init__(self, d_model, h):
        # d_model: размерность модели
        # h: число голов
        pass
    
    def forward(self, Q, K, V, mask=None):
        # Q, K, V: (batch, T, d_model)
        # mask: (batch, T, T) или None
        pass
```

---

### 2. Дополнительная часть (20% от общей оценки)

Задания этого раздела направлены на углублённое исследование и визуализацию ключевых механизмов Transformer. Выполнение рекомендуется, но не обязательно для получения зачёта (даёт дополнительные баллы).

#### 2.1. Визуализация позиционного кодирования

Реализуйте синусоидальное позиционное кодирование и создайте визуализации.

**Задания:**
- Реализовать функцию `positional_encoding(T, d_model)`
- Создать тепловую карту $PE$ для $T = 50, d_{\text{model}} = 64$
- Построить графики зависимости $PE$ от позиции для нескольких измерений (например, i=0, 4, 8, 16, 31)
- Объяснить, как разные частоты кодируют разные масштабы позиций
- Сравнить с обучаемыми позиционными эмбеддингами (опционально)

#### 2.2. Сравнение внимания с и без масштабирования

Проведите эксперимент, демонстрирующий важность масштабирования на $\sqrt{d_k}$.

**Задания:**
- Сгенерировать случайные Q, K для разных $d_k$ (4, 16, 64, 256)
- Вычислить распределение значений $QK^T$ и $QK^T/\sqrt{d_k}$
- Построить гистограммы распределений для каждого случая
- Визуализировать веса softmax для обоих случаев
- Объяснить, почему масштабирование критично для больших $d_k$

#### 2.3. Визуализация весов внимания

Для реального текста визуализируйте веса внимания из предобученной модели.

**Задания:**
- Загрузить предобученную модель через Hugging Face (BERT или GPT-2)
- Подать на вход простое предложение (5-10 слов)
- Извлечь веса внимания из одного слоя (или нескольких)
- Визуализировать тепловые карты для разных голов
- Интерпретировать, на какие связи обращают внимание разные головы

```python
from transformers import AutoModel, AutoTokenizer
import matplotlib.pyplot as plt

# Пример загрузки и визуализации
model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# ... остальной код
```

---

### 3. Исследовательская часть (20% от общей оценки)

Задания этого раздела направлены на изучение передовых тем и современных разработок в области архитектуры Transformer. Предназначены для студентов, стремящихся к глубокому пониманию предмета.

#### 3.1. Изучение Flash Attention

Прочитайте статью **"FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness"** (Dao et al., 2022) и выполните анализ.

**Задания:**
- Объяснить ключевые техники ускорения: tiling (разбиение на блоки), recomputation (перевычисление)
- Сравнить сложность по памяти стандартного Attention ($O(T^2)$) и Flash Attention ($O(T)$)
- Проанализировать, почему Flash Attention критичен для работы с длинными последовательностями (> 4096 токенов)
- Описать, какие изменения в реализации softmax позволяют уменьшить обращения к глобальной памяти

#### 3.2. Сравнение функций активации

Реализуйте FFN с различными функциями активации и проведите сравнительный анализ.

**Задания:**
- Реализовать FFN с функциями активации: ReLU, GELU, SwiGLU
- Сравнить число параметров для каждой функции (SwiGLU требует дополнительную проекцию)
- Сравнить время выполнения на синтетических данных разного размера
- Проанализировать, почему SwiGLU становится стандартом в современных LLM (LLaMA, Qwen, Mistral)

---

### 4. Требования к отчёту

#### 4.1. Структура отчёта

1. **Титульный лист** (название работы, ФИО студента, дата, курс)
2. **Введение** (краткое описание выполненной работы и целей)
3. **Теоретическая часть** (реферат по статье, описание архитектуры)
4. **Практическая часть** (код, скриншоты результатов, визуализации)
5. **Исследовательская часть** (анализ Flash Attention и функций активации)
6. **Выводы** (что нового узнали, какие были сложности, планы)
7. **Список литературы** (все использованные источники)

#### 4.2. Формат сдачи

**GitHub репозиторий:**
- Код на Python (`.py` файлы или Jupyter Notebook)
- README с инструкцией по установке и запуску
- Визуализации в папке `images/` (PNG/PDF)
- Ссылка на репозиторий в PDF-отчёте

**PDF отчёт:**
- Полная версия отчёта со всеми разделами (10-20 страниц)
- Включены схемы, графики, таблицы и листинги кода
- Код должен быть вставлен в виде форматированных листингов

#### 4.3. Критерии оценки

| Критерий | Вес | Описание |
|----------|-----|----------|
| **Понимание архитектуры** | 25% | Глубина описания компонентов, корректность схемы, качество реферата |
| **Качество реализации** | 25% | Работающий код, правильность вычислений, комментарии, тесты |
| **Анализ и визуализация** | 20% | Качество визуализаций, интерпретация результатов, научный подход |
| **Исследовательская часть** | 15% | Глубина анализа, понимание современных методов (Flash Attention) |
| **Оформление** | 15% | Структурированность, читаемость, полнота отчёта, оформление кода |

#### 4.4. Сроки сдачи

- **Дедлайн:** 2 недели с момента выдачи задания
- **Досрочная сдача:** приветствуется (бонусные баллы до +10%)
- **Штраф за просрочку:** -10% за каждый день просрочки (максимум -50%)
- **Пересдача:** возможность сдать доработанную работу через 1 неделю после проверки (максимальная оценка — 70%)

---

**Удачи в выполнении задания!** Глубокое понимание архитектуры Transformer — это фундамент для работы с любыми современными большими языковыми моделями. Не стесняйтесь задавать вопросы в чате курса и обращаться за помощью к преподавателям. Помните: практическая реализация ключевых компонентов — лучший способ понять, как работает архитектура "изнутри".

## ПОЛНАЯ РЕАЛИЗАЦИЯ TRANSFORMER НА NUMPY

In [ ]:
"""
================================================================================
ПОЛНАЯ РЕАЛИЗАЦИЯ TRANSFORMER НА NUMPY
================================================================================
Данный модуль содержит полную реализацию архитектуры Transformer
(энкодер-декодер) с использованием только NumPy. Предназначен для
образовательных целей: демонстрации внутреннего устройства модели.

Включает:
- Scaled Dot-Product Attention
- Multi-Head Attention
- Positional Encoding (синусоидальное)
- Feed-Forward Network (с ReLU)
- Encoder Layer
- Decoder Layer
- Полный Transformer (Encoder + Decoder)

Все классы сопровождаются документацией и проверкой размерностей.
================================================================================
"""

import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple, List

# ------------------------------------------------------------------------------
# 1. Scaled Dot-Product Attention
# ------------------------------------------------------------------------------

def scaled_dot_product_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    mask: Optional[np.ndarray] = None,
    d_k: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Scaled Dot-Product Attention.

    Формула: Attention(Q,K,V) = softmax(Q·K^T / sqrt(d_k)) · V

    Аргументы:
        Q: (batch, T_q, d_k) или (T_q, d_k) — запросы
        K: (batch, T_k, d_k) или (T_k, d_k) — ключи
        V: (batch, T_k, d_v) или (T_k, d_v) — значения
        mask: (batch, T_q, T_k) или (T_q, T_k) — маска (0 — разрешено, -inf — запрещено)
        d_k: размерность ключей (если None, вычисляется из Q)

    Возвращает:
        output: (batch, T_q, d_v) — результат внимания
        attn_weights: (batch, T_q, T_k) — веса внимания
    """
    # Приводим к 3D, если передан 2D
    if Q.ndim == 2:
        Q = Q[np.newaxis, ...]
        K = K[np.newaxis, ...]
        V = V[np.newaxis, ...]
        if mask is not None and mask.ndim == 2:
            mask = mask[np.newaxis, ...]
        squeeze_output = True
    else:
        squeeze_output = False

    batch, T_q, d_k_actual = Q.shape
    _, T_k, _ = K.shape
    d_v = V.shape[-1]

    if d_k is None:
        d_k = d_k_actual

    # 1. Вычисляем матрицу оценок S = Q·K^T
    # (batch, T_q, T_k)
    scores = Q @ K.transpose(0, 2, 1)

    # 2. Масштабирование
    scores = scores / np.sqrt(d_k)

    # 3. Применяем маску (если есть)
    if mask is not None:
        if mask.ndim == 2:
            mask = mask[np.newaxis, ...]
        # Маска: 0 для разрешённых, -inf для запрещённых
        scores = scores + mask  # маска должна быть (batch, T_q, T_k)

    # 4. Softmax по последней оси (по ключам)
    # Для численной стабильности вычитаем максимум
    scores_max = np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    attn_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

    # 5. Взвешенное суммирование значений
    output = attn_weights @ V  # (batch, T_q, d_v)

    if squeeze_output:
        output = output[0]
        attn_weights = attn_weights[0]

    return output, attn_weights


# ------------------------------------------------------------------------------
# 2. Multi-Head Attention
# ------------------------------------------------------------------------------

class MultiHeadAttention:
    """
    Многоголовое внимание (Multi-Head Attention).

    Атрибуты:
        d_model: размерность модели
        h: число голов
        d_k: размерность каждой головы (d_model / h)
        W_q, W_k, W_v: матрицы проекций для каждой головы (объединены в один тензор)
        W_o: финальная проекция
    """

    def __init__(self, d_model: int, h: int):
        """
        Инициализация параметров Multi-Head Attention.

        Аргументы:
            d_model: размерность модели
            h: число голов (должно делить d_model)
        """
        assert d_model % h == 0, "d_model должно делиться на h"
        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h
        self.d_v = self.d_k

        # Инициализация весов (Xavier)
        # Для каждой головы: W_q, W_k, W_v размером (h, d_model, d_k)
        scale_q = np.sqrt(2.0 / d_model)
        self.W_q = np.random.randn(h, d_model, self.d_k) * scale_q
        self.W_k = np.random.randn(h, d_model, self.d_k) * scale_q
        self.W_v = np.random.randn(h, d_model, self.d_v) * scale_q
        self.W_o = np.random.randn(d_model, d_model) * scale_q

    def forward(
        self,
        Q: np.ndarray,
        K: np.ndarray,
        V: np.ndarray,
        mask: Optional[np.ndarray] = None,
    ) -> Tuple[np.ndarray, List[np.ndarray]]:
        """
        Прямой проход Multi-Head Attention.

        Аргументы:
            Q: (batch, T_q, d_model) — запросы
            K: (batch, T_k, d_model) — ключи
            V: (batch, T_k, d_model) — значения
            mask: (batch, T_q, T_k) или (T_q, T_k) — маска

        Возвращает:
            output: (batch, T_q, d_model) — результат
            attn_weights: список весов внимания для каждой головы (h штук)
        """
        batch, T_q, _ = Q.shape
        _, T_k, _ = K.shape

        # 1. Линейные проекции для каждой головы
        # Q_proj: (batch, h, T_q, d_k), аналогично для K и V
        # i соответствует d_model (ось схлопывания при умножении)
        # j соответствует d_k / d_v (выходная ось)
        Q_proj = np.einsum('hij,bti->bhtj', self.W_q, Q)  # (batch, h, T_q, d_k)
        K_proj = np.einsum('hij,bti->bhtj', self.W_k, K)  # (batch, h, T_k, d_k)
        V_proj = np.einsum('hij,bti->bhtj', self.W_v, V)  # (batch, h, T_k, d_v)

        # 2. Применяем внимание для каждой головы
        # Переносим голову на первое измерение для удобства
        attn_weights_list = []
        outputs_list = []

        for head in range(self.h):
            q = Q_proj[:, head, :, :]  # (batch, T_q, d_k)
            k = K_proj[:, head, :, :]  # (batch, T_k, d_k)
            v = V_proj[:, head, :, :]  # (batch, T_k, d_v)

            out, attn = scaled_dot_product_attention(q, k, v, mask, self.d_k)
            outputs_list.append(out)
            attn_weights_list.append(attn)

        # 3. Конкатенация голов
        # (batch, T_q, h * d_v) = (batch, T_q, d_model)
        concat = np.concatenate(outputs_list, axis=-1)

        # 4. Финальная проекция
        output = concat @ self.W_o  # (batch, T_q, d_model)

        return output, attn_weights_list


# ------------------------------------------------------------------------------
# 3. Positional Encoding (синусоидальное)
# ------------------------------------------------------------------------------

def positional_encoding(T: int, d_model: int) -> np.ndarray:
    """
    Генерация синусоидального позиционного кодирования.

    Аргументы:
        T: длина последовательности (число позиций)
        d_model: размерность модели

    Возвращает:
        PE: (T, d_model) — позиционное кодирование
    """
    PE = np.zeros((T, d_model))
    for pos in range(T):
        for i in range(d_model // 2):
            freq = pos / (10000 ** (2 * i / d_model))
            PE[pos, 2 * i] = np.sin(freq)
            PE[pos, 2 * i + 1] = np.cos(freq)
    return PE


# ------------------------------------------------------------------------------
# 4. Feed-Forward Network
# ------------------------------------------------------------------------------

class FeedForward:
    """
    Position-wise Feed-Forward Network.
    Два линейных слоя с ReLU между ними.
    """

    def __init__(self, d_model: int, d_ff: int):
        """
        Инициализация FFN.

        Аргументы:
            d_model: размерность входа/выхода
            d_ff: размерность внутреннего слоя
        """
        self.d_model = d_model
        self.d_ff = d_ff

        scale = np.sqrt(2.0 / d_model)
        self.W1 = np.random.randn(d_ff, d_model) * scale
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_model, d_ff) * scale
        self.b2 = np.zeros(d_model)

    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        Прямой проход FFN.

        Аргументы:
            x: (batch, T, d_model)

        Возвращает:
            out: (batch, T, d_model)
        """
        # Первый слой: расширение
        hidden = x @ self.W1.T + self.b1  # (batch, T, d_ff)
        # ReLU
        hidden = np.maximum(0, hidden)
        # Второй слой: сжатие
        out = hidden @ self.W2.T + self.b2  # (batch, T, d_model)
        return out


# ------------------------------------------------------------------------------
# 5. Layer Normalization
# ------------------------------------------------------------------------------

class LayerNorm:
    """
    Layer Normalization.
    Нормализация по признаковому измерению с обучаемыми параметрами γ, β.
    """

    def __init__(self, d_model: int, eps: float = 1e-5):
        self.d_model = d_model
        self.eps = eps
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)

    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        Прямой проход LayerNorm.

        Аргументы:
            x: (batch, T, d_model)

        Возвращает:
            out: (batch, T, d_model)
        """
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        out = self.gamma * x_norm + self.beta
        return out


# ------------------------------------------------------------------------------
# 6. Encoder Layer
# ------------------------------------------------------------------------------

class EncoderLayer:
    """
    Один слой энкодера Transformer.
    Состоит из:
        - Multi-Head Self-Attention
        - Add & LayerNorm
        - Feed-Forward
        - Add & LayerNorm
    """

    def __init__(self, d_model: int, h: int, d_ff: int):
        self.self_attn = MultiHeadAttention(d_model, h)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)

    def forward(
        self, x: np.ndarray, mask: Optional[np.ndarray] = None
    ) -> Tuple[np.ndarray, List[np.ndarray]]:
        """
        Прямой проход слоя энкодера.

        Аргументы:
            x: (batch, T, d_model)
            mask: (batch, T, T) или None

        Возвращает:
            out: (batch, T, d_model)
            attn_weights: список весов внимания
        """
        # Self-Attention + остаточная связь + норм
        attn_out, attn_weights = self.self_attn.forward(x, x, x, mask)
        x = self.norm1.forward(x + attn_out)

        # FFN + остаточная связь + норм
        ffn_out = self.ffn.forward(x)
        x = self.norm2.forward(x + ffn_out)

        return x, attn_weights


# ------------------------------------------------------------------------------
# 7. Decoder Layer
# ------------------------------------------------------------------------------

class DecoderLayer:
    """
    Один слой декодера Transformer.
    Состоит из:
        - Masked Multi-Head Self-Attention
        - Add & LayerNorm
        - Cross-Attention (энкодер-декодер)
        - Add & LayerNorm
        - Feed-Forward
        - Add & LayerNorm
    """

    def __init__(self, d_model: int, h: int, d_ff: int):
        self.self_attn = MultiHeadAttention(d_model, h)
        self.cross_attn = MultiHeadAttention(d_model, h)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.norm3 = LayerNorm(d_model)

    def forward(
        self,
        x: np.ndarray,
        encoder_output: np.ndarray,
        src_mask: Optional[np.ndarray] = None,
        tgt_mask: Optional[np.ndarray] = None,
    ) -> Tuple[np.ndarray, List[np.ndarray], List[np.ndarray]]:
        """
        Прямой проход слоя декодера.

        Аргументы:
            x: (batch, T_tgt, d_model) — вход декодера (целевая последовательность)
            encoder_output: (batch, T_src, d_model) — выход энкодера
            src_mask: маска для источника (для Cross-Attention)
            tgt_mask: каузальная маска для Self-Attention

        Возвращает:
            out: (batch, T_tgt, d_model)
            self_attn_weights: веса Self-Attention
            cross_attn_weights: веса Cross-Attention
        """
        # Masked Self-Attention + остаточная + норм
        self_attn_out, self_attn_weights = self.self_attn.forward(x, x, x, tgt_mask)
        x = self.norm1.forward(x + self_attn_out)

        # Cross-Attention (Q из декодера, K,V из энкодера)
        cross_attn_out, cross_attn_weights = self.cross_attn.forward(
            x, encoder_output, encoder_output, src_mask
        )
        x = self.norm2.forward(x + cross_attn_out)

        # FFN + остаточная + норм
        ffn_out = self.ffn.forward(x)
        x = self.norm3.forward(x + ffn_out)

        return x, self_attn_weights, cross_attn_weights


# ------------------------------------------------------------------------------
# 8. Полный Transformer (Encoder + Decoder)
# ------------------------------------------------------------------------------

class Transformer:
    """
    Полная архитектура Transformer (энкодер-декодер).

    Атрибуты:
        d_model: размерность модели
        h: число голов
        d_ff: размерность скрытого слоя FFN
        N: число слоёв
        vocab_size: размер словаря
        encoder_layers: список слоёв энкодера
        decoder_layers: список слоёв декодера
        embedding: матрица эмбеддингов (vocab_size, d_model)
        pos_encoding: позиционное кодирование (максимальная длина)
        linear_out: выходной линейный слой (d_model, vocab_size)
    """

    def __init__(
        self,
        d_model: int,
        h: int,
        d_ff: int,
        N: int,
        vocab_size: int,
        max_len: int = 100,
    ):
        self.d_model = d_model
        self.h = h
        self.d_ff = d_ff
        self.N = N
        self.vocab_size = vocab_size
        self.max_len = max_len

        # Эмбеддинги
        scale = np.sqrt(2.0 / vocab_size)
        self.embedding = np.random.randn(vocab_size, d_model) * scale

        # Позиционное кодирование
        self.pos_encoding = positional_encoding(max_len, d_model)

        # Стеки слоёв
        self.encoder_layers = [EncoderLayer(d_model, h, d_ff) for _ in range(N)]
        self.decoder_layers = [DecoderLayer(d_model, h, d_ff) for _ in range(N)]

        # Выходной линейный слой
        self.linear_out = np.random.randn(d_model, vocab_size) * scale

    def _create_padding_mask(self, seq: np.ndarray) -> np.ndarray:
        """
        Создаёт маску для паддингов (токенов со значением 0).
        Аргументы:
            seq: (batch, T) — последовательности токенов
        Возвращает:
            mask: (batch, 1, T) — 0 для реальных токенов, -inf для паддингов
        """
        mask = (seq == 0).astype(np.float32) * (-1e9)
        return mask[:, np.newaxis, :]  # (batch, 1, T)

    def _create_causal_mask(self, T: int) -> np.ndarray:
        """
        Создаёт каузальную маску для декодера.
        Аргументы:
            T: длина последовательности
        Возвращает:
            mask: (T, T) — 0 для прошлых позиций, -inf для будущих
        """
        mask = np.triu(np.ones((T, T)) * (-1e9), k=1)
        return mask

    def _get_embeddings(self, tokens: np.ndarray) -> np.ndarray:
        """
        Преобразует токены в эмбеддинги и добавляет позиционное кодирование.
        tokens: (batch, T)
        Возвращает: (batch, T, d_model)
        """
        emb = self.embedding[tokens]  # (batch, T, d_model)
        # Добавляем позиционное кодирование (обрезаем до T)
        pos = self.pos_encoding[: tokens.shape[1], :]
        return emb + pos[np.newaxis, :, :]

    def forward(
        self,
        src_tokens: np.ndarray,
        tgt_tokens: np.ndarray,
    ) -> Tuple[np.ndarray, List, List, List, List]:
        """
        Прямой проход полного Transformer.

        Аргументы:
            src_tokens: (batch, T_src) — токены исходного текста
            tgt_tokens: (batch, T_tgt) — токены целевого текста (для обучения)

        Возвращает:
            logits: (batch, T_tgt, vocab_size) — логиты для каждого токена
            encoder_attn_weights: список весов для каждого слоя энкодера
            decoder_self_attn_weights: веса Self-Attention декодера
            decoder_cross_attn_weights: веса Cross-Attention декодера
        """
        batch, T_src = src_tokens.shape
        _, T_tgt = tgt_tokens.shape

        # Маски
        src_padding_mask = self._create_padding_mask(src_tokens)  # (batch, 1, T_src)
        tgt_padding_mask = self._create_padding_mask(tgt_tokens)  # (batch, 1, T_tgt)
        causal_mask = self._create_causal_mask(T_tgt)  # (T_tgt, T_tgt)

        # Комбинируем маски для декодера: padding + causal
        # tgt_padding_mask имеет форму (batch, 1, T_tgt), расширяем до (batch, T_tgt, T_tgt)
        tgt_mask = tgt_padding_mask + causal_mask[np.newaxis, :, :]
        src_mask = src_padding_mask  # для Cross-Attention (batch, 1, T_src)

        # Эмбеддинги
        src_emb = self._get_embeddings(src_tokens)  # (batch, T_src, d_model)
        tgt_emb = self._get_embeddings(tgt_tokens)  # (batch, T_tgt, d_model)

        # Энкодер
        encoder_output = src_emb
        encoder_attn_weights_all = []
        for layer in self.encoder_layers:
            encoder_output, attn_weights = layer.forward(encoder_output, src_mask)
            encoder_attn_weights_all.append(attn_weights)

        # Декодер
        decoder_output = tgt_emb
        decoder_self_attn_weights_all = []
        decoder_cross_attn_weights_all = []
        for layer in self.decoder_layers:
            decoder_output, self_attn, cross_attn = layer.forward(
                decoder_output, encoder_output, src_mask, tgt_mask
            )
            decoder_self_attn_weights_all.append(self_attn)
            decoder_cross_attn_weights_all.append(cross_attn)

        # Выходной слой
        logits = decoder_output @ self.linear_out  # (batch, T_tgt, vocab_size)

        return (
            logits,
            encoder_attn_weights_all,
            decoder_self_attn_weights_all,
            decoder_cross_attn_weights_all,
        )


# ------------------------------------------------------------------------------
# 9. ДЕМОНСТРАЦИЯ РАБОТЫ
# ------------------------------------------------------------------------------

def demo_transformer():
    """
    Демонстрация работы Transformer на маленьком синтетическом примере.
    """
    print("=" * 60)
    print("ДЕМОНСТРАЦИЯ РАБОТЫ TRANSFORMER НА NUMPY")
    print("=" * 60)

    # Параметры
    d_model = 8
    h = 2
    d_ff = 16
    N = 2
    vocab_size = 20
    max_len = 10
    batch = 1
    T_src = 4
    T_tgt = 5

    # Создаём модель
    transformer = Transformer(d_model, h, d_ff, N, vocab_size, max_len)

    # Генерируем случайные токены (0 — паддинг)
    src_tokens = np.random.randint(1, vocab_size, (batch, T_src))
    tgt_tokens = np.random.randint(1, vocab_size, (batch, T_tgt))

    print("\nИсходные токены (src):", src_tokens)
    print("Целевые токены (tgt):", tgt_tokens)

    # Прямой проход
    logits, enc_attns, dec_self_attns, dec_cross_attns = transformer.forward(
        src_tokens, tgt_tokens
    )

    print("\nВыходные логиты (batch=1, T_tgt, vocab_size):", logits.shape)
    print("Пример логита для первого токена:", logits[0, 0, :5].round(3))

    # Визуализация весов внимания
    # Возьмём последний слой энкодера и первую голову
    enc_attn_last = enc_attns[-1]  # список голов, каждая (batch, T_src, T_src)
    attn_matrix = enc_attn_last[0][0]  # первая голова, batch=0

    plt.figure(figsize=(6, 5))
    plt.imshow(attn_matrix, cmap='viridis', aspect='auto')
    plt.colorbar(label='Вес внимания')
    plt.title('Веса внимания энкодера (последний слой, голова 0)')
    plt.xlabel('Ключи (позиции)')
    plt.ylabel('Запросы (позиции)')
    plt.tight_layout()
    plt.show()

    # Сравнение с эталоном (PyTorch) — текстовая сводка
    print("\n" + "-" * 60)
    print("СРАВНЕНИЕ С ЭТАЛОННОЙ РЕАЛИЗАЦИЕЙ (PyTorch)")
    print("-" * 60)
    print("""
    Данная NumPy-реализация является образовательной и не оптимизирована
    для производительности. Она воспроизводит все ключевые компоненты
    Transformer (внимание, позиционное кодирование, остаточные связи,
    нормализацию, FFN) так, как они описаны в оригинальной статье.

    Ключевые отличия от эталонной реализации PyTorch:
    1. Отсутствие автоматического дифференцирования (обучение не предусмотрено).
    2. Используется только CPU, нет поддержки GPU.
    3. Реализованы основные операции, но без оптимизаций (например, Flash Attention).
    4. Размерности и вычисления полностью совпадают с теоретической моделью.

    Данная реализация полезна для понимания внутреннего устройства Transformer.
    Для реальных задач рекомендуется использовать библиотеки PyTorch или TensorFlow.
    """)


if __name__ == "__main__":
    demo_transformer()